In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip -q install lightgbm

import os
import json
import random
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    brier_score_loss
)

from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

In [ ]:
ROOT = "/content/drive/MyDrive"

CIFAR_OOF_DIR = os.path.join(
    ROOT,
    "OOF_Stacking_Results",
    "OOF_Probabilities"
)

CT_OOF_DIR = os.path.join(
    ROOT,
    "CT_OOF_Stacking_Results",
    "OOF_Probabilities"
)

PAPER_DIR = os.path.join(
    ROOT,
    "Paper_1_Matched_Protocol_Comparison"
)

EXT_DIR = os.path.join(
    PAPER_DIR,
    "Extended_Experiments"
)

INTEGRITY_DIR = os.path.join(EXT_DIR, "Integrity_Checks")
COVERAGE_DIR = os.path.join(EXT_DIR, "Coverage_Ablation")
HOLDOUT_DIR = os.path.join(EXT_DIR, "Repeated_Holdout")
OOF_REPEAT_DIR = os.path.join(EXT_DIR, "Repeated_OOF")
FIGURE_DIR = os.path.join(EXT_DIR, "Figures")
TABLE_DIR = os.path.join(EXT_DIR, "Tables")

for folder in [
    EXT_DIR,
    INTEGRITY_DIR,
    COVERAGE_DIR,
    HOLDOUT_DIR,
    OOF_REPEAT_DIR,
    FIGURE_DIR,
    TABLE_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Extended experiment folder:", EXT_DIR)

In [ ]:
def list_files(folder):
    print("\nFolder:", folder)
    if not os.path.exists(folder):
        print("Folder not found")
        return []

    files = sorted(os.listdir(folder))

    for file_name in files:
        print(file_name)

    return files


cifar_files = list_files(CIFAR_OOF_DIR)
ct_files = list_files(CT_OOF_DIR)

In [ ]:
X_cifar_oof_saved = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Training_40_Features.npy"
    )
)

X_cifar_test_saved = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Test_40_Features.npy"
    )
)

In [ ]:
import os
import numpy as np

ROOT = "/content/drive/MyDrive"

CIFAR_OOF_DIR = os.path.join(
    ROOT,
    "OOF_Stacking_Results",
    "OOF_Probabilities"
)

# Load CIFAR OOF probabilities
cifar_cnn_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_CNN_5Fold_OOF.npy"
    )
)

cifar_vgg_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_VGG16_5Fold_OOF.npy"
    )
)

cifar_resnet_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_ResNet50_5Fold_OOF.npy"
    )
)

cifar_effnet_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_EfficientNetB0_5Fold_OOF.npy"
    )
)

# Load CIFAR final-test probabilities
cifar_cnn_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_CNN_Test_Probabilities.npy"
    )
)

cifar_vgg_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_VGG16_Test_Probabilities.npy"
    )
)

cifar_resnet_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_ResNet50_Test_Probabilities.npy"
    )
)

cifar_effnet_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_EfficientNetB0_Test_Probabilities.npy"
    )
)

# Load labels
y_cifar_dev = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Training_Labels.npy"
    )
).reshape(-1)

y_cifar_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Test_Labels.npy"
    )
).reshape(-1)

print("Loaded successfully")
print("CNN OOF:", cifar_cnn_oof.shape)
print("VGG OOF:", cifar_vgg_oof.shape)
print("ResNet OOF:", cifar_resnet_oof.shape)
print("EfficientNet OOF:", cifar_effnet_oof.shape)
print("Development labels:", y_cifar_dev.shape)
print("Test labels:", y_cifar_test.shape)

In [ ]:
X_cifar_oof = np.concatenate(
    [
        cifar_cnn_oof,
        cifar_vgg_oof,
        cifar_resnet_oof,
        cifar_effnet_oof
    ],
    axis=1
)

X_cifar_test = np.concatenate(
    [
        cifar_cnn_test,
        cifar_vgg_test,
        cifar_resnet_test,
        cifar_effnet_test
    ],
    axis=1
)

print("X_cifar_oof shape:", X_cifar_oof.shape)
print("X_cifar_test shape:", X_cifar_test.shape)

In [ ]:
print("Constructed OOF:", X_cifar_oof.shape)
print("Saved OOF:", X_cifar_oof_saved.shape)

print("Maximum absolute OOF difference:",
      np.max(np.abs(X_cifar_oof - X_cifar_oof_saved)))

print("Mean absolute OOF difference:",
      np.mean(np.abs(X_cifar_oof - X_cifar_oof_saved)))

print("Constructed test:", X_cifar_test.shape)
print("Saved test:", X_cifar_test_saved.shape)

print("Maximum absolute test difference:",
      np.max(np.abs(X_cifar_test - X_cifar_test_saved)))

print("Mean absolute test difference:",
      np.mean(np.abs(X_cifar_test - X_cifar_test_saved)))

In [ ]:
from itertools import permutations

oof_models = {
    "CNN": cifar_cnn_oof,
    "VGG16": cifar_vgg_oof,
    "ResNet50": cifar_resnet_oof,
    "EfficientNetB0": cifar_effnet_oof
}

test_models = {
    "CNN": cifar_cnn_test,
    "VGG16": cifar_vgg_test,
    "ResNet50": cifar_resnet_test,
    "EfficientNetB0": cifar_effnet_test
}

comparison_rows = []

for order in permutations(oof_models.keys()):
    candidate_oof = np.concatenate(
        [oof_models[name] for name in order],
        axis=1
    )

    candidate_test = np.concatenate(
        [test_models[name] for name in order],
        axis=1
    )

    comparison_rows.append({
        "Order": " → ".join(order),
        "OOF_Mean_Abs_Difference": float(
            np.mean(np.abs(candidate_oof - X_cifar_oof_saved))
        ),
        "OOF_Max_Abs_Difference": float(
            np.max(np.abs(candidate_oof - X_cifar_oof_saved))
        ),
        "Test_Mean_Abs_Difference": float(
            np.mean(np.abs(candidate_test - X_cifar_test_saved))
        ),
        "Test_Max_Abs_Difference": float(
            np.max(np.abs(candidate_test - X_cifar_test_saved))
        )
    })

order_check = pd.DataFrame(comparison_rows).sort_values(
    [
        "OOF_Mean_Abs_Difference",
        "Test_Mean_Abs_Difference"
    ]
)

display(order_check.head(10))

In [ ]:
X_cifar_oof = np.concatenate(
    [
        cifar_cnn_oof,
        cifar_vgg_oof,
        cifar_effnet_oof,
        cifar_resnet_oof
    ],
    axis=1
)

X_cifar_test = np.concatenate(
    [
        cifar_cnn_test,
        cifar_vgg_test,
        cifar_effnet_test,
        cifar_resnet_test
    ],
    axis=1
)

In [ ]:
print(
    "OOF match:",
    np.allclose(
        X_cifar_oof,
        X_cifar_oof_saved,
        atol=1e-6
    )
)

print(
    "Test match:",
    np.allclose(
        X_cifar_test,
        X_cifar_test_saved,
        atol=1e-6
    )
)

In [ ]:
X_cifar_oof = X_cifar_oof_saved.copy()
X_cifar_test = X_cifar_test_saved.copy()

print("Using saved CIFAR meta-feature matrices")
print("OOF:", X_cifar_oof.shape)
print("Test:", X_cifar_test.shape)

In [ ]:
def check_meta_matrix(name, X, y, expected_columns):
    print("\n", name)
    print("Shape:", X.shape)
    print("Labels:", y.shape)
    print("Finite:", np.isfinite(X).all())
    print("Minimum:", X.min())
    print("Maximum:", X.max())
    print("Zero rows:", np.sum(np.isclose(X.sum(axis=1), 0.0)))

    assert X.shape[0] == len(y)
    assert X.shape[1] == expected_columns
    assert np.isfinite(X).all()
    assert X.min() >= 0.0
    assert X.max() <= 1.0
    assert not np.any(np.isclose(X.sum(axis=1), 0.0))


check_meta_matrix(
    "CIFAR OOF",
    X_cifar_oof,
    y_cifar_dev,
    40
)

check_meta_matrix(
    "CIFAR test",
    X_cifar_test,
    y_cifar_test,
    40
)

print("\nSaved CIFAR matrices passed integrity checks.")

In [ ]:
for block_index in range(4):
    start = block_index * 10
    end = start + 10

    oof_block = X_cifar_oof[:, start:end]
    test_block = X_cifar_test[:, start:end]

    print(
        f"Block {block_index + 1}:",
        "OOF mean row sum =",
        oof_block.sum(axis=1).mean(),
        "| Test mean row sum =",
        test_block.sum(axis=1).mean()
    )

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)
from lightgbm import LGBMClassifier

COVERAGE_LEVELS = [
    0.10,
    0.20,
    0.40,
    0.60,
    0.80,
    1.00
]

ABLATION_SEEDS = [
    42,
    123,
    2025,
    7,
    99,
    314,
    512,
    1001,
    2026,
    777
]

def build_logistic_regression(seed):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=3000,
                    random_state=seed
                )
            )
        ]
    )

def build_lightgbm(seed):
    return LGBMClassifier(
        random_state=seed,
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        verbosity=-1
    )

def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=15
):
    confidences = np.max(probabilities, axis=1)
    predictions = np.argmax(probabilities, axis=1)
    correctness = predictions == y_true

    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (
            (confidences > lower)
            & (confidences <= upper)
        )

        if np.any(mask):
            ece += (
                mask.mean()
                * abs(
                    correctness[mask].mean()
                    - confidences[mask].mean()
                )
            )

    return float(ece)

def multiclass_brier_score(
    y_true,
    probabilities
):
    one_hot = np.eye(
        probabilities.shape[1]
    )[y_true.astype(int)]

    return float(
        np.mean(
            np.sum(
                (probabilities - one_hot) ** 2,
                axis=1
            )
        )
    )

def evaluate_probabilities(
    y_true,
    probabilities
):
    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Macro_Precision": precision_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probabilities,
            multi_class="ovr",
            average="macro"
        ),
        "Log_Loss": log_loss(
            y_true,
            probabilities
        ),
        "Brier_Score": multiclass_brier_score(
            y_true,
            probabilities
        ),
        "ECE": expected_calibration_error(
            y_true,
            probabilities
        )
    }

print("Coverage-ablation functions ready.")

In [ ]:
coverage_rows = []

for coverage in COVERAGE_LEVELS:
    for seed in ABLATION_SEEDS:

        if coverage == 1.0:
            subset_indices = np.arange(
                len(y_cifar_dev)
            )
        else:
            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=coverage,
                random_state=seed
            )

            subset_indices, _ = next(
                splitter.split(
                    np.zeros(len(y_cifar_dev)),
                    y_cifar_dev
                )
            )

        X_subset = X_cifar_oof[
            subset_indices
        ]

        y_subset = y_cifar_dev[
            subset_indices
        ]

        models = {
            "LogisticRegression":
                build_logistic_regression(seed),
            "LightGBM":
                build_lightgbm(seed)
        }

        for method_name, model in models.items():

            model.fit(
                X_subset,
                y_subset
            )

            test_probabilities = (
                model.predict_proba(
                    X_cifar_test
                )
            )

            metrics = evaluate_probabilities(
                y_cifar_test,
                test_probabilities
            )

            row = {
                "Dataset": "CIFAR-10",
                "Coverage": coverage,
                "Coverage_Percent":
                    int(coverage * 100),
                "Seed": seed,
                "Method": method_name,
                "Meta_Training_Samples":
                    len(subset_indices)
            }

            row.update(metrics)
            coverage_rows.append(row)

            print(
                f"Coverage={coverage:.0%} | "
                f"Seed={seed} | "
                f"{method_name} | "
                f"Accuracy={metrics['Accuracy']:.4f}"
            )

cifar_coverage_results = pd.DataFrame(
    coverage_rows
)

cifar_coverage_results.to_csv(
    os.path.join(
        COVERAGE_DIR,
        "CIFAR10_Coverage_Ablation_All_Runs.csv"
    ),
    index=False
)

print("\nCIFAR coverage ablation completed.")
display(cifar_coverage_results.head())

In [ ]:
cifar_coverage_summary = (
    cifar_coverage_results
    .groupby(
        [
            "Coverage_Percent",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Meta_Training_Samples=(
            "Meta_Training_Samples",
            "first"
        ),
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Accuracy_Min=("Accuracy", "min"),
        Accuracy_Max=("Accuracy", "max"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std"),
        ROC_AUC_Mean=("ROC_AUC", "mean"),
        ROC_AUC_SD=("ROC_AUC", "std"),
        Log_Loss_Mean=("Log_Loss", "mean"),
        Log_Loss_SD=("Log_Loss", "std"),
        Brier_Mean=("Brier_Score", "mean"),
        Brier_SD=("Brier_Score", "std"),
        ECE_Mean=("ECE", "mean"),
        ECE_SD=("ECE", "std")
    )
)

for column in [
    "Accuracy_Mean",
    "Accuracy_SD",
    "Accuracy_Min",
    "Accuracy_Max",
    "Macro_F1_Mean",
    "Macro_F1_SD",
    "ROC_AUC_Mean",
    "ROC_AUC_SD"
]:
    cifar_coverage_summary[column] *= 100

cifar_coverage_summary.to_csv(
    os.path.join(
        TABLE_DIR,
        "CIFAR10_Coverage_Ablation_Summary.csv"
    ),
    index=False
)

display(
    cifar_coverage_summary.round(4)
)

In [ ]:
CIFAR_SOFT_VOTING_ACCURACY = 0.9147

cifar_coverage_results[
    "Beats_Soft_Voting"
] = (
    cifar_coverage_results["Accuracy"]
    > CIFAR_SOFT_VOTING_ACCURACY
)

cifar_beat_summary = (
    cifar_coverage_results
    .groupby(
        [
            "Coverage_Percent",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Runs=("Seed", "count"),
        Runs_Beating_Soft_Voting=(
            "Beats_Soft_Voting",
            "sum"
        ),
        Proportion_Beating_Soft_Voting=(
            "Beats_Soft_Voting",
            "mean"
        )
    )
)

cifar_beat_summary[
    "Proportion_Beating_Soft_Voting"
] *= 100

display(cifar_beat_summary)

cifar_beat_summary.to_csv(
    os.path.join(
        TABLE_DIR,
        "CIFAR10_Coverage_Beating_Soft_Voting.csv"
    ),
    index=False
)

In [ ]:
plt.figure(figsize=(8, 5))

for method in [
    "LogisticRegression",
    "LightGBM"
]:
    method_df = cifar_coverage_summary[
        cifar_coverage_summary["Method"] == method
    ]

    plt.errorbar(
        method_df["Coverage_Percent"],
        method_df["Accuracy_Mean"],
        yerr=method_df["Accuracy_SD"],
        marker="o",
        capsize=4,
        label=method
    )

plt.axhline(
    y=91.47,
    linestyle="--",
    label="Soft voting"
)

plt.xlabel("OOF meta-training coverage (%)")
plt.ylabel("Test accuracy (%)")
plt.title(
    "CIFAR-10: Effect of OOF meta-training coverage"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "CIFAR10_Coverage_vs_Accuracy.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

def build_logistic_regression(seed):
    return LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        multi_class="auto",
        random_state=seed,
        n_jobs=-1
    )


def build_lightgbm(seed):
    return LGBMClassifier(
        objective="multiclass",
        num_class=10,
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.20,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

print("Original CIFAR meta-learner settings restored.")

In [ ]:
reproduction_rows = []

for seed in [42]:

    models = {
        "LogisticRegression":
            build_logistic_regression(seed),

        "LightGBM":
            build_lightgbm(seed)
    }

    for method_name, model in models.items():

        model.fit(
            X_cifar_oof,
            y_cifar_dev
        )

        probabilities = model.predict_proba(
            X_cifar_test
        )

        metrics = evaluate_probabilities(
            y_cifar_test,
            probabilities
        )

        reproduction_rows.append({
            "Method": method_name,
            **metrics
        })

reproduction_check = pd.DataFrame(
    reproduction_rows
)

display(reproduction_check)

In [ ]:
saved_logistic_probs = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Logistic_Meta_Test_Probabilities.npy"
    )
)

saved_lightgbm_probs = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_LightGBM_Meta_Test_Probabilities.npy"
    )
)

new_logistic = build_logistic_regression(42)
new_logistic.fit(
    X_cifar_oof,
    y_cifar_dev
)
new_logistic_probs = new_logistic.predict_proba(
    X_cifar_test
)

new_lightgbm = build_lightgbm(42)
new_lightgbm.fit(
    X_cifar_oof,
    y_cifar_dev
)
new_lightgbm_probs = new_lightgbm.predict_proba(
    X_cifar_test
)

print(
    "Logistic probability match:",
    np.allclose(
        new_logistic_probs,
        saved_logistic_probs,
        atol=1e-6
    )
)

print(
    "LightGBM probability match:",
    np.allclose(
        new_lightgbm_probs,
        saved_lightgbm_probs,
        atol=1e-6
    )
)

print(
    "Logistic mean absolute difference:",
    np.mean(
        np.abs(
            new_logistic_probs
            - saved_logistic_probs
        )
    )
)

print(
    "LightGBM mean absolute difference:",
    np.mean(
        np.abs(
            new_lightgbm_probs
            - saved_lightgbm_probs
        )
    )
)

In [ ]:
# Delete previous coverage-ablation results from memory
variables_to_delete = [
    "coverage_rows",
    "cifar_coverage_results",
    "cifar_coverage_summary",
    "cifar_beat_summary",
    "all_coverage_results",
    "full_coverage_results"
]

for variable_name in variables_to_delete:
    if variable_name in globals():
        del globals()[variable_name]
        print("Deleted:", variable_name)

# Create a fresh empty result list
coverage_rows = []

print("\nOld coverage results cleared.")
print("A new empty coverage_rows list has been created.")

In [ ]:
# ============================================================
# CORRECTED CIFAR-10 OOF META-TRAINING COVERAGE ABLATION
# Uses the exact original Logistic Regression and LightGBM settings
# ============================================================

coverage_rows = []

for coverage in COVERAGE_LEVELS:

    for seed in ABLATION_SEEDS:

        # Use all samples at 100% coverage.
        # For lower coverage, draw a stratified subset.
        if coverage >= 1.0:

            subset_indices = np.arange(
                len(y_cifar_dev)
            )

        else:

            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=coverage,
                random_state=seed
            )

            subset_indices, _ = next(
                splitter.split(
                    np.zeros(len(y_cifar_dev)),
                    y_cifar_dev
                )
            )

        X_subset = X_cifar_oof[
            subset_indices
        ]

        y_subset = y_cifar_dev[
            subset_indices
        ]

        models = {
            "LogisticRegression":
                build_logistic_regression(seed),

            "LightGBM":
                build_lightgbm(seed)
        }

        for method_name, model in models.items():

            model.fit(
                X_subset,
                y_subset
            )

            test_probabilities = model.predict_proba(
                X_cifar_test
            )

            metrics = evaluate_probabilities(
                y_cifar_test,
                test_probabilities
            )

            result_row = {
                "Dataset": "CIFAR-10",
                "Coverage": coverage,
                "Coverage_Percent": int(
                    coverage * 100
                ),
                "Seed": seed,
                "Method": method_name,
                "Meta_Training_Samples": len(
                    subset_indices
                )
            }

            result_row.update(metrics)

            coverage_rows.append(
                result_row
            )

            print(
                f"Coverage={coverage:>4.0%} | "
                f"Seed={seed:>4} | "
                f"{method_name:<20} | "
                f"Accuracy={metrics['Accuracy']:.4f} | "
                f"F1={metrics['Macro_F1']:.4f}"
            )

# Convert results to a DataFrame
cifar_coverage_results = pd.DataFrame(
    coverage_rows
)

# Save immediately
cifar_coverage_output_path = os.path.join(
    COVERAGE_DIR,
    "CIFAR10_Coverage_Ablation_All_Runs.csv"
)

cifar_coverage_results.to_csv(
    cifar_coverage_output_path,
    index=False
)

print("\n" + "=" * 72)
print("Corrected CIFAR coverage ablation completed.")
print("Rows generated:", len(cifar_coverage_results))
print("Saved to:", cifar_coverage_output_path)
print("=" * 72)

display(
    cifar_coverage_results.head()
)

In [ ]:
# ============================================================
# VERIFY 100% COVERAGE AND BUILD CIFAR SUMMARY TABLE
# ============================================================

# 1. Check the full-coverage results
full_coverage_results = (
    cifar_coverage_results[
        cifar_coverage_results["Coverage_Percent"] == 100
    ]
    .groupby("Method", as_index=False)
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Log_Loss_Mean=("Log_Loss", "mean"),
        ECE_Mean=("ECE", "mean")
    )
)

display(full_coverage_results.round(6))

expected_accuracy = {
    "LogisticRegression": 0.9185,
    "LightGBM": 0.9198
}

for _, row in full_coverage_results.iterrows():
    method = row["Method"]

    assert np.isclose(
        row["Accuracy_Mean"],
        expected_accuracy[method],
        atol=1e-10
    ), (
        f"{method} does not reproduce the original "
        f"accuracy. Obtained {row['Accuracy_Mean']}"
    )

print("Full-coverage results exactly reproduce the original CIFAR results.")

# 2. Generate the complete coverage summary
cifar_coverage_summary = (
    cifar_coverage_results
    .groupby(
        [
            "Coverage_Percent",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Meta_Training_Samples=(
            "Meta_Training_Samples",
            "first"
        ),
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Accuracy_Min=("Accuracy", "min"),
        Accuracy_Max=("Accuracy", "max"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std"),
        ROC_AUC_Mean=("ROC_AUC", "mean"),
        ROC_AUC_SD=("ROC_AUC", "std"),
        Log_Loss_Mean=("Log_Loss", "mean"),
        Log_Loss_SD=("Log_Loss", "std"),
        Brier_Mean=("Brier_Score", "mean"),
        Brier_SD=("Brier_Score", "std"),
        ECE_Mean=("ECE", "mean"),
        ECE_SD=("ECE", "std")
    )
)

# Convert selected metrics to percentage form
percentage_columns = [
    "Accuracy_Mean",
    "Accuracy_SD",
    "Accuracy_Min",
    "Accuracy_Max",
    "Macro_F1_Mean",
    "Macro_F1_SD",
    "ROC_AUC_Mean",
    "ROC_AUC_SD"
]

for column in percentage_columns:
    cifar_coverage_summary[column] *= 100

summary_path = os.path.join(
    TABLE_DIR,
    "CIFAR10_Coverage_Ablation_Summary.csv"
)

cifar_coverage_summary.to_csv(
    summary_path,
    index=False
)

print("\nSummary saved to:")
print(summary_path)

display(
    cifar_coverage_summary.round(4)
)

In [ ]:
# ============================================================
# RESET AND RELOAD THE EXACT ARCHIVED CIFAR META-FEATURE MATRICES
# ============================================================

import os
import numpy as np
import pandas as pd

# Delete incorrect in-memory results
for variable_name in [
    "coverage_rows",
    "cifar_coverage_results",
    "cifar_coverage_summary",
    "full_coverage_results"
]:
    if variable_name in globals():
        del globals()[variable_name]

# Reload the exact archived matrices used for the original results
X_cifar_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Training_40_Features.npy"
    )
).astype(np.float32)

X_cifar_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Test_40_Features.npy"
    )
).astype(np.float32)

y_cifar_dev = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Training_Labels.npy"
    )
).reshape(-1).astype(np.int64)

y_cifar_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Test_Labels.npy"
    )
).reshape(-1).astype(np.int64)

print("Exact archived matrices reloaded:")
print("X_cifar_oof:", X_cifar_oof.shape)
print("X_cifar_test:", X_cifar_test.shape)
print("y_cifar_dev:", y_cifar_dev.shape)
print("y_cifar_test:", y_cifar_test.shape)

# Confirm probability blocks
for block_index in range(4):
    start = block_index * 10
    end = start + 10

    assert np.allclose(
        X_cifar_oof[:, start:end].sum(axis=1),
        1.0,
        atol=1e-5
    )

    assert np.allclose(
        X_cifar_test[:, start:end].sum(axis=1),
        1.0,
        atol=1e-5
    )

print("All four probability blocks are valid.")

In [ ]:
# ============================================================
# DIRECT REPRODUCTION CHECK BEFORE RERUNNING THE ABLATION
# ============================================================

logistic_check = build_logistic_regression(42)
logistic_check.fit(
    X_cifar_oof,
    y_cifar_dev
)

logistic_probabilities = logistic_check.predict_proba(
    X_cifar_test
)

lightgbm_check = build_lightgbm(42)
lightgbm_check.fit(
    X_cifar_oof,
    y_cifar_dev
)

lightgbm_probabilities = lightgbm_check.predict_proba(
    X_cifar_test
)

logistic_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(logistic_probabilities, axis=1)
)

lightgbm_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(lightgbm_probabilities, axis=1)
)

print("Logistic Regression accuracy:", logistic_accuracy)
print("LightGBM accuracy:", lightgbm_accuracy)

assert np.isclose(
    logistic_accuracy,
    0.9185,
    atol=1e-10
)

assert np.isclose(
    lightgbm_accuracy,
    0.9198,
    atol=1e-10
)

print("Exact original results reproduced successfully.")

In [ ]:
# ============================================================
# REBUILD THE EXACT CIFAR META-FEATURE MATRICES
# FROM THE CURRENT INDIVIDUAL PROBABILITY FILES
# ============================================================

import os
import numpy as np

# Reload individual OOF arrays
cifar_cnn_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_CNN_5Fold_OOF.npy"
    )
)

cifar_vgg_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_VGG16_5Fold_OOF.npy"
    )
)

cifar_resnet_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_ResNet50_5Fold_OOF.npy"
    )
)

cifar_effnet_oof = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_EfficientNetB0_5Fold_OOF.npy"
    )
)

# Reload individual final-test arrays
cifar_cnn_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_CNN_Test_Probabilities.npy"
    )
)

cifar_vgg_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_VGG16_Test_Probabilities.npy"
    )
)

cifar_resnet_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_ResNet50_Test_Probabilities.npy"
    )
)

cifar_effnet_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_EfficientNetB0_Test_Probabilities.npy"
    )
)

# Exact original order:
# CNN → VGG16 → ResNet50 → EfficientNetB0
X_cifar_oof = np.concatenate(
    [
        cifar_cnn_oof,
        cifar_vgg_oof,
        cifar_resnet_oof,
        cifar_effnet_oof
    ],
    axis=1
).astype(np.float32)

X_cifar_test = np.concatenate(
    [
        cifar_cnn_test,
        cifar_vgg_test,
        cifar_resnet_test,
        cifar_effnet_test
    ],
    axis=1
).astype(np.float32)

y_cifar_dev = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Training_Labels.npy"
    )
).reshape(-1).astype(np.int64)

y_cifar_test = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Meta_Test_Labels.npy"
    )
).reshape(-1).astype(np.int64)

print("Rebuilt OOF matrix:", X_cifar_oof.shape)
print("Rebuilt test matrix:", X_cifar_test.shape)

# Validate each probability block
for block_index in range(4):
    start = block_index * 10
    end = start + 10

    assert np.allclose(
        X_cifar_oof[:, start:end].sum(axis=1),
        1.0,
        atol=1e-5
    )

    assert np.allclose(
        X_cifar_test[:, start:end].sum(axis=1),
        1.0,
        atol=1e-5
    )

print("All rebuilt probability blocks are valid.")

In [ ]:
# ============================================================
# FINAL REPRODUCTION CHECK
# ============================================================

logistic_check = build_logistic_regression(42)
logistic_check.fit(
    X_cifar_oof,
    y_cifar_dev
)

logistic_probs = logistic_check.predict_proba(
    X_cifar_test
)

lightgbm_check = build_lightgbm(42)
lightgbm_check.fit(
    X_cifar_oof,
    y_cifar_dev
)

lightgbm_probs = lightgbm_check.predict_proba(
    X_cifar_test
)

logistic_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(logistic_probs, axis=1)
)

lightgbm_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(lightgbm_probs, axis=1)
)

print("Logistic Regression accuracy:", logistic_accuracy)
print("LightGBM accuracy:", lightgbm_accuracy)

saved_logistic_probs = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Logistic_Meta_Test_Probabilities.npy"
    )
)

saved_lightgbm_probs = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_LightGBM_Meta_Test_Probabilities.npy"
    )
)

print(
    "Logistic probability match:",
    np.allclose(
        logistic_probs,
        saved_logistic_probs,
        atol=1e-7
    )
)

print(
    "LightGBM probability match:",
    np.allclose(
        lightgbm_probs,
        saved_lightgbm_probs,
        atol=1e-7
    )
)

In [ ]:
# ============================================================
# FINAL REPRODUCTION CHECK
# ============================================================

logistic_check = build_logistic_regression(42)
logistic_check.fit(
    X_cifar_oof,
    y_cifar_dev
)

logistic_probs = logistic_check.predict_proba(
    X_cifar_test
)

lightgbm_check = build_lightgbm(42)
lightgbm_check.fit(
    X_cifar_oof,
    y_cifar_dev
)

lightgbm_probs = lightgbm_check.predict_proba(
    X_cifar_test
)

logistic_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(logistic_probs, axis=1)
)

lightgbm_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(lightgbm_probs, axis=1)
)

print("Logistic Regression accuracy:", logistic_accuracy)
print("LightGBM accuracy:", lightgbm_accuracy)

saved_logistic_probs = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_Logistic_Meta_Test_Probabilities.npy"
    )
)

saved_lightgbm_probs = np.load(
    os.path.join(
        CIFAR_OOF_DIR,
        "CIFAR_LightGBM_Meta_Test_Probabilities.npy"
    )
)

print(
    "Logistic probability match:",
    np.allclose(
        logistic_probs,
        saved_logistic_probs,
        atol=1e-7
    )
)

print(
    "LightGBM probability match:",
    np.allclose(
        lightgbm_probs,
        saved_lightgbm_probs,
        atol=1e-7
    )
)

In [ ]:
# ============================================================
# QUANTIFY DIFFERENCES FROM ARCHIVED META-LEARNER PROBABILITIES
# ============================================================

saved_logistic_predictions = np.argmax(
    saved_logistic_probs,
    axis=1
)

saved_lightgbm_predictions = np.argmax(
    saved_lightgbm_probs,
    axis=1
)

new_logistic_predictions = np.argmax(
    logistic_probs,
    axis=1
)

new_lightgbm_predictions = np.argmax(
    lightgbm_probs,
    axis=1
)

print("LOGISTIC REGRESSION")
print(
    "Mean absolute probability difference:",
    np.mean(np.abs(logistic_probs - saved_logistic_probs))
)
print(
    "Maximum absolute probability difference:",
    np.max(np.abs(logistic_probs - saved_logistic_probs))
)
print(
    "Prediction agreement:",
    np.mean(
        new_logistic_predictions
        == saved_logistic_predictions
    )
)
print(
    "Different predicted labels:",
    np.sum(
        new_logistic_predictions
        != saved_logistic_predictions
    )
)

print("\nLIGHTGBM")
print(
    "Mean absolute probability difference:",
    np.mean(np.abs(lightgbm_probs - saved_lightgbm_probs))
)
print(
    "Maximum absolute probability difference:",
    np.max(np.abs(lightgbm_probs - saved_lightgbm_probs))
)
print(
    "Prediction agreement:",
    np.mean(
        new_lightgbm_predictions
        == saved_lightgbm_predictions
    )
)
print(
    "Different predicted labels:",
    np.sum(
        new_lightgbm_predictions
        != saved_lightgbm_predictions
    )
)

In [ ]:
# ============================================================
# COMPARE SAVED AND RECREATED METRICS
# ============================================================

comparison_rows = []

for method, recreated, archived in [
    (
        "LogisticRegression",
        logistic_probs,
        saved_logistic_probs
    ),
    (
        "LightGBM",
        lightgbm_probs,
        saved_lightgbm_probs
    )
]:
    recreated_metrics = evaluate_probabilities(
        y_cifar_test,
        recreated
    )

    archived_metrics = evaluate_probabilities(
        y_cifar_test,
        archived
    )

    for metric_name in recreated_metrics:
        comparison_rows.append(
            {
                "Method": method,
                "Metric": metric_name,
                "Recreated": recreated_metrics[
                    metric_name
                ],
                "Archived": archived_metrics[
                    metric_name
                ],
                "Absolute_Difference": abs(
                    recreated_metrics[metric_name]
                    - archived_metrics[metric_name]
                )
            }
        )

metric_comparison = pd.DataFrame(
    comparison_rows
)

display(
    metric_comparison.round(8)
)

In [ ]:
# ============================================================
# SAVE CORRECTED CIFAR OOF META-LEARNER BASELINE OUTPUTS
# ============================================================

CORRECTED_BASELINE_DIR = os.path.join(
    EXT_DIR,
    "Corrected_CIFAR_Baseline"
)

os.makedirs(
    CORRECTED_BASELINE_DIR,
    exist_ok=True
)

# Predictions
corrected_logistic_predictions = np.argmax(
    logistic_probs,
    axis=1
)

corrected_lightgbm_predictions = np.argmax(
    lightgbm_probs,
    axis=1
)

# Save corrected probabilities and predictions
np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_Logistic_Corrected_Test_Probabilities.npy"
    ),
    logistic_probs.astype(np.float32)
)

np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_Logistic_Corrected_Test_Predictions.npy"
    ),
    corrected_logistic_predictions.astype(np.int64)
)

np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_LightGBM_Corrected_Test_Probabilities.npy"
    ),
    lightgbm_probs.astype(np.float32)
)

np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_LightGBM_Corrected_Test_Predictions.npy"
    ),
    corrected_lightgbm_predictions.astype(np.int64)
)

# Save the exact rebuilt meta-feature matrices used
np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_Rebuilt_OOF_Meta_Features.npy"
    ),
    X_cifar_oof.astype(np.float32)
)

np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_Rebuilt_Test_Meta_Features.npy"
    ),
    X_cifar_test.astype(np.float32)
)

np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_Development_Labels.npy"
    ),
    y_cifar_dev.astype(np.int64)
)

np.save(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_Test_Labels.npy"
    ),
    y_cifar_test.astype(np.int64)
)

# Save corrected metrics
corrected_baseline_rows = []

for method_name, probabilities in [
    ("LogisticRegression", logistic_probs),
    ("LightGBM", lightgbm_probs)
]:
    metrics = evaluate_probabilities(
        y_cifar_test,
        probabilities
    )

    corrected_baseline_rows.append({
        "Dataset": "CIFAR-10",
        "Protocol": "Five-fold OOF",
        "Method": method_name,
        **metrics
    })

corrected_baseline_results = pd.DataFrame(
    corrected_baseline_rows
)

corrected_baseline_results.to_csv(
    os.path.join(
        CORRECTED_BASELINE_DIR,
        "CIFAR_Corrected_OOF_Baseline_Metrics.csv"
    ),
    index=False
)

print("Corrected CIFAR baseline files saved to:")
print(CORRECTED_BASELINE_DIR)

display(
    corrected_baseline_results.round(6)
)

In [ ]:
# ============================================================
# CLEAR THE INVALID COVERAGE-ABLATION RESULTS
# ============================================================

for variable_name in [
    "coverage_rows",
    "cifar_coverage_results",
    "cifar_coverage_summary",
    "cifar_beat_summary",
    "full_coverage_results"
]:
    if variable_name in globals():
        del globals()[variable_name]
        print("Deleted from memory:", variable_name)

invalid_files = [
    os.path.join(
        COVERAGE_DIR,
        "CIFAR10_Coverage_Ablation_All_Runs.csv"
    ),
    os.path.join(
        TABLE_DIR,
        "CIFAR10_Coverage_Ablation_Summary.csv"
    ),
    os.path.join(
        TABLE_DIR,
        "CIFAR10_Coverage_Beating_Soft_Voting.csv"
    ),
    os.path.join(
        FIGURE_DIR,
        "CIFAR10_Coverage_vs_Accuracy.png"
    )
]

for file_path in invalid_files:
    if os.path.exists(file_path):
        os.remove(file_path)
        print("Removed invalid file:", file_path)

coverage_rows = []

print("\nInvalid ablation outputs cleared.")

In [ ]:
# ============================================================
# FINAL CIFAR-10 OOF META-TRAINING COVERAGE ABLATION
# ============================================================

from sklearn.model_selection import StratifiedShuffleSplit

# Remove previous invalid results from memory
for variable_name in [
    "coverage_rows",
    "cifar_coverage_results",
    "cifar_coverage_summary",
    "cifar_beat_summary",
    "full_coverage_results"
]:
    if variable_name in globals():
        del globals()[variable_name]

coverage_rows = []

for coverage in COVERAGE_LEVELS:

    # At 100% coverage there is no subset-selection variation.
    # Therefore, use only the original seed 42.
    if coverage >= 1.0:
        subset_seeds = [42]
    else:
        subset_seeds = ABLATION_SEEDS

    for subset_seed in subset_seeds:

        # ----------------------------------------------------
        # Select the meta-training observations
        # ----------------------------------------------------
        if coverage >= 1.0:

            subset_indices = np.arange(
                len(y_cifar_dev)
            )

        else:

            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=coverage,
                random_state=subset_seed
            )

            subset_indices, _ = next(
                splitter.split(
                    np.zeros(len(y_cifar_dev)),
                    y_cifar_dev
                )
            )

        X_subset = X_cifar_oof[
            subset_indices
        ]

        y_subset = y_cifar_dev[
            subset_indices
        ]

        # ----------------------------------------------------
        # Keep model seed fixed.
        # Only meta-training coverage and subset should vary.
        # ----------------------------------------------------
        models = {
            "LogisticRegression":
                build_logistic_regression(42),

            "LightGBM":
                build_lightgbm(42)
        }

        for method_name, model in models.items():

            model.fit(
                X_subset,
                y_subset
            )

            test_probabilities = model.predict_proba(
                X_cifar_test
            )

            metrics = evaluate_probabilities(
                y_cifar_test,
                test_probabilities
            )

            result_row = {
                "Dataset": "CIFAR-10",
                "Protocol": "Five-fold OOF",
                "Coverage": coverage,
                "Coverage_Percent": int(
                    coverage * 100
                ),
                "Subset_Seed": subset_seed,
                "Model_Seed": 42,
                "Method": method_name,
                "Meta_Training_Samples": len(
                    subset_indices
                )
            }

            result_row.update(metrics)

            coverage_rows.append(
                result_row
            )

            print(
                f"Coverage={coverage:>4.0%} | "
                f"Subset seed={subset_seed:>4} | "
                f"{method_name:<20} | "
                f"Accuracy={metrics['Accuracy']:.4f} | "
                f"Macro-F1={metrics['Macro_F1']:.4f}"
            )

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------
cifar_coverage_results = pd.DataFrame(
    coverage_rows
)

# ------------------------------------------------------------
# Validate expected row count
# Five partial coverages × 10 seeds × 2 models = 100
# Full coverage × 1 seed × 2 models = 2
# Total = 102
# ------------------------------------------------------------
assert len(cifar_coverage_results) == 102, (
    "Expected 102 rows but obtained "
    f"{len(cifar_coverage_results)}"
)

# ------------------------------------------------------------
# Validate exact full-coverage results
# ------------------------------------------------------------
full_coverage_check = cifar_coverage_results[
    cifar_coverage_results[
        "Coverage_Percent"
    ] == 100
][
    [
        "Method",
        "Accuracy",
        "Macro_F1",
        "ROC_AUC",
        "Log_Loss",
        "Brier_Score",
        "ECE"
    ]
]

display(full_coverage_check.round(6))

logistic_full_accuracy = float(
    full_coverage_check.loc[
        full_coverage_check["Method"]
        == "LogisticRegression",
        "Accuracy"
    ].iloc[0]
)

lightgbm_full_accuracy = float(
    full_coverage_check.loc[
        full_coverage_check["Method"]
        == "LightGBM",
        "Accuracy"
    ].iloc[0]
)

assert np.isclose(
    logistic_full_accuracy,
    0.9185,
    atol=1e-10
)

assert np.isclose(
    lightgbm_full_accuracy,
    0.9198,
    atol=1e-10
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
output_path = os.path.join(
    COVERAGE_DIR,
    "CIFAR10_Coverage_Ablation_Final_Valid.csv"
)

cifar_coverage_results.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 72)
print("FINAL VALID CIFAR COVERAGE ABLATION COMPLETED")
print("Rows generated:", len(cifar_coverage_results))
print("100% Logistic Regression accuracy:",
      logistic_full_accuracy)
print("100% LightGBM accuracy:",
      lightgbm_full_accuracy)
print("Saved to:", output_path)
print("=" * 72)

In [ ]:
# ============================================================
# CIFAR COVERAGE SUMMARY + SOFT-VOTING COMPARISON
# ============================================================

# Soft-voting baseline from the original CIFAR experiment
CIFAR_SOFT_VOTING_ACCURACY = 0.9147

# ------------------------------------------------------------
# 1. Build summary statistics
# ------------------------------------------------------------
cifar_coverage_summary = (
    cifar_coverage_results
    .groupby(
        [
            "Coverage_Percent",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Meta_Training_Samples=(
            "Meta_Training_Samples",
            "first"
        ),
        Runs=("Subset_Seed", "count"),

        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Accuracy_Min=("Accuracy", "min"),
        Accuracy_Max=("Accuracy", "max"),

        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std"),

        ROC_AUC_Mean=("ROC_AUC", "mean"),
        ROC_AUC_SD=("ROC_AUC", "std"),

        Log_Loss_Mean=("Log_Loss", "mean"),
        Log_Loss_SD=("Log_Loss", "std"),

        Brier_Mean=("Brier_Score", "mean"),
        Brier_SD=("Brier_Score", "std"),

        ECE_Mean=("ECE", "mean"),
        ECE_SD=("ECE", "std")
    )
)

# Standard deviation is undefined for the single 100% run.
# Replace that NaN with zero.
sd_columns = [
    "Accuracy_SD",
    "Macro_F1_SD",
    "ROC_AUC_SD",
    "Log_Loss_SD",
    "Brier_SD",
    "ECE_SD"
]

cifar_coverage_summary[
    sd_columns
] = cifar_coverage_summary[
    sd_columns
].fillna(0.0)

# ------------------------------------------------------------
# 2. Add comparison with soft voting
# ------------------------------------------------------------
cifar_coverage_results[
    "Beats_Soft_Voting"
] = (
    cifar_coverage_results["Accuracy"]
    > CIFAR_SOFT_VOTING_ACCURACY
)

cifar_beat_summary = (
    cifar_coverage_results
    .groupby(
        [
            "Coverage_Percent",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Runs=("Subset_Seed", "count"),
        Runs_Beating_Soft_Voting=(
            "Beats_Soft_Voting",
            "sum"
        ),
        Proportion_Beating_Soft_Voting=(
            "Beats_Soft_Voting",
            "mean"
        )
    )
)

cifar_beat_summary[
    "Proportion_Beating_Soft_Voting"
] *= 100

# ------------------------------------------------------------
# 3. Add the soft-voting comparison to the summary table
# ------------------------------------------------------------
cifar_coverage_summary = (
    cifar_coverage_summary
    .merge(
        cifar_beat_summary[
            [
                "Coverage_Percent",
                "Method",
                "Runs_Beating_Soft_Voting",
                "Proportion_Beating_Soft_Voting"
            ]
        ],
        on=[
            "Coverage_Percent",
            "Method"
        ],
        how="left"
    )
)

cifar_coverage_summary[
    "Accuracy_Gain_Over_Soft_Voting"
] = (
    cifar_coverage_summary["Accuracy_Mean"]
    - CIFAR_SOFT_VOTING_ACCURACY
)

# ------------------------------------------------------------
# 4. Create a publication-friendly percentage table
# ------------------------------------------------------------
cifar_coverage_summary_display = (
    cifar_coverage_summary.copy()
)

percentage_columns = [
    "Accuracy_Mean",
    "Accuracy_SD",
    "Accuracy_Min",
    "Accuracy_Max",
    "Macro_F1_Mean",
    "Macro_F1_SD",
    "ROC_AUC_Mean",
    "ROC_AUC_SD",
    "Accuracy_Gain_Over_Soft_Voting"
]

for column in percentage_columns:
    cifar_coverage_summary_display[
        column
    ] *= 100

# ------------------------------------------------------------
# 5. Save both raw and publication-friendly summaries
# ------------------------------------------------------------
raw_summary_path = os.path.join(
    TABLE_DIR,
    "CIFAR10_Coverage_Ablation_Summary_Raw.csv"
)

display_summary_path = os.path.join(
    TABLE_DIR,
    "CIFAR10_Coverage_Ablation_Summary_Percent.csv"
)

beat_summary_path = os.path.join(
    TABLE_DIR,
    "CIFAR10_Coverage_Beating_Soft_Voting.csv"
)

cifar_coverage_summary.to_csv(
    raw_summary_path,
    index=False
)

cifar_coverage_summary_display.to_csv(
    display_summary_path,
    index=False
)

cifar_beat_summary.to_csv(
    beat_summary_path,
    index=False
)

print("Summary files saved:")
print(raw_summary_path)
print(display_summary_path)
print(beat_summary_path)

display(
    cifar_coverage_summary_display.round(4)
)

In [ ]:
# ============================================================
# FINAL CIFAR COVERAGE FIGURE
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8.5, 5.5))

for method in [
    "LogisticRegression",
    "LightGBM"
]:
    method_df = (
        cifar_coverage_summary_display[
            cifar_coverage_summary_display[
                "Method"
            ] == method
        ]
        .sort_values("Coverage_Percent")
    )

    plt.errorbar(
        method_df["Coverage_Percent"],
        method_df["Accuracy_Mean"],
        yerr=method_df["Accuracy_SD"],
        marker="o",
        capsize=4,
        linewidth=1.8,
        label=method
    )

plt.axhline(
    y=91.47,
    linestyle="--",
    linewidth=1.5,
    label="Soft voting"
)

plt.xlabel("OOF meta-training coverage (%)")
plt.ylabel("Test accuracy (%)")
plt.title(
    "CIFAR-10: Effect of OOF meta-training coverage"
)

plt.xticks(
    [10, 20, 40, 60, 80, 100]
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

final_figure_path = os.path.join(
    FIGURE_DIR,
    "CIFAR10_Coverage_vs_Accuracy_Final.png"
)

plt.savefig(
    final_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Final figure saved to:")
print(final_figure_path)

In [ ]:
# ============================================================
# LOAD AND PREPARE CT OOF META-FEATURES
# ============================================================

import os
import pickle
import numpy as np

CT_DATA_PATH = (
    "/content/drive/MyDrive/"
    "PhD_CT_Transfer_Learning/data/"
    "ct_preprocessed.pkl"
)

# Load CT base-model OOF and final-test probabilities
ct_resnet_oof = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_ResNet50_5Fold_OOF.npy"
    )
)

ct_effnet_oof = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_EfficientNetB0_5Fold_OOF.npy"
    )
)

ct_resnet_test = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_ResNet50_Test_Probabilities.npy"
    )
)

ct_effnet_test = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_EfficientNetB0_Test_Probabilities.npy"
    )
)

# Load CT labels from the preprocessed dataset
with open(CT_DATA_PATH, "rb") as file:
    ct_data = pickle.load(file)

y_ct_train = np.asarray(
    ct_data["y_train"]
).reshape(-1).astype(np.int64)

y_ct_val = np.asarray(
    ct_data["y_val"]
).reshape(-1).astype(np.int64)

y_ct_dev = np.concatenate(
    [
        y_ct_train,
        y_ct_val
    ],
    axis=0
)

y_ct_test = np.asarray(
    ct_data["y_test"]
).reshape(-1).astype(np.int64)

# Build exact CT meta-feature matrices:
# ResNet50 probabilities followed by EfficientNetB0 probabilities
X_ct_oof = np.concatenate(
    [
        ct_resnet_oof,
        ct_effnet_oof
    ],
    axis=1
).astype(np.float32)

X_ct_test = np.concatenate(
    [
        ct_resnet_test,
        ct_effnet_test
    ],
    axis=1
).astype(np.float32)

print("CT OOF meta-feature shape:", X_ct_oof.shape)
print("CT test meta-feature shape:", X_ct_test.shape)
print("CT development labels:", y_ct_dev.shape)
print("CT test labels:", y_ct_test.shape)
print("Development class counts:", np.bincount(y_ct_dev))
print("Test class counts:", np.bincount(y_ct_test))

# Validate probability blocks
for start, end, model_name in [
    (0, 2, "ResNet50"),
    (2, 4, "EfficientNetB0")
]:
    assert np.allclose(
        X_ct_oof[:, start:end].sum(axis=1),
        1.0,
        atol=1e-5
    )

    assert np.allclose(
        X_ct_test[:, start:end].sum(axis=1),
        1.0,
        atol=1e-5
    )

    print(model_name, "probability blocks verified.")

assert X_ct_oof.shape == (1984, 4)
assert X_ct_test.shape == (497, 4)

print("\nCT meta-feature preparation completed successfully.")

In [ ]:
# ============================================================
# CT FULL-COVERAGE BASELINE REPRODUCTION
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

# Logistic Regression configuration
# Use scaling for the CT binary meta-features
def build_ct_logistic(seed=42):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=3000,
                    solver="lbfgs",
                    random_state=seed
                )
            )
        ]
    )

# LightGBM configuration
def build_ct_lightgbm(seed=42):
    return LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.20,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

# Train on all CT OOF meta-features
ct_logistic_model = build_ct_logistic(42)
ct_logistic_model.fit(
    X_ct_oof,
    y_ct_dev
)

ct_logistic_probs = ct_logistic_model.predict_proba(
    X_ct_test
)

ct_lightgbm_model = build_ct_lightgbm(42)
ct_lightgbm_model.fit(
    X_ct_oof,
    y_ct_dev
)

ct_lightgbm_probs = ct_lightgbm_model.predict_proba(
    X_ct_test
)

# Evaluate
ct_logistic_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(ct_logistic_probs, axis=1)
)

ct_lightgbm_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(ct_lightgbm_probs, axis=1)
)

print("CT Logistic Regression accuracy:", ct_logistic_accuracy)
print("CT LightGBM accuracy:", ct_lightgbm_accuracy)

# Full metrics
ct_baseline_rows = []

for method_name, probabilities in [
    ("LogisticRegression", ct_logistic_probs),
    ("LightGBM", ct_lightgbm_probs)
]:
    metrics = evaluate_probabilities(
        y_ct_test,
        probabilities
    )

    ct_baseline_rows.append(
        {
            "Dataset": "CT",
            "Coverage_Percent": 100,
            "Method": method_name,
            **metrics
        }
    )

ct_full_coverage_baseline = pd.DataFrame(
    ct_baseline_rows
)

display(
    ct_full_coverage_baseline.round(6)
)

In [ ]:
# ============================================================
# CORRECTED EVALUATION FUNCTION
# Works for both binary and multiclass classification
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

def evaluate_probabilities(
    y_true,
    probabilities
):
    y_true = np.asarray(y_true).reshape(-1)
    probabilities = np.asarray(probabilities)

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    result = {
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Macro_Precision": precision_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Log_Loss": log_loss(
            y_true,
            probabilities
        ),
        "Brier_Score": multiclass_brier_score(
            y_true,
            probabilities
        ),
        "ECE": expected_calibration_error(
            y_true,
            probabilities
        )
    }

    # Binary classification
    if probabilities.shape[1] == 2:
        result["ROC_AUC"] = roc_auc_score(
            y_true,
            probabilities[:, 1]
        )

    # Multiclass classification
    else:
        result["ROC_AUC"] = roc_auc_score(
            y_true,
            probabilities,
            multi_class="ovr",
            average="macro"
        )

    return result

print("Corrected evaluation function is ready.")

In [ ]:
# ============================================================
# RECALCULATE CT FULL-COVERAGE METRICS
# ============================================================

ct_baseline_rows = []

for method_name, probabilities in [
    ("LogisticRegression", ct_logistic_probs),
    ("LightGBM", ct_lightgbm_probs)
]:
    metrics = evaluate_probabilities(
        y_ct_test,
        probabilities
    )

    ct_baseline_rows.append(
        {
            "Dataset": "CT",
            "Coverage_Percent": 100,
            "Method": method_name,
            **metrics
        }
    )

ct_full_coverage_baseline = pd.DataFrame(
    ct_baseline_rows
)

display(
    ct_full_coverage_baseline.round(6)
)

In [ ]:
# ============================================================
# CT OOF META-TRAINING COVERAGE ABLATION
# ============================================================

CT_COVERAGE_LEVELS = [
    0.10,
    0.20,
    0.40,
    0.60,
    0.80,
    1.00
]

CT_ABLATION_SEEDS = [
    42,
    123,
    2025,
    7,
    99,
    314,
    512,
    1001,
    2026,
    777
]

ct_coverage_rows = []

for coverage in CT_COVERAGE_LEVELS:

    # At full coverage there is no subset-selection variation
    subset_seeds = (
        [42]
        if coverage >= 1.0
        else CT_ABLATION_SEEDS
    )

    for subset_seed in subset_seeds:

        if coverage >= 1.0:
            subset_indices = np.arange(
                len(y_ct_dev)
            )

        else:
            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=coverage,
                random_state=subset_seed
            )

            subset_indices, _ = next(
                splitter.split(
                    np.zeros(len(y_ct_dev)),
                    y_ct_dev
                )
            )

        X_subset = X_ct_oof[
            subset_indices
        ]

        y_subset = y_ct_dev[
            subset_indices
        ]

        models = {
            "LogisticRegression":
                build_ct_logistic(42),

            "LightGBM":
                build_ct_lightgbm(42)
        }

        for method_name, model in models.items():

            model.fit(
                X_subset,
                y_subset
            )

            test_probabilities = model.predict_proba(
                X_ct_test
            )

            metrics = evaluate_probabilities(
                y_ct_test,
                test_probabilities
            )

            result_row = {
                "Dataset": "CT",
                "Protocol": "Five-fold OOF",
                "Coverage": coverage,
                "Coverage_Percent": int(
                    coverage * 100
                ),
                "Subset_Seed": subset_seed,
                "Model_Seed": 42,
                "Method": method_name,
                "Meta_Training_Samples":
                    len(subset_indices)
            }

            result_row.update(metrics)
            ct_coverage_rows.append(
                result_row
            )

            print(
                f"Coverage={coverage:>4.0%} | "
                f"Subset seed={subset_seed:>4} | "
                f"{method_name:<20} | "
                f"Accuracy={metrics['Accuracy']:.4f} | "
                f"Macro-F1={metrics['Macro_F1']:.4f}"
            )

ct_coverage_results = pd.DataFrame(
    ct_coverage_rows
)

assert len(ct_coverage_results) == 102, (
    "Expected 102 rows but obtained "
    f"{len(ct_coverage_results)}"
)

# Validate full-coverage results
ct_full_check = ct_coverage_results[
    ct_coverage_results[
        "Coverage_Percent"
    ] == 100
][
    [
        "Method",
        "Accuracy",
        "Macro_F1",
        "ROC_AUC",
        "Log_Loss",
        "Brier_Score",
        "ECE"
    ]
]

display(
    ct_full_check.round(6)
)

for method in [
    "LogisticRegression",
    "LightGBM"
]:
    full_accuracy = float(
        ct_full_check.loc[
            ct_full_check["Method"] == method,
            "Accuracy"
        ].iloc[0]
    )

    assert np.isclose(
        full_accuracy,
        0.9637826961770624,
        atol=1e-10
    )

ct_output_path = os.path.join(
    COVERAGE_DIR,
    "CT_Coverage_Ablation_Final_Valid.csv"
)

ct_coverage_results.to_csv(
    ct_output_path,
    index=False
)

print("\n" + "=" * 72)
print("FINAL VALID CT COVERAGE ABLATION COMPLETED")
print("Rows generated:", len(ct_coverage_results))
print("Saved to:", ct_output_path)
print("=" * 72)

In [ ]:
# ============================================================
# CT COVERAGE SUMMARY + SOFT-VOTING COMPARISON
# ============================================================

# Recalculate CT soft-voting baseline from the same test probabilities
ct_soft_voting_probs = (
    ct_resnet_test + ct_effnet_test
) / 2.0

ct_soft_voting_metrics = evaluate_probabilities(
    y_ct_test,
    ct_soft_voting_probs
)

CT_SOFT_VOTING_ACCURACY = (
    ct_soft_voting_metrics["Accuracy"]
)

print("CT soft-voting metrics:")
print(ct_soft_voting_metrics)

# ------------------------------------------------------------
# 1. Build summary statistics
# ------------------------------------------------------------
ct_coverage_summary = (
    ct_coverage_results
    .groupby(
        [
            "Coverage_Percent",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Meta_Training_Samples=(
            "Meta_Training_Samples",
            "first"
        ),
        Runs=("Subset_Seed", "count"),

        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_SD=("Accuracy", "std"),
        Accuracy_Min=("Accuracy", "min"),
        Accuracy_Max=("Accuracy", "max"),

        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_SD=("Macro_F1", "std"),

        ROC_AUC_Mean=("ROC_AUC", "mean"),
        ROC_AUC_SD=("ROC_AUC", "std"),

        Log_Loss_Mean=("Log_Loss", "mean"),
        Log_Loss_SD=("Log_Loss", "std"),

        Brier_Mean=("Brier_Score", "mean"),
        Brier_SD=("Brier_Score", "std"),

        ECE_Mean=("ECE", "mean"),
        ECE_SD=("ECE", "std")
    )
)

# Replace undefined SD for the single 100% run with zero
sd_columns = [
    "Accuracy_SD",
    "Macro_F1_SD",
    "ROC_AUC_SD",
    "Log_Loss_SD",
    "Brier_SD",
    "ECE_SD"
]

ct_coverage_summary[
    sd_columns
] = ct_coverage_summary[
    sd_columns
].fillna(0.0)

# ------------------------------------------------------------
# 2. Compare every run with soft voting
# ------------------------------------------------------------
ct_coverage_results[
    "Beats_Soft_Voting"
] = (
    ct_coverage_results["Accuracy"]
    > CT_SOFT_VOTING_ACCURACY
)

ct_beat_summary = (
    ct_coverage_results
    .groupby(
        [
            "Coverage_Percent",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Runs=("Subset_Seed", "count"),
        Runs_Beating_Soft_Voting=(
            "Beats_Soft_Voting",
            "sum"
        ),
        Proportion_Beating_Soft_Voting=(
            "Beats_Soft_Voting",
            "mean"
        )
    )
)

ct_beat_summary[
    "Proportion_Beating_Soft_Voting"
] *= 100

# ------------------------------------------------------------
# 3. Merge soft-voting comparison into summary
# ------------------------------------------------------------
ct_coverage_summary = (
    ct_coverage_summary
    .merge(
        ct_beat_summary[
            [
                "Coverage_Percent",
                "Method",
                "Runs_Beating_Soft_Voting",
                "Proportion_Beating_Soft_Voting"
            ]
        ],
        on=[
            "Coverage_Percent",
            "Method"
        ],
        how="left"
    )
)

ct_coverage_summary[
    "Accuracy_Gain_Over_Soft_Voting"
] = (
    ct_coverage_summary["Accuracy_Mean"]
    - CT_SOFT_VOTING_ACCURACY
)

# ------------------------------------------------------------
# 4. Create publication-friendly percentage table
# ------------------------------------------------------------
ct_coverage_summary_display = (
    ct_coverage_summary.copy()
)

percentage_columns = [
    "Accuracy_Mean",
    "Accuracy_SD",
    "Accuracy_Min",
    "Accuracy_Max",
    "Macro_F1_Mean",
    "Macro_F1_SD",
    "ROC_AUC_Mean",
    "ROC_AUC_SD",
    "Accuracy_Gain_Over_Soft_Voting"
]

for column in percentage_columns:
    ct_coverage_summary_display[
        column
    ] *= 100

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------
ct_raw_summary_path = os.path.join(
    TABLE_DIR,
    "CT_Coverage_Ablation_Summary_Raw.csv"
)

ct_percent_summary_path = os.path.join(
    TABLE_DIR,
    "CT_Coverage_Ablation_Summary_Percent.csv"
)

ct_beat_summary_path = os.path.join(
    TABLE_DIR,
    "CT_Coverage_Beating_Soft_Voting.csv"
)

ct_soft_voting_path = os.path.join(
    TABLE_DIR,
    "CT_Soft_Voting_Baseline.csv"
)

ct_coverage_summary.to_csv(
    ct_raw_summary_path,
    index=False
)

ct_coverage_summary_display.to_csv(
    ct_percent_summary_path,
    index=False
)

ct_beat_summary.to_csv(
    ct_beat_summary_path,
    index=False
)

pd.DataFrame(
    [
        {
            "Dataset": "CT",
            "Method": "SoftVoting",
            **ct_soft_voting_metrics
        }
    ]
).to_csv(
    ct_soft_voting_path,
    index=False
)

print("\nSaved:")
print(ct_raw_summary_path)
print(ct_percent_summary_path)
print(ct_beat_summary_path)
print(ct_soft_voting_path)

display(
    ct_coverage_summary_display.round(4)
)

In [ ]:
# ============================================================
# FINAL CT COVERAGE FIGURE
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8.5, 5.5))

for method in [
    "LogisticRegression",
    "LightGBM"
]:
    method_df = (
        ct_coverage_summary_display[
            ct_coverage_summary_display[
                "Method"
            ] == method
        ]
        .sort_values("Coverage_Percent")
    )

    plt.errorbar(
        method_df["Coverage_Percent"],
        method_df["Accuracy_Mean"],
        yerr=method_df["Accuracy_SD"],
        marker="o",
        capsize=4,
        linewidth=1.8,
        label=method
    )

plt.axhline(
    y=CT_SOFT_VOTING_ACCURACY * 100,
    linestyle="--",
    linewidth=1.5,
    label="Soft voting"
)

plt.xlabel("OOF meta-training coverage (%)")
plt.ylabel("Test accuracy (%)")
plt.title(
    "CT: Effect of OOF meta-training coverage"
)

plt.xticks(
    [10, 20, 40, 60, 80, 100]
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

ct_final_figure_path = os.path.join(
    FIGURE_DIR,
    "CT_Coverage_vs_Accuracy_Final.png"
)

plt.savefig(
    ct_final_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Final CT figure saved to:")
print(ct_final_figure_path)

In [ ]:
# ============================================================
# PRINT KEY CT COVERAGE FINDINGS
# ============================================================

for method in [
    "LogisticRegression",
    "LightGBM"
]:
    method_df = (
        ct_coverage_summary_display[
            ct_coverage_summary_display[
                "Method"
            ] == method
        ]
        .sort_values("Coverage_Percent")
    )

    print("\n", method)
    print(
        method_df[
            [
                "Coverage_Percent",
                "Accuracy_Mean",
                "Accuracy_SD",
                "Runs_Beating_Soft_Voting",
                "Proportion_Beating_Soft_Voting",
                "Accuracy_Gain_Over_Soft_Voting",
                "ECE_Mean"
            ]
        ].round(4).to_string(index=False)
    )

print("\nCT soft-voting accuracy (%):")
print(round(CT_SOFT_VOTING_ACCURACY * 100, 4))

In [ ]:
# ============================================================
# COMBINE CIFAR-10 AND CT COVERAGE RESULTS
# ============================================================

combined_coverage_summary = pd.concat(
    [
        cifar_coverage_summary_display.assign(
            Dataset="CIFAR-10"
        ),
        ct_coverage_summary_display.assign(
            Dataset="CT"
        )
    ],
    ignore_index=True
)

combined_columns = [
    "Dataset",
    "Coverage_Percent",
    "Method",
    "Meta_Training_Samples",
    "Runs",
    "Accuracy_Mean",
    "Accuracy_SD",
    "Macro_F1_Mean",
    "ROC_AUC_Mean",
    "Log_Loss_Mean",
    "Brier_Mean",
    "ECE_Mean",
    "Runs_Beating_Soft_Voting",
    "Proportion_Beating_Soft_Voting",
    "Accuracy_Gain_Over_Soft_Voting"
]

combined_coverage_summary = (
    combined_coverage_summary[
        combined_columns
    ]
    .sort_values(
        [
            "Dataset",
            "Coverage_Percent",
            "Method"
        ]
    )
)

combined_summary_path = os.path.join(
    TABLE_DIR,
    "Cross_Domain_Coverage_Ablation_Summary.csv"
)

combined_coverage_summary.to_csv(
    combined_summary_path,
    index=False
)

display(
    combined_coverage_summary.round(4)
)

print("Combined summary saved to:")
print(combined_summary_path)

In [ ]:
# ============================================================
# CROSS-DOMAIN COVERAGE FIGURE
# ============================================================

for dataset_name, summary_df, soft_voting_accuracy in [
    (
        "CIFAR-10",
        cifar_coverage_summary_display,
        91.47
    ),
    (
        "CT",
        ct_coverage_summary_display,
        CT_SOFT_VOTING_ACCURACY * 100
    )
]:
    plt.figure(figsize=(8.5, 5.5))

    for method in [
        "LogisticRegression",
        "LightGBM"
    ]:
        method_df = (
            summary_df[
                summary_df["Method"] == method
            ]
            .sort_values("Coverage_Percent")
        )

        plt.errorbar(
            method_df["Coverage_Percent"],
            method_df["Accuracy_Mean"],
            yerr=method_df["Accuracy_SD"],
            marker="o",
            capsize=4,
            linewidth=1.8,
            label=method
        )

    plt.axhline(
        y=soft_voting_accuracy,
        linestyle="--",
        linewidth=1.5,
        label="Soft voting"
    )

    plt.xlabel("OOF meta-training coverage (%)")
    plt.ylabel("Test accuracy (%)")
    plt.title(
        f"{dataset_name}: Coverage sensitivity of stacking"
    )
    plt.xticks([10, 20, 40, 60, 80, 100])
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    output_path = os.path.join(
        FIGURE_DIR,
        f"{dataset_name.replace('-', '')}_Coverage_Sensitivity_Final.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Saved:", output_path)

In [ ]:
# ============================================================
# REPEATED FIXED 80:20 HOLD-OUT — CT SETUP
# ============================================================

import os
import gc
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

HOLDOUT_SEEDS = [
    42,
    123,
    2025
]

CT_REPEATED_HOLDOUT_DIR = os.path.join(
    EXT_DIR,
    "Repeated_Holdout",
    "CT"
)

os.makedirs(
    CT_REPEATED_HOLDOUT_DIR,
    exist_ok=True
)

def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# Load CT images if they are not already available
if "X_ct_development" not in globals():

    X_ct_train = np.asarray(
        ct_data["X_train"],
        dtype=np.float32
    )

    X_ct_val = np.asarray(
        ct_data["X_val"],
        dtype=np.float32
    )

    X_ct_test_images = np.asarray(
        ct_data["X_test"],
        dtype=np.float32
    )

    X_ct_development = np.concatenate(
        [
            X_ct_train,
            X_ct_val
        ],
        axis=0
    )

    if X_ct_development.max() > 1.0:
        X_ct_development /= 255.0

    if X_ct_test_images.max() > 1.0:
        X_ct_test_images /= 255.0

    X_ct_development = np.clip(
        X_ct_development,
        0.0,
        1.0
    ).astype(np.float32)

    X_ct_test_images = np.clip(
        X_ct_test_images,
        0.0,
        1.0
    ).astype(np.float32)

print("CT development images:", X_ct_development.shape)
print("CT development labels:", y_ct_dev.shape)
print("CT final test images:", X_ct_test_images.shape)
print("CT final test labels:", y_ct_test.shape)

assert X_ct_development.shape == (
    1984,
    224,
    224,
    1
)

assert X_ct_test_images.shape == (
    497,
    224,
    224,
    1
)

# Create and save the three stratified 80:20 splits
split_rows = []

for holdout_seed in HOLDOUT_SEEDS:

    all_indices = np.arange(
        len(y_ct_dev)
    )

    base_indices, meta_indices = train_test_split(
        all_indices,
        test_size=0.20,
        stratify=y_ct_dev,
        random_state=holdout_seed
    )

    seed_dir = os.path.join(
        CT_REPEATED_HOLDOUT_DIR,
        f"Seed_{holdout_seed}"
    )

    os.makedirs(
        seed_dir,
        exist_ok=True
    )

    np.save(
        os.path.join(
            seed_dir,
            "Base_Training_Indices.npy"
        ),
        base_indices
    )

    np.save(
        os.path.join(
            seed_dir,
            "Meta_Holdout_Indices.npy"
        ),
        meta_indices
    )

    split_rows.append(
        {
            "Dataset": "CT",
            "Seed": holdout_seed,
            "Base_Training_Samples":
                len(base_indices),
            "Meta_Training_Samples":
                len(meta_indices),
            "Base_Class_0":
                int(np.sum(
                    y_ct_dev[base_indices] == 0
                )),
            "Base_Class_1":
                int(np.sum(
                    y_ct_dev[base_indices] == 1
                )),
            "Meta_Class_0":
                int(np.sum(
                    y_ct_dev[meta_indices] == 0
                )),
            "Meta_Class_1":
                int(np.sum(
                    y_ct_dev[meta_indices] == 1
                ))
        }
    )

ct_repeated_holdout_splits = pd.DataFrame(
    split_rows
)

ct_repeated_holdout_splits.to_csv(
    os.path.join(
        CT_REPEATED_HOLDOUT_DIR,
        "CT_Repeated_Holdout_Split_Summary.csv"
    ),
    index=False
)

display(ct_repeated_holdout_splits)

print(
    "\nRepeated CT hold-out splits created successfully."
)

In [ ]:
# ============================================================
# REPEATED FIXED 80:20 HOLD-OUT — CT DATA SETUP
# Correct keys from ct_preprocessed.pkl
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

HOLDOUT_SEEDS = [42, 123, 2025]

CT_REPEATED_HOLDOUT_DIR = os.path.join(
    EXT_DIR,
    "Repeated_Holdout",
    "CT"
)

os.makedirs(
    CT_REPEATED_HOLDOUT_DIR,
    exist_ok=True
)

def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# ------------------------------------------------------------
# Load baseline arrays for ResNet50
# ------------------------------------------------------------

X_ct_resnet_development = np.concatenate(
    [
        np.asarray(
            ct_data["X_train_baseline"],
            dtype=np.float32
        ),
        np.asarray(
            ct_data["X_val_baseline"],
            dtype=np.float32
        )
    ],
    axis=0
)

X_ct_resnet_test = np.asarray(
    ct_data["X_test_baseline"],
    dtype=np.float32
)

# ------------------------------------------------------------
# Load enhanced arrays for EfficientNetB0
# ------------------------------------------------------------

X_ct_effnet_development = np.concatenate(
    [
        np.asarray(
            ct_data["X_train_enhanced"],
            dtype=np.float32
        ),
        np.asarray(
            ct_data["X_val_enhanced"],
            dtype=np.float32
        )
    ],
    axis=0
)

X_ct_effnet_test = np.asarray(
    ct_data["X_test_enhanced"],
    dtype=np.float32
)

# Labels are shared by both variants
y_ct_dev = np.concatenate(
    [
        np.asarray(
            ct_data["y_train"]
        ).reshape(-1),
        np.asarray(
            ct_data["y_val"]
        ).reshape(-1)
    ],
    axis=0
).astype(np.int64)

y_ct_test = np.asarray(
    ct_data["y_test"]
).reshape(-1).astype(np.int64)

# ------------------------------------------------------------
# Normalize only when needed
# ------------------------------------------------------------

def normalize_ct_array(array):
    array = np.asarray(
        array,
        dtype=np.float32
    )

    if array.max() > 1.0:
        array = array / 255.0

    return np.clip(
        array,
        0.0,
        1.0
    ).astype(np.float32)

X_ct_resnet_development = normalize_ct_array(
    X_ct_resnet_development
)

X_ct_resnet_test = normalize_ct_array(
    X_ct_resnet_test
)

X_ct_effnet_development = normalize_ct_array(
    X_ct_effnet_development
)

X_ct_effnet_test = normalize_ct_array(
    X_ct_effnet_test
)

# ------------------------------------------------------------
# Validate dimensions
# ------------------------------------------------------------

print("ResNet development:",
      X_ct_resnet_development.shape)
print("ResNet test:",
      X_ct_resnet_test.shape)

print("EfficientNet development:",
      X_ct_effnet_development.shape)
print("EfficientNet test:",
      X_ct_effnet_test.shape)

print("Development labels:",
      y_ct_dev.shape)
print("Test labels:",
      y_ct_test.shape)

print("Development class counts:",
      np.bincount(y_ct_dev))
print("Test class counts:",
      np.bincount(y_ct_test))

assert X_ct_resnet_development.shape == (
    1984, 224, 224, 1
)

assert X_ct_effnet_development.shape == (
    1984, 224, 224, 1
)

assert X_ct_resnet_test.shape == (
    497, 224, 224, 1
)

assert X_ct_effnet_test.shape == (
    497, 224, 224, 1
)

assert len(y_ct_dev) == 1984
assert len(y_ct_test) == 497

# ------------------------------------------------------------
# Create repeated stratified 80:20 splits
# ------------------------------------------------------------

split_rows = []

for holdout_seed in HOLDOUT_SEEDS:

    all_indices = np.arange(
        len(y_ct_dev)
    )

    base_indices, meta_indices = train_test_split(
        all_indices,
        test_size=0.20,
        stratify=y_ct_dev,
        random_state=holdout_seed
    )

    seed_dir = os.path.join(
        CT_REPEATED_HOLDOUT_DIR,
        f"Seed_{holdout_seed}"
    )

    os.makedirs(
        seed_dir,
        exist_ok=True
    )

    np.save(
        os.path.join(
            seed_dir,
            "Base_Training_Indices.npy"
        ),
        base_indices
    )

    np.save(
        os.path.join(
            seed_dir,
            "Meta_Holdout_Indices.npy"
        ),
        meta_indices
    )

    split_rows.append(
        {
            "Dataset": "CT",
            "Seed": holdout_seed,
            "Base_Training_Samples":
                len(base_indices),
            "Meta_Training_Samples":
                len(meta_indices),
            "Base_Class_0":
                int(np.sum(
                    y_ct_dev[base_indices] == 0
                )),
            "Base_Class_1":
                int(np.sum(
                    y_ct_dev[base_indices] == 1
                )),
            "Meta_Class_0":
                int(np.sum(
                    y_ct_dev[meta_indices] == 0
                )),
            "Meta_Class_1":
                int(np.sum(
                    y_ct_dev[meta_indices] == 1
                ))
        }
    )

ct_repeated_holdout_splits = pd.DataFrame(
    split_rows
)

split_summary_path = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    "CT_Repeated_Holdout_Split_Summary.csv"
)

ct_repeated_holdout_splits.to_csv(
    split_summary_path,
    index=False
)

display(ct_repeated_holdout_splits)

print("\nCT repeated hold-out setup completed.")
print("Saved to:", split_summary_path)

In [ ]:
# ============================================================
# CT REPEATED HOLD-OUT
# TRAIN RESNET50 FOR SEED 42
# ============================================================

import os
import numpy as np
import tensorflow as tf

HOLDOUT_SEED = 42

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet50"
)

os.makedirs(
    resnet_dir,
    exist_ok=True
)

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

X_base = X_ct_resnet_development[base_indices]
y_base = y_ct_dev[base_indices]

X_meta = X_ct_resnet_development[meta_indices]
y_meta = y_ct_dev[meta_indices]

print("Base training:", X_base.shape, y_base.shape)
print("Meta hold-out:", X_meta.shape, y_meta.shape)
print("Final test:", X_ct_resnet_test.shape, y_ct_test.shape)

set_all_seeds(HOLDOUT_SEED)
clear_training_memory()

# ------------------------------------------------------------
# Build ResNet50 model
# Uses grayscale-to-RGB conversion inside the model
# ------------------------------------------------------------

def build_ct_resnet50_holdout():
    inputs = tf.keras.Input(
        shape=(224, 224, 1)
    )

    rgb = tf.keras.layers.Concatenate()(
        [inputs, inputs, inputs]
    )

    backbone = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=rgb
    )

    backbone.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
    x = tf.keras.layers.Dropout(0.30)(x)
    outputs = tf.keras.layers.Dense(
        2,
        activation="softmax"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model_path = os.path.join(
    resnet_dir,
    "CT_ResNet50_Seed42_Final.keras"
)

meta_prob_path = os.path.join(
    resnet_dir,
    "CT_ResNet50_Seed42_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    resnet_dir,
    "CT_ResNet50_Seed42_Test_Probabilities.npy"
)

history_path = os.path.join(
    resnet_dir,
    "CT_ResNet50_Seed42_History.csv"
)

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):
    print("Saved ResNet50 probabilities already exist. Loading them.")

    resnet_meta_probs_seed42 = np.load(
        meta_prob_path
    )

    resnet_test_probs_seed42 = np.load(
        test_prob_path
    )

else:
    resnet_model_seed42 = build_ct_resnet50_holdout()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=model_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    history = resnet_model_seed42.fit(
        X_base,
        y_base,
        validation_split=0.15,
        epochs=20,
        batch_size=16,
        shuffle=True,
        callbacks=callbacks,
        verbose=1
    )

    pd.DataFrame(
        history.history
    ).to_csv(
        history_path,
        index=False
    )

    resnet_meta_probs_seed42 = (
        resnet_model_seed42.predict(
            X_meta,
            batch_size=16,
            verbose=1
        )
    )

    resnet_test_probs_seed42 = (
        resnet_model_seed42.predict(
            X_ct_resnet_test,
            batch_size=16,
            verbose=1
        )
    )

    np.save(
        meta_prob_path,
        resnet_meta_probs_seed42.astype(
            np.float32
        )
    )

    np.save(
        test_prob_path,
        resnet_test_probs_seed42.astype(
            np.float32
        )
    )

# ------------------------------------------------------------
# Validate outputs
# ------------------------------------------------------------

assert resnet_meta_probs_seed42.shape == (
    397,
    2
)

assert resnet_test_probs_seed42.shape == (
    497,
    2
)

assert np.allclose(
    resnet_meta_probs_seed42.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    resnet_test_probs_seed42.sum(axis=1),
    1.0,
    atol=1e-5
)

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        resnet_meta_probs_seed42,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(
        resnet_test_probs_seed42,
        axis=1
    )
)

print("\nResNet50 seed 42 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", resnet_dir)

In [ ]:
# ============================================================
# REMOVE INVALID SIMPLIFIED RESNET50 SEED-42 OUTPUTS
# ============================================================

import os
import shutil

invalid_resnet_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    "Seed_42",
    "ResNet50"
)

if os.path.exists(invalid_resnet_dir):
    shutil.rmtree(invalid_resnet_dir)
    print("Removed invalid directory:")
    print(invalid_resnet_dir)
else:
    print("Invalid directory was not present.")

# Remove related variables from memory
for variable_name in [
    "resnet_model_seed42",
    "resnet_meta_probs_seed42",
    "resnet_test_probs_seed42",
    "X_base",
    "X_meta",
    "y_base",
    "y_meta"
]:
    if variable_name in globals():
        del globals()[variable_name]

clear_training_memory()

print("Invalid ResNet50 seed-42 run cleared.")

In [ ]:
# ============================================================
# MATCHED CT RESNET50 CONFIGURATION
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, regularizers

CT_L2 = 1e-4
CT_DROPOUT = 0.50
CT_HEAD_EPOCHS = 10
CT_FINE_TUNE_EPOCHS = 20
CT_BATCH_SIZE = 16

def build_ct_augmentation():
    return tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(
                factor=0.03,
                fill_mode="nearest"
            ),
            layers.RandomTranslation(
                height_factor=0.03,
                width_factor=0.03,
                fill_mode="nearest"
            ),
            layers.RandomZoom(
                height_factor=(-0.05, 0.05),
                width_factor=(-0.05, 0.05),
                fill_mode="nearest"
            )
        ],
        name="ct_augmentation"
    )


def build_matched_ct_resnet50():
    inputs = tf.keras.Input(
        shape=(224, 224, 1),
        name="ct_input"
    )

    x = build_ct_augmentation()(inputs)

    # Convert grayscale to three channels
    x = tf.keras.layers.Concatenate(
        name="grayscale_to_rgb"
    )([x, x, x])

    # ResNet applications expect ImageNet-style preprocessing
    x = tf.keras.applications.resnet.preprocess_input(
        x * 255.0
    )

    backbone = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False
    )

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.BatchNormalization(
        name="head_batch_normalization"
    )(x)

    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizers.l2(
            CT_L2
        ),
        name="head_dense_256"
    )(x)

    x = layers.Dropout(
        CT_DROPOUT,
        name="head_dropout"
    )(x)

    outputs = layers.Dense(
        2,
        activation="softmax",
        name="classification_output"
    )(x)

    model = tf.keras.Model(
        inputs,
        outputs,
        name="CT_ResNet50_Matched"
    )

    return model, backbone


def compile_ct_resnet_head(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )


def prepare_ct_resnet_fine_tuning(
    model,
    backbone
):
    backbone.trainable = True

    # Freeze most of the backbone and fine-tune only
    # the upper ResNet block.
    for layer in backbone.layers[:-30]:
        layer.trainable = False

    # Keep BatchNormalization layers frozen for stability.
    for layer in backbone.layers:
        if isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        ):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-5
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

print("Matched CT ResNet50 configuration defined.")

In [ ]:
# ============================================================
# RETRAIN MATCHED CT RESNET50 — HOLD-OUT SEED 42
# ============================================================

import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

HOLDOUT_SEED = 42

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet50_Matched"
)

os.makedirs(
    resnet_dir,
    exist_ok=True
)

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

X_base_all = X_ct_resnet_development[
    base_indices
]

y_base_all = y_ct_dev[
    base_indices
]

X_meta = X_ct_resnet_development[
    meta_indices
]

y_meta = y_ct_dev[
    meta_indices
]

# Internal validation comes only from the 80% base partition.
train_indices_local, val_indices_local = (
    train_test_split(
        np.arange(len(y_base_all)),
        test_size=0.10,
        stratify=y_base_all,
        random_state=HOLDOUT_SEED
    )
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

print(
    "Internal training:",
    X_train_internal.shape,
    y_train_internal.shape
)

print(
    "Internal validation:",
    X_val_internal.shape,
    y_val_internal.shape
)

print(
    "Untouched meta hold-out:",
    X_meta.shape,
    y_meta.shape
)

set_all_seeds(HOLDOUT_SEED)
clear_training_memory()

model, backbone = build_matched_ct_resnet50()
compile_ct_resnet_head(model)

head_checkpoint = os.path.join(
    resnet_dir,
    "ResNet50_Seed42_Head_Best.weights.h5"
)

fine_checkpoint = os.path.join(
    resnet_dir,
    "ResNet50_Seed42_FineTune_Best.weights.h5"
)

common_callbacks_head = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        head_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 1: Training classification head")

history_head = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_HEAD_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=common_callbacks_head,
    verbose=1
)

prepare_ct_resnet_fine_tuning(
    model,
    backbone
)

common_callbacks_fine = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        fine_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 2: Fine-tuning upper ResNet layers")

history_fine = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_FINE_TUNE_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=common_callbacks_fine,
    verbose=1
)

# Save histories
pd.DataFrame(
    history_head.history
).to_csv(
    os.path.join(
        resnet_dir,
        "ResNet50_Seed42_Head_History.csv"
    ),
    index=False
)

pd.DataFrame(
    history_fine.history
).to_csv(
    os.path.join(
        resnet_dir,
        "ResNet50_Seed42_FineTune_History.csv"
    ),
    index=False
)

# Generate leakage-safe meta probabilities
resnet_meta_probs_seed42 = model.predict(
    X_meta,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

resnet_test_probs_seed42 = model.predict(
    X_ct_resnet_test,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

np.save(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed42_Meta_Probabilities.npy"
    ),
    resnet_meta_probs_seed42.astype(
        np.float32
    )
)

np.save(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed42_Test_Probabilities.npy"
    ),
    resnet_test_probs_seed42.astype(
        np.float32
    )
)

model.save_weights(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed42_Final.weights.h5"
    )
)

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        resnet_meta_probs_seed42,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(
        resnet_test_probs_seed42,
        axis=1
    )
)

print("\nMatched ResNet50 seed 42 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", resnet_dir)

In [ ]:
# ============================================================
# MATCHED CT EFFICIENTNETB0 — HOLD-OUT SEED 42
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, regularizers

HOLDOUT_SEED = 42

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

effnet_dir = os.path.join(
    seed_dir,
    "EfficientNetB0_Matched"
)

os.makedirs(
    effnet_dir,
    exist_ok=True
)

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

X_base_all = X_ct_effnet_development[
    base_indices
]

y_base_all = y_ct_dev[
    base_indices
]

X_meta = X_ct_effnet_development[
    meta_indices
]

y_meta = y_ct_dev[
    meta_indices
]

# Internal validation comes only from the base-training partition
train_indices_local, val_indices_local = train_test_split(
    np.arange(len(y_base_all)),
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_ct_effnet_test.shape)

set_all_seeds(HOLDOUT_SEED)
clear_training_memory()

# ------------------------------------------------------------
# Build matched EfficientNetB0
# ------------------------------------------------------------

def build_matched_ct_efficientnetb0():
    inputs = tf.keras.Input(
        shape=(224, 224, 1),
        name="ct_input"
    )

    x = build_ct_augmentation()(inputs)

    x = tf.keras.layers.Concatenate(
        name="grayscale_to_rgb"
    )([x, x, x])

    # EfficientNetB0 in Keras includes rescaling internally
    x = x * 255.0

    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False
    )

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.BatchNormalization(
        name="head_batch_normalization"
    )(x)

    x = layers.Dense(
        256,
        activation="swish",
        kernel_regularizer=regularizers.l2(
            CT_L2
        ),
        name="head_dense_256"
    )(x)

    x = layers.Dropout(
        CT_DROPOUT,
        name="head_dropout"
    )(x)

    outputs = layers.Dense(
        2,
        activation="softmax",
        name="classification_output"
    )(x)

    model = tf.keras.Model(
        inputs,
        outputs,
        name="CT_EfficientNetB0_Matched"
    )

    return model, backbone


def compile_ct_effnet_head(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )


def prepare_ct_effnet_fine_tuning(
    model,
    backbone
):
    backbone.trainable = True

    # Fine-tune only upper EfficientNet layers
    for layer in backbone.layers[:-30]:
        layer.trainable = False

    for layer in backbone.layers:
        if isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        ):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-5
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )


model, backbone = build_matched_ct_efficientnetb0()
compile_ct_effnet_head(model)

head_checkpoint = os.path.join(
    effnet_dir,
    "EfficientNetB0_Seed42_Head_Best.weights.h5"
)

fine_checkpoint = os.path.join(
    effnet_dir,
    "EfficientNetB0_Seed42_FineTune_Best.weights.h5"
)

head_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        head_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 1: Training EfficientNetB0 head")

history_head = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_HEAD_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=head_callbacks,
    verbose=1
)

prepare_ct_effnet_fine_tuning(
    model,
    backbone
)

fine_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        fine_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 2: Fine-tuning upper EfficientNetB0 layers")

history_fine = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_FINE_TUNE_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=fine_callbacks,
    verbose=1
)

pd.DataFrame(
    history_head.history
).to_csv(
    os.path.join(
        effnet_dir,
        "EfficientNetB0_Seed42_Head_History.csv"
    ),
    index=False
)

pd.DataFrame(
    history_fine.history
).to_csv(
    os.path.join(
        effnet_dir,
        "EfficientNetB0_Seed42_FineTune_History.csv"
    ),
    index=False
)

effnet_meta_probs_seed42 = model.predict(
    X_meta,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

effnet_test_probs_seed42 = model.predict(
    X_ct_effnet_test,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

np.save(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed42_Meta_Probabilities.npy"
    ),
    effnet_meta_probs_seed42.astype(
        np.float32
    )
)

np.save(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed42_Test_Probabilities.npy"
    ),
    effnet_test_probs_seed42.astype(
        np.float32
    )
)

model.save_weights(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed42_Final.weights.h5"
    )
)

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        effnet_meta_probs_seed42,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(
        effnet_test_probs_seed42,
        axis=1
    )
)

print("\nMatched EfficientNetB0 seed 42 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", effnet_dir)

In [ ]:
# ============================================================
# CT HOLD-OUT SEED 42 — META-LEARNERS AND SOFT VOTING
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

HOLDOUT_SEED = 42

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet50_Matched"
)

effnet_dir = os.path.join(
    seed_dir,
    "EfficientNetB0_Matched"
)

meta_dir = os.path.join(
    seed_dir,
    "Meta_Learners"
)

os.makedirs(meta_dir, exist_ok=True)

# ------------------------------------------------------------
# Load saved base-model probabilities
# ------------------------------------------------------------

resnet_meta_probs = np.load(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed42_Meta_Probabilities.npy"
    )
)

resnet_test_probs = np.load(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed42_Test_Probabilities.npy"
    )
)

effnet_meta_probs = np.load(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed42_Meta_Probabilities.npy"
    )
)

effnet_test_probs = np.load(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed42_Test_Probabilities.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

y_meta = y_ct_dev[meta_indices]

# ------------------------------------------------------------
# Build four-dimensional meta-feature matrices
# ------------------------------------------------------------

X_meta_train = np.concatenate(
    [
        resnet_meta_probs,
        effnet_meta_probs
    ],
    axis=1
).astype(np.float32)

X_meta_test = np.concatenate(
    [
        resnet_test_probs,
        effnet_test_probs
    ],
    axis=1
).astype(np.float32)

print("Meta-training matrix:", X_meta_train.shape)
print("Meta-test matrix:", X_meta_test.shape)
print("Meta labels:", y_meta.shape)

assert X_meta_train.shape == (397, 4)
assert X_meta_test.shape == (497, 4)

# ------------------------------------------------------------
# Soft voting
# ------------------------------------------------------------

soft_voting_probs = (
    resnet_test_probs + effnet_test_probs
) / 2.0

# ------------------------------------------------------------
# Logistic Regression with original CT tuning procedure
# ------------------------------------------------------------

meta_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

logistic_parameter_grid = {
    "classifier__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "classifier__penalty": [
        "l1",
        "l2"
    ]
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_parameter_grid,
    scoring="f1_macro",
    cv=meta_cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)

logistic_search.fit(
    X_meta_train,
    y_meta
)

logistic_model = logistic_search.best_estimator_

logistic_probs = logistic_model.predict_proba(
    X_meta_test
)

# ------------------------------------------------------------
# LightGBM with original CT configuration
# ------------------------------------------------------------

lightgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=150,
    learning_rate=0.03,
    num_leaves=7,
    max_depth=3,
    min_child_samples=30,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lightgbm_model.fit(
    X_meta_train,
    y_meta
)

lightgbm_probs = lightgbm_model.predict_proba(
    X_meta_test
)

# ------------------------------------------------------------
# Evaluate all methods
# ------------------------------------------------------------

seed42_rows = []

for method_name, probabilities in [
    ("SoftVoting", soft_voting_probs),
    ("LogisticRegression", logistic_probs),
    ("LightGBM", lightgbm_probs)
]:
    metrics = evaluate_probabilities(
        y_ct_test,
        probabilities
    )

    seed42_rows.append(
        {
            "Dataset": "CT",
            "Protocol": "Repeated fixed hold-out",
            "Holdout_Seed": HOLDOUT_SEED,
            "Method": method_name,
            "Meta_Training_Samples": len(y_meta),
            **metrics
        }
    )

ct_holdout_seed42_results = pd.DataFrame(
    seed42_rows
)

display(
    ct_holdout_seed42_results.round(6)
)

print(
    "Best Logistic parameters:",
    logistic_search.best_params_
)

print(
    "Best Logistic CV macro-F1:",
    logistic_search.best_score_
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed42_SoftVoting_Test_Probabilities.npy"
    ),
    soft_voting_probs.astype(np.float32)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed42_Logistic_Test_Probabilities.npy"
    ),
    logistic_probs.astype(np.float32)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed42_LightGBM_Test_Probabilities.npy"
    ),
    lightgbm_probs.astype(np.float32)
)

joblib.dump(
    logistic_model,
    os.path.join(
        meta_dir,
        "CT_Seed42_Logistic_Model.joblib"
    )
)

joblib.dump(
    lightgbm_model,
    os.path.join(
        meta_dir,
        "CT_Seed42_LightGBM_Model.joblib"
    )
)

ct_holdout_seed42_results.to_csv(
    os.path.join(
        meta_dir,
        "CT_Seed42_Holdout_Results.csv"
    ),
    index=False
)

print("\nSeed-42 stacking evaluation completed.")
print("Saved under:", meta_dir)

In [ ]:
# ============================================================
# MATCHED CT RESNET50 — HOLD-OUT SEED 123
# ============================================================

HOLDOUT_SEED = 123

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet50_Matched"
)

os.makedirs(resnet_dir, exist_ok=True)

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

X_base_all = X_ct_resnet_development[base_indices]
y_base_all = y_ct_dev[base_indices]

X_meta = X_ct_resnet_development[meta_indices]
y_meta = y_ct_dev[meta_indices]

train_indices_local, val_indices_local = train_test_split(
    np.arange(len(y_base_all)),
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[train_indices_local]
y_train_internal = y_base_all[train_indices_local]

X_val_internal = X_base_all[val_indices_local]
y_val_internal = y_base_all[val_indices_local]

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)

set_all_seeds(HOLDOUT_SEED)
clear_training_memory()

model, backbone = build_matched_ct_resnet50()
compile_ct_resnet_head(model)

head_checkpoint = os.path.join(
    resnet_dir,
    "ResNet50_Seed123_Head_Best.weights.h5"
)

fine_checkpoint = os.path.join(
    resnet_dir,
    "ResNet50_Seed123_FineTune_Best.weights.h5"
)

head_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        head_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 1: Training ResNet50 head for seed 123")

history_head = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_HEAD_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=head_callbacks,
    verbose=1
)

prepare_ct_resnet_fine_tuning(
    model,
    backbone
)

fine_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        fine_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 2: Fine-tuning ResNet50 for seed 123")

history_fine = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_FINE_TUNE_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=fine_callbacks,
    verbose=1
)

pd.DataFrame(
    history_head.history
).to_csv(
    os.path.join(
        resnet_dir,
        "ResNet50_Seed123_Head_History.csv"
    ),
    index=False
)

pd.DataFrame(
    history_fine.history
).to_csv(
    os.path.join(
        resnet_dir,
        "ResNet50_Seed123_FineTune_History.csv"
    ),
    index=False
)

resnet_meta_probs_seed123 = model.predict(
    X_meta,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

resnet_test_probs_seed123 = model.predict(
    X_ct_resnet_test,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

np.save(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed123_Meta_Probabilities.npy"
    ),
    resnet_meta_probs_seed123.astype(np.float32)
)

np.save(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed123_Test_Probabilities.npy"
    ),
    resnet_test_probs_seed123.astype(np.float32)
)

model.save_weights(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed123_Final.weights.h5"
    )
)

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        resnet_meta_probs_seed123,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(
        resnet_test_probs_seed123,
        axis=1
    )
)

print("\nMatched ResNet50 seed 123 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", resnet_dir)

In [ ]:
# ============================================================
# MATCHED CT EFFICIENTNETB0 — HOLD-OUT SEED 123
# ============================================================

HOLDOUT_SEED = 123

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

effnet_dir = os.path.join(
    seed_dir,
    "EfficientNetB0_Matched"
)

os.makedirs(effnet_dir, exist_ok=True)

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

X_base_all = X_ct_effnet_development[base_indices]
y_base_all = y_ct_dev[base_indices]

X_meta = X_ct_effnet_development[meta_indices]
y_meta = y_ct_dev[meta_indices]

train_indices_local, val_indices_local = train_test_split(
    np.arange(len(y_base_all)),
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[train_indices_local]
y_train_internal = y_base_all[train_indices_local]

X_val_internal = X_base_all[val_indices_local]
y_val_internal = y_base_all[val_indices_local]

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)

set_all_seeds(HOLDOUT_SEED)
clear_training_memory()

model, backbone = build_matched_ct_efficientnetb0()
compile_ct_effnet_head(model)

head_checkpoint = os.path.join(
    effnet_dir,
    "EfficientNetB0_Seed123_Head_Best.weights.h5"
)

fine_checkpoint = os.path.join(
    effnet_dir,
    "EfficientNetB0_Seed123_FineTune_Best.weights.h5"
)

head_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        head_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 1: Training EfficientNetB0 head for seed 123")

history_head = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_HEAD_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=head_callbacks,
    verbose=1
)

prepare_ct_effnet_fine_tuning(
    model,
    backbone
)

fine_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        fine_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 2: Fine-tuning EfficientNetB0 for seed 123")

history_fine = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_FINE_TUNE_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=fine_callbacks,
    verbose=1
)

pd.DataFrame(
    history_head.history
).to_csv(
    os.path.join(
        effnet_dir,
        "EfficientNetB0_Seed123_Head_History.csv"
    ),
    index=False
)

pd.DataFrame(
    history_fine.history
).to_csv(
    os.path.join(
        effnet_dir,
        "EfficientNetB0_Seed123_FineTune_History.csv"
    ),
    index=False
)

effnet_meta_probs_seed123 = model.predict(
    X_meta,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

effnet_test_probs_seed123 = model.predict(
    X_ct_effnet_test,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

np.save(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed123_Meta_Probabilities.npy"
    ),
    effnet_meta_probs_seed123.astype(np.float32)
)

np.save(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed123_Test_Probabilities.npy"
    ),
    effnet_test_probs_seed123.astype(np.float32)
)

model.save_weights(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed123_Final.weights.h5"
    )
)

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        effnet_meta_probs_seed123,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(
        effnet_test_probs_seed123,
        axis=1
    )
)

print("\nMatched EfficientNetB0 seed 123 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", effnet_dir)

In [ ]:
# ============================================================
# CT HOLD-OUT SEED 123 — META-LEARNERS AND SOFT VOTING
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

HOLDOUT_SEED = 123

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet50_Matched"
)

effnet_dir = os.path.join(
    seed_dir,
    "EfficientNetB0_Matched"
)

meta_dir = os.path.join(
    seed_dir,
    "Meta_Learners"
)

os.makedirs(meta_dir, exist_ok=True)

# Load saved probabilities
resnet_meta_probs = np.load(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed123_Meta_Probabilities.npy"
    )
)

resnet_test_probs = np.load(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed123_Test_Probabilities.npy"
    )
)

effnet_meta_probs = np.load(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed123_Meta_Probabilities.npy"
    )
)

effnet_test_probs = np.load(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed123_Test_Probabilities.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

y_meta = y_ct_dev[meta_indices]

# Build meta-feature matrices
X_meta_train = np.concatenate(
    [
        resnet_meta_probs,
        effnet_meta_probs
    ],
    axis=1
).astype(np.float32)

X_meta_test = np.concatenate(
    [
        resnet_test_probs,
        effnet_test_probs
    ],
    axis=1
).astype(np.float32)

print("Meta-training matrix:", X_meta_train.shape)
print("Meta-test matrix:", X_meta_test.shape)
print("Meta labels:", y_meta.shape)

assert X_meta_train.shape == (397, 4)
assert X_meta_test.shape == (497, 4)

# Soft voting
soft_voting_probs = (
    resnet_test_probs + effnet_test_probs
) / 2.0

# Logistic Regression with original CT tuning procedure
meta_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

logistic_parameter_grid = {
    "classifier__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "classifier__penalty": [
        "l1",
        "l2"
    ]
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_parameter_grid,
    scoring="f1_macro",
    cv=meta_cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)

logistic_search.fit(
    X_meta_train,
    y_meta
)

logistic_model = logistic_search.best_estimator_

logistic_probs = logistic_model.predict_proba(
    X_meta_test
)

# LightGBM with original CT configuration
lightgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=150,
    learning_rate=0.03,
    num_leaves=7,
    max_depth=3,
    min_child_samples=30,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lightgbm_model.fit(
    X_meta_train,
    y_meta
)

lightgbm_probs = lightgbm_model.predict_proba(
    X_meta_test
)

# Evaluate all methods
seed123_rows = []

for method_name, probabilities in [
    ("SoftVoting", soft_voting_probs),
    ("LogisticRegression", logistic_probs),
    ("LightGBM", lightgbm_probs)
]:
    metrics = evaluate_probabilities(
        y_ct_test,
        probabilities
    )

    seed123_rows.append(
        {
            "Dataset": "CT",
            "Protocol": "Repeated fixed hold-out",
            "Holdout_Seed": HOLDOUT_SEED,
            "Method": method_name,
            "Meta_Training_Samples": len(y_meta),
            **metrics
        }
    )

ct_holdout_seed123_results = pd.DataFrame(
    seed123_rows
)

display(
    ct_holdout_seed123_results.round(6)
)

print(
    "Best Logistic parameters:",
    logistic_search.best_params_
)

print(
    "Best Logistic CV macro-F1:",
    logistic_search.best_score_
)

# Save outputs
np.save(
    os.path.join(
        meta_dir,
        "CT_Seed123_SoftVoting_Test_Probabilities.npy"
    ),
    soft_voting_probs.astype(np.float32)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed123_Logistic_Test_Probabilities.npy"
    ),
    logistic_probs.astype(np.float32)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed123_LightGBM_Test_Probabilities.npy"
    ),
    lightgbm_probs.astype(np.float32)
)

joblib.dump(
    logistic_model,
    os.path.join(
        meta_dir,
        "CT_Seed123_Logistic_Model.joblib"
    )
)

joblib.dump(
    lightgbm_model,
    os.path.join(
        meta_dir,
        "CT_Seed123_LightGBM_Model.joblib"
    )
)

ct_holdout_seed123_results.to_csv(
    os.path.join(
        meta_dir,
        "CT_Seed123_Holdout_Results.csv"
    ),
    index=False
)

print("\nSeed-123 stacking evaluation completed.")
print("Saved under:", meta_dir)

In [ ]:
# ============================================================
# MATCHED CT RESNET50 — HOLD-OUT SEED 2025
# ============================================================

HOLDOUT_SEED = 2025

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet50_Matched"
)

os.makedirs(resnet_dir, exist_ok=True)

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

X_base_all = X_ct_resnet_development[base_indices]
y_base_all = y_ct_dev[base_indices]

X_meta = X_ct_resnet_development[meta_indices]
y_meta = y_ct_dev[meta_indices]

# Internal validation is drawn only from the base-training partition
train_indices_local, val_indices_local = train_test_split(
    np.arange(len(y_base_all)),
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[train_indices_local]
y_train_internal = y_base_all[train_indices_local]

X_val_internal = X_base_all[val_indices_local]
y_val_internal = y_base_all[val_indices_local]

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_ct_resnet_test.shape)

set_all_seeds(HOLDOUT_SEED)
clear_training_memory()

model, backbone = build_matched_ct_resnet50()
compile_ct_resnet_head(model)

head_checkpoint = os.path.join(
    resnet_dir,
    "ResNet50_Seed2025_Head_Best.weights.h5"
)

fine_checkpoint = os.path.join(
    resnet_dir,
    "ResNet50_Seed2025_FineTune_Best.weights.h5"
)

head_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        head_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 1: Training ResNet50 head for seed 2025")

history_head = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_HEAD_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=head_callbacks,
    verbose=1
)

prepare_ct_resnet_fine_tuning(
    model,
    backbone
)

fine_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        fine_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print("\nSTAGE 2: Fine-tuning ResNet50 for seed 2025")

history_fine = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_FINE_TUNE_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=fine_callbacks,
    verbose=1
)

# Save training histories
pd.DataFrame(
    history_head.history
).to_csv(
    os.path.join(
        resnet_dir,
        "ResNet50_Seed2025_Head_History.csv"
    ),
    index=False
)

pd.DataFrame(
    history_fine.history
).to_csv(
    os.path.join(
        resnet_dir,
        "ResNet50_Seed2025_FineTune_History.csv"
    ),
    index=False
)

# Generate meta-hold-out and final-test probabilities
resnet_meta_probs_seed2025 = model.predict(
    X_meta,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

resnet_test_probs_seed2025 = model.predict(
    X_ct_resnet_test,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

# Validate probability outputs
assert resnet_meta_probs_seed2025.shape == (397, 2)
assert resnet_test_probs_seed2025.shape == (497, 2)

assert np.allclose(
    resnet_meta_probs_seed2025.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    resnet_test_probs_seed2025.sum(axis=1),
    1.0,
    atol=1e-5
)

# Save probabilities
np.save(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed2025_Meta_Probabilities.npy"
    ),
    resnet_meta_probs_seed2025.astype(np.float32)
)

np.save(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed2025_Test_Probabilities.npy"
    ),
    resnet_test_probs_seed2025.astype(np.float32)
)

model.save_weights(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed2025_Final.weights.h5"
    )
)

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        resnet_meta_probs_seed2025,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(
        resnet_test_probs_seed2025,
        axis=1
    )
)

print("\nMatched ResNet50 seed 2025 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", resnet_dir)

In [ ]:
# ============================================================
# MATCHED CT EFFICIENTNETB0 — HOLD-OUT SEED 2025
# ============================================================

HOLDOUT_SEED = 2025

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

effnet_dir = os.path.join(
    seed_dir,
    "EfficientNetB0_Matched"
)

os.makedirs(effnet_dir, exist_ok=True)

# ------------------------------------------------------------
# Load the saved 80:20 split
# ------------------------------------------------------------

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

# EfficientNetB0 uses the enhanced CT images
X_base_all = X_ct_effnet_development[base_indices]
y_base_all = y_ct_dev[base_indices]

X_meta = X_ct_effnet_development[meta_indices]
y_meta = y_ct_dev[meta_indices]

# ------------------------------------------------------------
# Internal validation from base-training data only
# ------------------------------------------------------------

train_indices_local, val_indices_local = train_test_split(
    np.arange(len(y_base_all)),
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[train_indices_local]
y_train_internal = y_base_all[train_indices_local]

X_val_internal = X_base_all[val_indices_local]
y_val_internal = y_base_all[val_indices_local]

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_ct_effnet_test.shape)

print(
    "Training class counts:",
    np.bincount(y_train_internal)
)

print(
    "Validation class counts:",
    np.bincount(y_val_internal)
)

print(
    "Meta class counts:",
    np.bincount(y_meta)
)

# ------------------------------------------------------------
# Reset memory and random states
# ------------------------------------------------------------

set_all_seeds(HOLDOUT_SEED)
clear_training_memory()

# ------------------------------------------------------------
# Build and compile the matched model
# ------------------------------------------------------------

model, backbone = build_matched_ct_efficientnetb0()
compile_ct_effnet_head(model)

head_checkpoint = os.path.join(
    effnet_dir,
    "EfficientNetB0_Seed2025_Head_Best.weights.h5"
)

fine_checkpoint = os.path.join(
    effnet_dir,
    "EfficientNetB0_Seed2025_FineTune_Best.weights.h5"
)

# ------------------------------------------------------------
# Stage 1: train classification head
# ------------------------------------------------------------

head_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=head_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print(
    "\nSTAGE 1: Training EfficientNetB0 head "
    "for seed 2025"
)

history_head = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_HEAD_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=head_callbacks,
    verbose=1
)

# ------------------------------------------------------------
# Stage 2: fine-tune upper EfficientNetB0 layers
# ------------------------------------------------------------

prepare_ct_effnet_fine_tuning(
    model,
    backbone
)

fine_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=fine_checkpoint,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

print(
    "\nSTAGE 2: Fine-tuning EfficientNetB0 "
    "for seed 2025"
)

history_fine = model.fit(
    X_train_internal,
    y_train_internal,
    validation_data=(
        X_val_internal,
        y_val_internal
    ),
    epochs=CT_FINE_TUNE_EPOCHS,
    batch_size=CT_BATCH_SIZE,
    shuffle=True,
    callbacks=fine_callbacks,
    verbose=1
)

# ------------------------------------------------------------
# Save training histories
# ------------------------------------------------------------

pd.DataFrame(
    history_head.history
).to_csv(
    os.path.join(
        effnet_dir,
        "EfficientNetB0_Seed2025_Head_History.csv"
    ),
    index=False
)

pd.DataFrame(
    history_fine.history
).to_csv(
    os.path.join(
        effnet_dir,
        "EfficientNetB0_Seed2025_FineTune_History.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Generate meta-hold-out and final-test probabilities
# ------------------------------------------------------------

effnet_meta_probs_seed2025 = model.predict(
    X_meta,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

effnet_test_probs_seed2025 = model.predict(
    X_ct_effnet_test,
    batch_size=CT_BATCH_SIZE,
    verbose=1
)

# ------------------------------------------------------------
# Validate probability arrays
# ------------------------------------------------------------

assert effnet_meta_probs_seed2025.shape == (397, 2)
assert effnet_test_probs_seed2025.shape == (497, 2)

assert np.allclose(
    effnet_meta_probs_seed2025.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    effnet_test_probs_seed2025.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.isfinite(
    effnet_meta_probs_seed2025
).all()

assert np.isfinite(
    effnet_test_probs_seed2025
).all()

# ------------------------------------------------------------
# Save probabilities and final weights
# ------------------------------------------------------------

np.save(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed2025_Meta_Probabilities.npy"
    ),
    effnet_meta_probs_seed2025.astype(np.float32)
)

np.save(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed2025_Test_Probabilities.npy"
    ),
    effnet_test_probs_seed2025.astype(np.float32)
)

model.save_weights(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed2025_Final.weights.h5"
    )
)

# ------------------------------------------------------------
# Report base-model performance
# ------------------------------------------------------------

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        effnet_meta_probs_seed2025,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_ct_test,
    np.argmax(
        effnet_test_probs_seed2025,
        axis=1
    )
)

print(
    "\nMatched EfficientNetB0 seed 2025 completed."
)

print(
    "Meta hold-out accuracy:",
    meta_accuracy
)

print(
    "Final test accuracy:",
    test_accuracy
)

print(
    "Saved under:",
    effnet_dir
)

In [ ]:
# ============================================================
# CT HOLD-OUT SEED 2025 — META-LEARNERS AND SOFT VOTING
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

HOLDOUT_SEED = 2025

seed_dir = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet50_Matched"
)

effnet_dir = os.path.join(
    seed_dir,
    "EfficientNetB0_Matched"
)

meta_dir = os.path.join(
    seed_dir,
    "Meta_Learners"
)

os.makedirs(meta_dir, exist_ok=True)

# ------------------------------------------------------------
# Load saved ResNet50 probability matrices
# ------------------------------------------------------------

resnet_meta_probs = np.load(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed2025_Meta_Probabilities.npy"
    )
)

resnet_test_probs = np.load(
    os.path.join(
        resnet_dir,
        "CT_ResNet50_Seed2025_Test_Probabilities.npy"
    )
)

# ------------------------------------------------------------
# Load saved EfficientNetB0 probability matrices
# ------------------------------------------------------------

effnet_meta_probs = np.load(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed2025_Meta_Probabilities.npy"
    )
)

effnet_test_probs = np.load(
    os.path.join(
        effnet_dir,
        "CT_EfficientNetB0_Seed2025_Test_Probabilities.npy"
    )
)

# ------------------------------------------------------------
# Load the corresponding meta-training labels
# ------------------------------------------------------------

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

y_meta = y_ct_dev[meta_indices]

# ------------------------------------------------------------
# Validate base-model probability matrices
# ------------------------------------------------------------

assert resnet_meta_probs.shape == (397, 2)
assert effnet_meta_probs.shape == (397, 2)

assert resnet_test_probs.shape == (497, 2)
assert effnet_test_probs.shape == (497, 2)

assert np.isfinite(resnet_meta_probs).all()
assert np.isfinite(effnet_meta_probs).all()

assert np.isfinite(resnet_test_probs).all()
assert np.isfinite(effnet_test_probs).all()

assert np.allclose(
    resnet_meta_probs.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    effnet_meta_probs.sum(axis=1),
    1.0,
    atol=1e-5
)

# ------------------------------------------------------------
# Build four-dimensional meta-feature matrices
# Feature order:
# ResNet class 0, ResNet class 1,
# EfficientNet class 0, EfficientNet class 1
# ------------------------------------------------------------

X_meta_train = np.concatenate(
    [
        resnet_meta_probs,
        effnet_meta_probs
    ],
    axis=1
).astype(np.float32)

X_meta_test = np.concatenate(
    [
        resnet_test_probs,
        effnet_test_probs
    ],
    axis=1
).astype(np.float32)

print("Meta-training matrix:", X_meta_train.shape)
print("Meta-test matrix:", X_meta_test.shape)
print("Meta labels:", y_meta.shape)

assert X_meta_train.shape == (397, 4)
assert X_meta_test.shape == (497, 4)
assert y_meta.shape == (397,)

# ------------------------------------------------------------
# 1. Soft voting
# ------------------------------------------------------------

soft_voting_probs = (
    resnet_test_probs + effnet_test_probs
) / 2.0

# ------------------------------------------------------------
# 2. Logistic Regression stacking
# ------------------------------------------------------------

meta_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_pipeline = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

logistic_parameter_grid = {
    "classifier__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "classifier__penalty": [
        "l1",
        "l2"
    ]
}

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_parameter_grid,
    scoring="f1_macro",
    cv=meta_cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)

logistic_search.fit(
    X_meta_train,
    y_meta
)

logistic_model = (
    logistic_search.best_estimator_
)

logistic_probs = logistic_model.predict_proba(
    X_meta_test
)

# ------------------------------------------------------------
# 3. LightGBM stacking
# ------------------------------------------------------------

lightgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=150,
    learning_rate=0.03,
    num_leaves=7,
    max_depth=3,
    min_child_samples=30,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lightgbm_model.fit(
    X_meta_train,
    y_meta
)

lightgbm_probs = lightgbm_model.predict_proba(
    X_meta_test
)

# ------------------------------------------------------------
# Validate ensemble probabilities
# ------------------------------------------------------------

for probability_name, probabilities in [
    ("Soft Voting", soft_voting_probs),
    ("Logistic Regression", logistic_probs),
    ("LightGBM", lightgbm_probs)
]:
    assert probabilities.shape == (497, 2)
    assert np.isfinite(probabilities).all()

    assert np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-5
    ), f"Invalid probabilities: {probability_name}"

# ------------------------------------------------------------
# Evaluate all three methods
# ------------------------------------------------------------

seed2025_rows = []

for method_name, probabilities in [
    (
        "SoftVoting",
        soft_voting_probs
    ),
    (
        "LogisticRegression",
        logistic_probs
    ),
    (
        "LightGBM",
        lightgbm_probs
    )
]:
    metrics = evaluate_probabilities(
        y_ct_test,
        probabilities
    )

    seed2025_rows.append(
        {
            "Dataset": "CT",
            "Protocol": "Repeated fixed hold-out",
            "Holdout_Seed": HOLDOUT_SEED,
            "Method": method_name,
            "Meta_Training_Samples": len(y_meta),
            **metrics
        }
    )

ct_holdout_seed2025_results = pd.DataFrame(
    seed2025_rows
)

display(
    ct_holdout_seed2025_results.round(6)
)

print(
    "\nBest Logistic parameters:",
    logistic_search.best_params_
)

print(
    "Best Logistic CV macro-F1:",
    logistic_search.best_score_
)

# ------------------------------------------------------------
# Save test probability matrices
# ------------------------------------------------------------

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed2025_SoftVoting_Test_Probabilities.npy"
    ),
    soft_voting_probs.astype(np.float32)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed2025_Logistic_Test_Probabilities.npy"
    ),
    logistic_probs.astype(np.float32)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed2025_LightGBM_Test_Probabilities.npy"
    ),
    lightgbm_probs.astype(np.float32)
)

# ------------------------------------------------------------
# Save predictions for later statistical testing
# ------------------------------------------------------------

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed2025_SoftVoting_Test_Predictions.npy"
    ),
    np.argmax(
        soft_voting_probs,
        axis=1
    ).astype(np.int64)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed2025_Logistic_Test_Predictions.npy"
    ),
    np.argmax(
        logistic_probs,
        axis=1
    ).astype(np.int64)
)

np.save(
    os.path.join(
        meta_dir,
        "CT_Seed2025_LightGBM_Test_Predictions.npy"
    ),
    np.argmax(
        lightgbm_probs,
        axis=1
    ).astype(np.int64)
)

# ------------------------------------------------------------
# Save fitted meta-learners
# ------------------------------------------------------------

joblib.dump(
    logistic_model,
    os.path.join(
        meta_dir,
        "CT_Seed2025_Logistic_Model.joblib"
    )
)

joblib.dump(
    lightgbm_model,
    os.path.join(
        meta_dir,
        "CT_Seed2025_LightGBM_Model.joblib"
    )
)

# ------------------------------------------------------------
# Save Logistic Regression search information
# ------------------------------------------------------------

logistic_search_summary = {
    "Holdout_Seed": HOLDOUT_SEED,
    "Best_C": logistic_search.best_params_[
        "classifier__C"
    ],
    "Best_Penalty": logistic_search.best_params_[
        "classifier__penalty"
    ],
    "Best_Meta_CV_Macro_F1": float(
        logistic_search.best_score_
    )
}

pd.DataFrame(
    [logistic_search_summary]
).to_csv(
    os.path.join(
        meta_dir,
        "CT_Seed2025_Logistic_Search_Summary.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save final results
# ------------------------------------------------------------

results_path = os.path.join(
    meta_dir,
    "CT_Seed2025_Holdout_Results.csv"
)

ct_holdout_seed2025_results.to_csv(
    results_path,
    index=False
)

print(
    "\nSeed-2025 stacking evaluation completed."
)

print(
    "Results saved to:",
    results_path
)

In [ ]:
# ============================================================
# CT REPEATED HOLD-OUT — COMBINE SEEDS 42, 123, AND 2025
# ============================================================

import os
import numpy as np
import pandas as pd

HOLDOUT_SEEDS = [42, 123, 2025]

all_seed_results = []

for seed in HOLDOUT_SEEDS:

    result_path = os.path.join(
        CT_REPEATED_HOLDOUT_DIR,
        f"Seed_{seed}",
        "Meta_Learners",
        f"CT_Seed{seed}_Holdout_Results.csv"
    )

    if not os.path.exists(result_path):
        raise FileNotFoundError(
            f"Missing result file:\n{result_path}"
        )

    seed_df = pd.read_csv(result_path)

    required_methods = {
        "SoftVoting",
        "LogisticRegression",
        "LightGBM"
    }

    found_methods = set(
        seed_df["Method"].astype(str)
    )

    if found_methods != required_methods:
        raise ValueError(
            f"Unexpected methods for seed {seed}.\n"
            f"Expected: {required_methods}\n"
            f"Found: {found_methods}"
        )

    all_seed_results.append(seed_df)

ct_repeated_holdout_all = pd.concat(
    all_seed_results,
    ignore_index=True
)

ct_repeated_holdout_all = (
    ct_repeated_holdout_all
    .sort_values(
        ["Holdout_Seed", "Method"]
    )
    .reset_index(drop=True)
)

print("Combined rows:", len(ct_repeated_holdout_all))
display(ct_repeated_holdout_all.round(6))

# ------------------------------------------------------------
# Save complete seed-level results
# ------------------------------------------------------------

all_results_path = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    "CT_Repeated_Holdout_All_Seed_Results.csv"
)

ct_repeated_holdout_all.to_csv(
    all_results_path,
    index=False
)

# ------------------------------------------------------------
# Calculate mean, SD, minimum, maximum, and range
# ------------------------------------------------------------

metric_columns = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Log_Loss",
    "Brier_Score",
    "ECE",
    "ROC_AUC"
]

summary_rows = []

for method_name, method_df in (
    ct_repeated_holdout_all.groupby("Method")
):

    row = {
        "Dataset": "CT",
        "Protocol": "Repeated fixed hold-out",
        "Method": method_name,
        "Number_of_Seeds": method_df[
            "Holdout_Seed"
        ].nunique()
    }

    for metric in metric_columns:

        values = method_df[
            metric
        ].astype(float).to_numpy()

        row[f"{metric}_Mean"] = float(
            np.mean(values)
        )

        row[f"{metric}_SD"] = float(
            np.std(values, ddof=1)
        )

        row[f"{metric}_Min"] = float(
            np.min(values)
        )

        row[f"{metric}_Max"] = float(
            np.max(values)
        )

        row[f"{metric}_Range"] = float(
            np.max(values) - np.min(values)
        )

    summary_rows.append(row)

ct_repeated_holdout_summary = pd.DataFrame(
    summary_rows
)

ct_repeated_holdout_summary = (
    ct_repeated_holdout_summary
    .sort_values(
        "Accuracy_Mean",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Add convenient percentage-formatted columns
# ------------------------------------------------------------

ct_repeated_holdout_summary[
    "Accuracy_Mean_Percent"
] = (
    ct_repeated_holdout_summary[
        "Accuracy_Mean"
    ] * 100
)

ct_repeated_holdout_summary[
    "Accuracy_SD_Percent"
] = (
    ct_repeated_holdout_summary[
        "Accuracy_SD"
    ] * 100
)

ct_repeated_holdout_summary[
    "Macro_F1_Mean_Percent"
] = (
    ct_repeated_holdout_summary[
        "Macro_F1_Mean"
    ] * 100
)

ct_repeated_holdout_summary[
    "Macro_F1_SD_Percent"
] = (
    ct_repeated_holdout_summary[
        "Macro_F1_SD"
    ] * 100
)

ct_repeated_holdout_summary[
    "Accuracy_Mean_SD"
] = (
    ct_repeated_holdout_summary[
        "Accuracy_Mean_Percent"
    ].map(lambda x: f"{x:.3f}")
    + " ± "
    + ct_repeated_holdout_summary[
        "Accuracy_SD_Percent"
    ].map(lambda x: f"{x:.3f}")
)

ct_repeated_holdout_summary[
    "Macro_F1_Mean_SD"
] = (
    ct_repeated_holdout_summary[
        "Macro_F1_Mean_Percent"
    ].map(lambda x: f"{x:.3f}")
    + " ± "
    + ct_repeated_holdout_summary[
        "Macro_F1_SD_Percent"
    ].map(lambda x: f"{x:.3f}")
)

# ------------------------------------------------------------
# Add mean-accuracy ranking
# ------------------------------------------------------------

ct_repeated_holdout_summary[
    "Accuracy_Rank"
] = (
    ct_repeated_holdout_summary[
        "Accuracy_Mean"
    ]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)

summary_path = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    "CT_Repeated_Holdout_Mean_SD_Summary.csv"
)

ct_repeated_holdout_summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# Compact manuscript-ready table
# ------------------------------------------------------------

compact_columns = [
    "Accuracy_Rank",
    "Method",
    "Number_of_Seeds",
    "Accuracy_Mean_SD",
    "Macro_F1_Mean_SD",
    "ROC_AUC_Mean",
    "ROC_AUC_SD",
    "Log_Loss_Mean",
    "Log_Loss_SD",
    "Brier_Score_Mean",
    "Brier_Score_SD",
    "ECE_Mean",
    "ECE_SD",
    "Accuracy_Min",
    "Accuracy_Max",
    "Accuracy_Range"
]

ct_repeated_holdout_compact = (
    ct_repeated_holdout_summary[
        compact_columns
    ].copy()
)

compact_path = os.path.join(
    CT_REPEATED_HOLDOUT_DIR,
    "CT_Repeated_Holdout_Compact_Table.csv"
)

ct_repeated_holdout_compact.to_csv(
    compact_path,
    index=False
)

print("\nCT repeated hold-out summary:")
display(
    ct_repeated_holdout_compact.round(6)
)

print("\nSaved complete results to:")
print(all_results_path)

print("\nSaved detailed summary to:")
print(summary_path)

print("\nSaved compact manuscript table to:")
print(compact_path)

In [ ]:
# ============================================================
# CT: REPEATED HOLD-OUT VERSUS FIVE-FOLD OOF COMPARISON
# Accuracy figure with hold-out mean ± SD
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Output locations
# ------------------------------------------------------------

comparison_dir = os.path.join(
    EXT_DIR,
    "Figures"
)

table_dir = os.path.join(
    EXT_DIR,
    "Tables"
)

os.makedirs(comparison_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)

# ------------------------------------------------------------
# Repeated hold-out summary already calculated
# ------------------------------------------------------------

holdout_summary = (
    ct_repeated_holdout_summary[
        [
            "Method",
            "Accuracy_Mean",
            "Accuracy_SD",
            "Macro_F1_Mean",
            "Macro_F1_SD",
            "ROC_AUC_Mean",
            "ROC_AUC_SD",
            "Log_Loss_Mean",
            "Log_Loss_SD",
            "Brier_Score_Mean",
            "Brier_Score_SD",
            "ECE_Mean",
            "ECE_SD"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# Full-coverage five-fold OOF results
#
# These values correspond to the completed CT OOF experiment:
# Soft Voting: 95.9759%
# Logistic Regression: 96.3783%
# LightGBM: 96.3783%
# ------------------------------------------------------------

ct_oof_accuracy = {
    "SoftVoting": 0.9597585513078471,
    "LogisticRegression": 0.9637826961770624,
    "LightGBM": 0.9637826961770624
}

# Full OOF metrics previously obtained for the two meta-learners
ct_oof_details = {
    "LogisticRegression": {
        "ROC_AUC": 0.993587,
        "Log_Loss": 0.115673,
        "Brier_Score": 0.060919,
        "ECE": 0.024239
    },
    "LightGBM": {
        "ROC_AUC": 0.994583,
        "Log_Loss": 0.101805,
        "Brier_Score": 0.056421,
        "ECE": 0.010289
    }
}

# ------------------------------------------------------------
# Create comparison table
# ------------------------------------------------------------

comparison_rows = []

method_order = [
    "SoftVoting",
    "LogisticRegression",
    "LightGBM"
]

for method in method_order:

    holdout_row = holdout_summary[
        holdout_summary["Method"] == method
    ].iloc[0]

    row = {
        "Dataset": "CT",
        "Method": method,

        "Repeated_Holdout_Accuracy_Mean":
            holdout_row["Accuracy_Mean"],

        "Repeated_Holdout_Accuracy_SD":
            holdout_row["Accuracy_SD"],

        "Repeated_Holdout_Accuracy_Min":
            ct_repeated_holdout_all.loc[
                ct_repeated_holdout_all["Method"] == method,
                "Accuracy"
            ].min(),

        "Repeated_Holdout_Accuracy_Max":
            ct_repeated_holdout_all.loc[
                ct_repeated_holdout_all["Method"] == method,
                "Accuracy"
            ].max(),

        "OOF_Accuracy":
            ct_oof_accuracy[method],

        "OOF_Minus_Holdout_Mean":
            (
                ct_oof_accuracy[method]
                - holdout_row["Accuracy_Mean"]
            )
    }

    if method in ct_oof_details:

        row.update(
            {
                "Repeated_Holdout_ROC_AUC_Mean":
                    holdout_row["ROC_AUC_Mean"],

                "OOF_ROC_AUC":
                    ct_oof_details[method]["ROC_AUC"],

                "Repeated_Holdout_Log_Loss_Mean":
                    holdout_row["Log_Loss_Mean"],

                "OOF_Log_Loss":
                    ct_oof_details[method]["Log_Loss"],

                "Repeated_Holdout_Brier_Mean":
                    holdout_row["Brier_Score_Mean"],

                "OOF_Brier":
                    ct_oof_details[method]["Brier_Score"],

                "Repeated_Holdout_ECE_Mean":
                    holdout_row["ECE_Mean"],

                "OOF_ECE":
                    ct_oof_details[method]["ECE"]
            }
        )

    comparison_rows.append(row)

ct_protocol_comparison = pd.DataFrame(
    comparison_rows
)

# Percentage-point improvement
ct_protocol_comparison[
    "OOF_Minus_Holdout_Mean_pp"
] = (
    ct_protocol_comparison[
        "OOF_Minus_Holdout_Mean"
    ] * 100
)

comparison_table_path = os.path.join(
    table_dir,
    "CT_Repeated_Holdout_vs_OOF_Comparison.csv"
)

ct_protocol_comparison.to_csv(
    comparison_table_path,
    index=False
)

print("CT protocol comparison:")
display(
    ct_protocol_comparison.round(6)
)

# ------------------------------------------------------------
# Accuracy comparison figure
# ------------------------------------------------------------

plot_df = (
    ct_protocol_comparison
    .set_index("Method")
    .loc[method_order]
    .reset_index()
)

x_positions = np.arange(len(method_order))
bar_width = 0.34

holdout_accuracy_percent = (
    plot_df[
        "Repeated_Holdout_Accuracy_Mean"
    ].to_numpy() * 100
)

holdout_sd_percent = (
    plot_df[
        "Repeated_Holdout_Accuracy_SD"
    ].to_numpy() * 100
)

oof_accuracy_percent = (
    plot_df[
        "OOF_Accuracy"
    ].to_numpy() * 100
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

holdout_bars = ax.bar(
    x_positions - bar_width / 2,
    holdout_accuracy_percent,
    bar_width,
    yerr=holdout_sd_percent,
    capsize=5,
    label="Repeated fixed hold-out: mean ± SD"
)

oof_bars = ax.bar(
    x_positions + bar_width / 2,
    oof_accuracy_percent,
    bar_width,
    label="Five-fold OOF"
)

ax.set_ylabel("Test accuracy (%)")
ax.set_xlabel("Ensemble method")

ax.set_title(
    "CT Protocol Comparison: "
    "Repeated Fixed Hold-Out versus Five-Fold OOF"
)

ax.set_xticks(x_positions)

ax.set_xticklabels(
    [
        "Soft Voting",
        "Logistic Regression",
        "LightGBM"
    ]
)

minimum_value = min(
    np.min(
        holdout_accuracy_percent
        - holdout_sd_percent
    ),
    np.min(oof_accuracy_percent)
)

ax.set_ylim(
    max(90.0, minimum_value - 1.0),
    97.5
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

ax.legend()

# Add values above hold-out bars
for bar, mean_value, sd_value in zip(
    holdout_bars,
    holdout_accuracy_percent,
    holdout_sd_percent
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + sd_value + 0.12,
        f"{mean_value:.2f} ± {sd_value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# Add values above OOF bars
for bar, value in zip(
    oof_bars,
    oof_accuracy_percent
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.12,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

figure_path = os.path.join(
    comparison_dir,
    "CT_Repeated_Holdout_vs_OOF_Accuracy.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nComparison table saved to:")
print(comparison_table_path)

print("\nFigure saved to:")
print(figure_path)

In [ ]:
# ============================================================
# LOAD CIFAR-10 FROM EXISTING DRIVE NPY FILES
# ============================================================

import os
import numpy as np

CIFAR_CACHE_DIR = "/content/drive/MyDrive/CIFAR10_Cache"

x_train_path = os.path.join(
    CIFAR_CACHE_DIR,
    "X_cf_train.npy"
)

y_train_path = os.path.join(
    CIFAR_CACHE_DIR,
    "y_cf_train.npy"
)

x_test_path = os.path.join(
    CIFAR_CACHE_DIR,
    "X_cf_test.npy"
)

y_test_path = os.path.join(
    CIFAR_CACHE_DIR,
    "y_cf_test.npy"
)

for path in [
    x_train_path,
    y_train_path,
    x_test_path,
    y_test_path
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )

X_cifar_development = np.load(
    x_train_path,
    allow_pickle=False
)

y_cifar_development = np.load(
    y_train_path,
    allow_pickle=False
)

X_cifar_test = np.load(
    x_test_path,
    allow_pickle=False
)

y_cifar_test = np.load(
    y_test_path,
    allow_pickle=False
)

# Flatten labels
y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

# Convert images to float32
X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

# Normalize only when stored in 0–255 range
if X_cifar_development.max() > 1.0:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.0:
    X_cifar_test /= 255.0

X_cifar_development = np.clip(
    X_cifar_development,
    0.0,
    1.0
)

X_cifar_test = np.clip(
    X_cifar_test,
    0.0,
    1.0
)

print("CIFAR-10 loaded from Drive.")
print(
    "Development images:",
    X_cifar_development.shape
)
print(
    "Development labels:",
    y_cifar_development.shape
)
print(
    "Test images:",
    X_cifar_test.shape
)
print(
    "Test labels:",
    y_cifar_test.shape
)

print(
    "Development range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

print(
    "Test range:",
    float(X_cifar_test.min()),
    "to",
    float(X_cifar_test.max())
)

print(
    "Development class counts:",
    np.bincount(
        y_cifar_development,
        minlength=10
    )
)

print(
    "Test class counts:",
    np.bincount(
        y_cifar_test,
        minlength=10
    )
)

assert X_cifar_development.shape == (
    50000,
    32,
    32,
    3
)

assert y_cifar_development.shape == (
    50000,
)

assert X_cifar_test.shape == (
    10000,
    32,
    32,
    3
)

assert y_cifar_test.shape == (
    10000,
)

print(
    "\nDrive-based CIFAR-10 loading completed successfully."
)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED 80:20 HOLD-OUT SPLITS
# Uses already loaded Drive arrays
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

HOLDOUT_SEEDS = [42, 123, 2025]

CIFAR_REPEATED_HOLDOUT_DIR = os.path.join(
    EXT_DIR,
    "Repeated_Holdout",
    "CIFAR10"
)

os.makedirs(
    CIFAR_REPEATED_HOLDOUT_DIR,
    exist_ok=True
)

all_indices = np.arange(
    len(y_cifar_development)
)

split_rows = []

for holdout_seed in HOLDOUT_SEEDS:

    base_indices, meta_indices = train_test_split(
        all_indices,
        test_size=0.20,
        stratify=y_cifar_development,
        random_state=holdout_seed
    )

    seed_dir = os.path.join(
        CIFAR_REPEATED_HOLDOUT_DIR,
        f"Seed_{holdout_seed}"
    )

    os.makedirs(
        seed_dir,
        exist_ok=True
    )

    # Save split indices
    np.save(
        os.path.join(
            seed_dir,
            "Base_Training_Indices.npy"
        ),
        base_indices.astype(np.int64)
    )

    np.save(
        os.path.join(
            seed_dir,
            "Meta_Holdout_Indices.npy"
        ),
        meta_indices.astype(np.int64)
    )

    # Validate sizes and leakage
    assert len(base_indices) == 40000
    assert len(meta_indices) == 10000

    overlap = np.intersect1d(
        base_indices,
        meta_indices
    )

    assert len(overlap) == 0

    assert len(
        np.union1d(
            base_indices,
            meta_indices
        )
    ) == 50000

    base_class_counts = np.bincount(
        y_cifar_development[base_indices],
        minlength=10
    )

    meta_class_counts = np.bincount(
        y_cifar_development[meta_indices],
        minlength=10
    )

    row = {
        "Dataset": "CIFAR-10",
        "Seed": holdout_seed,
        "Base_Training_Samples":
            len(base_indices),
        "Meta_Training_Samples":
            len(meta_indices)
    }

    for class_id in range(10):

        row[
            f"Base_Class_{class_id}"
        ] = int(
            base_class_counts[class_id]
        )

        row[
            f"Meta_Class_{class_id}"
        ] = int(
            meta_class_counts[class_id]
        )

    split_rows.append(row)

cifar_repeated_holdout_splits = pd.DataFrame(
    split_rows
)

split_summary_path = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Repeated_Holdout_Split_Summary.csv"
)

cifar_repeated_holdout_splits.to_csv(
    split_summary_path,
    index=False
)

display(
    cifar_repeated_holdout_splits
)

print(
    "\nCIFAR-10 repeated hold-out splits completed."
)

print(
    "Saved to:",
    split_summary_path
)

In [ ]:
# ============================================================
# LOCATE EXISTING CIFAR-10 FIXED HOLD-OUT FILES IN DRIVE
# ============================================================

import os

SEARCH_ROOTS = [
    "/content/drive/MyDrive/Paper_1_Matched_Protocol_Comparison",
    "/content/drive/MyDrive/OOF_Stacking_Results",
    "/content/drive/MyDrive/Leakage_Free_OOF_Hybrid_Stacking",
    "/content/drive/MyDrive/CIFAR10_Cache"
]

keywords = [
    "holdout",
    "hold_out",
    "meta_train",
    "meta_training",
    "cnn",
    "vgg",
    "resnet",
    "efficient",
    "probabil",
    "40_features"
]

candidate_files = []

for search_root in SEARCH_ROOTS:

    if not os.path.exists(search_root):
        continue

    for root, directories, files in os.walk(search_root):

        for filename in files:

            filename_lower = filename.lower()

            if (
                filename_lower.endswith(
                    (".npy", ".npz", ".csv", ".pkl")
                )
                and any(
                    keyword in filename_lower
                    for keyword in keywords
                )
            ):
                full_path = os.path.join(
                    root,
                    filename
                )

                candidate_files.append(full_path)

print(
    "Number of candidate files:",
    len(candidate_files)
)

for path in sorted(candidate_files):
    print(path)

In [ ]:
# ============================================================
# VALIDATE AND REUSE EXISTING CIFAR-10 SEED-42 HOLD-OUT FILES
# No deep-model retraining
# ============================================================

import os
import numpy as np
import pandas as pd

CIFAR_MATCHED_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison/CIFAR10"
)

CIFAR_HOLDOUT_PROB_DIR = os.path.join(
    CIFAR_MATCHED_ROOT,
    "Holdout_Probabilities"
)

CIFAR_TEST_PROB_DIR = (
    "/content/drive/MyDrive/"
    "OOF_Stacking_Results/OOF_Probabilities"
)

seed42_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "Seed_42"
)

os.makedirs(seed42_dir, exist_ok=True)

meta_paths = {
    "CNN": os.path.join(
        CIFAR_HOLDOUT_PROB_DIR,
        "CIFAR_CNN_Holdout_Meta_Probabilities.npy"
    ),
    "VGG16": os.path.join(
        CIFAR_HOLDOUT_PROB_DIR,
        "CIFAR_VGG16_Holdout_Meta_Probabilities.npy"
    ),
    "ResNet50": os.path.join(
        CIFAR_HOLDOUT_PROB_DIR,
        "CIFAR_ResNet50_Holdout_Meta_Probabilities.npy"
    ),
    "EfficientNetB0": os.path.join(
        CIFAR_HOLDOUT_PROB_DIR,
        "CIFAR_EfficientNetB0_Holdout_Meta_Probabilities.npy"
    )
}

test_paths = {
    "CNN": os.path.join(
        CIFAR_TEST_PROB_DIR,
        "CIFAR_CNN_Test_Probabilities.npy"
    ),
    "VGG16": os.path.join(
        CIFAR_TEST_PROB_DIR,
        "CIFAR_VGG16_Test_Probabilities.npy"
    ),
    "ResNet50": os.path.join(
        CIFAR_TEST_PROB_DIR,
        "CIFAR_ResNet50_Test_Probabilities.npy"
    ),
    "EfficientNetB0": os.path.join(
        CIFAR_TEST_PROB_DIR,
        "CIFAR_EfficientNetB0_Test_Probabilities.npy"
    )
}

combined_meta_path = os.path.join(
    CIFAR_HOLDOUT_PROB_DIR,
    "CIFAR10_Holdout_Meta_40_Features.npy"
)

meta_labels_path = os.path.join(
    CIFAR_HOLDOUT_PROB_DIR,
    "CIFAR10_Holdout_Meta_Labels.npy"
)

all_required_paths = (
    list(meta_paths.values())
    + list(test_paths.values())
    + [combined_meta_path, meta_labels_path]
)

for path in all_required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )

# ------------------------------------------------------------
# Load individual model probabilities
# ------------------------------------------------------------

seed42_meta_probs = {
    model_name: np.load(path)
    for model_name, path in meta_paths.items()
}

seed42_test_probs = {
    model_name: np.load(path)
    for model_name, path in test_paths.items()
}

seed42_meta_features_saved = np.load(
    combined_meta_path
)

seed42_meta_labels = np.load(
    meta_labels_path
).reshape(-1).astype(np.int64)

# ------------------------------------------------------------
# Validate shapes, values and class ordering
# ------------------------------------------------------------

model_order = [
    "CNN",
    "VGG16",
    "ResNet50",
    "EfficientNetB0"
]

for model_name in model_order:

    meta_probs = seed42_meta_probs[model_name]
    test_probs = seed42_test_probs[model_name]

    print(
        model_name,
        "meta:",
        meta_probs.shape,
        "test:",
        test_probs.shape
    )

    assert meta_probs.shape == (10000, 10)
    assert test_probs.shape == (10000, 10)

    assert np.isfinite(meta_probs).all()
    assert np.isfinite(test_probs).all()

    assert np.allclose(
        meta_probs.sum(axis=1),
        1.0,
        atol=1e-5
    )

    assert np.allclose(
        test_probs.sum(axis=1),
        1.0,
        atol=1e-5
    )

reconstructed_meta_features = np.concatenate(
    [
        seed42_meta_probs[model_name]
        for model_name in model_order
    ],
    axis=1
).astype(np.float32)

reconstructed_test_features = np.concatenate(
    [
        seed42_test_probs[model_name]
        for model_name in model_order
    ],
    axis=1
).astype(np.float32)

print(
    "\nReconstructed meta features:",
    reconstructed_meta_features.shape
)

print(
    "Saved combined meta features:",
    seed42_meta_features_saved.shape
)

print(
    "Reconstructed test features:",
    reconstructed_test_features.shape
)

print(
    "Meta labels:",
    seed42_meta_labels.shape
)

assert reconstructed_meta_features.shape == (
    10000,
    40
)

assert reconstructed_test_features.shape == (
    10000,
    40
)

assert seed42_meta_features_saved.shape == (
    10000,
    40
)

assert seed42_meta_labels.shape == (
    10000,
)

# Check whether saved combined matrix matches current individual files
saved_matrix_matches = np.allclose(
    reconstructed_meta_features,
    seed42_meta_features_saved,
    atol=1e-7
)

print(
    "\nDoes saved combined matrix match "
    "the individual arrays?",
    saved_matrix_matches
)

if not saved_matrix_matches:
    print(
        "Warning: use the reconstructed matrix from "
        "the individual files, not the saved combined file."
    )

# ------------------------------------------------------------
# Check whether the old seed-42 meta indices match our new split
# ------------------------------------------------------------

old_meta_indices_path = os.path.join(
    CIFAR_MATCHED_ROOT,
    "Split_Indices",
    "CIFAR10_Meta_Holdout_Indices.npy"
)

new_meta_indices_path = os.path.join(
    seed42_dir,
    "Meta_Holdout_Indices.npy"
)

old_meta_indices = np.load(
    old_meta_indices_path
)

new_meta_indices = np.load(
    new_meta_indices_path
)

indices_match = np.array_equal(
    old_meta_indices,
    new_meta_indices
)

print(
    "Do old and newly generated seed-42 "
    "meta indices match?",
    indices_match
)

if not indices_match:
    raise ValueError(
        "The old seed-42 split does not match the "
        "newly generated split. Do not combine them."
    )

labels_match = np.array_equal(
    seed42_meta_labels,
    y_cifar_development[new_meta_indices]
)

print(
    "Do saved meta labels match the split indices?",
    labels_match
)

if not labels_match:
    raise ValueError(
        "Saved meta labels do not align with the "
        "seed-42 meta indices."
    )

# ------------------------------------------------------------
# Save aliases in the repeated-holdout folder
# ------------------------------------------------------------

np.save(
    os.path.join(
        seed42_dir,
        "CIFAR10_Seed42_Meta_40_Features.npy"
    ),
    reconstructed_meta_features
)

np.save(
    os.path.join(
        seed42_dir,
        "CIFAR10_Seed42_Test_40_Features.npy"
    ),
    reconstructed_test_features
)

np.save(
    os.path.join(
        seed42_dir,
        "CIFAR10_Seed42_Meta_Labels.npy"
    ),
    seed42_meta_labels
)

print("\nExisting CIFAR seed-42 hold-out files")

In [ ]:
# ============================================================
# CHECK EXISTING CIFAR REPEATED-HOLDOUT RESULTS
# Seeds 42, 123, 2025
# ============================================================

import os
import numpy as np
import pandas as pd

HOLDOUT_SEEDS = [42, 123, 2025]

model_names = [
    "CNN",
    "VGG16",
    "ResNet50",
    "EfficientNetB0"
]

check_rows = []

for seed in HOLDOUT_SEEDS:

    seed_dir = os.path.join(
        CIFAR_REPEATED_HOLDOUT_DIR,
        f"Seed_{seed}"
    )

    print("\n" + "=" * 70)
    print("SEED:", seed)
    print("Folder:", seed_dir)
    print("=" * 70)

    if not os.path.exists(seed_dir):
        print("Seed folder does not exist.")

        check_rows.append(
            {
                "Seed": seed,
                "Seed_Folder_Exists": False,
                "Meta_Indices_Exist": False,
                "Base_Indices_Exist": False,
                "Meta_Probabilities_Complete": False,
                "Test_Probabilities_Complete": False,
                "Results_CSV_Exists": False
            }
        )

        continue

    base_index_path = os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )

    meta_index_path = os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )

    base_indices_exist = os.path.exists(
        base_index_path
    )

    meta_indices_exist = os.path.exists(
        meta_index_path
    )

    print("Base indices:", base_indices_exist)
    print("Meta indices:", meta_indices_exist)

    if base_indices_exist:
        base_indices = np.load(
            base_index_path
        )

        print(
            "Base index shape:",
            base_indices.shape
        )

    if meta_indices_exist:
        meta_indices = np.load(
            meta_index_path
        )

        print(
            "Meta index shape:",
            meta_indices.shape
        )

    # --------------------------------------------------------
    # Search recursively for saved probability files
    # --------------------------------------------------------

    all_seed_files = []

    for root, directories, files in os.walk(
        seed_dir
    ):
        for filename in files:
            all_seed_files.append(
                os.path.join(
                    root,
                    filename
                )
            )

    probability_files = [
        path
        for path in all_seed_files
        if (
            path.lower().endswith(".npy")
            and "probabil" in os.path.basename(
                path
            ).lower()
        )
    ]

    results_csv_files = [
        path
        for path in all_seed_files
        if (
            path.lower().endswith(".csv")
            and "result" in os.path.basename(
                path
            ).lower()
        )
    ]

    print(
        "Probability files found:",
        len(probability_files)
    )

    for path in probability_files:
        print("  ", path)

    print(
        "Result CSV files found:",
        len(results_csv_files)
    )

    for path in results_csv_files:
        print("  ", path)

    # --------------------------------------------------------
    # Determine model-level coverage
    # --------------------------------------------------------

    meta_models_found = []
    test_models_found = []

    for model_name in model_names:

        normalized_model = (
            model_name
            .lower()
            .replace("b0", "")
        )

        for path in probability_files:

            filename_lower = os.path.basename(
                path
            ).lower()

            if normalized_model in filename_lower:

                if "meta" in filename_lower:
                    meta_models_found.append(
                        model_name
                    )

                if "test" in filename_lower:
                    test_models_found.append(
                        model_name
                    )

    meta_models_found = sorted(
        set(meta_models_found)
    )

    test_models_found = sorted(
        set(test_models_found)
    )

    meta_complete = (
        set(meta_models_found)
        == set(model_names)
    )

    test_complete = (
        set(test_models_found)
        == set(model_names)
    )

    print(
        "Models with meta probabilities:",
        meta_models_found
    )

    print(
        "Models with test probabilities:",
        test_models_found
    )

    check_rows.append(
        {
            "Seed": seed,
            "Seed_Folder_Exists": True,
            "Base_Indices_Exist":
                base_indices_exist,
            "Meta_Indices_Exist":
                meta_indices_exist,
            "Number_of_Probability_Files":
                len(probability_files),
            "Meta_Models_Found":
                ", ".join(meta_models_found),
            "Test_Models_Found":
                ", ".join(test_models_found),
            "Meta_Probabilities_Complete":
                meta_complete,
            "Test_Probabilities_Complete":
                test_complete,
            "Results_CSV_Exists":
                len(results_csv_files) > 0
        }
    )

cifar_existing_results_check = pd.DataFrame(
    check_rows
)

display(
    cifar_existing_results_check
)

check_output_path = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Existing_Repeated_Holdout_Check.csv"
)

cifar_existing_results_check.to_csv(
    check_output_path,
    index=False
)

print("\nSaved inspection summary to:")
print(check_output_path)

In [ ]:
# ============================================================
# REGISTER EXISTING CIFAR-10 SEED-42 HOLD-OUT RESULTS
# No deep-model retraining
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd

SEED = 42

seed42_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "Seed_42"
)

seed42_probability_dir = os.path.join(
    seed42_dir,
    "Base_Model_Probabilities"
)

seed42_meta_dir = os.path.join(
    seed42_dir,
    "Meta_Learners"
)

os.makedirs(seed42_probability_dir, exist_ok=True)
os.makedirs(seed42_meta_dir, exist_ok=True)

original_holdout_dir = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison/"
    "CIFAR10/Holdout_Probabilities"
)

original_test_dir = (
    "/content/drive/MyDrive/"
    "OOF_Stacking_Results/OOF_Probabilities"
)

original_results_path = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison/"
    "CIFAR10/Results/"
    "CIFAR10_Matched_Holdout_vs_OOF_Results.csv"
)

model_file_map = {
    "CNN": {
        "meta": os.path.join(
            original_holdout_dir,
            "CIFAR_CNN_Holdout_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            original_test_dir,
            "CIFAR_CNN_Test_Probabilities.npy"
        )
    },
    "VGG16": {
        "meta": os.path.join(
            original_holdout_dir,
            "CIFAR_VGG16_Holdout_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            original_test_dir,
            "CIFAR_VGG16_Test_Probabilities.npy"
        )
    },
    "ResNet50": {
        "meta": os.path.join(
            original_holdout_dir,
            "CIFAR_ResNet50_Holdout_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            original_test_dir,
            "CIFAR_ResNet50_Test_Probabilities.npy"
        )
    },
    "EfficientNetB0": {
        "meta": os.path.join(
            original_holdout_dir,
            "CIFAR_EfficientNetB0_Holdout_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            original_test_dir,
            "CIFAR_EfficientNetB0_Test_Probabilities.npy"
        )
    }
}

model_order = [
    "CNN",
    "VGG16",
    "ResNet50",
    "EfficientNetB0"
]

meta_arrays = []
test_arrays = []

for model_name in model_order:

    source_meta = model_file_map[model_name]["meta"]
    source_test = model_file_map[model_name]["test"]

    if not os.path.exists(source_meta):
        raise FileNotFoundError(source_meta)

    if not os.path.exists(source_test):
        raise FileNotFoundError(source_test)

    meta_probs = np.load(source_meta)
    test_probs = np.load(source_test)

    assert meta_probs.shape == (10000, 10)
    assert test_probs.shape == (10000, 10)

    assert np.allclose(
        meta_probs.sum(axis=1),
        1.0,
        atol=1e-5
    )

    assert np.allclose(
        test_probs.sum(axis=1),
        1.0,
        atol=1e-5
    )

    destination_meta = os.path.join(
        seed42_probability_dir,
        f"CIFAR10_{model_name}_Seed42_Meta_Probabilities.npy"
    )

    destination_test = os.path.join(
        seed42_probability_dir,
        f"CIFAR10_{model_name}_Seed42_Test_Probabilities.npy"
    )

    np.save(
        destination_meta,
        meta_probs.astype(np.float32)
    )

    np.save(
        destination_test,
        test_probs.astype(np.float32)
    )

    meta_arrays.append(meta_probs)
    test_arrays.append(test_probs)

    print(
        model_name,
        "registered:",
        meta_probs.shape,
        test_probs.shape
    )

# ------------------------------------------------------------
# Save reconstructed 40-feature matrices
# ------------------------------------------------------------

seed42_meta_features = np.concatenate(
    meta_arrays,
    axis=1
).astype(np.float32)

seed42_test_features = np.concatenate(
    test_arrays,
    axis=1
).astype(np.float32)

meta_indices = np.load(
    os.path.join(
        seed42_dir,
        "Meta_Holdout_Indices.npy"
    )
)

seed42_meta_labels = (
    y_cifar_development[meta_indices]
    .reshape(-1)
    .astype(np.int64)
)

assert seed42_meta_features.shape == (10000, 40)
assert seed42_test_features.shape == (10000, 40)
assert seed42_meta_labels.shape == (10000,)

np.save(
    os.path.join(
        seed42_dir,
        "CIFAR10_Seed42_Meta_40_Features.npy"
    ),
    seed42_meta_features
)

np.save(
    os.path.join(
        seed42_dir,
        "CIFAR10_Seed42_Test_40_Features.npy"
    ),
    seed42_test_features
)

np.save(
    os.path.join(
        seed42_dir,
        "CIFAR10_Seed42_Meta_Labels.npy"
    ),
    seed42_meta_labels
)

# ------------------------------------------------------------
# Copy or extract original seed-42 result rows
# ------------------------------------------------------------

if not os.path.exists(original_results_path):
    raise FileNotFoundError(
        original_results_path
    )

original_results = pd.read_csv(
    original_results_path
)

print("\nOriginal result columns:")
print(original_results.columns.tolist())

# Flexible filtering for fixed hold-out rows
protocol_column = next(
    (
        column
        for column in original_results.columns
        if "protocol" in column.lower()
    ),
    None
)

if protocol_column is not None:

    seed42_results = original_results[
        original_results[protocol_column]
        .astype(str)
        .str.lower()
        .str.contains("hold")
    ].copy()

else:
    seed42_results = original_results.copy()

# Keep the three ensemble methods only
method_column = next(
    (
        column
        for column in seed42_results.columns
        if "method" in column.lower()
    ),
    None
)

if method_column is not None:

    seed42_results = seed42_results[
        seed42_results[method_column]
        .astype(str)
        .str.lower()
        .str.contains(
            "soft|logistic|lightgbm",
            regex=True
        )
    ].copy()

seed42_results["Holdout_Seed"] = 42
seed42_results["Meta_Training_Samples"] = 10000

seed42_results_path = os.path.join(
    seed42_meta_dir,
    "CIFAR10_Seed42_Holdout_Results_Original.csv"
)

seed42_results.to_csv(
    seed42_results_path,
    index=False
)

display(seed42_results)

print("\nSeed-42 registration completed.")
print("Meta-feature shape:", seed42_meta_features.shape)
print("Test-feature shape:", seed42_test_features.shape)
print("Results saved to:", seed42_results_path)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED HOLD-OUT
# CUSTOM CNN — SEED 123
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

HOLDOUT_SEED = 123

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

cnn_dir = os.path.join(
    seed_dir,
    "Custom_CNN"
)

os.makedirs(cnn_dir, exist_ok=True)

# ------------------------------------------------------------
# Reproducibility and memory helpers
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()

# ------------------------------------------------------------
# Load saved repeated-holdout split
# ------------------------------------------------------------

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)

X_base_all = X_cifar_development[
    base_indices
]

y_base_all = y_cifar_development[
    base_indices
]

X_meta = X_cifar_development[
    meta_indices
]

y_meta = y_cifar_development[
    meta_indices
]

# ------------------------------------------------------------
# Internal 90:10 train-validation split
# Taken only from the 40,000-image base partition
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Untouched meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

print(
    "Training class counts:",
    np.bincount(
        y_train_internal,
        minlength=10
    )
)

print(
    "Validation class counts:",
    np.bincount(
        y_val_internal,
        minlength=10
    )
)

print(
    "Meta class counts:",
    np.bincount(
        y_meta,
        minlength=10
    )
)

assert X_train_internal.shape[0] == 36000
assert X_val_internal.shape[0] == 4000
assert X_meta.shape[0] == 10000

# ------------------------------------------------------------
# Matched augmentation
# ------------------------------------------------------------

def build_cnn_cifar_seed123():

    augmentation = tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="cnn_cifar_augmentation"
    )

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_cnn_input"
    )

    x = augmentation(inputs)

    # Block 1
    x = layers.Conv2D(
        64,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        64,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.20)(x)

    # Block 2
    x = layers.Conv2D(
        128,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        128,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.30)(x)

    # Block 3
    x = layers.Conv2D(
        256,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        256,
        3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.40)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_cnn_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_CNN_Holdout_Seed123"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

model_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed123_Best.keras"
)

history_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed123_History.csv"
)

meta_prob_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed123_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed123_Test_Probabilities.npy"
)

# ------------------------------------------------------------
# Train unless valid probabilities already exist
# ------------------------------------------------------------

existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):

    try:
        cnn_meta_probs_seed123 = np.load(
            meta_prob_path
        )

        cnn_test_probs_seed123 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            cnn_meta_probs_seed123.shape
            == (10000, 10)
            and cnn_test_probs_seed123.shape
            == (10000, 10)
            and np.isfinite(
                cnn_meta_probs_seed123
            ).all()
            and np.isfinite(
                cnn_test_probs_seed123
            ).all()
        )

    except Exception:
        existing_outputs_valid = False

if existing_outputs_valid:

    print(
        "\nValid saved CNN seed-123 probabilities "
        "already exist. Training skipped."
    )

else:

    set_all_seeds(HOLDOUT_SEED)
    clear_training_memory()

    cnn_model_seed123 = (
        build_cnn_cifar_seed123()
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=CIFAR_EARLY_STOPPING_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=model_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    history = cnn_model_seed123.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=2
    )

    pd.DataFrame(
        history.history
    ).to_csv(
        history_path,
        index=False
    )

    cnn_meta_probs_seed123 = (
        cnn_model_seed123.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    cnn_test_probs_seed123 = (
        cnn_model_seed123.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    np.save(
        meta_prob_path,
        cnn_meta_probs_seed123.astype(
            np.float32
        )
    )

    np.save(
        test_prob_path,
        cnn_test_probs_seed123.astype(
            np.float32
        )
    )

# ------------------------------------------------------------
# Validate saved probability matrices
# ------------------------------------------------------------

assert cnn_meta_probs_seed123.shape == (
    10000,
    10
)

assert cnn_test_probs_seed123.shape == (
    10000,
    10
)

assert np.isfinite(
    cnn_meta_probs_seed123
).all()

assert np.isfinite(
    cnn_test_probs_seed123
).all()

assert np.allclose(
    cnn_meta_probs_seed123.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    cnn_test_probs_seed123.sum(axis=1),
    1.0,
    atol=1e-5
)

# ------------------------------------------------------------
# Report performance
# ------------------------------------------------------------

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        cnn_meta_probs_seed123,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(
        cnn_test_probs_seed123,
        axis=1
    )
)

cnn_seed123_summary = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": 123,
            "Model": "Custom CNN",
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Training_Samples": 10000,
            "Meta_Holdout_Accuracy":
                meta_accuracy,
            "Final_Test_Accuracy":
                test_accuracy
        }
    ]
)

summary_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed123_Summary.csv"
)

cnn_seed123_summary.to_csv(
    summary_path,
    index=False
)

display(
    cnn_seed123_summary.round(6)
)

print("\nCustom CNN seed 123 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", cnn_dir)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED HOLD-OUT
# VGG16-STYLE MODEL — SEED 123
# WITH CHECKPOINT RELOAD
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

HOLDOUT_SEED = 123

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

vgg_dir = os.path.join(
    seed_dir,
    "VGG16_Style"
)

os.makedirs(vgg_dir, exist_ok=True)

# ------------------------------------------------------------
# Reproducibility helpers
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()

# ------------------------------------------------------------
# Load saved repeated-holdout split
# ------------------------------------------------------------

base_indices = np.load(
    os.path.join(
        seed_dir,
        "Base_Training_Indices.npy"
    )
)

meta_indices = np.load(
    os.path.join(
        seed_dir,
        "Meta_Holdout_Indices.npy"
    )
)

X_base_all = X_cifar_development[base_indices]
y_base_all = y_cifar_development[base_indices]

X_meta = X_cifar_development[meta_indices]
y_meta = y_cifar_development[meta_indices]

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Untouched meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

assert X_train_internal.shape[0] == 36000
assert X_val_internal.shape[0] == 4000
assert X_meta.shape[0] == 10000

# ------------------------------------------------------------
# Matched VGG16-style model
# ------------------------------------------------------------

def build_vgg16_cifar_seed123():

    augmentation = tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="vgg16_cifar_augmentation"
    )

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_vgg16_input"
    )

    x = augmentation(inputs)

    # Stage 1: 64 × 2
    for _ in range(2):
        x = layers.Conv2D(
            64,
            3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal"
        )(x)

        x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.20)(x)

    # Stage 2: 128 × 2
    for _ in range(2):
        x = layers.Conv2D(
            128,
            3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal"
        )(x)

        x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.30)(x)

    # Stage 3: 256 × 3
    for _ in range(3):
        x = layers.Conv2D(
            256,
            3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal"
        )(x)

        x = layers.BatchNormalization()(x)

    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.40)(x)

    # Stage 4: 512 × 3
    for _ in range(3):
        x = layers.Conv2D(
            512,
            3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal"
        )(x)

        x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    x = layers.Dense(
        256,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_vgg16_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_VGG16_Holdout_Seed123"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

model_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed123_Best.keras"
)

history_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed123_History.csv"
)

meta_prob_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed123_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed123_Test_Probabilities.npy"
)

# ------------------------------------------------------------
# Reuse valid saved probabilities when available
# ------------------------------------------------------------

existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):
    try:
        vgg_meta_probs_seed123 = np.load(
            meta_prob_path
        )

        vgg_test_probs_seed123 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            vgg_meta_probs_seed123.shape
            == (10000, 10)
            and vgg_test_probs_seed123.shape
            == (10000, 10)
            and np.isfinite(
                vgg_meta_probs_seed123
            ).all()
            and np.isfinite(
                vgg_test_probs_seed123
            ).all()
        )

    except Exception:
        existing_outputs_valid = False

if existing_outputs_valid:

    print(
        "\nValid saved VGG16-style seed-123 "
        "probabilities already exist. Training skipped."
    )

else:

    set_all_seeds(HOLDOUT_SEED)
    clear_training_memory()

    if os.path.exists(model_path):

        print(
            "\nLoading existing VGG16-style checkpoint:"
        )
        print(model_path)

        vgg_model_seed123 = (
            tf.keras.models.load_model(
                model_path
            )
        )

    else:

        vgg_model_seed123 = (
            build_vgg16_cifar_seed123()
        )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=CIFAR_EARLY_STOPPING_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=model_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    history = vgg_model_seed123.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=2
    )

    pd.DataFrame(
        history.history
    ).to_csv(
        history_path,
        index=False
    )

    vgg_meta_probs_seed123 = (
        vgg_model_seed123.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    vgg_test_probs_seed123 = (
        vgg_model_seed123.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    np.save(
        meta_prob_path,
        vgg_meta_probs_seed123.astype(
            np.float32
        )
    )

    np.save(
        test_prob_path,
        vgg_test_probs_seed123.astype(
            np.float32
        )
    )

# ------------------------------------------------------------
# Validate outputs
# ------------------------------------------------------------

assert vgg_meta_probs_seed123.shape == (
    10000,
    10
)

assert vgg_test_probs_seed123.shape == (
    10000,
    10
)

assert np.allclose(
    vgg_meta_probs_seed123.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    vgg_test_probs_seed123.sum(axis=1),
    1.0,
    atol=1e-5
)

meta_accuracy = accuracy_score(
    y_meta,
    np.argmax(
        vgg_meta_probs_seed123,
        axis=1
    )
)

test_accuracy = accuracy_score(
    y_cifar_test,
    np.argmax(
        vgg_test_probs_seed123,
        axis=1
    )
)

vgg_seed123_summary = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": 123,
            "Model": "VGG16-style",
            "Meta_Holdout_Accuracy":
                meta_accuracy,
            "Final_Test_Accuracy":
                test_accuracy
        }
    ]
)

vgg_seed123_summary.to_csv(
    os.path.join(
        vgg_dir,
        "CIFAR10_VGG16_Seed123_Summary.csv"
    ),
    index=False
)

display(
    vgg_seed123_summary.round(6)
)

print("\nVGG16-style seed 123 completed.")
print("Meta hold-out accuracy:", meta_accuracy)
print("Final test accuracy:", test_accuracy)
print("Saved under:", vgg_dir)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# MATCHED RESNET-STYLE MODEL — SEED 123
# COMPLETE RESTART-SAFE CELL
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    roc_auc_score
)

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

HOLDOUT_SEED = 123

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

MODEL_NAME = "ResNet50"
MODEL_DISPLAY_NAME = "ResNet-style"

# CIFAR_REPEATED_HOLDOUT_DIR must already be defined.
# Expected:
# /content/drive/MyDrive/Paper_1_Matched_Protocol_Comparison/
# Extended_Experiments/Repeated_Holdout/CIFAR10

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet_Style"
)

os.makedirs(
    resnet_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. REPRODUCIBILITY HELPERS
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()


set_all_seeds(
    HOLDOUT_SEED
)

# ------------------------------------------------------------
# 3. CHECK REQUIRED DATA VARIABLES
# ------------------------------------------------------------

required_variables = [
    "X_cifar_development",
    "y_cifar_development",
    "X_cifar_test",
    "y_cifar_test"
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise NameError(
        "Required CIFAR variables are missing: "
        f"{missing_variables}. "
        "Run the CIFAR loading cell first."
    )

X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

# Normalize only when required.
if X_cifar_development.max() > 1.5:
    X_cifar_development = (
        X_cifar_development / 255.0
    )

if X_cifar_test.max() > 1.5:
    X_cifar_test = (
        X_cifar_test / 255.0
    )

assert X_cifar_development.shape == (
    50000,
    32,
    32,
    3
)

assert X_cifar_test.shape == (
    10000,
    32,
    32,
    3
)

assert y_cifar_development.shape == (
    50000,
)

assert y_cifar_test.shape == (
    10000,
)

print(
    "Development pixel range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

print(
    "Test pixel range:",
    float(X_cifar_test.min()),
    "to",
    float(X_cifar_test.max())
)

# ------------------------------------------------------------
# 4. LOAD SAVED BASE/META SPLIT INDICES
# ------------------------------------------------------------

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(
    base_indices_path
):
    raise FileNotFoundError(
        f"Missing base indices:\n{base_indices_path}"
    )

if not os.path.exists(
    meta_indices_path
):
    raise FileNotFoundError(
        f"Missing meta indices:\n{meta_indices_path}"
    )

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert base_indices.shape == (
    40000,
)

assert meta_indices.shape == (
    10000,
)

assert len(
    np.intersect1d(
        base_indices,
        meta_indices
    )
) == 0

assert np.array_equal(
    np.sort(
        np.concatenate(
            [
                base_indices,
                meta_indices
            ]
        )
    ),
    np.arange(50000)
)

X_base_all = X_cifar_development[
    base_indices
]

y_base_all = y_cifar_development[
    base_indices
]

X_meta = X_cifar_development[
    meta_indices
]

y_meta = y_cifar_development[
    meta_indices
]

# ------------------------------------------------------------
# 5. INTERNAL VALIDATION SPLIT
# Only from the 40,000-image base partition
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = (
    train_test_split(
        local_indices,
        test_size=0.10,
        stratify=y_base_all,
        random_state=HOLDOUT_SEED
    )
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

assert X_train_internal.shape == (
    36000,
    32,
    32,
    3
)

assert X_val_internal.shape == (
    4000,
    32,
    32,
    3
)

assert X_meta.shape == (
    10000,
    32,
    32,
    3
)

print("\nSplit verification")
print("------------------")
print("Base partition:", X_base_all.shape)
print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

print(
    "Base class counts:",
    np.bincount(
        y_base_all,
        minlength=10
    )
)

print(
    "Training class counts:",
    np.bincount(
        y_train_internal,
        minlength=10
    )
)

print(
    "Validation class counts:",
    np.bincount(
        y_val_internal,
        minlength=10
    )
)

print(
    "Meta class counts:",
    np.bincount(
        y_meta,
        minlength=10
    )
)

# ------------------------------------------------------------
# 6. MATCHED DATA AUGMENTATION
# ------------------------------------------------------------

def build_resnet_augmentation():

    return tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="resnet_cifar_augmentation"
    )

# ------------------------------------------------------------
# 7. RESIDUAL BLOCK
# ------------------------------------------------------------

def cifar_residual_block(
    inputs,
    filters,
    stride=1,
    block_name="residual_block"
):

    shortcut = inputs

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv1"
    )(inputs)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{block_name}_relu1"
    )(x)

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv2"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn2"
    )(x)

    input_channels = int(
        inputs.shape[-1]
    )

    if (
        stride != 1
        or input_channels != filters
    ):

        shortcut = layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{block_name}_shortcut_conv"
        )(shortcut)

        shortcut = layers.BatchNormalization(
            name=f"{block_name}_shortcut_bn"
        )(shortcut)

    x = layers.Add(
        name=f"{block_name}_add"
    )(
        [
            x,
            shortcut
        ]
    )

    x = layers.Activation(
        "relu",
        name=f"{block_name}_output"
    )(x)

    return x

# ------------------------------------------------------------
# 8. MATCHED CIFAR RESNET BUILDER
# ------------------------------------------------------------

def build_resnet50_cifar():

    augmentation = (
        build_resnet_augmentation()
    )

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_resnet_input"
    )

    x = augmentation(
        inputs
    )

    # CIFAR stem: 3x3 convolution, no initial max pooling
    x = layers.Conv2D(
        filters=64,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv"
    )(x)

    x = layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = layers.Activation(
        "relu",
        name="stem_relu"
    )(x)

    # Stage 1: two blocks, 64 filters
    x = cifar_residual_block(
        x,
        filters=64,
        stride=1,
        block_name="stage1_block1"
    )

    x = cifar_residual_block(
        x,
        filters=64,
        stride=1,
        block_name="stage1_block2"
    )

    x = layers.Dropout(
        0.20,
        name="stage1_dropout"
    )(x)

    # Stage 2: two blocks, 128 filters
    x = cifar_residual_block(
        x,
        filters=128,
        stride=2,
        block_name="stage2_block1"
    )

    x = cifar_residual_block(
        x,
        filters=128,
        stride=1,
        block_name="stage2_block2"
    )

    x = layers.Dropout(
        0.30,
        name="stage2_dropout"
    )(x)

    # Stage 3: two blocks, 256 filters
    x = cifar_residual_block(
        x,
        filters=256,
        stride=2,
        block_name="stage3_block1"
    )

    x = cifar_residual_block(
        x,
        filters=256,
        stride=1,
        block_name="stage3_block2"
    )

    x = layers.Dropout(
        0.40,
        name="stage3_dropout"
    )(x)

    # Stage 4: two blocks, 512 filters
    x = cifar_residual_block(
        x,
        filters=512,
        stride=2,
        block_name="stage4_block1"
    )

    x = cifar_residual_block(
        x,
        filters=512,
        stride=1,
        block_name="stage4_block2"
    )

    x = layers.Dropout(
        0.40,
        name="stage4_dropout"
    )(x)

    # Classification head
    x = layers.GlobalAveragePooling2D(
        name="global_average_pool"
    )(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense"
    )(x)

    x = layers.BatchNormalization(
        name="classification_bn"
    )(x)

    x = layers.Dropout(
        0.50,
        name="classification_dropout"
    )(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_resnet_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_ResNet_Reconstructed"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss=(
            "sparse_categorical_crossentropy"
        ),
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# 9. OUTPUT FILES
# ------------------------------------------------------------

best_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed123_Best.keras"
)

latest_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed123_Latest.keras"
)

history_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed123_History.csv"
)

training_log_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed123_Training_Log.csv"
)

meta_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed123_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed123_Test_Probabilities.npy"
)

summary_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed123_Summary.csv"
)

# ------------------------------------------------------------
# 10. VALIDATE EXISTING PROBABILITY FILES
# ------------------------------------------------------------

def valid_probability_matrix(
    array,
    expected_shape
):

    return (
        isinstance(
            array,
            np.ndarray
        )
        and array.shape == expected_shape
        and np.isfinite(
            array
        ).all()
        and np.all(
            array >= 0.0
        )
        and np.all(
            array <= 1.0
        )
        and np.allclose(
            array.sum(axis=1),
            1.0,
            atol=1e-5
        )
    )


existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):

    try:

        resnet_meta_probs_seed123 = np.load(
            meta_prob_path
        )

        resnet_test_probs_seed123 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            valid_probability_matrix(
                resnet_meta_probs_seed123,
                (10000, 10)
            )
            and valid_probability_matrix(
                resnet_test_probs_seed123,
                (10000, 10)
            )
        )

    except Exception as error:

        print(
            "Could not validate saved probabilities:",
            error
        )

        existing_outputs_valid = False

# ------------------------------------------------------------
# 11. BUILD, RELOAD OR TRAIN MODEL
# ------------------------------------------------------------

if existing_outputs_valid:

    print(
        "\nValid ResNet seed-123 probability "
        "files already exist."
    )

    print(
        "Deep-model training is skipped."
    )

else:

    clear_training_memory()
    set_all_seeds(
        HOLDOUT_SEED
    )

    # Reload best checkpoint when available.
    if os.path.exists(
        best_model_path
    ):

        print(
            "\nLoading existing best checkpoint:"
        )

        print(
            best_model_path
        )

        resnet_model_seed123 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    elif os.path.exists(
        latest_model_path
    ):

        print(
            "\nLoading latest saved model:"
        )

        print(
            latest_model_path
        )

        resnet_model_seed123 = (
            tf.keras.models.load_model(
                latest_model_path,
                compile=True
            )
        )

    else:

        print(
            "\nBuilding a new matched "
            "CIFAR ResNet model."
        )

        resnet_model_seed123 = (
            build_resnet50_cifar()
        )

    print(
        "Model parameters:",
        f"{resnet_model_seed123.count_params():,}"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=(
                CIFAR_EARLY_STOPPING_PATIENCE
            ),
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=latest_model_path,
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),

        CSVLogger(
            training_log_path,
            append=os.path.exists(
                training_log_path
            )
        )
    ]

    history = resnet_model_seed123.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    new_history_df = pd.DataFrame(
        history.history
    )

    if (
        os.path.exists(history_path)
        and os.path.getsize(history_path) > 0
    ):

        old_history_df = pd.read_csv(
            history_path
        )

        combined_history_df = pd.concat(
            [
                old_history_df,
                new_history_df
            ],
            ignore_index=True
        )

    else:

        combined_history_df = (
            new_history_df
        )

    combined_history_df.to_csv(
        history_path,
        index=False
    )

    # Always reload the best validation checkpoint
    # before producing probabilities.
    if os.path.exists(
        best_model_path
    ):

        clear_training_memory()

        resnet_model_seed123 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    print(
        "\nGenerating meta-holdout "
        "probabilities..."
    )

    resnet_meta_probs_seed123 = (
        resnet_model_seed123.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    print(
        "\nGenerating final-test "
        "probabilities..."
    )

    resnet_test_probs_seed123 = (
        resnet_model_seed123.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    resnet_meta_probs_seed123 = (
        np.asarray(
            resnet_meta_probs_seed123,
            dtype=np.float32
        )
    )

    resnet_test_probs_seed123 = (
        np.asarray(
            resnet_test_probs_seed123,
            dtype=np.float32
        )
    )

    np.save(
        meta_prob_path,
        resnet_meta_probs_seed123
    )

    np.save(
        test_prob_path,
        resnet_test_probs_seed123
    )

# ------------------------------------------------------------
# 12. FINAL OUTPUT VALIDATION
# ------------------------------------------------------------

if not valid_probability_matrix(
    resnet_meta_probs_seed123,
    (10000, 10)
):
    raise ValueError(
        "Invalid ResNet meta-probability matrix."
    )

if not valid_probability_matrix(
    resnet_test_probs_seed123,
    (10000, 10)
):
    raise ValueError(
        "Invalid ResNet test-probability matrix."
    )

# ------------------------------------------------------------
# 13. PERFORMANCE METRICS
# ------------------------------------------------------------

meta_predictions = np.argmax(
    resnet_meta_probs_seed123,
    axis=1
)

test_predictions = np.argmax(
    resnet_test_probs_seed123,
    axis=1
)

meta_accuracy = accuracy_score(
    y_meta,
    meta_predictions
)

meta_macro_precision = precision_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_macro_recall = recall_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_macro_f1 = f1_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_log_loss = log_loss(
    y_meta,
    resnet_meta_probs_seed123,
    labels=np.arange(10)
)

meta_roc_auc = roc_auc_score(
    y_meta,
    resnet_meta_probs_seed123,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

test_accuracy = accuracy_score(
    y_cifar_test,
    test_predictions
)

test_macro_precision = precision_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_macro_recall = recall_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_macro_f1 = f1_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_log_loss = log_loss(
    y_cifar_test,
    resnet_test_probs_seed123,
    labels=np.arange(10)
)

test_roc_auc = roc_auc_score(
    y_cifar_test,
    resnet_test_probs_seed123,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

resnet_seed123_summary = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": HOLDOUT_SEED,
            "Model": MODEL_DISPLAY_NAME,
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Holdout_Samples": 10000,
            "Final_Test_Samples": 10000,
            "Meta_Accuracy": meta_accuracy,
            "Meta_Macro_Precision":
                meta_macro_precision,
            "Meta_Macro_Recall":
                meta_macro_recall,
            "Meta_Macro_F1":
                meta_macro_f1,
            "Meta_ROC_AUC":
                meta_roc_auc,
            "Meta_Log_Loss":
                meta_log_loss,
            "Test_Accuracy":
                test_accuracy,
            "Test_Macro_Precision":
                test_macro_precision,
            "Test_Macro_Recall":
                test_macro_recall,
            "Test_Macro_F1":
                test_macro_f1,
            "Test_ROC_AUC":
                test_roc_auc,
            "Test_Log_Loss":
                test_log_loss
        }
    ]
)

resnet_seed123_summary.to_csv(
    summary_path,
    index=False
)

display(
    resnet_seed123_summary.round(6)
)

print(
    "\nResNet-style seed 123 completed successfully."
)

print(
    "Meta hold-out accuracy:",
    f"{meta_accuracy * 100:.4f}%"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.4f}%"
)

print(
    "Meta probability shape:",
    resnet_meta_probs_seed123.shape
)

print(
    "Test probability shape:",
    resnet_test_probs_seed123.shape
)

print(
    "\nBest checkpoint:"
)

print(
    best_model_path
)

print(
    "\nMeta probabilities:"
)

print(
    meta_prob_path
)

print(
    "\nTest probabilities:"
)

print(
    test_prob_path
)

print(
    "\nSummary:"
)

print(
    summary_path
)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# MATCHED EFFICIENTNET-STYLE MODEL — SEED 123
# COMPLETE CHECKPOINTED CELL
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    roc_auc_score
)

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

HOLDOUT_SEED = 123

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

efficientnet_dir = os.path.join(
    seed_dir,
    "EfficientNet_Style"
)

os.makedirs(
    efficientnet_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. REPRODUCIBILITY HELPERS
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()


set_all_seeds(HOLDOUT_SEED)

# ------------------------------------------------------------
# 3. CHECK REQUIRED CIFAR VARIABLES
# ------------------------------------------------------------

required_variables = [
    "X_cifar_development",
    "y_cifar_development",
    "X_cifar_test",
    "y_cifar_test"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Required CIFAR variables are missing: "
        f"{missing_variables}"
    )

X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

assert X_cifar_development.shape == (
    50000,
    32,
    32,
    3
)

assert X_cifar_test.shape == (
    10000,
    32,
    32,
    3
)

assert y_cifar_development.shape == (50000,)
assert y_cifar_test.shape == (10000,)

print(
    "Development pixel range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

# ------------------------------------------------------------
# 4. LOAD SAVED SEED-123 SPLIT
# ------------------------------------------------------------

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(base_indices_path):
    raise FileNotFoundError(base_indices_path)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(meta_indices_path)

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)

assert len(
    np.intersect1d(
        base_indices,
        meta_indices
    )
) == 0

X_base_all = X_cifar_development[
    base_indices
]

y_base_all = y_cifar_development[
    base_indices
]

X_meta = X_cifar_development[
    meta_indices
]

y_meta = y_cifar_development[
    meta_indices
]

# ------------------------------------------------------------
# 5. INTERNAL VALIDATION SPLIT
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

assert X_train_internal.shape == (
    36000,
    32,
    32,
    3
)

assert X_val_internal.shape == (
    4000,
    32,
    32,
    3
)

assert X_meta.shape == (
    10000,
    32,
    32,
    3
)

print("\nSplit verification")
print("------------------")
print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

print(
    "Meta class counts:",
    np.bincount(
        y_meta,
        minlength=10
    )
)

# ------------------------------------------------------------
# 6. AUGMENTATION
# ------------------------------------------------------------

def build_efficientnet_augmentation():

    return tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="efficientnet_cifar_augmentation"
    )

# ------------------------------------------------------------
# 7. SQUEEZE-AND-EXCITATION BLOCK
# ------------------------------------------------------------

def squeeze_excite_block(
    inputs,
    se_ratio=0.25,
    name="se"
):

    filters = int(
        inputs.shape[-1]
    )

    reduced_filters = max(
        1,
        int(filters * se_ratio)
    )

    x = layers.GlobalAveragePooling2D(
        keepdims=True,
        name=f"{name}_gap"
    )(inputs)

    x = layers.Conv2D(
        reduced_filters,
        kernel_size=1,
        activation="swish",
        padding="same",
        name=f"{name}_reduce"
    )(x)

    x = layers.Conv2D(
        filters,
        kernel_size=1,
        activation="sigmoid",
        padding="same",
        name=f"{name}_expand"
    )(x)

    return layers.Multiply(
        name=f"{name}_scale"
    )(
        [inputs, x]
    )

# ------------------------------------------------------------
# 8. MBConv BLOCK
# ------------------------------------------------------------

def mbconv_block(
    inputs,
    output_filters,
    kernel_size,
    stride,
    expansion_ratio,
    dropout_rate=0.0,
    block_name="mbconv"
):

    input_filters = int(
        inputs.shape[-1]
    )

    expanded_filters = (
        input_filters
        * expansion_ratio
    )

    x = inputs

    if expansion_ratio != 1:

        x = layers.Conv2D(
            expanded_filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            name=f"{block_name}_expand_conv"
        )(x)

        x = layers.BatchNormalization(
            name=f"{block_name}_expand_bn"
        )(x)

        x = layers.Activation(
            "swish",
            name=f"{block_name}_expand_activation"
        )(x)

    x = layers.DepthwiseConv2D(
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        use_bias=False,
        name=f"{block_name}_depthwise"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_depthwise_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name=f"{block_name}_depthwise_activation"
    )(x)

    x = squeeze_excite_block(
        x,
        se_ratio=0.25,
        name=f"{block_name}_se"
    )

    x = layers.Conv2D(
        output_filters,
        kernel_size=1,
        padding="same",
        use_bias=False,
        name=f"{block_name}_project_conv"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_project_bn"
    )(x)

    if (
        stride == 1
        and input_filters == output_filters
    ):

        if dropout_rate > 0:

            x = layers.Dropout(
                dropout_rate,
                name=f"{block_name}_dropout"
            )(x)

        x = layers.Add(
            name=f"{block_name}_add"
        )(
            [inputs, x]
        )

    return x

# ------------------------------------------------------------
# 9. MATCHED EFFICIENTNET-STYLE BUILDER
# ------------------------------------------------------------

def build_efficientnet_cifar():

    augmentation = (
        build_efficientnet_augmentation()
    )

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_efficientnet_input"
    )

    x = augmentation(
        inputs
    )

    # Stem
    x = layers.Conv2D(
        32,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv"
    )(x)

    x = layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name="stem_activation"
    )(x)

    # MBConv sequence
    x = mbconv_block(
        x,
        output_filters=16,
        kernel_size=3,
        stride=1,
        expansion_ratio=1,
        dropout_rate=0.05,
        block_name="block1"
    )

    x = mbconv_block(
        x,
        output_filters=24,
        kernel_size=3,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.05,
        block_name="block2a"
    )

    x = mbconv_block(
        x,
        output_filters=24,
        kernel_size=3,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.05,
        block_name="block2b"
    )

    x = mbconv_block(
        x,
        output_filters=40,
        kernel_size=5,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.10,
        block_name="block3a"
    )

    x = mbconv_block(
        x,
        output_filters=40,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.10,
        block_name="block3b"
    )

    x = mbconv_block(
        x,
        output_filters=80,
        kernel_size=3,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.15,
        block_name="block4a"
    )

    x = mbconv_block(
        x,
        output_filters=80,
        kernel_size=3,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.15,
        block_name="block4b"
    )

    x = mbconv_block(
        x,
        output_filters=112,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.20,
        block_name="block5"
    )

    x = mbconv_block(
        x,
        output_filters=192,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.20,
        block_name="block6"
    )

    # Head
    x = layers.Conv2D(
        1280,
        kernel_size=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="head_conv"
    )(x)

    x = layers.BatchNormalization(
        name="head_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name="head_activation"
    )(x)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.Dropout(
        0.50,
        name="head_dropout"
    )(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_efficientnet_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_EfficientNet_Reconstructed"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# 10. OUTPUT PATHS
# ------------------------------------------------------------

best_model_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed123_Best.keras"
)

latest_model_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed123_Latest.keras"
)

history_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed123_History.csv"
)

training_log_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed123_Training_Log.csv"
)

meta_prob_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed123_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed123_Test_Probabilities.npy"
)

summary_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed123_Summary.csv"
)

# ------------------------------------------------------------
# 11. PROBABILITY VALIDATION
# ------------------------------------------------------------

def valid_probability_matrix(
    array,
    expected_shape
):

    return (
        isinstance(array, np.ndarray)
        and array.shape == expected_shape
        and np.isfinite(array).all()
        and np.all(array >= 0.0)
        and np.all(array <= 1.0)
        and np.allclose(
            array.sum(axis=1),
            1.0,
            atol=1e-5
        )
    )


existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):

    try:

        efficientnet_meta_probs_seed123 = np.load(
            meta_prob_path
        )

        efficientnet_test_probs_seed123 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            valid_probability_matrix(
                efficientnet_meta_probs_seed123,
                (10000, 10)
            )
            and valid_probability_matrix(
                efficientnet_test_probs_seed123,
                (10000, 10)
            )
        )

    except Exception as error:

        print(
            "Saved probability validation failed:",
            error
        )

# ------------------------------------------------------------
# 12. TRAIN OR RELOAD
# ------------------------------------------------------------

if existing_outputs_valid:

    print(
        "\nValid EfficientNet seed-123 probabilities "
        "already exist. Training skipped."
    )

else:

    clear_training_memory()
    set_all_seeds(HOLDOUT_SEED)

    if os.path.exists(best_model_path):

        print("\nLoading best checkpoint:")
        print(best_model_path)

        efficientnet_model_seed123 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    elif os.path.exists(latest_model_path):

        print("\nLoading latest checkpoint:")
        print(latest_model_path)

        efficientnet_model_seed123 = (
            tf.keras.models.load_model(
                latest_model_path,
                compile=True
            )
        )

    else:

        print(
            "\nBuilding new matched "
            "EfficientNet-style model."
        )

        efficientnet_model_seed123 = (
            build_efficientnet_cifar()
        )

    print(
        "Model parameters:",
        f"{efficientnet_model_seed123.count_params():,}"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=(
                CIFAR_EARLY_STOPPING_PATIENCE
            ),
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=latest_model_path,
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),

        CSVLogger(
            training_log_path,
            append=os.path.exists(
                training_log_path
            )
        )
    ]

    history = efficientnet_model_seed123.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    new_history_df = pd.DataFrame(
        history.history
    )

    if (
        os.path.exists(history_path)
        and os.path.getsize(history_path) > 0
    ):

        old_history_df = pd.read_csv(
            history_path
        )

        combined_history_df = pd.concat(
            [
                old_history_df,
                new_history_df
            ],
            ignore_index=True
        )

    else:

        combined_history_df = (
            new_history_df
        )

    combined_history_df.to_csv(
        history_path,
        index=False
    )

    # Reload best validation model before predictions
    if os.path.exists(best_model_path):

        clear_training_memory()

        efficientnet_model_seed123 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    efficientnet_meta_probs_seed123 = (
        efficientnet_model_seed123.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    efficientnet_test_probs_seed123 = (
        efficientnet_model_seed123.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    efficientnet_meta_probs_seed123 = np.asarray(
        efficientnet_meta_probs_seed123,
        dtype=np.float32
    )

    efficientnet_test_probs_seed123 = np.asarray(
        efficientnet_test_probs_seed123,
        dtype=np.float32
    )

    np.save(
        meta_prob_path,
        efficientnet_meta_probs_seed123
    )

    np.save(
        test_prob_path,
        efficientnet_test_probs_seed123
    )

# ------------------------------------------------------------
# 13. FINAL VALIDATION
# ------------------------------------------------------------

if not valid_probability_matrix(
    efficientnet_meta_probs_seed123,
    (10000, 10)
):
    raise ValueError(
        "Invalid EfficientNet meta probabilities."
    )

if not valid_probability_matrix(
    efficientnet_test_probs_seed123,
    (10000, 10)
):
    raise ValueError(
        "Invalid EfficientNet test probabilities."
    )

# ------------------------------------------------------------
# 14. METRICS
# ------------------------------------------------------------

meta_predictions = np.argmax(
    efficientnet_meta_probs_seed123,
    axis=1
)

test_predictions = np.argmax(
    efficientnet_test_probs_seed123,
    axis=1
)

meta_accuracy = accuracy_score(
    y_meta,
    meta_predictions
)

meta_precision = precision_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_recall = recall_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_f1 = f1_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_auc = roc_auc_score(
    y_meta,
    efficientnet_meta_probs_seed123,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

meta_loss = log_loss(
    y_meta,
    efficientnet_meta_probs_seed123,
    labels=np.arange(10)
)

test_accuracy = accuracy_score(
    y_cifar_test,
    test_predictions
)

test_precision = precision_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_auc = roc_auc_score(
    y_cifar_test,
    efficientnet_test_probs_seed123,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

test_loss = log_loss(
    y_cifar_test,
    efficientnet_test_probs_seed123,
    labels=np.arange(10)
)

summary_df = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": HOLDOUT_SEED,
            "Model": "EfficientNet-style",
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Holdout_Samples": 10000,
            "Final_Test_Samples": 10000,
            "Meta_Accuracy": meta_accuracy,
            "Meta_Macro_Precision":
                meta_precision,
            "Meta_Macro_Recall":
                meta_recall,
            "Meta_Macro_F1":
                meta_f1,
            "Meta_ROC_AUC":
                meta_auc,
            "Meta_Log_Loss":
                meta_loss,
            "Test_Accuracy":
                test_accuracy,
            "Test_Macro_Precision":
                test_precision,
            "Test_Macro_Recall":
                test_recall,
            "Test_Macro_F1":
                test_f1,
            "Test_ROC_AUC":
                test_auc,
            "Test_Log_Loss":
                test_loss
        }
    ]
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df.round(6)
)

print(
    "\nEfficientNet-style seed 123 completed."
)

print(
    "Meta hold-out accuracy:",
    f"{meta_accuracy * 100:.4f}%"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.4f}%"
)

print(
    "Meta probability shape:",
    efficientnet_meta_probs_seed123.shape
)

print(
    "Test probability shape:",
    efficientnet_test_probs_seed123.shape
)

print("\nBest checkpoint:")
print(best_model_path)

print("\nMeta probabilities:")
print(meta_prob_path)

print("\nTest probabilities:")
print(test_prob_path)

print("\nSummary:")
print(summary_path)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# META-LEARNERS AND SOFT VOTING — SEED 123
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

# LightGBM must be installed in the Colab environment.
try:
    from lightgbm import LGBMClassifier
except ImportError:
    !pip -q install lightgbm
    from lightgbm import LGBMClassifier

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

HOLDOUT_SEED = 123
NUM_CLASSES = 10

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

meta_output_dir = os.path.join(
    seed_dir,
    "Meta_Learners"
)

os.makedirs(
    meta_output_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. CHECK LABEL VARIABLES
# ------------------------------------------------------------

if "y_cifar_development" not in globals():
    raise NameError(
        "y_cifar_development is missing. "
        "Run the CIFAR data-loading cell first."
    )

if "y_cifar_test" not in globals():
    raise NameError(
        "y_cifar_test is missing. "
        "Run the CIFAR data-loading cell first."
    )

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

assert y_cifar_development.shape == (50000,)
assert y_cifar_test.shape == (10000,)

# ------------------------------------------------------------
# 3. LOAD META INDICES AND LABELS
# ------------------------------------------------------------

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(
        meta_indices_path
    )

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert meta_indices.shape == (10000,)

y_meta = y_cifar_development[
    meta_indices
]

assert y_meta.shape == (10000,)

print(
    "Meta-label counts:",
    np.bincount(
        y_meta,
        minlength=NUM_CLASSES
    )
)

print(
    "Test-label counts:",
    np.bincount(
        y_cifar_test,
        minlength=NUM_CLASSES
    )
)

# ------------------------------------------------------------
# 4. BASE-MODEL PROBABILITY PATHS
# Exact feature order:
# CNN -> VGG16 -> ResNet50 -> EfficientNetB0
# ------------------------------------------------------------

probability_paths = {
    "CNN": {
        "meta": os.path.join(
            seed_dir,
            "Custom_CNN",
            "CIFAR10_CNN_Seed123_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "Custom_CNN",
            "CIFAR10_CNN_Seed123_Test_Probabilities.npy"
        )
    },

    "VGG16": {
        "meta": os.path.join(
            seed_dir,
            "VGG16_Style",
            "CIFAR10_VGG16_Seed123_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "VGG16_Style",
            "CIFAR10_VGG16_Seed123_Test_Probabilities.npy"
        )
    },

    "ResNet50": {
        "meta": os.path.join(
            seed_dir,
            "ResNet_Style",
            "CIFAR10_ResNet50_Seed123_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "ResNet_Style",
            "CIFAR10_ResNet50_Seed123_Test_Probabilities.npy"
        )
    },

    "EfficientNetB0": {
        "meta": os.path.join(
            seed_dir,
            "EfficientNet_Style",
            "CIFAR10_EfficientNetB0_Seed123_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "EfficientNet_Style",
            "CIFAR10_EfficientNetB0_Seed123_Test_Probabilities.npy"
        )
    }
}

model_order = [
    "CNN",
    "VGG16",
    "ResNet50",
    "EfficientNetB0"
]

# ------------------------------------------------------------
# 5. VALIDATION FUNCTION
# ------------------------------------------------------------

def validate_probability_matrix(
    probabilities,
    expected_shape,
    model_name,
    split_name
):
    probabilities = np.asarray(
        probabilities
    )

    if probabilities.shape != expected_shape:
        raise ValueError(
            f"{model_name} {split_name} shape is "
            f"{probabilities.shape}; expected {expected_shape}."
        )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{model_name} {split_name} contains "
            "NaN or infinite values."
        )

    if np.any(probabilities < 0.0):
        raise ValueError(
            f"{model_name} {split_name} contains "
            "negative probabilities."
        )

    if np.any(probabilities > 1.0):
        raise ValueError(
            f"{model_name} {split_name} contains "
            "probabilities greater than 1."
        )

    row_sums = probabilities.sum(
        axis=1
    )

    if not np.allclose(
        row_sums,
        1.0,
        atol=1e-5
    ):
        raise ValueError(
            f"{model_name} {split_name} probability "
            "rows do not sum to 1."
        )

    return probabilities.astype(
        np.float32
    )

# ------------------------------------------------------------
# 6. LOAD INDIVIDUAL PROBABILITY ARRAYS
# ------------------------------------------------------------

meta_probability_arrays = []
test_probability_arrays = []

base_model_accuracy_rows = []

for model_name in model_order:

    meta_path = probability_paths[
        model_name
    ]["meta"]

    test_path = probability_paths[
        model_name
    ]["test"]

    if not os.path.exists(meta_path):
        raise FileNotFoundError(
            f"Missing {model_name} meta probabilities:\n"
            f"{meta_path}"
        )

    if not os.path.exists(test_path):
        raise FileNotFoundError(
            f"Missing {model_name} test probabilities:\n"
            f"{test_path}"
        )

    meta_probabilities = (
        validate_probability_matrix(
            np.load(meta_path),
            (10000, NUM_CLASSES),
            model_name,
            "meta"
        )
    )

    test_probabilities = (
        validate_probability_matrix(
            np.load(test_path),
            (10000, NUM_CLASSES),
            model_name,
            "test"
        )
    )

    meta_probability_arrays.append(
        meta_probabilities
    )

    test_probability_arrays.append(
        test_probabilities
    )

    meta_predictions = np.argmax(
        meta_probabilities,
        axis=1
    )

    test_predictions = np.argmax(
        test_probabilities,
        axis=1
    )

    base_model_accuracy_rows.append(
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": HOLDOUT_SEED,
            "Model": model_name,
            "Meta_Accuracy": accuracy_score(
                y_meta,
                meta_predictions
            ),
            "Test_Accuracy": accuracy_score(
                y_cifar_test,
                test_predictions
            )
        }
    )

    print(
        model_name,
        "loaded:",
        meta_probabilities.shape,
        test_probabilities.shape
    )

base_model_accuracy_df = pd.DataFrame(
    base_model_accuracy_rows
)

display(
    base_model_accuracy_df.round(6)
)

# ------------------------------------------------------------
# 7. CONSTRUCT 40-FEATURE STACKING MATRICES
# ------------------------------------------------------------

X_meta_stack = np.concatenate(
    meta_probability_arrays,
    axis=1
).astype(np.float32)

X_test_stack = np.concatenate(
    test_probability_arrays,
    axis=1
).astype(np.float32)

assert X_meta_stack.shape == (10000, 40)
assert X_test_stack.shape == (10000, 40)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_Meta_40_Features.npy"
    ),
    X_meta_stack
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_Test_40_Features.npy"
    ),
    X_test_stack
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_Meta_Labels.npy"
    ),
    y_meta
)

print(
    "\nMeta feature shape:",
    X_meta_stack.shape
)

print(
    "Test feature shape:",
    X_test_stack.shape
)

# ------------------------------------------------------------
# 8. METRIC FUNCTIONS
# ------------------------------------------------------------

def multiclass_brier_score(
    y_true,
    probabilities,
    number_of_classes
):
    one_hot = np.eye(
        number_of_classes,
        dtype=np.float32
    )[y_true]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot
                ) ** 2,
                axis=1
            )
        )
    )


def expected_calibration_error(
    y_true,
    probabilities,
    number_of_bins=15
):
    predictions = np.argmax(
        probabilities,
        axis=1
    )

    confidences = np.max(
        probabilities,
        axis=1
    )

    correctness = (
        predictions == y_true
    ).astype(np.float64)

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1
    )

    ece = 0.0

    for bin_index in range(
        number_of_bins
    ):
        lower_bound = bin_edges[
            bin_index
        ]

        upper_bound = bin_edges[
            bin_index + 1
        ]

        if bin_index == 0:
            in_bin = (
                (confidences >= lower_bound)
                & (confidences <= upper_bound)
            )
        else:
            in_bin = (
                (confidences > lower_bound)
                & (confidences <= upper_bound)
            )

        bin_count = np.sum(
            in_bin
        )

        if bin_count == 0:
            continue

        bin_accuracy = np.mean(
            correctness[in_bin]
        )

        bin_confidence = np.mean(
            confidences[in_bin]
        )

        ece += (
            bin_count
            / len(y_true)
        ) * abs(
            bin_accuracy
            - bin_confidence
        )

    return float(ece)


def evaluate_multiclass_method(
    method_name,
    probabilities,
    y_true
):
    probabilities = validate_probability_matrix(
        probabilities,
        (
            len(y_true),
            NUM_CLASSES
        ),
        method_name,
        "test"
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "Dataset": "CIFAR-10",
        "Protocol": "Repeated fixed hold-out",
        "Holdout_Seed": HOLDOUT_SEED,
        "Method": method_name,

        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "Macro_Precision": precision_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro_Recall": recall_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro_ROC_AUC_OVR": roc_auc_score(
            y_true,
            probabilities,
            multi_class="ovr",
            average="macro",
            labels=np.arange(
                NUM_CLASSES
            )
        ),

        "Log_Loss": log_loss(
            y_true,
            probabilities,
            labels=np.arange(
                NUM_CLASSES
            )
        ),

        "Multiclass_Brier": (
            multiclass_brier_score(
                y_true,
                probabilities,
                NUM_CLASSES
            )
        ),

        "ECE_15_Bins": (
            expected_calibration_error(
                y_true,
                probabilities,
                number_of_bins=15
            )
        )
    }

# ------------------------------------------------------------
# 9. SOFT VOTING
# ------------------------------------------------------------

soft_voting_test_probabilities = np.mean(
    np.stack(
        test_probability_arrays,
        axis=0
    ),
    axis=0
).astype(np.float32)

soft_voting_test_probabilities /= (
    soft_voting_test_probabilities.sum(
        axis=1,
        keepdims=True
    )
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_SoftVoting_Test_Probabilities.npy"
    ),
    soft_voting_test_probabilities
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_SoftVoting_Predicted_Labels.npy"
    ),
    np.argmax(
        soft_voting_test_probabilities,
        axis=1
    ).astype(np.int64)
)

# ------------------------------------------------------------
# 10. LOGISTIC REGRESSION WITH MATCHED GRID SEARCH
# ------------------------------------------------------------

logistic_pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

logistic_parameter_grid = {
    "classifier__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "classifier__penalty": [
        "l1",
        "l2"
    ]
}

logistic_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_parameter_grid,
    scoring="f1_macro",
    cv=logistic_cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
    return_train_score=False
)

print(
    "\nTraining Logistic Regression..."
)

logistic_grid_search.fit(
    X_meta_stack,
    y_meta
)

best_logistic_model = (
    logistic_grid_search.best_estimator_
)

logistic_test_probabilities = (
    best_logistic_model.predict_proba(
        X_test_stack
    )
)

# Ensure class-column order is 0 to 9.
logistic_classes = (
    best_logistic_model
    .named_steps["classifier"]
    .classes_
)

if not np.array_equal(
    logistic_classes,
    np.arange(NUM_CLASSES)
):
    reordered_probabilities = np.zeros(
        (
            len(X_test_stack),
            NUM_CLASSES
        ),
        dtype=np.float64
    )

    reordered_probabilities[
        :,
        logistic_classes
    ] = logistic_test_probabilities

    logistic_test_probabilities = (
        reordered_probabilities
    )

logistic_test_probabilities = np.asarray(
    logistic_test_probabilities,
    dtype=np.float32
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_LogisticRegression_Test_Probabilities.npy"
    ),
    logistic_test_probabilities
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_LogisticRegression_Predicted_Labels.npy"
    ),
    np.argmax(
        logistic_test_probabilities,
        axis=1
    ).astype(np.int64)
)

logistic_best_parameters_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed123_LogisticRegression_Best_Parameters.json"
)

with open(
    logistic_best_parameters_path,
    "w"
) as file:
    json.dump(
        {
            "best_parameters":
                logistic_grid_search.best_params_,
            "best_cv_macro_f1":
                float(
                    logistic_grid_search.best_score_
                )
        },
        file,
        indent=4
    )

print(
    "Best Logistic Regression parameters:",
    logistic_grid_search.best_params_
)

print(
    "Best CV macro F1:",
    logistic_grid_search.best_score_
)

# ------------------------------------------------------------
# 11. LIGHTGBM
# ------------------------------------------------------------

lightgbm_model = LGBMClassifier(
    objective="multiclass",
    num_class=NUM_CLASSES,
    n_estimators=150,
    learning_rate=0.03,
    num_leaves=7,
    max_depth=3,
    min_child_samples=30,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print(
    "\nTraining LightGBM..."
)

lightgbm_model.fit(
    X_meta_stack,
    y_meta
)

lightgbm_test_probabilities = (
    lightgbm_model.predict_proba(
        X_test_stack
    )
)

lightgbm_test_probabilities = np.asarray(
    lightgbm_test_probabilities,
    dtype=np.float32
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_LightGBM_Test_Probabilities.npy"
    ),
    lightgbm_test_probabilities
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed123_LightGBM_Predicted_Labels.npy"
    ),
    np.argmax(
        lightgbm_test_probabilities,
        axis=1
    ).astype(np.int64)
)

# ------------------------------------------------------------
# 12. EVALUATE ALL THREE ENSEMBLE METHODS
# ------------------------------------------------------------

seed123_result_rows = [
    evaluate_multiclass_method(
        "Soft Voting",
        soft_voting_test_probabilities,
        y_cifar_test
    ),

    evaluate_multiclass_method(
        "Logistic Regression",
        logistic_test_probabilities,
        y_cifar_test
    ),

    evaluate_multiclass_method(
        "LightGBM",
        lightgbm_test_probabilities,
        y_cifar_test
    )
]

seed123_results_df = pd.DataFrame(
    seed123_result_rows
)

results_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed123_Repeated_Holdout_Results.csv"
)

seed123_results_df.to_csv(
    results_path,
    index=False
)

base_model_results_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed123_Base_Model_Accuracy.csv"
)

base_model_accuracy_df.to_csv(
    base_model_results_path,
    index=False
)

display(
    seed123_results_df.round(6)
)

print(
    "\nSeed-123 meta-learning completed successfully."
)

print(
    "\nResults saved to:"
)

print(
    results_path
)

print(
    "\nStacking feature order:"
)

print(
    "CNN -> VGG16 -> ResNet50 -> EfficientNetB0"
)

print(
    "\nMeta feature shape:",
    X_meta_stack.shape
)

print(
    "Test feature shape:",
    X_test_stack.shape
)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# CUSTOM CNN — SEED 2025
# COMPLETE CHECKPOINTED CELL
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    roc_auc_score
)

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

HOLDOUT_SEED = 2025

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

cnn_dir = os.path.join(
    seed_dir,
    "Custom_CNN"
)

os.makedirs(cnn_dir, exist_ok=True)

# ------------------------------------------------------------
# 2. REPRODUCIBILITY
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()


set_all_seeds(HOLDOUT_SEED)

# ------------------------------------------------------------
# 3. CHECK AND PREPARE CIFAR DATA
# ------------------------------------------------------------

required_variables = [
    "X_cifar_development",
    "y_cifar_development",
    "X_cifar_test",
    "y_cifar_test"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Missing CIFAR variables: "
        f"{missing_variables}. "
        "Run the CIFAR data-loading cell first."
    )

X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

assert X_cifar_development.shape == (50000, 32, 32, 3)
assert X_cifar_test.shape == (10000, 32, 32, 3)
assert y_cifar_development.shape == (50000,)
assert y_cifar_test.shape == (10000,)

print(
    "Development pixel range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

# ------------------------------------------------------------
# 4. LOAD SEED-2025 BASE AND META INDICES
# ------------------------------------------------------------

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(base_indices_path):
    raise FileNotFoundError(base_indices_path)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(meta_indices_path)

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)

assert len(
    np.intersect1d(
        base_indices,
        meta_indices
    )
) == 0

assert np.array_equal(
    np.sort(
        np.concatenate(
            [base_indices, meta_indices]
        )
    ),
    np.arange(50000)
)

X_base_all = X_cifar_development[
    base_indices
]

y_base_all = y_cifar_development[
    base_indices
]

X_meta = X_cifar_development[
    meta_indices
]

y_meta = y_cifar_development[
    meta_indices
]

# ------------------------------------------------------------
# 5. INTERNAL 90:10 VALIDATION SPLIT
# Taken only from the 40,000-image base partition
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

assert X_train_internal.shape == (36000, 32, 32, 3)
assert X_val_internal.shape == (4000, 32, 32, 3)
assert X_meta.shape == (10000, 32, 32, 3)

print("\nSplit verification")
print("------------------")
print("Base partition:", X_base_all.shape)
print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

print(
    "Training class counts:",
    np.bincount(
        y_train_internal,
        minlength=10
    )
)

print(
    "Validation class counts:",
    np.bincount(
        y_val_internal,
        minlength=10
    )
)

print(
    "Meta class counts:",
    np.bincount(
        y_meta,
        minlength=10
    )
)

# ------------------------------------------------------------
# 6. MATCHED CUSTOM CNN BUILDER
# ------------------------------------------------------------

def build_cnn_cifar():

    augmentation = tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="cnn_cifar_augmentation"
    )

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_cnn_input"
    )

    x = augmentation(inputs)

    # Block 1
    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.20)(x)

    # Block 2
    x = layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.30)(x)

    # Block 3
    x = layers.Conv2D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.40)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_cnn_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_CNN_Holdout_Seed2025"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# 7. OUTPUT PATHS
# ------------------------------------------------------------

best_model_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed2025_Best.keras"
)

latest_model_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed2025_Latest.keras"
)

history_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed2025_History.csv"
)

training_log_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed2025_Training_Log.csv"
)

meta_prob_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed2025_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed2025_Test_Probabilities.npy"
)

summary_path = os.path.join(
    cnn_dir,
    "CIFAR10_CNN_Seed2025_Summary.csv"
)

# ------------------------------------------------------------
# 8. VALIDATE EXISTING PROBABILITY FILES
# ------------------------------------------------------------

def valid_probability_matrix(
    array,
    expected_shape
):
    return (
        isinstance(array, np.ndarray)
        and array.shape == expected_shape
        and np.isfinite(array).all()
        and np.all(array >= 0.0)
        and np.all(array <= 1.0)
        and np.allclose(
            array.sum(axis=1),
            1.0,
            atol=1e-5
        )
    )


existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):
    try:
        cnn_meta_probs_seed2025 = np.load(
            meta_prob_path
        )

        cnn_test_probs_seed2025 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            valid_probability_matrix(
                cnn_meta_probs_seed2025,
                (10000, 10)
            )
            and valid_probability_matrix(
                cnn_test_probs_seed2025,
                (10000, 10)
            )
        )

    except Exception as error:
        print(
            "Saved probability validation failed:",
            error
        )

# ------------------------------------------------------------
# 9. TRAIN OR RELOAD
# ------------------------------------------------------------

if existing_outputs_valid:

    print(
        "\nValid CNN seed-2025 probability files "
        "already exist. Training skipped."
    )

else:

    clear_training_memory()
    set_all_seeds(HOLDOUT_SEED)

    if os.path.exists(best_model_path):

        print("\nLoading best checkpoint:")
        print(best_model_path)

        cnn_model_seed2025 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    elif os.path.exists(latest_model_path):

        print("\nLoading latest checkpoint:")
        print(latest_model_path)

        cnn_model_seed2025 = (
            tf.keras.models.load_model(
                latest_model_path,
                compile=True
            )
        )

    else:

        print(
            "\nBuilding a new Custom CNN "
            "for seed 2025."
        )

        cnn_model_seed2025 = build_cnn_cifar()

    print(
        "Model parameters:",
        f"{cnn_model_seed2025.count_params():,}"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=CIFAR_EARLY_STOPPING_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=latest_model_path,
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),

        CSVLogger(
            training_log_path,
            append=os.path.exists(
                training_log_path
            )
        )
    ]

    history = cnn_model_seed2025.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    new_history_df = pd.DataFrame(
        history.history
    )

    if (
        os.path.exists(history_path)
        and os.path.getsize(history_path) > 0
    ):
        old_history_df = pd.read_csv(
            history_path
        )

        combined_history_df = pd.concat(
            [
                old_history_df,
                new_history_df
            ],
            ignore_index=True
        )

    else:
        combined_history_df = new_history_df

    combined_history_df.to_csv(
        history_path,
        index=False
    )

    # Reload best checkpoint before predictions.
    clear_training_memory()

    cnn_model_seed2025 = (
        tf.keras.models.load_model(
            best_model_path,
            compile=True
        )
    )

    print(
        "\nGenerating seed-2025 meta probabilities..."
    )

    cnn_meta_probs_seed2025 = (
        cnn_model_seed2025.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    print(
        "\nGenerating seed-2025 test probabilities..."
    )

    cnn_test_probs_seed2025 = (
        cnn_model_seed2025.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    cnn_meta_probs_seed2025 = np.asarray(
        cnn_meta_probs_seed2025,
        dtype=np.float32
    )

    cnn_test_probs_seed2025 = np.asarray(
        cnn_test_probs_seed2025,
        dtype=np.float32
    )

    np.save(
        meta_prob_path,
        cnn_meta_probs_seed2025
    )

    np.save(
        test_prob_path,
        cnn_test_probs_seed2025
    )

# ------------------------------------------------------------
# 10. FINAL PROBABILITY VALIDATION
# ------------------------------------------------------------

if not valid_probability_matrix(
    cnn_meta_probs_seed2025,
    (10000, 10)
):
    raise ValueError(
        "Invalid CNN seed-2025 meta probabilities."
    )

if not valid_probability_matrix(
    cnn_test_probs_seed2025,
    (10000, 10)
):
    raise ValueError(
        "Invalid CNN seed-2025 test probabilities."
    )

# ------------------------------------------------------------
# 11. METRICS
# ------------------------------------------------------------

meta_predictions = np.argmax(
    cnn_meta_probs_seed2025,
    axis=1
)

test_predictions = np.argmax(
    cnn_test_probs_seed2025,
    axis=1
)

meta_accuracy = accuracy_score(
    y_meta,
    meta_predictions
)

meta_precision = precision_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_recall = recall_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_f1 = f1_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_auc = roc_auc_score(
    y_meta,
    cnn_meta_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

meta_loss = log_loss(
    y_meta,
    cnn_meta_probs_seed2025,
    labels=np.arange(10)
)

test_accuracy = accuracy_score(
    y_cifar_test,
    test_predictions
)

test_precision = precision_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_auc = roc_auc_score(
    y_cifar_test,
    cnn_test_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

test_loss = log_loss(
    y_cifar_test,
    cnn_test_probs_seed2025,
    labels=np.arange(10)
)

summary_df = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": HOLDOUT_SEED,
            "Model": "Custom CNN",
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Holdout_Samples": 10000,
            "Final_Test_Samples": 10000,
            "Meta_Accuracy": meta_accuracy,
            "Meta_Macro_Precision":
                meta_precision,
            "Meta_Macro_Recall":
                meta_recall,
            "Meta_Macro_F1":
                meta_f1,
            "Meta_ROC_AUC":
                meta_auc,
            "Meta_Log_Loss":
                meta_loss,
            "Test_Accuracy":
                test_accuracy,
            "Test_Macro_Precision":
                test_precision,
            "Test_Macro_Recall":
                test_recall,
            "Test_Macro_F1":
                test_f1,
            "Test_ROC_AUC":
                test_auc,
            "Test_Log_Loss":
                test_loss
        }
    ]
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df.round(6)
)

print(
    "\nCustom CNN seed 2025 completed successfully."
)

print(
    "Meta hold-out accuracy:",
    f"{meta_accuracy * 100:.4f}%"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.4f}%"
)

print(
    "Meta probability shape:",
    cnn_meta_probs_seed2025.shape
)

print(
    "Test probability shape:",
    cnn_test_probs_seed2025.shape
)

print("\nBest checkpoint:")
print(best_model_path)

print("\nMeta probabilities:")
print(meta_prob_path)

print("\nTest probabilities:")
print(test_prob_path)

print("\nSummary:")
print(summary_path)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# VGG16-STYLE MODEL — SEED 2025
# COMPLETE CHECKPOINTED CELL
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

HOLDOUT_SEED = 2025

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

vgg_dir = os.path.join(
    seed_dir,
    "VGG16_Style"
)

os.makedirs(vgg_dir, exist_ok=True)

# ------------------------------------------------------------
# 2. REPRODUCIBILITY
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()


set_all_seeds(HOLDOUT_SEED)

# ------------------------------------------------------------
# 3. CHECK CIFAR VARIABLES
# ------------------------------------------------------------

required_variables = [
    "X_cifar_development",
    "y_cifar_development",
    "X_cifar_test",
    "y_cifar_test"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Missing CIFAR variables: "
        f"{missing_variables}. "
        "Run the CIFAR loading cell first."
    )

X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

assert X_cifar_development.shape == (
    50000,
    32,
    32,
    3
)

assert X_cifar_test.shape == (
    10000,
    32,
    32,
    3
)

assert y_cifar_development.shape == (50000,)
assert y_cifar_test.shape == (10000,)

print(
    "Development pixel range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

# ------------------------------------------------------------
# 4. LOAD SEED-2025 SPLIT
# ------------------------------------------------------------

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(base_indices_path):
    raise FileNotFoundError(base_indices_path)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(meta_indices_path)

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)

assert len(
    np.intersect1d(
        base_indices,
        meta_indices
    )
) == 0

assert np.array_equal(
    np.sort(
        np.concatenate(
            [
                base_indices,
                meta_indices
            ]
        )
    ),
    np.arange(50000)
)

X_base_all = X_cifar_development[
    base_indices
]

y_base_all = y_cifar_development[
    base_indices
]

X_meta = X_cifar_development[
    meta_indices
]

y_meta = y_cifar_development[
    meta_indices
]

# ------------------------------------------------------------
# 5. INTERNAL VALIDATION SPLIT
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

assert X_train_internal.shape == (
    36000,
    32,
    32,
    3
)

assert X_val_internal.shape == (
    4000,
    32,
    32,
    3
)

assert X_meta.shape == (
    10000,
    32,
    32,
    3
)

print("\nSplit verification")
print("------------------")
print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

print(
    "Training class counts:",
    np.bincount(
        y_train_internal,
        minlength=10
    )
)

print(
    "Validation class counts:",
    np.bincount(
        y_val_internal,
        minlength=10
    )
)

print(
    "Meta class counts:",
    np.bincount(
        y_meta,
        minlength=10
    )
)

# ------------------------------------------------------------
# 6. MATCHED VGG16-STYLE MODEL
# ------------------------------------------------------------

def build_vgg16_cifar():

    augmentation = tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="vgg16_cifar_augmentation"
    )

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_vgg16_input"
    )

    x = augmentation(inputs)

    # Stage 1: 64 filters × 2
    for block_index in range(2):
        x = layers.Conv2D(
            64,
            kernel_size=3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal",
            name=f"stage1_conv{block_index + 1}"
        )(x)

        x = layers.BatchNormalization(
            name=f"stage1_bn{block_index + 1}"
        )(x)

    x = layers.MaxPooling2D(
        pool_size=2,
        name="stage1_pool"
    )(x)

    x = layers.Dropout(
        0.20,
        name="stage1_dropout"
    )(x)

    # Stage 2: 128 filters × 2
    for block_index in range(2):
        x = layers.Conv2D(
            128,
            kernel_size=3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal",
            name=f"stage2_conv{block_index + 1}"
        )(x)

        x = layers.BatchNormalization(
            name=f"stage2_bn{block_index + 1}"
        )(x)

    x = layers.MaxPooling2D(
        pool_size=2,
        name="stage2_pool"
    )(x)

    x = layers.Dropout(
        0.30,
        name="stage2_dropout"
    )(x)

    # Stage 3: 256 filters × 3
    for block_index in range(3):
        x = layers.Conv2D(
            256,
            kernel_size=3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal",
            name=f"stage3_conv{block_index + 1}"
        )(x)

        x = layers.BatchNormalization(
            name=f"stage3_bn{block_index + 1}"
        )(x)

    x = layers.MaxPooling2D(
        pool_size=2,
        name="stage3_pool"
    )(x)

    x = layers.Dropout(
        0.40,
        name="stage3_dropout"
    )(x)

    # Stage 4: 512 filters × 3
    for block_index in range(3):
        x = layers.Conv2D(
            512,
            kernel_size=3,
            padding="same",
            activation="relu",
            kernel_initializer="he_normal",
            name=f"stage4_conv{block_index + 1}"
        )(x)

        x = layers.BatchNormalization(
            name=f"stage4_bn{block_index + 1}"
        )(x)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pool"
    )(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal",
        name="dense_512"
    )(x)

    x = layers.BatchNormalization(
        name="dense_512_bn"
    )(x)

    x = layers.Dropout(
        0.50,
        name="dense_512_dropout"
    )(x)

    x = layers.Dense(
        256,
        activation="relu",
        kernel_initializer="he_normal",
        name="dense_256"
    )(x)

    x = layers.BatchNormalization(
        name="dense_256_bn"
    )(x)

    x = layers.Dropout(
        0.50,
        name="dense_256_dropout"
    )(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_vgg16_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_VGG16_Holdout_Seed2025"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# 7. OUTPUT PATHS
# ------------------------------------------------------------

best_model_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed2025_Best.keras"
)

latest_model_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed2025_Latest.keras"
)

history_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed2025_History.csv"
)

training_log_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed2025_Training_Log.csv"
)

meta_prob_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed2025_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed2025_Test_Probabilities.npy"
)

summary_path = os.path.join(
    vgg_dir,
    "CIFAR10_VGG16_Seed2025_Summary.csv"
)

# ------------------------------------------------------------
# 8. PROBABILITY VALIDATION
# ------------------------------------------------------------

def valid_probability_matrix(
    array,
    expected_shape
):

    return (
        isinstance(array, np.ndarray)
        and array.shape == expected_shape
        and np.isfinite(array).all()
        and np.all(array >= 0.0)
        and np.all(array <= 1.0)
        and np.allclose(
            array.sum(axis=1),
            1.0,
            atol=1e-5
        )
    )


existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):

    try:
        vgg_meta_probs_seed2025 = np.load(
            meta_prob_path
        )

        vgg_test_probs_seed2025 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            valid_probability_matrix(
                vgg_meta_probs_seed2025,
                (10000, 10)
            )
            and valid_probability_matrix(
                vgg_test_probs_seed2025,
                (10000, 10)
            )
        )

    except Exception as error:
        print(
            "Saved probability validation failed:",
            error
        )

# ------------------------------------------------------------
# 9. TRAIN OR RELOAD
# ------------------------------------------------------------

if existing_outputs_valid:

    print(
        "\nValid VGG16 seed-2025 probability "
        "files already exist. Training skipped."
    )

else:

    clear_training_memory()
    set_all_seeds(HOLDOUT_SEED)

    if os.path.exists(best_model_path):

        print("\nLoading best checkpoint:")
        print(best_model_path)

        vgg_model_seed2025 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    elif os.path.exists(latest_model_path):

        print("\nLoading latest checkpoint:")
        print(latest_model_path)

        vgg_model_seed2025 = (
            tf.keras.models.load_model(
                latest_model_path,
                compile=True
            )
        )

    else:

        print(
            "\nBuilding a new VGG16-style "
            "model for seed 2025."
        )

        vgg_model_seed2025 = (
            build_vgg16_cifar()
        )

    print(
        "Model parameters:",
        f"{vgg_model_seed2025.count_params():,}"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=CIFAR_EARLY_STOPPING_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=latest_model_path,
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),

        CSVLogger(
            training_log_path,
            append=os.path.exists(
                training_log_path
            )
        )
    ]

    history = vgg_model_seed2025.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    new_history_df = pd.DataFrame(
        history.history
    )

    if (
        os.path.exists(history_path)
        and os.path.getsize(history_path) > 0
    ):

        old_history_df = pd.read_csv(
            history_path
        )

        combined_history_df = pd.concat(
            [
                old_history_df,
                new_history_df
            ],
            ignore_index=True
        )

    else:
        combined_history_df = new_history_df

    combined_history_df.to_csv(
        history_path,
        index=False
    )

    # Reload the best validation checkpoint before prediction.
    clear_training_memory()

    if not os.path.exists(best_model_path):
        raise FileNotFoundError(
            "Best VGG16 checkpoint was not saved."
        )

    vgg_model_seed2025 = (
        tf.keras.models.load_model(
            best_model_path,
            compile=True
        )
    )

    print(
        "\nGenerating seed-2025 "
        "meta-holdout probabilities..."
    )

    vgg_meta_probs_seed2025 = (
        vgg_model_seed2025.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    print(
        "\nGenerating seed-2025 "
        "final-test probabilities..."
    )

    vgg_test_probs_seed2025 = (
        vgg_model_seed2025.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    vgg_meta_probs_seed2025 = np.asarray(
        vgg_meta_probs_seed2025,
        dtype=np.float32
    )

    vgg_test_probs_seed2025 = np.asarray(
        vgg_test_probs_seed2025,
        dtype=np.float32
    )

    np.save(
        meta_prob_path,
        vgg_meta_probs_seed2025
    )

    np.save(
        test_prob_path,
        vgg_test_probs_seed2025
    )

# ------------------------------------------------------------
# 10. FINAL VALIDATION
# ------------------------------------------------------------

if not valid_probability_matrix(
    vgg_meta_probs_seed2025,
    (10000, 10)
):
    raise ValueError(
        "Invalid VGG16 seed-2025 meta probabilities."
    )

if not valid_probability_matrix(
    vgg_test_probs_seed2025,
    (10000, 10)
):
    raise ValueError(
        "Invalid VGG16 seed-2025 test probabilities."
    )

# ------------------------------------------------------------
# 11. METRICS
# ------------------------------------------------------------

meta_predictions = np.argmax(
    vgg_meta_probs_seed2025,
    axis=1
)

test_predictions = np.argmax(
    vgg_test_probs_seed2025,
    axis=1
)

meta_accuracy = accuracy_score(
    y_meta,
    meta_predictions
)

meta_precision = precision_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_recall = recall_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_f1 = f1_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_auc = roc_auc_score(
    y_meta,
    vgg_meta_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

meta_loss = log_loss(
    y_meta,
    vgg_meta_probs_seed2025,
    labels=np.arange(10)
)

test_accuracy = accuracy_score(
    y_cifar_test,
    test_predictions
)

test_precision = precision_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_auc = roc_auc_score(
    y_cifar_test,
    vgg_test_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

test_loss = log_loss(
    y_cifar_test,
    vgg_test_probs_seed2025,
    labels=np.arange(10)
)

summary_df = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": HOLDOUT_SEED,
            "Model": "VGG16-style",
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Holdout_Samples": 10000,
            "Final_Test_Samples": 10000,
            "Meta_Accuracy": meta_accuracy,
            "Meta_Macro_Precision":
                meta_precision,
            "Meta_Macro_Recall":
                meta_recall,
            "Meta_Macro_F1":
                meta_f1,
            "Meta_ROC_AUC":
                meta_auc,
            "Meta_Log_Loss":
                meta_loss,
            "Test_Accuracy":
                test_accuracy,
            "Test_Macro_Precision":
                test_precision,
            "Test_Macro_Recall":
                test_recall,
            "Test_Macro_F1":
                test_f1,
            "Test_ROC_AUC":
                test_auc,
            "Test_Log_Loss":
                test_loss
        }
    ]
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df.round(6)
)

print(
    "\nVGG16-style seed 2025 completed successfully."
)

print(
    "Meta hold-out accuracy:",
    f"{meta_accuracy * 100:.4f}%"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.4f}%"
)

print(
    "Meta probability shape:",
    vgg_meta_probs_seed2025.shape
)

print(
    "Test probability shape:",
    vgg_test_probs_seed2025.shape
)

print("\nBest checkpoint:")
print(best_model_path)

print("\nMeta probabilities:")
print(meta_prob_path)

print("\nTest probabilities:")
print(test_prob_path)

print("\nSummary:")
print(summary_path)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# RESNET-STYLE MODEL — SEED 2025
# COMPLETE CHECKPOINTED CELL
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

HOLDOUT_SEED = 2025

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet_Style"
)

os.makedirs(
    resnet_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. REPRODUCIBILITY
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()


set_all_seeds(HOLDOUT_SEED)

# ------------------------------------------------------------
# 3. CHECK CIFAR DATA
# ------------------------------------------------------------

required_variables = [
    "X_cifar_development",
    "y_cifar_development",
    "X_cifar_test",
    "y_cifar_test"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Missing CIFAR variables: "
        f"{missing_variables}. "
        "Run the CIFAR data-loading cell first."
    )

X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

assert X_cifar_development.shape == (
    50000,
    32,
    32,
    3
)

assert X_cifar_test.shape == (
    10000,
    32,
    32,
    3
)

assert y_cifar_development.shape == (50000,)
assert y_cifar_test.shape == (10000,)

print(
    "Development pixel range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

# ------------------------------------------------------------
# 4. LOAD SEED-2025 BASE/META INDICES
# ------------------------------------------------------------

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(base_indices_path):
    raise FileNotFoundError(base_indices_path)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(meta_indices_path)

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)

assert len(
    np.intersect1d(
        base_indices,
        meta_indices
    )
) == 0

assert np.array_equal(
    np.sort(
        np.concatenate(
            [
                base_indices,
                meta_indices
            ]
        )
    ),
    np.arange(50000)
)

X_base_all = X_cifar_development[
    base_indices
]

y_base_all = y_cifar_development[
    base_indices
]

X_meta = X_cifar_development[
    meta_indices
]

y_meta = y_cifar_development[
    meta_indices
]

# ------------------------------------------------------------
# 5. INTERNAL VALIDATION SPLIT
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

assert X_train_internal.shape == (
    36000,
    32,
    32,
    3
)

assert X_val_internal.shape == (
    4000,
    32,
    32,
    3
)

assert X_meta.shape == (
    10000,
    32,
    32,
    3
)

print("\nSplit verification")
print("------------------")
print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

print(
    "Training class counts:",
    np.bincount(
        y_train_internal,
        minlength=10
    )
)

print(
    "Validation class counts:",
    np.bincount(
        y_val_internal,
        minlength=10
    )
)

print(
    "Meta class counts:",
    np.bincount(
        y_meta,
        minlength=10
    )
)

# ------------------------------------------------------------
# 6. DATA AUGMENTATION
# ------------------------------------------------------------

def build_resnet_augmentation():

    return tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="resnet_cifar_augmentation"
    )

# ------------------------------------------------------------
# 7. RESIDUAL BLOCK
# ------------------------------------------------------------

def cifar_residual_block(
    inputs,
    filters,
    stride=1,
    block_name="residual_block"
):

    shortcut = inputs

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv1"
    )(inputs)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{block_name}_relu1"
    )(x)

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv2"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn2"
    )(x)

    input_filters = int(
        inputs.shape[-1]
    )

    if (
        stride != 1
        or input_filters != filters
    ):

        shortcut = layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{block_name}_shortcut_conv"
        )(shortcut)

        shortcut = layers.BatchNormalization(
            name=f"{block_name}_shortcut_bn"
        )(shortcut)

    x = layers.Add(
        name=f"{block_name}_add"
    )(
        [
            x,
            shortcut
        ]
    )

    x = layers.Activation(
        "relu",
        name=f"{block_name}_output"
    )(x)

    return x

# ------------------------------------------------------------
# 8. MATCHED CIFAR RESNET BUILDER
# ------------------------------------------------------------

def build_resnet50_cifar():

    augmentation = (
        build_resnet_augmentation()
    )

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_resnet_input"
    )

    x = augmentation(inputs)

    # CIFAR stem: 3x3 convolution, no initial max pooling
    x = layers.Conv2D(
        filters=64,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv"
    )(x)

    x = layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = layers.Activation(
        "relu",
        name="stem_relu"
    )(x)

    # Stage 1
    x = cifar_residual_block(
        x,
        filters=64,
        stride=1,
        block_name="stage1_block1"
    )

    x = cifar_residual_block(
        x,
        filters=64,
        stride=1,
        block_name="stage1_block2"
    )

    x = layers.Dropout(
        0.20,
        name="stage1_dropout"
    )(x)

    # Stage 2
    x = cifar_residual_block(
        x,
        filters=128,
        stride=2,
        block_name="stage2_block1"
    )

    x = cifar_residual_block(
        x,
        filters=128,
        stride=1,
        block_name="stage2_block2"
    )

    x = layers.Dropout(
        0.30,
        name="stage2_dropout"
    )(x)

    # Stage 3
    x = cifar_residual_block(
        x,
        filters=256,
        stride=2,
        block_name="stage3_block1"
    )

    x = cifar_residual_block(
        x,
        filters=256,
        stride=1,
        block_name="stage3_block2"
    )

    x = layers.Dropout(
        0.40,
        name="stage3_dropout"
    )(x)

    # Stage 4
    x = cifar_residual_block(
        x,
        filters=512,
        stride=2,
        block_name="stage4_block1"
    )

    x = cifar_residual_block(
        x,
        filters=512,
        stride=1,
        block_name="stage4_block2"
    )

    x = layers.Dropout(
        0.40,
        name="stage4_dropout"
    )(x)

    # Classification head
    x = layers.GlobalAveragePooling2D(
        name="global_average_pool"
    )(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense"
    )(x)

    x = layers.BatchNormalization(
        name="classification_bn"
    )(x)

    x = layers.Dropout(
        0.50,
        name="classification_dropout"
    )(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_resnet_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_ResNet_Holdout_Seed2025"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# 9. OUTPUT PATHS
# ------------------------------------------------------------

best_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Best.keras"
)

latest_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Latest.keras"
)

history_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_History.csv"
)

training_log_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Training_Log.csv"
)

meta_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Test_Probabilities.npy"
)

summary_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Summary.csv"
)

# ------------------------------------------------------------
# 10. PROBABILITY VALIDATION
# ------------------------------------------------------------

def valid_probability_matrix(
    array,
    expected_shape
):

    return (
        isinstance(array, np.ndarray)
        and array.shape == expected_shape
        and np.isfinite(array).all()
        and np.all(array >= 0.0)
        and np.all(array <= 1.0)
        and np.allclose(
            array.sum(axis=1),
            1.0,
            atol=1e-5
        )
    )


existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):

    try:

        resnet_meta_probs_seed2025 = np.load(
            meta_prob_path
        )

        resnet_test_probs_seed2025 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            valid_probability_matrix(
                resnet_meta_probs_seed2025,
                (10000, 10)
            )
            and valid_probability_matrix(
                resnet_test_probs_seed2025,
                (10000, 10)
            )
        )

    except Exception as error:

        print(
            "Saved probability validation failed:",
            error
        )

# ------------------------------------------------------------
# 11. TRAIN OR RELOAD CHECKPOINT
# ------------------------------------------------------------

if existing_outputs_valid:

    print(
        "\nValid ResNet seed-2025 probability "
        "files already exist. Training skipped."
    )

else:

    clear_training_memory()
    set_all_seeds(HOLDOUT_SEED)

    if os.path.exists(best_model_path):

        print("\nLoading existing best checkpoint:")
        print(best_model_path)

        resnet_model_seed2025 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    elif os.path.exists(latest_model_path):

        print("\nLoading latest checkpoint:")
        print(latest_model_path)

        resnet_model_seed2025 = (
            tf.keras.models.load_model(
                latest_model_path,
                compile=True
            )
        )

    else:

        print(
            "\nBuilding a new matched "
            "ResNet-style model for seed 2025."
        )

        resnet_model_seed2025 = (
            build_resnet50_cifar()
        )

    print(
        "Model parameters:",
        f"{resnet_model_seed2025.count_params():,}"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=CIFAR_EARLY_STOPPING_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=latest_model_path,
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),

        CSVLogger(
            training_log_path,
            append=os.path.exists(
                training_log_path
            )
        )
    ]

    history = resnet_model_seed2025.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    new_history_df = pd.DataFrame(
        history.history
    )

    if (
        os.path.exists(history_path)
        and os.path.getsize(history_path) > 0
    ):

        old_history_df = pd.read_csv(
            history_path
        )

        combined_history_df = pd.concat(
            [
                old_history_df,
                new_history_df
            ],
            ignore_index=True
        )

    else:

        combined_history_df = (
            new_history_df
        )

    combined_history_df.to_csv(
        history_path,
        index=False
    )

    # Reload the best validation model before prediction
    clear_training_memory()

    if not os.path.exists(best_model_path):
        raise FileNotFoundError(
            "Best ResNet checkpoint was not saved."
        )

    resnet_model_seed2025 = (
        tf.keras.models.load_model(
            best_model_path,
            compile=True
        )
    )

    print(
        "\nGenerating seed-2025 "
        "meta-holdout probabilities..."
    )

    resnet_meta_probs_seed2025 = (
        resnet_model_seed2025.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    print(
        "\nGenerating seed-2025 "
        "final-test probabilities..."
    )

    resnet_test_probs_seed2025 = (
        resnet_model_seed2025.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    )

    resnet_meta_probs_seed2025 = np.asarray(
        resnet_meta_probs_seed2025,
        dtype=np.float32
    )

    resnet_test_probs_seed2025 = np.asarray(
        resnet_test_probs_seed2025,
        dtype=np.float32
    )

    np.save(
        meta_prob_path,
        resnet_meta_probs_seed2025
    )

    np.save(
        test_prob_path,
        resnet_test_probs_seed2025
    )

# ------------------------------------------------------------
# 12. FINAL PROBABILITY VALIDATION
# ------------------------------------------------------------

if not valid_probability_matrix(
    resnet_meta_probs_seed2025,
    (10000, 10)
):
    raise ValueError(
        "Invalid ResNet seed-2025 meta probabilities."
    )

if not valid_probability_matrix(
    resnet_test_probs_seed2025,
    (10000, 10)
):
    raise ValueError(
        "Invalid ResNet seed-2025 test probabilities."
    )

# ------------------------------------------------------------
# 13. METRICS
# ------------------------------------------------------------

meta_predictions = np.argmax(
    resnet_meta_probs_seed2025,
    axis=1
)

test_predictions = np.argmax(
    resnet_test_probs_seed2025,
    axis=1
)

meta_accuracy = accuracy_score(
    y_meta,
    meta_predictions
)

meta_precision = precision_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_recall = recall_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_f1 = f1_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_auc = roc_auc_score(
    y_meta,
    resnet_meta_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

meta_loss = log_loss(
    y_meta,
    resnet_meta_probs_seed2025,
    labels=np.arange(10)
)

test_accuracy = accuracy_score(
    y_cifar_test,
    test_predictions
)

test_precision = precision_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_auc = roc_auc_score(
    y_cifar_test,
    resnet_test_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

test_loss = log_loss(
    y_cifar_test,
    resnet_test_probs_seed2025,
    labels=np.arange(10)
)

summary_df = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": HOLDOUT_SEED,
            "Model": "ResNet-style",
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Holdout_Samples": 10000,
            "Final_Test_Samples": 10000,
            "Meta_Accuracy": meta_accuracy,
            "Meta_Macro_Precision":
                meta_precision,
            "Meta_Macro_Recall":
                meta_recall,
            "Meta_Macro_F1":
                meta_f1,
            "Meta_ROC_AUC":
                meta_auc,
            "Meta_Log_Loss":
                meta_loss,
            "Test_Accuracy":
                test_accuracy,
            "Test_Macro_Precision":
                test_precision,
            "Test_Macro_Recall":
                test_recall,
            "Test_Macro_F1":
                test_f1,
            "Test_ROC_AUC":
                test_auc,
            "Test_Log_Loss":
                test_loss
        }
    ]
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df.round(6)
)

print(
    "\nResNet-style seed 2025 completed successfully."
)

print(
    "Meta hold-out accuracy:",
    f"{meta_accuracy * 100:.4f}%"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.4f}%"
)

print(
    "Meta probability shape:",
    resnet_meta_probs_seed2025.shape
)

print(
    "Test probability shape:",
    resnet_test_probs_seed2025.shape
)

print("\nBest checkpoint:")
print(best_model_path)

print("\nMeta probabilities:")
print(meta_prob_path)

print("\nTest probabilities:")
print(test_prob_path)

print("\nSummary:")
print(summary_path)

In [ ]:
# ============================================================
# RELOAD CIFAR-10 DATA AFTER COLAB DISCONNECTION
# ============================================================

import os
import numpy as np

CIFAR_CACHE_DIR = (
    "/content/drive/MyDrive/CIFAR10_Cache"
)

development_images_path = os.path.join(
    CIFAR_CACHE_DIR,
    "X_cf_train.npy"
)

development_labels_path = os.path.join(
    CIFAR_CACHE_DIR,
    "y_cf_train.npy"
)

test_images_path = os.path.join(
    CIFAR_CACHE_DIR,
    "X_cf_test.npy"
)

test_labels_path = os.path.join(
    CIFAR_CACHE_DIR,
    "y_cf_test.npy"
)

required_files = [
    development_images_path,
    development_labels_path,
    test_images_path,
    test_labels_path
]

missing_files = [
    path
    for path in required_files
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "Missing CIFAR cache files:\n"
        + "\n".join(missing_files)
    )

X_cifar_development = np.load(
    development_images_path
).astype(np.float32)

y_cifar_development = np.load(
    development_labels_path
).reshape(-1).astype(np.int64)

X_cifar_test = np.load(
    test_images_path
).astype(np.float32)

y_cifar_test = np.load(
    test_labels_path
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

assert X_cifar_development.shape == (
    50000,
    32,
    32,
    3
)

assert y_cifar_development.shape == (
    50000,
)

assert X_cifar_test.shape == (
    10000,
    32,
    32,
    3
)

assert y_cifar_test.shape == (
    10000,
)

print(
    "Development images:",
    X_cifar_development.shape
)

print(
    "Development labels:",
    y_cifar_development.shape
)

print(
    "Test images:",
    X_cifar_test.shape
)

print(
    "Test labels:",
    y_cifar_test.shape
)

print(
    "Development pixel range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

In [ ]:
# ============================================================
# VERIFY SEED-2025 FILES BEFORE RESUMING RESNET
# ============================================================

import os
import numpy as np

HOLDOUT_SEED = 2025

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "Seed_2025"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet_Style"
)

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

best_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Best.keras"
)

latest_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Latest.keras"
)

meta_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Test_Probabilities.npy"
)

print("Seed folder:", os.path.exists(seed_dir))
print("Base indices:", os.path.exists(base_indices_path))
print("Meta indices:", os.path.exists(meta_indices_path))
print("Best checkpoint:", os.path.exists(best_model_path))
print("Latest checkpoint:", os.path.exists(latest_model_path))
print("Meta probabilities:", os.path.exists(meta_prob_path))
print("Test probabilities:", os.path.exists(test_prob_path))

if os.path.exists(base_indices_path):
    print(
        "Base indices shape:",
        np.load(base_indices_path).shape
    )

if os.path.exists(meta_indices_path):
    print(
        "Meta indices shape:",
        np.load(meta_indices_path).shape
    )

In [ ]:
import os

CIFAR_REPEATED_HOLDOUT_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison/"
    "Extended_Experiments/Repeated_Holdout/CIFAR10"
)

print("Path:", CIFAR_REPEATED_HOLDOUT_DIR)
print("Folder exists:", os.path.exists(CIFAR_REPEATED_HOLDOUT_DIR))

In [ ]:
import os
import numpy as np

CIFAR_CACHE_DIR = "/content/drive/MyDrive/CIFAR10_Cache"

X_cifar_development = np.load(
    os.path.join(CIFAR_CACHE_DIR, "X_cf_train.npy")
).astype(np.float32)

y_cifar_development = np.load(
    os.path.join(CIFAR_CACHE_DIR, "y_cf_train.npy")
).reshape(-1).astype(np.int64)

X_cifar_test = np.load(
    os.path.join(CIFAR_CACHE_DIR, "X_cf_test.npy")
).astype(np.float32)

y_cifar_test = np.load(
    os.path.join(CIFAR_CACHE_DIR, "y_cf_test.npy")
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

print("Development:", X_cifar_development.shape, y_cifar_development.shape)
print("Test:", X_cifar_test.shape, y_cifar_test.shape)
print(
    "Pixel range:",
    float(X_cifar_development.min()),
    "to",
    float(X_cifar_development.max())
)

In [ ]:
# ============================================================
# VERIFY SEED-2025 FILES BEFORE RESUMING RESNET
# ============================================================

import os
import numpy as np

HOLDOUT_SEED = 2025

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet_Style"
)

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

best_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Best.keras"
)

latest_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Latest.keras"
)

meta_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Test_Probabilities.npy"
)

history_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_History.csv"
)

training_log_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Training_Log.csv"
)

summary_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Summary.csv"
)

check_items = {
    "Seed folder": seed_dir,
    "ResNet folder": resnet_dir,
    "Base indices": base_indices_path,
    "Meta indices": meta_indices_path,
    "Best checkpoint": best_model_path,
    "Latest checkpoint": latest_model_path,
    "Meta probabilities": meta_prob_path,
    "Test probabilities": test_prob_path,
    "History": history_path,
    "Training log": training_log_path,
    "Summary": summary_path
}

print("=" * 70)
print("SEED-2025 RESNET RESTART CHECK")
print("=" * 70)

for item_name, item_path in check_items.items():
    print(
        f"{item_name:<22}:",
        os.path.exists(item_path)
    )

if not os.path.exists(base_indices_path):
    raise FileNotFoundError(
        f"Base indices are missing:\n{base_indices_path}"
    )

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(
        f"Meta indices are missing:\n{meta_indices_path}"
    )

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

print("\nBase indices shape:", base_indices.shape)
print("Meta indices shape:", meta_indices.shape)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)

overlap_count = len(
    np.intersect1d(
        base_indices,
        meta_indices
    )
)

print("Base/meta overlap:", overlap_count)

assert overlap_count == 0

# ------------------------------------------------------------
# Check saved probability files when present
# ------------------------------------------------------------

def inspect_probability_file(
    file_path,
    label
):
    if not os.path.exists(file_path):
        print(f"{label}: not found")
        return False

    try:
        probabilities = np.load(
            file_path
        )

        valid = (
            probabilities.shape == (10000, 10)
            and np.isfinite(probabilities).all()
            and np.all(probabilities >= 0.0)
            and np.all(probabilities <= 1.0)
            and np.allclose(
                probabilities.sum(axis=1),
                1.0,
                atol=1e-5
            )
        )

        print(
            f"{label}:",
            probabilities.shape,
            "| valid:",
            valid
        )

        return valid

    except Exception as error:
        print(
            f"{label}: could not be loaded:",
            error
        )
        return False


meta_probs_valid = inspect_probability_file(
    meta_prob_path,
    "Meta probabilities"
)

test_probs_valid = inspect_probability_file(
    test_prob_path,
    "Test probabilities"
)

print("\n" + "=" * 70)

if meta_probs_valid and test_probs_valid:
    print(
        "STATUS: Training is already complete. "
        "The ResNet cell will skip retraining."
    )

elif os.path.exists(best_model_path):
    print(
        "STATUS: Best checkpoint exists. "
        "The ResNet cell will load it and resume."
    )

elif os.path.exists(latest_model_path):
    print(
        "STATUS: Latest checkpoint exists. "
        "The ResNet cell will load it and resume."
    )

else:
    print(
        "STATUS: No checkpoint or complete probability files found. "
        "Training will restart from the beginning."
    )

print("=" * 70)

In [ ]:
import os

resnet_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "Seed_2025",
    "ResNet_Style"
)

os.makedirs(
    resnet_dir,
    exist_ok=True
)

print("ResNet folder:", resnet_dir)
print("Folder exists:", os.path.exists(resnet_dir))

In [ ]:
# ============================================================
# CELL 1: SETUP + DATA SPLIT + RESNET MODEL — SEED 2025
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

HOLDOUT_SEED = 2025
CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

CIFAR_REPEATED_HOLDOUT_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison/"
    "Extended_Experiments/Repeated_Holdout/CIFAR10"
)

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "Seed_2025"
)

resnet_dir = os.path.join(
    seed_dir,
    "ResNet_Style"
)

os.makedirs(
    resnet_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()


set_all_seeds(HOLDOUT_SEED)

# ------------------------------------------------------------
# Verify CIFAR arrays
# ------------------------------------------------------------

required_variables = [
    "X_cifar_development",
    "y_cifar_development",
    "X_cifar_test",
    "y_cifar_test"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Missing variables: {missing_variables}. "
        "Run the CIFAR reload cell first."
    )

X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

assert X_cifar_development.shape == (50000, 32, 32, 3)
assert X_cifar_test.shape == (10000, 32, 32, 3)
assert y_cifar_development.shape == (50000,)
assert y_cifar_test.shape == (10000,)

# ------------------------------------------------------------
# Load saved seed-2025 indices
# ------------------------------------------------------------

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(base_indices_path):
    raise FileNotFoundError(base_indices_path)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(meta_indices_path)

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)
assert len(np.intersect1d(base_indices, meta_indices)) == 0

X_base_all = X_cifar_development[base_indices]
y_base_all = y_cifar_development[base_indices]

X_meta = X_cifar_development[meta_indices]
y_meta = y_cifar_development[meta_indices]

# ------------------------------------------------------------
# Internal validation split
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[train_indices_local]
y_train_internal = y_base_all[train_indices_local]

X_val_internal = X_base_all[val_indices_local]
y_val_internal = y_base_all[val_indices_local]

assert X_train_internal.shape == (36000, 32, 32, 3)
assert X_val_internal.shape == (4000, 32, 32, 3)
assert X_meta.shape == (10000, 32, 32, 3)

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

# ------------------------------------------------------------
# Augmentation
# ------------------------------------------------------------

def build_resnet_augmentation():

    return tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal"),

            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),

            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="resnet_cifar_augmentation"
    )

# ------------------------------------------------------------
# Residual block
# ------------------------------------------------------------

def cifar_residual_block(
    inputs,
    filters,
    stride=1,
    block_name="residual_block"
):

    shortcut = inputs

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv1"
    )(inputs)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{block_name}_relu1"
    )(x)

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv2"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn2"
    )(x)

    input_filters = int(
        inputs.shape[-1]
    )

    if stride != 1 or input_filters != filters:

        shortcut = layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{block_name}_shortcut_conv"
        )(shortcut)

        shortcut = layers.BatchNormalization(
            name=f"{block_name}_shortcut_bn"
        )(shortcut)

    x = layers.Add(
        name=f"{block_name}_add"
    )([x, shortcut])

    x = layers.Activation(
        "relu",
        name=f"{block_name}_output"
    )(x)

    return x

# ------------------------------------------------------------
# CIFAR ResNet builder
# ------------------------------------------------------------

def build_resnet50_cifar():

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_resnet_input"
    )

    x = build_resnet_augmentation()(inputs)

    x = layers.Conv2D(
        filters=64,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv"
    )(x)

    x = layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = layers.Activation(
        "relu",
        name="stem_relu"
    )(x)

    # Stage 1
    x = cifar_residual_block(
        x, 64, stride=1,
        block_name="stage1_block1"
    )

    x = cifar_residual_block(
        x, 64, stride=1,
        block_name="stage1_block2"
    )

    x = layers.Dropout(
        0.20,
        name="stage1_dropout"
    )(x)

    # Stage 2
    x = cifar_residual_block(
        x, 128, stride=2,
        block_name="stage2_block1"
    )

    x = cifar_residual_block(
        x, 128, stride=1,
        block_name="stage2_block2"
    )

    x = layers.Dropout(
        0.30,
        name="stage2_dropout"
    )(x)

    # Stage 3
    x = cifar_residual_block(
        x, 256, stride=2,
        block_name="stage3_block1"
    )

    x = cifar_residual_block(
        x, 256, stride=1,
        block_name="stage3_block2"
    )

    x = layers.Dropout(
        0.40,
        name="stage3_dropout"
    )(x)

    # Stage 4
    x = cifar_residual_block(
        x, 512, stride=2,
        block_name="stage4_block1"
    )

    x = cifar_residual_block(
        x, 512, stride=1,
        block_name="stage4_block2"
    )

    x = layers.Dropout(
        0.40,
        name="stage4_dropout"
    )(x)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pool"
    )(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense"
    )(x)

    x = layers.BatchNormalization(
        name="classification_bn"
    )(x)

    x = layers.Dropout(
        0.50,
        name="classification_dropout"
    )(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_resnet_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_ResNet_Holdout_Seed2025"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("Cell 1 completed successfully.")

In [ ]:
# ============================================================
# CELL 2: TRAIN / RESUME RESNET — SEED 2025
# ============================================================

best_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Best.keras"
)

latest_model_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Latest.keras"
)

history_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_History.csv"
)

training_log_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Training_Log.csv"
)

meta_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Test_Probabilities.npy"
)

summary_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Summary.csv"
)


def valid_probability_matrix(
    array,
    expected_shape
):
    return (
        isinstance(array, np.ndarray)
        and array.shape == expected_shape
        and np.isfinite(array).all()
        and np.all(array >= 0.0)
        and np.all(array <= 1.0)
        and np.allclose(
            array.sum(axis=1),
            1.0,
            atol=1e-5
        )
    )


existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):
    try:
        resnet_meta_probs_seed2025 = np.load(
            meta_prob_path
        )

        resnet_test_probs_seed2025 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            valid_probability_matrix(
                resnet_meta_probs_seed2025,
                (10000, 10)
            )
            and valid_probability_matrix(
                resnet_test_probs_seed2025,
                (10000, 10)
            )
        )

    except Exception:
        existing_outputs_valid = False


if existing_outputs_valid:

    print(
        "Valid seed-2025 ResNet probabilities "
        "already exist. Training skipped."
    )

else:

    clear_training_memory()
    set_all_seeds(HOLDOUT_SEED)

    if os.path.exists(best_model_path):

        print("Loading best checkpoint:")
        print(best_model_path)

        resnet_model_seed2025 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    elif os.path.exists(latest_model_path):

        print("Loading latest checkpoint:")
        print(latest_model_path)

        resnet_model_seed2025 = (
            tf.keras.models.load_model(
                latest_model_path,
                compile=True
            )
        )

    else:

        print(
            "No checkpoint found. "
            "Building model from scratch."
        )

        resnet_model_seed2025 = (
            build_resnet50_cifar()
        )

    print(
        "Model parameters:",
        f"{resnet_model_seed2025.count_params():,}"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=CIFAR_EARLY_STOPPING_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=latest_model_path,
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),

        CSVLogger(
            training_log_path,
            append=os.path.exists(
                training_log_path
            )
        )
    ]

    history = resnet_model_seed2025.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    new_history_df = pd.DataFrame(
        history.history
    )

    if (
        os.path.exists(history_path)
        and os.path.getsize(history_path) > 0
    ):
        old_history_df = pd.read_csv(
            history_path
        )

        history_df = pd.concat(
            [
                old_history_df,
                new_history_df
            ],
            ignore_index=True
        )

    else:
        history_df = new_history_df

    history_df.to_csv(
        history_path,
        index=False
    )

    if not os.path.exists(best_model_path):
        raise FileNotFoundError(
            "Best checkpoint was not created."
        )

    clear_training_memory()

    resnet_model_seed2025 = (
        tf.keras.models.load_model(
            best_model_path,
            compile=True
        )
    )

    print(
        "Generating meta-holdout probabilities..."
    )

    resnet_meta_probs_seed2025 = (
        resnet_model_seed2025.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    ).astype(np.float32)

    print(
        "Generating final-test probabilities..."
    )

    resnet_test_probs_seed2025 = (
        resnet_model_seed2025.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    ).astype(np.float32)

    np.save(
        meta_prob_path,
        resnet_meta_probs_seed2025
    )

    np.save(
        test_prob_path,
        resnet_test_probs_seed2025
    )

assert valid_probability_matrix(
    resnet_meta_probs_seed2025,
    (10000, 10)
)

assert valid_probability_matrix(
    resnet_test_probs_seed2025,
    (10000, 10)
)

print("Meta probabilities:", resnet_meta_probs_seed2025.shape)
print("Test probabilities:", resnet_test_probs_seed2025.shape)
print("Best checkpoint:", best_model_path)

In [ ]:
# ============================================================
# CELL 3: RESNET SEED-2025 METRICS AND SUMMARY
# ============================================================

meta_predictions = np.argmax(
    resnet_meta_probs_seed2025,
    axis=1
)

test_predictions = np.argmax(
    resnet_test_probs_seed2025,
    axis=1
)

meta_accuracy = accuracy_score(
    y_meta,
    meta_predictions
)

meta_precision = precision_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_recall = recall_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_f1 = f1_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_auc = roc_auc_score(
    y_meta,
    resnet_meta_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

meta_loss = log_loss(
    y_meta,
    resnet_meta_probs_seed2025,
    labels=np.arange(10)
)

test_accuracy = accuracy_score(
    y_cifar_test,
    test_predictions
)

test_precision = precision_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_auc = roc_auc_score(
    y_cifar_test,
    resnet_test_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

test_loss = log_loss(
    y_cifar_test,
    resnet_test_probs_seed2025,
    labels=np.arange(10)
)

summary_df = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": 2025,
            "Model": "ResNet-style",
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Holdout_Samples": 10000,
            "Final_Test_Samples": 10000,
            "Meta_Accuracy": meta_accuracy,
            "Meta_Macro_Precision": meta_precision,
            "Meta_Macro_Recall": meta_recall,
            "Meta_Macro_F1": meta_f1,
            "Meta_ROC_AUC": meta_auc,
            "Meta_Log_Loss": meta_loss,
            "Test_Accuracy": test_accuracy,
            "Test_Macro_Precision": test_precision,
            "Test_Macro_Recall": test_recall,
            "Test_Macro_F1": test_f1,
            "Test_ROC_AUC": test_auc,
            "Test_Log_Loss": test_loss
        }
    ]
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df.round(6)
)

print(
    "Meta hold-out accuracy:",
    f"{meta_accuracy * 100:.4f}%"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.4f}%"
)

print("Summary saved to:")
print(summary_path)

In [ ]:
import os
import numpy as np

resnet_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "Seed_2025",
    "ResNet_Style"
)

files_to_check = {
    "Best checkpoint":
        "CIFAR10_ResNet50_Seed2025_Best.keras",
    "Meta probabilities":
        "CIFAR10_ResNet50_Seed2025_Meta_Probabilities.npy",
    "Test probabilities":
        "CIFAR10_ResNet50_Seed2025_Test_Probabilities.npy",
    "Summary":
        "CIFAR10_ResNet50_Seed2025_Summary.csv"
}

for label, filename in files_to_check.items():
    path = os.path.join(
        resnet_dir,
        filename
    )
    print(
        f"{label:<20}:",
        os.path.exists(path)
    )

meta_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Meta_Probabilities.npy"
)

test_path = os.path.join(
    resnet_dir,
    "CIFAR10_ResNet50_Seed2025_Test_Probabilities.npy"
)

if os.path.exists(meta_path):
    print(
        "Meta shape:",
        np.load(meta_path).shape
    )

if os.path.exists(test_path):
    print(
        "Test shape:",
        np.load(test_path).shape
    )

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# EFFICIENTNET-STYLE MODEL — SEED 2025
# COMPLETE CHECKPOINTED CELL
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

HOLDOUT_SEED = 2025

CIFAR_EPOCHS = 30
CIFAR_BATCH_SIZE = 64
CIFAR_LEARNING_RATE = 1e-3
CIFAR_EARLY_STOPPING_PATIENCE = 6

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "Seed_2025"
)

efficientnet_dir = os.path.join(
    seed_dir,
    "EfficientNet_Style"
)

os.makedirs(
    efficientnet_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. REPRODUCIBILITY
# ------------------------------------------------------------

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def clear_training_memory():
    tf.keras.backend.clear_session()
    gc.collect()


set_all_seeds(HOLDOUT_SEED)

# ------------------------------------------------------------
# 3. VERIFY CIFAR ARRAYS
# ------------------------------------------------------------

required_variables = [
    "X_cifar_development",
    "y_cifar_development",
    "X_cifar_test",
    "y_cifar_test"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Missing CIFAR variables: "
        f"{missing_variables}. "
        "Run the CIFAR reload cell first."
    )

X_cifar_development = np.asarray(
    X_cifar_development,
    dtype=np.float32
)

X_cifar_test = np.asarray(
    X_cifar_test,
    dtype=np.float32
)

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

if X_cifar_development.max() > 1.5:
    X_cifar_development /= 255.0

if X_cifar_test.max() > 1.5:
    X_cifar_test /= 255.0

assert X_cifar_development.shape == (50000, 32, 32, 3)
assert y_cifar_development.shape == (50000,)
assert X_cifar_test.shape == (10000, 32, 32, 3)
assert y_cifar_test.shape == (10000,)

# ------------------------------------------------------------
# 4. LOAD SEED-2025 SPLIT
# ------------------------------------------------------------

base_indices_path = os.path.join(
    seed_dir,
    "Base_Training_Indices.npy"
)

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(base_indices_path):
    raise FileNotFoundError(base_indices_path)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(meta_indices_path)

base_indices = np.load(
    base_indices_path
).reshape(-1).astype(np.int64)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert base_indices.shape == (40000,)
assert meta_indices.shape == (10000,)

assert len(
    np.intersect1d(
        base_indices,
        meta_indices
    )
) == 0

X_base_all = X_cifar_development[
    base_indices
]

y_base_all = y_cifar_development[
    base_indices
]

X_meta = X_cifar_development[
    meta_indices
]

y_meta = y_cifar_development[
    meta_indices
]

# ------------------------------------------------------------
# 5. INTERNAL VALIDATION SPLIT
# ------------------------------------------------------------

local_indices = np.arange(
    len(y_base_all)
)

train_indices_local, val_indices_local = train_test_split(
    local_indices,
    test_size=0.10,
    stratify=y_base_all,
    random_state=HOLDOUT_SEED
)

X_train_internal = X_base_all[
    train_indices_local
]

y_train_internal = y_base_all[
    train_indices_local
]

X_val_internal = X_base_all[
    val_indices_local
]

y_val_internal = y_base_all[
    val_indices_local
]

assert X_train_internal.shape == (36000, 32, 32, 3)
assert X_val_internal.shape == (4000, 32, 32, 3)
assert X_meta.shape == (10000, 32, 32, 3)

print("Internal training:", X_train_internal.shape)
print("Internal validation:", X_val_internal.shape)
print("Meta hold-out:", X_meta.shape)
print("Final test:", X_cifar_test.shape)

# ------------------------------------------------------------
# 6. AUGMENTATION
# ------------------------------------------------------------

def build_efficientnet_augmentation():

    return tf.keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal"
            ),

            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),

            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name="efficientnet_cifar_augmentation"
    )

# ------------------------------------------------------------
# 7. SQUEEZE-AND-EXCITATION BLOCK
# ------------------------------------------------------------

def squeeze_excite_block(
    inputs,
    se_ratio=0.25,
    name="se"
):

    filters = int(
        inputs.shape[-1]
    )

    reduced_filters = max(
        1,
        int(filters * se_ratio)
    )

    x = layers.GlobalAveragePooling2D(
        keepdims=True,
        name=f"{name}_gap"
    )(inputs)

    x = layers.Conv2D(
        reduced_filters,
        kernel_size=1,
        activation="swish",
        padding="same",
        name=f"{name}_reduce"
    )(x)

    x = layers.Conv2D(
        filters,
        kernel_size=1,
        activation="sigmoid",
        padding="same",
        name=f"{name}_expand"
    )(x)

    return layers.Multiply(
        name=f"{name}_scale"
    )([inputs, x])

# ------------------------------------------------------------
# 8. MBConv BLOCK
# ------------------------------------------------------------

def mbconv_block(
    inputs,
    output_filters,
    kernel_size,
    stride,
    expansion_ratio,
    dropout_rate=0.0,
    block_name="mbconv"
):

    input_filters = int(
        inputs.shape[-1]
    )

    expanded_filters = (
        input_filters * expansion_ratio
    )

    x = inputs

    if expansion_ratio != 1:

        x = layers.Conv2D(
            expanded_filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            name=f"{block_name}_expand_conv"
        )(x)

        x = layers.BatchNormalization(
            name=f"{block_name}_expand_bn"
        )(x)

        x = layers.Activation(
            "swish",
            name=f"{block_name}_expand_activation"
        )(x)

    x = layers.DepthwiseConv2D(
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        use_bias=False,
        name=f"{block_name}_depthwise"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_depthwise_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name=f"{block_name}_depthwise_activation"
    )(x)

    x = squeeze_excite_block(
        x,
        se_ratio=0.25,
        name=f"{block_name}_se"
    )

    x = layers.Conv2D(
        output_filters,
        kernel_size=1,
        padding="same",
        use_bias=False,
        name=f"{block_name}_project_conv"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_project_bn"
    )(x)

    if (
        stride == 1
        and input_filters == output_filters
    ):

        if dropout_rate > 0:

            x = layers.Dropout(
                dropout_rate,
                name=f"{block_name}_dropout"
            )(x)

        x = layers.Add(
            name=f"{block_name}_add"
        )([inputs, x])

    return x

# ------------------------------------------------------------
# 9. MATCHED EFFICIENTNET-STYLE BUILDER
# ------------------------------------------------------------

def build_efficientnet_cifar():

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_efficientnet_input"
    )

    x = build_efficientnet_augmentation()(
        inputs
    )

    # Stem
    x = layers.Conv2D(
        32,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv"
    )(x)

    x = layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name="stem_activation"
    )(x)

    # MBConv sequence
    x = mbconv_block(
        x,
        output_filters=16,
        kernel_size=3,
        stride=1,
        expansion_ratio=1,
        dropout_rate=0.05,
        block_name="block1"
    )

    x = mbconv_block(
        x,
        output_filters=24,
        kernel_size=3,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.05,
        block_name="block2a"
    )

    x = mbconv_block(
        x,
        output_filters=24,
        kernel_size=3,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.05,
        block_name="block2b"
    )

    x = mbconv_block(
        x,
        output_filters=40,
        kernel_size=5,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.10,
        block_name="block3a"
    )

    x = mbconv_block(
        x,
        output_filters=40,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.10,
        block_name="block3b"
    )

    x = mbconv_block(
        x,
        output_filters=80,
        kernel_size=3,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.15,
        block_name="block4a"
    )

    x = mbconv_block(
        x,
        output_filters=80,
        kernel_size=3,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.15,
        block_name="block4b"
    )

    x = mbconv_block(
        x,
        output_filters=112,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.20,
        block_name="block5"
    )

    x = mbconv_block(
        x,
        output_filters=192,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.20,
        block_name="block6"
    )

    # Head
    x = layers.Conv2D(
        1280,
        kernel_size=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="head_conv"
    )(x)

    x = layers.BatchNormalization(
        name="head_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name="head_activation"
    )(x)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.Dropout(
        0.50,
        name="head_dropout"
    )(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_efficientnet_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_EfficientNet_Holdout_Seed2025"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=CIFAR_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ------------------------------------------------------------
# 10. OUTPUT PATHS
# ------------------------------------------------------------

best_model_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed2025_Best.keras"
)

latest_model_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed2025_Latest.keras"
)

history_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed2025_History.csv"
)

training_log_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed2025_Training_Log.csv"
)

meta_prob_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed2025_Meta_Probabilities.npy"
)

test_prob_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed2025_Test_Probabilities.npy"
)

summary_path = os.path.join(
    efficientnet_dir,
    "CIFAR10_EfficientNetB0_Seed2025_Summary.csv"
)

# ------------------------------------------------------------
# 11. PROBABILITY VALIDATION
# ------------------------------------------------------------

def valid_probability_matrix(
    array,
    expected_shape
):

    return (
        isinstance(array, np.ndarray)
        and array.shape == expected_shape
        and np.isfinite(array).all()
        and np.all(array >= 0.0)
        and np.all(array <= 1.0)
        and np.allclose(
            array.sum(axis=1),
            1.0,
            atol=1e-5
        )
    )


existing_outputs_valid = False

if (
    os.path.exists(meta_prob_path)
    and os.path.exists(test_prob_path)
):

    try:

        efficientnet_meta_probs_seed2025 = np.load(
            meta_prob_path
        )

        efficientnet_test_probs_seed2025 = np.load(
            test_prob_path
        )

        existing_outputs_valid = (
            valid_probability_matrix(
                efficientnet_meta_probs_seed2025,
                (10000, 10)
            )
            and valid_probability_matrix(
                efficientnet_test_probs_seed2025,
                (10000, 10)
            )
        )

    except Exception as error:

        print(
            "Saved probability validation failed:",
            error
        )

# ------------------------------------------------------------
# 12. TRAIN OR RELOAD
# ------------------------------------------------------------

if existing_outputs_valid:

    print(
        "Valid EfficientNet seed-2025 probability "
        "files already exist. Training skipped."
    )

else:

    clear_training_memory()
    set_all_seeds(HOLDOUT_SEED)

    if os.path.exists(best_model_path):

        print("Loading best checkpoint:")
        print(best_model_path)

        efficientnet_model_seed2025 = (
            tf.keras.models.load_model(
                best_model_path,
                compile=True
            )
        )

    elif os.path.exists(latest_model_path):

        print("Loading latest checkpoint:")
        print(latest_model_path)

        efficientnet_model_seed2025 = (
            tf.keras.models.load_model(
                latest_model_path,
                compile=True
            )
        )

    else:

        print(
            "No EfficientNet checkpoint found. "
            "Building from scratch."
        )

        efficientnet_model_seed2025 = (
            build_efficientnet_cifar()
        )

    print(
        "Model parameters:",
        f"{efficientnet_model_seed2025.count_params():,}"
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=CIFAR_EARLY_STOPPING_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=best_model_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        ModelCheckpoint(
            filepath=latest_model_path,
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            verbose=0
        ),

        CSVLogger(
            training_log_path,
            append=os.path.exists(
                training_log_path
            )
        )
    ]

    history = efficientnet_model_seed2025.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=CIFAR_EPOCHS,
        batch_size=CIFAR_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    new_history_df = pd.DataFrame(
        history.history
    )

    if (
        os.path.exists(history_path)
        and os.path.getsize(history_path) > 0
    ):

        old_history_df = pd.read_csv(
            history_path
        )

        history_df = pd.concat(
            [
                old_history_df,
                new_history_df
            ],
            ignore_index=True
        )

    else:

        history_df = new_history_df

    history_df.to_csv(
        history_path,
        index=False
    )

    if not os.path.exists(best_model_path):
        raise FileNotFoundError(
            "Best EfficientNet checkpoint was not created."
        )

    clear_training_memory()

    efficientnet_model_seed2025 = (
        tf.keras.models.load_model(
            best_model_path,
            compile=True
        )
    )

    print(
        "Generating seed-2025 meta probabilities..."
    )

    efficientnet_meta_probs_seed2025 = (
        efficientnet_model_seed2025.predict(
            X_meta,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    ).astype(np.float32)

    print(
        "Generating seed-2025 test probabilities..."
    )

    efficientnet_test_probs_seed2025 = (
        efficientnet_model_seed2025.predict(
            X_cifar_test,
            batch_size=CIFAR_BATCH_SIZE,
            verbose=1
        )
    ).astype(np.float32)

    np.save(
        meta_prob_path,
        efficientnet_meta_probs_seed2025
    )

    np.save(
        test_prob_path,
        efficientnet_test_probs_seed2025
    )

# ------------------------------------------------------------
# 13. FINAL VALIDATION
# ------------------------------------------------------------

assert valid_probability_matrix(
    efficientnet_meta_probs_seed2025,
    (10000, 10)
)

assert valid_probability_matrix(
    efficientnet_test_probs_seed2025,
    (10000, 10)
)

# ------------------------------------------------------------
# 14. METRICS AND SUMMARY
# ------------------------------------------------------------

meta_predictions = np.argmax(
    efficientnet_meta_probs_seed2025,
    axis=1
)

test_predictions = np.argmax(
    efficientnet_test_probs_seed2025,
    axis=1
)

meta_accuracy = accuracy_score(
    y_meta,
    meta_predictions
)

meta_precision = precision_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_recall = recall_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_f1 = f1_score(
    y_meta,
    meta_predictions,
    average="macro",
    zero_division=0
)

meta_auc = roc_auc_score(
    y_meta,
    efficientnet_meta_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

meta_loss = log_loss(
    y_meta,
    efficientnet_meta_probs_seed2025,
    labels=np.arange(10)
)

test_accuracy = accuracy_score(
    y_cifar_test,
    test_predictions
)

test_precision = precision_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_cifar_test,
    test_predictions,
    average="macro",
    zero_division=0
)

test_auc = roc_auc_score(
    y_cifar_test,
    efficientnet_test_probs_seed2025,
    multi_class="ovr",
    average="macro",
    labels=np.arange(10)
)

test_loss = log_loss(
    y_cifar_test,
    efficientnet_test_probs_seed2025,
    labels=np.arange(10)
)

summary_df = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": 2025,
            "Model": "EfficientNet-style",
            "Base_Training_Samples": 40000,
            "Internal_Training_Samples": 36000,
            "Internal_Validation_Samples": 4000,
            "Meta_Holdout_Samples": 10000,
            "Final_Test_Samples": 10000,
            "Meta_Accuracy": meta_accuracy,
            "Meta_Macro_Precision":
                meta_precision,
            "Meta_Macro_Recall":
                meta_recall,
            "Meta_Macro_F1":
                meta_f1,
            "Meta_ROC_AUC":
                meta_auc,
            "Meta_Log_Loss":
                meta_loss,
            "Test_Accuracy":
                test_accuracy,
            "Test_Macro_Precision":
                test_precision,
            "Test_Macro_Recall":
                test_recall,
            "Test_Macro_F1":
                test_f1,
            "Test_ROC_AUC":
                test_auc,
            "Test_Log_Loss":
                test_loss
        }
    ]
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df.round(6)
)

print(
    "\nEfficientNet-style seed 2025 completed."
)

print(
    "Meta hold-out accuracy:",
    f"{meta_accuracy * 100:.4f}%"
)

print(
    "Final test accuracy:",
    f"{test_accuracy * 100:.4f}%"
)

print(
    "Meta probability shape:",
    efficientnet_meta_probs_seed2025.shape
)

print(
    "Test probability shape:",
    efficientnet_test_probs_seed2025.shape
)

print("\nBest checkpoint:")
print(best_model_path)

print("\nSummary:")
print(summary_path)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# META-LEARNERS + SOFT VOTING — SEED 2025
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

try:
    from lightgbm import LGBMClassifier
except ImportError:
    !pip -q install lightgbm
    from lightgbm import LGBMClassifier

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

HOLDOUT_SEED = 2025
NUM_CLASSES = 10

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

seed_dir = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    f"Seed_{HOLDOUT_SEED}"
)

meta_output_dir = os.path.join(
    seed_dir,
    "Meta_Learners"
)

os.makedirs(
    meta_output_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Verify labels
# ------------------------------------------------------------

if "y_cifar_development" not in globals():
    raise NameError(
        "y_cifar_development is missing. "
        "Run the CIFAR reload cell first."
    )

if "y_cifar_test" not in globals():
    raise NameError(
        "y_cifar_test is missing. "
        "Run the CIFAR reload cell first."
    )

y_cifar_development = np.asarray(
    y_cifar_development
).reshape(-1).astype(np.int64)

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

assert y_cifar_development.shape == (50000,)
assert y_cifar_test.shape == (10000,)

# ------------------------------------------------------------
# 3. Load seed-2025 meta labels
# ------------------------------------------------------------

meta_indices_path = os.path.join(
    seed_dir,
    "Meta_Holdout_Indices.npy"
)

if not os.path.exists(meta_indices_path):
    raise FileNotFoundError(meta_indices_path)

meta_indices = np.load(
    meta_indices_path
).reshape(-1).astype(np.int64)

assert meta_indices.shape == (10000,)

y_meta = y_cifar_development[
    meta_indices
]

assert y_meta.shape == (10000,)

print(
    "Meta class counts:",
    np.bincount(
        y_meta,
        minlength=NUM_CLASSES
    )
)

print(
    "Test class counts:",
    np.bincount(
        y_cifar_test,
        minlength=NUM_CLASSES
    )
)

# ------------------------------------------------------------
# 4. Exact base-model probability paths
# Feature order:
# CNN -> VGG16 -> ResNet50 -> EfficientNetB0
# ------------------------------------------------------------

probability_paths = {
    "CNN": {
        "meta": os.path.join(
            seed_dir,
            "Custom_CNN",
            "CIFAR10_CNN_Seed2025_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "Custom_CNN",
            "CIFAR10_CNN_Seed2025_Test_Probabilities.npy"
        )
    },

    "VGG16": {
        "meta": os.path.join(
            seed_dir,
            "VGG16_Style",
            "CIFAR10_VGG16_Seed2025_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "VGG16_Style",
            "CIFAR10_VGG16_Seed2025_Test_Probabilities.npy"
        )
    },

    "ResNet50": {
        "meta": os.path.join(
            seed_dir,
            "ResNet_Style",
            "CIFAR10_ResNet50_Seed2025_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "ResNet_Style",
            "CIFAR10_ResNet50_Seed2025_Test_Probabilities.npy"
        )
    },

    "EfficientNetB0": {
        "meta": os.path.join(
            seed_dir,
            "EfficientNet_Style",
            "CIFAR10_EfficientNetB0_Seed2025_Meta_Probabilities.npy"
        ),
        "test": os.path.join(
            seed_dir,
            "EfficientNet_Style",
            "CIFAR10_EfficientNetB0_Seed2025_Test_Probabilities.npy"
        )
    }
}

model_order = [
    "CNN",
    "VGG16",
    "ResNet50",
    "EfficientNetB0"
]

# ------------------------------------------------------------
# 5. Probability validation
# ------------------------------------------------------------

def validate_probability_matrix(
    probabilities,
    expected_shape,
    model_name,
    split_name
):
    probabilities = np.asarray(
        probabilities
    )

    if probabilities.shape != expected_shape:
        raise ValueError(
            f"{model_name} {split_name} shape "
            f"{probabilities.shape}; expected {expected_shape}."
        )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{model_name} {split_name} contains "
            "NaN or infinite values."
        )

    if np.any(probabilities < 0.0):
        raise ValueError(
            f"{model_name} {split_name} has "
            "negative probabilities."
        )

    if np.any(probabilities > 1.0):
        raise ValueError(
            f"{model_name} {split_name} has "
            "probabilities greater than 1."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-5
    ):
        raise ValueError(
            f"{model_name} {split_name} rows "
            "do not sum to 1."
        )

    return probabilities.astype(
        np.float32
    )

# ------------------------------------------------------------
# 6. Load all four probability pairs
# ------------------------------------------------------------

meta_probability_arrays = []
test_probability_arrays = []
base_model_rows = []

for model_name in model_order:

    meta_path = probability_paths[
        model_name
    ]["meta"]

    test_path = probability_paths[
        model_name
    ]["test"]

    if not os.path.exists(meta_path):
        raise FileNotFoundError(
            f"Missing {model_name} meta probabilities:\n"
            f"{meta_path}"
        )

    if not os.path.exists(test_path):
        raise FileNotFoundError(
            f"Missing {model_name} test probabilities:\n"
            f"{test_path}"
        )

    meta_probabilities = validate_probability_matrix(
        np.load(meta_path),
        (10000, NUM_CLASSES),
        model_name,
        "meta"
    )

    test_probabilities = validate_probability_matrix(
        np.load(test_path),
        (10000, NUM_CLASSES),
        model_name,
        "test"
    )

    meta_probability_arrays.append(
        meta_probabilities
    )

    test_probability_arrays.append(
        test_probabilities
    )

    base_model_rows.append(
        {
            "Dataset": "CIFAR-10",
            "Holdout_Seed": HOLDOUT_SEED,
            "Model": model_name,
            "Meta_Accuracy": accuracy_score(
                y_meta,
                np.argmax(
                    meta_probabilities,
                    axis=1
                )
            ),
            "Test_Accuracy": accuracy_score(
                y_cifar_test,
                np.argmax(
                    test_probabilities,
                    axis=1
                )
            )
        }
    )

    print(
        f"{model_name:<15}",
        "meta:",
        meta_probabilities.shape,
        "| test:",
        test_probabilities.shape
    )

base_model_accuracy_df = pd.DataFrame(
    base_model_rows
)

display(
    base_model_accuracy_df.round(6)
)

# ------------------------------------------------------------
# 7. Create 40-feature stacking matrices
# ------------------------------------------------------------

X_meta_stack = np.concatenate(
    meta_probability_arrays,
    axis=1
).astype(np.float32)

X_test_stack = np.concatenate(
    test_probability_arrays,
    axis=1
).astype(np.float32)

assert X_meta_stack.shape == (10000, 40)
assert X_test_stack.shape == (10000, 40)

meta_features_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed2025_Meta_40_Features.npy"
)

test_features_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed2025_Test_40_Features.npy"
)

meta_labels_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed2025_Meta_Labels.npy"
)

np.save(
    meta_features_path,
    X_meta_stack
)

np.save(
    test_features_path,
    X_test_stack
)

np.save(
    meta_labels_path,
    y_meta
)

print("\nMeta features:", X_meta_stack.shape)
print("Test features:", X_test_stack.shape)

# ------------------------------------------------------------
# 8. Metric functions
# ------------------------------------------------------------

def multiclass_brier_score(
    y_true,
    probabilities,
    number_of_classes
):
    one_hot_targets = np.eye(
        number_of_classes,
        dtype=np.float32
    )[y_true]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot_targets
                ) ** 2,
                axis=1
            )
        )
    )


def expected_calibration_error(
    y_true,
    probabilities,
    number_of_bins=15
):
    predictions = np.argmax(
        probabilities,
        axis=1
    )

    confidences = np.max(
        probabilities,
        axis=1
    )

    correctness = (
        predictions == y_true
    ).astype(np.float64)

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1
    )

    ece = 0.0

    for bin_index in range(
        number_of_bins
    ):
        lower = bin_edges[
            bin_index
        ]

        upper = bin_edges[
            bin_index + 1
        ]

        if bin_index == 0:
            in_bin = (
                (confidences >= lower)
                & (confidences <= upper)
            )
        else:
            in_bin = (
                (confidences > lower)
                & (confidences <= upper)
            )

        count = np.sum(
            in_bin
        )

        if count == 0:
            continue

        bin_accuracy = np.mean(
            correctness[in_bin]
        )

        bin_confidence = np.mean(
            confidences[in_bin]
        )

        ece += (
            count / len(y_true)
        ) * abs(
            bin_accuracy
            - bin_confidence
        )

    return float(ece)


def evaluate_method(
    method_name,
    probabilities
):
    probabilities = validate_probability_matrix(
        probabilities,
        (len(y_cifar_test), NUM_CLASSES),
        method_name,
        "test"
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "Dataset": "CIFAR-10",
        "Protocol": "Repeated fixed hold-out",
        "Holdout_Seed": HOLDOUT_SEED,
        "Method": method_name,

        "Accuracy": accuracy_score(
            y_cifar_test,
            predictions
        ),

        "Macro_Precision": precision_score(
            y_cifar_test,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro_Recall": recall_score(
            y_cifar_test,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_cifar_test,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro_ROC_AUC_OVR": roc_auc_score(
            y_cifar_test,
            probabilities,
            multi_class="ovr",
            average="macro",
            labels=np.arange(
                NUM_CLASSES
            )
        ),

        "Log_Loss": log_loss(
            y_cifar_test,
            probabilities,
            labels=np.arange(
                NUM_CLASSES
            )
        ),

        "Multiclass_Brier": multiclass_brier_score(
            y_cifar_test,
            probabilities,
            NUM_CLASSES
        ),

        "ECE_15_Bins": expected_calibration_error(
            y_cifar_test,
            probabilities,
            number_of_bins=15
        )
    }

# ------------------------------------------------------------
# 9. Soft voting
# ------------------------------------------------------------

soft_voting_test_probabilities = np.mean(
    np.stack(
        test_probability_arrays,
        axis=0
    ),
    axis=0
).astype(np.float32)

soft_voting_test_probabilities /= (
    soft_voting_test_probabilities.sum(
        axis=1,
        keepdims=True
    )
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed2025_SoftVoting_Test_Probabilities.npy"
    ),
    soft_voting_test_probabilities
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed2025_SoftVoting_Predicted_Labels.npy"
    ),
    np.argmax(
        soft_voting_test_probabilities,
        axis=1
    ).astype(np.int64)
)

# ------------------------------------------------------------
# 10. Logistic Regression
# ------------------------------------------------------------

logistic_pipeline = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                max_iter=5000,
                random_state=42
            )
        )
    ]
)

logistic_parameter_grid = {
    "classifier__C": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0
    ],
    "classifier__penalty": [
        "l1",
        "l2"
    ]
}

logistic_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_parameter_grid,
    scoring="f1_macro",
    cv=logistic_cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

print("\nTraining Logistic Regression...")

logistic_grid_search.fit(
    X_meta_stack,
    y_meta
)

best_logistic_model = (
    logistic_grid_search.best_estimator_
)

logistic_test_probabilities = (
    best_logistic_model.predict_proba(
        X_test_stack
    )
)

logistic_classes = (
    best_logistic_model
    .named_steps["classifier"]
    .classes_
)

if not np.array_equal(
    logistic_classes,
    np.arange(NUM_CLASSES)
):
    reordered = np.zeros(
        (
            len(X_test_stack),
            NUM_CLASSES
        ),
        dtype=np.float64
    )

    reordered[
        :,
        logistic_classes
    ] = logistic_test_probabilities

    logistic_test_probabilities = reordered

logistic_test_probabilities = np.asarray(
    logistic_test_probabilities,
    dtype=np.float32
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed2025_LogisticRegression_Test_Probabilities.npy"
    ),
    logistic_test_probabilities
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed2025_LogisticRegression_Predicted_Labels.npy"
    ),
    np.argmax(
        logistic_test_probabilities,
        axis=1
    ).astype(np.int64)
)

with open(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed2025_LogisticRegression_Best_Parameters.json"
    ),
    "w"
) as json_file:
    json.dump(
        {
            "best_parameters":
                logistic_grid_search.best_params_,
            "best_cv_macro_f1":
                float(
                    logistic_grid_search.best_score_
                )
        },
        json_file,
        indent=4
    )

print(
    "Best LR parameters:",
    logistic_grid_search.best_params_
)

print(
    "Best LR CV macro F1:",
    logistic_grid_search.best_score_
)

# ------------------------------------------------------------
# 11. LightGBM
# ------------------------------------------------------------

lightgbm_model = LGBMClassifier(
    objective="multiclass",
    num_class=NUM_CLASSES,
    n_estimators=150,
    learning_rate=0.03,
    num_leaves=7,
    max_depth=3,
    min_child_samples=30,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print("\nTraining LightGBM...")

lightgbm_model.fit(
    X_meta_stack,
    y_meta
)

lightgbm_test_probabilities = np.asarray(
    lightgbm_model.predict_proba(
        X_test_stack
    ),
    dtype=np.float32
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed2025_LightGBM_Test_Probabilities.npy"
    ),
    lightgbm_test_probabilities
)

np.save(
    os.path.join(
        meta_output_dir,
        "CIFAR10_Seed2025_LightGBM_Predicted_Labels.npy"
    ),
    np.argmax(
        lightgbm_test_probabilities,
        axis=1
    ).astype(np.int64)
)

# ------------------------------------------------------------
# 12. Evaluate and save results
# ------------------------------------------------------------

seed2025_results_df = pd.DataFrame(
    [
        evaluate_method(
            "Soft Voting",
            soft_voting_test_probabilities
        ),

        evaluate_method(
            "Logistic Regression",
            logistic_test_probabilities
        ),

        evaluate_method(
            "LightGBM",
            lightgbm_test_probabilities
        )
    ]
)

results_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed2025_Repeated_Holdout_Results.csv"
)

base_results_path = os.path.join(
    meta_output_dir,
    "CIFAR10_Seed2025_Base_Model_Accuracy.csv"
)

seed2025_results_df.to_csv(
    results_path,
    index=False
)

base_model_accuracy_df.to_csv(
    base_results_path,
    index=False
)

display(
    seed2025_results_df.round(6)
)

print(
    "\nSeed-2025 meta-learning completed successfully."
)

print("Results saved to:")
print(results_path)

print("\nFeature order:")
print(
    "CNN -> VGG16 -> ResNet50 -> EfficientNetB0"
)

print(
    "Meta feature shape:",
    X_meta_stack.shape
)

print(
    "Test feature shape:",
    X_test_stack.shape
)

In [ ]:
# ============================================================
# FIND ALL CIFAR-10 REPEATED-HOLDOUT RESULT FILES
# ============================================================

import os
import glob

search_roots = [
    CIFAR_REPEATED_HOLDOUT_DIR,

    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison/CIFAR10",

    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
]

found_csv_files = []

for root in search_roots:

    print("\nSearching:")
    print(root)

    if not os.path.exists(root):
        print("Folder does not exist.")
        continue

    matches = glob.glob(
        os.path.join(
            root,
            "**",
            "*.csv"
        ),
        recursive=True
    )

    relevant_matches = [
        path
        for path in matches
        if (
            "result" in os.path.basename(path).lower()
            or "holdout" in os.path.basename(path).lower()
        )
    ]

    for path in sorted(relevant_matches):
        print(path)
        found_csv_files.append(path)

print("\nTotal relevant CSV files found:", len(found_csv_files))

In [ ]:
# ============================================================
# RECONSTRUCT COMPLETE CIFAR-10 SEED-42 RESULTS
# Adds:
#   1. Soft Voting
#   2. Multiclass Brier score
#   3. ECE with 15 bins
# Recalculates all metrics consistently for all three methods
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

HOLDOUT_SEED = 42
NUM_CLASSES = 10

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

SEED42_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Repeated_Holdout",
    "CIFAR10",
    "Seed_42"
)

SEED42_META_DIR = os.path.join(
    SEED42_DIR,
    "Meta_Learners"
)

ORIGINAL_PREDICTION_DIR = os.path.join(
    PROJECT_ROOT,
    "CIFAR10",
    "Test_Predictions"
)

os.makedirs(
    SEED42_META_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Verify final test labels
# ------------------------------------------------------------

if "y_cifar_test" not in globals():

    test_label_path = (
        "/content/drive/MyDrive/"
        "CIFAR10_Cache/y_cf_test.npy"
    )

    if not os.path.exists(test_label_path):
        raise FileNotFoundError(
            f"Test-label file not found:\n{test_label_path}"
        )

    y_cifar_test = np.load(
        test_label_path
    )

y_cifar_test = np.asarray(
    y_cifar_test
).reshape(-1).astype(np.int64)

assert y_cifar_test.shape == (10000,)

print(
    "Test labels:",
    y_cifar_test.shape
)

# ------------------------------------------------------------
# 2. Helper to locate saved probability files
# ------------------------------------------------------------

def find_probability_file(
    preferred_paths,
    search_patterns,
    label
):
    for path in preferred_paths:
        if os.path.exists(path):
            print(f"{label}:")
            print(path)
            return path

    matches = []

    for pattern in search_patterns:
        matches.extend(
            glob.glob(
                pattern,
                recursive=True
            )
        )

    matches = sorted(
        set(matches)
    )

    if not matches:
        raise FileNotFoundError(
            f"No probability file found for {label}."
        )

    print(f"{label}:")
    print(matches[0])

    if len(matches) > 1:
        print(
            f"Note: {len(matches)} possible files found; "
            "using the first matching file."
        )

    return matches[0]

# ------------------------------------------------------------
# 3. Locate LR and LightGBM hold-out probabilities
# ------------------------------------------------------------

lr_probability_path = find_probability_file(
    preferred_paths=[
        os.path.join(
            ORIGINAL_PREDICTION_DIR,
            "Holdout_LogisticRegression_Test_Probabilities.npy"
        ),
        os.path.join(
            SEED42_META_DIR,
            "CIFAR10_Seed42_LogisticRegression_Test_Probabilities.npy"
        )
    ],
    search_patterns=[
        os.path.join(
            PROJECT_ROOT,
            "**",
            "*Holdout*LogisticRegression*Test*Probabilities*.npy"
        ),
        os.path.join(
            PROJECT_ROOT,
            "**",
            "*Seed42*LogisticRegression*Probabilities*.npy"
        )
    ],
    label="Seed-42 Logistic Regression probabilities"
)

lgbm_probability_path = find_probability_file(
    preferred_paths=[
        os.path.join(
            ORIGINAL_PREDICTION_DIR,
            "Holdout_LightGBM_Test_Probabilities.npy"
        ),
        os.path.join(
            SEED42_META_DIR,
            "CIFAR10_Seed42_LightGBM_Test_Probabilities.npy"
        )
    ],
    search_patterns=[
        os.path.join(
            PROJECT_ROOT,
            "**",
            "*Holdout*LightGBM*Test*Probabilities*.npy"
        ),
        os.path.join(
            PROJECT_ROOT,
            "**",
            "*Seed42*LightGBM*Probabilities*.npy"
        )
    ],
    label="Seed-42 LightGBM probabilities"
)

lr_test_probabilities = np.load(
    lr_probability_path
)

lgbm_test_probabilities = np.load(
    lgbm_probability_path
)

# ------------------------------------------------------------
# 4. Try to load an existing hold-out soft-voting file
# ------------------------------------------------------------

soft_voting_matches = []

soft_voting_patterns = [
    os.path.join(
        ORIGINAL_PREDICTION_DIR,
        "*Holdout*Soft*Voting*Test*Probabilities*.npy"
    ),
    os.path.join(
        ORIGINAL_PREDICTION_DIR,
        "*Holdout*SoftVoting*Test*Probabilities*.npy"
    ),
    os.path.join(
        SEED42_META_DIR,
        "*SoftVoting*Test*Probabilities*.npy"
    ),
    os.path.join(
        PROJECT_ROOT,
        "**",
        "*Holdout*SoftVoting*Test*Probabilities*.npy"
    )
]

for pattern in soft_voting_patterns:
    soft_voting_matches.extend(
        glob.glob(
            pattern,
            recursive=True
        )
    )

soft_voting_matches = sorted(
    set(soft_voting_matches)
)

# ------------------------------------------------------------
# 5. If no saved soft voting exists, reconstruct it from
#    the four seed-42 base-model test probability arrays
# ------------------------------------------------------------

if soft_voting_matches:

    soft_voting_probability_path = (
        soft_voting_matches[0]
    )

    print(
        "\nExisting soft-voting probabilities found:"
    )
    print(soft_voting_probability_path)

    soft_voting_test_probabilities = np.load(
        soft_voting_probability_path
    )

else:

    print(
        "\nNo saved hold-out soft-voting array found."
    )

    print(
        "Reconstructing soft voting from the four "
        "saved base-model test arrays."
    )

    OOF_PROBABILITY_DIR = (
        "/content/drive/MyDrive/"
        "OOF_Stacking_Results/OOF_Probabilities"
    )

    base_probability_paths = {
        "CNN": os.path.join(
            OOF_PROBABILITY_DIR,
            "CIFAR_CNN_Test_Probabilities.npy"
        ),

        "VGG16": os.path.join(
            OOF_PROBABILITY_DIR,
            "CIFAR_VGG16_Test_Probabilities.npy"
        ),

        "ResNet50": os.path.join(
            OOF_PROBABILITY_DIR,
            "CIFAR_ResNet50_Test_Probabilities.npy"
        ),

        "EfficientNetB0": os.path.join(
            OOF_PROBABILITY_DIR,
            "CIFAR_EfficientNetB0_Test_Probabilities.npy"
        )
    }

    base_test_probabilities = []

    for model_name, path in base_probability_paths.items():

        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{model_name} test probabilities missing:\n"
                f"{path}"
            )

        probabilities = np.load(
            path
        )

        if probabilities.shape != (10000, 10):
            raise ValueError(
                f"{model_name} probability shape is "
                f"{probabilities.shape}; expected (10000, 10)."
            )

        base_test_probabilities.append(
            probabilities
        )

        print(
            f"{model_name:<15}:",
            probabilities.shape
        )

    soft_voting_test_probabilities = np.mean(
        np.stack(
            base_test_probabilities,
            axis=0
        ),
        axis=0
    )

    soft_voting_test_probabilities = (
        soft_voting_test_probabilities
        / soft_voting_test_probabilities.sum(
            axis=1,
            keepdims=True
        )
    )

    soft_voting_probability_path = os.path.join(
        SEED42_META_DIR,
        "CIFAR10_Seed42_SoftVoting_Test_Probabilities.npy"
    )

    np.save(
        soft_voting_probability_path,
        soft_voting_test_probabilities.astype(
            np.float32
        )
    )

    print(
        "\nReconstructed soft-voting probabilities saved:"
    )
    print(soft_voting_probability_path)

# ------------------------------------------------------------
# 6. Validate all three probability matrices
# ------------------------------------------------------------

def validate_probability_matrix(
    probabilities,
    label
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    if probabilities.shape != (10000, 10):
        raise ValueError(
            f"{label} shape is {probabilities.shape}; "
            "expected (10000, 10)."
        )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{label} contains NaN or infinite values."
        )

    if np.any(probabilities < 0.0):
        raise ValueError(
            f"{label} contains negative values."
        )

    row_sums = probabilities.sum(
        axis=1,
        keepdims=True
    )

    if np.any(row_sums <= 0):
        raise ValueError(
            f"{label} has invalid probability rows."
        )

    # Normalize harmless floating-point differences
    probabilities = probabilities / row_sums

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-6
    ):
        raise ValueError(
            f"{label} rows do not sum to one."
        )

    return probabilities


lr_test_probabilities = validate_probability_matrix(
    lr_test_probabilities,
    "Logistic Regression"
)

lgbm_test_probabilities = validate_probability_matrix(
    lgbm_test_probabilities,
    "LightGBM"
)

soft_voting_test_probabilities = (
    validate_probability_matrix(
        soft_voting_test_probabilities,
        "Soft Voting"
    )
)

# ------------------------------------------------------------
# 7. Calibration metrics
# ------------------------------------------------------------

def multiclass_brier_score(
    y_true,
    probabilities,
    number_of_classes=10
):
    one_hot_targets = np.eye(
        number_of_classes
    )[y_true]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot_targets
                ) ** 2,
                axis=1
            )
        )
    )


def expected_calibration_error(
    y_true,
    probabilities,
    number_of_bins=15
):
    predicted_labels = np.argmax(
        probabilities,
        axis=1
    )

    confidences = np.max(
        probabilities,
        axis=1
    )

    correct = (
        predicted_labels == y_true
    ).astype(float)

    bin_boundaries = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1
    )

    ece = 0.0

    for index in range(number_of_bins):

        lower = bin_boundaries[index]
        upper = bin_boundaries[index + 1]

        if index == 0:
            mask = (
                (confidences >= lower)
                & (confidences <= upper)
            )
        else:
            mask = (
                (confidences > lower)
                & (confidences <= upper)
            )

        count = np.sum(mask)

        if count == 0:
            continue

        bin_accuracy = np.mean(
            correct[mask]
        )

        bin_confidence = np.mean(
            confidences[mask]
        )

        ece += (
            count / len(y_true)
        ) * abs(
            bin_accuracy
            - bin_confidence
        )

    return float(ece)

# ------------------------------------------------------------
# 8. Consistent evaluation function
# ------------------------------------------------------------

def evaluate_seed42_method(
    method_name,
    probabilities
):
    predicted_labels = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "Dataset": "CIFAR-10",
        "Protocol": "Repeated fixed hold-out",
        "Method": method_name,
        "Accuracy": accuracy_score(
            y_cifar_test,
            predicted_labels
        ),
        "Macro_Precision": precision_score(
            y_cifar_test,
            predicted_labels,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_cifar_test,
            predicted_labels,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_cifar_test,
            predicted_labels,
            average="macro",
            zero_division=0
        ),
        "Macro_ROC_AUC_OVR": roc_auc_score(
            y_cifar_test,
            probabilities,
            multi_class="ovr",
            average="macro",
            labels=np.arange(NUM_CLASSES)
        ),
        "Log_Loss": log_loss(
            y_cifar_test,
            probabilities,
            labels=np.arange(NUM_CLASSES)
        ),
        "Multiclass_Brier": multiclass_brier_score(
            y_cifar_test,
            probabilities,
            NUM_CLASSES
        ),
        "ECE_15_Bins": expected_calibration_error(
            y_cifar_test,
            probabilities,
            number_of_bins=15
        ),
        "Seed": HOLDOUT_SEED,
        "Holdout_Seed": HOLDOUT_SEED,
        "Meta_Training_Samples": 10000
    }

# ------------------------------------------------------------
# 9. Build the complete seed-42 table
# ------------------------------------------------------------

seed42_complete_df = pd.DataFrame(
    [
        evaluate_seed42_method(
            "Soft Voting",
            soft_voting_test_probabilities
        ),

        evaluate_seed42_method(
            "Logistic Regression",
            lr_test_probabilities
        ),

        evaluate_seed42_method(
            "LightGBM",
            lgbm_test_probabilities
        )
    ]
)

# ------------------------------------------------------------
# 10. Save complete seed-42 results
# ------------------------------------------------------------

seed42_complete_path = os.path.join(
    SEED42_META_DIR,
    "CIFAR10_Seed42_Repeated_Holdout_Results.csv"
)

seed42_complete_df.to_csv(
    seed42_complete_path,
    index=False
)

display(
    seed42_complete_df.round(6)
)

print(
    "\nComplete seed-42 results saved:"
)
print(seed42_complete_path)

print(
    "\nSeed-42 methods:",
    list(seed42_complete_df["Method"])
)

In [ ]:
# ============================================================
# CIFAR-10 REPEATED FIXED HOLD-OUT
# COMBINE SEEDS 42, 123, 2025
# CALCULATE MEAN ± SAMPLE STANDARD DEVIATION
# COMPLETE CORRECTED CELL
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEEDS = [42, 123, 2025]

METHOD_ORDER = [
    "Soft Voting",
    "Logistic Regression",
    "LightGBM"
]

METRIC_COLUMNS = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Macro_ROC_AUC_OVR",
    "Log_Loss",
    "Multiclass_Brier",
    "ECE_15_Bins"
]

PERCENTAGE_METRICS = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Macro_ROC_AUC_OVR"
]

if "CIFAR_REPEATED_HOLDOUT_DIR" not in globals():
    CIFAR_REPEATED_HOLDOUT_DIR = (
        "/content/drive/MyDrive/"
        "Paper_1_Matched_Protocol_Comparison/"
        "Extended_Experiments/Repeated_Holdout/CIFAR10"
    )

extended_experiments_dir = os.path.dirname(
    CIFAR_REPEATED_HOLDOUT_DIR
)

tables_dir = os.path.join(
    extended_experiments_dir,
    "Tables"
)

os.makedirs(
    CIFAR_REPEATED_HOLDOUT_DIR,
    exist_ok=True
)

os.makedirs(
    tables_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Exact corrected result paths
# ------------------------------------------------------------

seed_result_paths = {
    42: os.path.join(
        CIFAR_REPEATED_HOLDOUT_DIR,
        "Seed_42",
        "Meta_Learners",
        "CIFAR10_Seed42_Repeated_Holdout_Results.csv"
    ),

    123: os.path.join(
        CIFAR_REPEATED_HOLDOUT_DIR,
        "Seed_123",
        "Meta_Learners",
        "CIFAR10_Seed123_Repeated_Holdout_Results.csv"
    ),

    2025: os.path.join(
        CIFAR_REPEATED_HOLDOUT_DIR,
        "Seed_2025",
        "Meta_Learners",
        "CIFAR10_Seed2025_Repeated_Holdout_Results.csv"
    )
}

print("=" * 78)
print("CIFAR-10 REPEATED HOLD-OUT RESULT FILE CHECK")
print("=" * 78)

for seed, path in seed_result_paths.items():

    file_exists = os.path.exists(
        path
    )

    print(
        f"Seed {seed:<4}:",
        file_exists
    )

    print(path)

    if not file_exists:
        raise FileNotFoundError(
            f"\nSeed-{seed} result file is missing:\n"
            f"{path}"
        )

# ------------------------------------------------------------
# 3. Normalize method labels
# ------------------------------------------------------------

def normalize_method_name(value):

    value = str(value).strip()

    replacements = {
        "SoftVoting": "Soft Voting",
        "Soft_Voting": "Soft Voting",
        "Soft-Voting": "Soft Voting",
        "Voting": "Soft Voting",

        "LR": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression Stacking":
            "Logistic Regression",

        "LGBM": "LightGBM",
        "Light GBM": "LightGBM",
        "LightGBM Stacking": "LightGBM"
    }

    return replacements.get(
        value,
        value
    )

# ------------------------------------------------------------
# 4. Normalize possible column-name differences
# ------------------------------------------------------------

def normalize_result_columns(
    dataframe,
    seed
):

    dataframe = dataframe.copy()

    rename_map = {
        "Random_Seed": "Holdout_Seed",

        "Model": "Method",
        "Meta_Learner": "Method",
        "Classifier": "Method",

        "Precision": "Macro_Precision",
        "Macro Precision": "Macro_Precision",
        "MacroPrecision": "Macro_Precision",

        "Recall": "Macro_Recall",
        "Macro Recall": "Macro_Recall",
        "MacroRecall": "Macro_Recall",

        "F1": "Macro_F1",
        "F1_Score": "Macro_F1",
        "Macro_F1_Score": "Macro_F1",
        "Macro F1": "Macro_F1",
        "MacroF1": "Macro_F1",

        "AUC": "Macro_ROC_AUC_OVR",
        "ROC_AUC": "Macro_ROC_AUC_OVR",
        "ROC_AUC_OVR": "Macro_ROC_AUC_OVR",
        "Macro_AUC": "Macro_ROC_AUC_OVR",
        "Macro ROC AUC": "Macro_ROC_AUC_OVR",

        "LogLoss": "Log_Loss",
        "Log Loss": "Log_Loss",

        "Brier": "Multiclass_Brier",
        "Brier_Score": "Multiclass_Brier",
        "Brier Score": "Multiclass_Brier",

        "ECE": "ECE_15_Bins",
        "ECE_15": "ECE_15_Bins",
        "ECE15": "ECE_15_Bins"
    }

    for old_column, new_column in rename_map.items():

        if (
            old_column in dataframe.columns
            and new_column not in dataframe.columns
        ):

            dataframe = dataframe.rename(
                columns={
                    old_column: new_column
                }
            )

    if "Method" not in dataframe.columns:
        raise ValueError(
            f"Seed {seed} has no recognizable Method column.\n"
            f"Available columns:\n"
            f"{list(dataframe.columns)}"
        )

    dataframe["Method"] = (
        dataframe["Method"]
        .apply(
            normalize_method_name
        )
    )

    dataframe["Holdout_Seed"] = seed
    dataframe["Dataset"] = "CIFAR-10"
    dataframe[
        "Protocol"
    ] = "Repeated fixed hold-out"

    return dataframe

# ------------------------------------------------------------
# 5. Load and validate each seed result table
# ------------------------------------------------------------

seed_dataframes = []

for seed in SEEDS:

    seed_path = seed_result_paths[
        seed
    ]

    seed_df = pd.read_csv(
        seed_path
    )

    print(
        f"\nSeed {seed} original columns:"
    )

    print(
        list(seed_df.columns)
    )

    seed_df = normalize_result_columns(
        seed_df,
        seed
    )

    # Retain only required methods
    seed_df = seed_df[
        seed_df["Method"].isin(
            METHOD_ORDER
        )
    ].copy()

    # Remove duplicate rows for the same method, if any
    seed_df = seed_df.drop_duplicates(
        subset=["Method"],
        keep="last"
    )

    missing_methods = [
        method
        for method in METHOD_ORDER
        if method not in seed_df[
            "Method"
        ].values
    ]

    if missing_methods:
        raise ValueError(
            f"\nSeed {seed} is missing methods:\n"
            f"{missing_methods}\n\n"
            f"Rows found:\n"
            f"{seed_df}"
        )

    missing_metrics = [
        metric
        for metric in METRIC_COLUMNS
        if metric not in seed_df.columns
    ]

    if missing_metrics:
        raise ValueError(
            f"\nSeed {seed} is missing metric columns:\n"
            f"{missing_metrics}\n\n"
            f"Available columns:\n"
            f"{list(seed_df.columns)}"
        )

    for metric in METRIC_COLUMNS:

        seed_df[metric] = pd.to_numeric(
            seed_df[metric],
            errors="raise"
        )

        if not np.isfinite(
            seed_df[metric]
        ).all():

            raise ValueError(
                f"Seed {seed}, metric {metric}, "
                "contains NaN or infinite values."
            )

    # Keep only required reporting columns
    seed_df = seed_df[
        [
            "Dataset",
            "Protocol",
            "Holdout_Seed",
            "Method"
        ] + METRIC_COLUMNS
    ].copy()

    seed_dataframes.append(
        seed_df
    )

    print(
        f"Seed {seed} loaded successfully:"
    )

    print(
        list(seed_df["Method"])
    )

# ------------------------------------------------------------
# 6. Combine all three seeds
# ------------------------------------------------------------

all_seed_results_df = pd.concat(
    seed_dataframes,
    ignore_index=True
)

all_seed_results_df = (
    all_seed_results_df
    .drop_duplicates(
        subset=[
            "Holdout_Seed",
            "Method"
        ],
        keep="last"
    )
)

expected_row_count = (
    len(SEEDS)
    * len(METHOD_ORDER)
)

if len(all_seed_results_df) != expected_row_count:

    raise ValueError(
        f"Expected {expected_row_count} rows "
        f"(3 seeds × 3 methods), "
        f"but found {len(all_seed_results_df)}."
    )

# ------------------------------------------------------------
# 7. Verify all seed-method combinations
# ------------------------------------------------------------

expected_combinations = {
    (
        seed,
        method
    )
    for seed in SEEDS
    for method in METHOD_ORDER
}

actual_combinations = {
    (
        int(row["Holdout_Seed"]),
        str(row["Method"])
    )
    for _, row in all_seed_results_df.iterrows()
}

missing_combinations = (
    expected_combinations
    - actual_combinations
)

unexpected_combinations = (
    actual_combinations
    - expected_combinations
)

if missing_combinations:
    raise ValueError(
        "Missing seed-method combinations:\n"
        f"{sorted(missing_combinations)}"
    )

if unexpected_combinations:
    raise ValueError(
        "Unexpected seed-method combinations:\n"
        f"{sorted(unexpected_combinations)}"
    )

# ------------------------------------------------------------
# 8. Sort final seed-level table
# ------------------------------------------------------------

method_rank = {
    method: index
    for index, method in enumerate(
        METHOD_ORDER
    )
}

all_seed_results_df[
    "_Method_Order"
] = all_seed_results_df[
    "Method"
].map(
    method_rank
)

all_seed_results_df = (
    all_seed_results_df
    .sort_values(
        by=[
            "Holdout_Seed",
            "_Method_Order"
        ]
    )
    .drop(
        columns=[
            "_Method_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)

print("\n" + "=" * 78)
print("COMBINED SEED-LEVEL RESULTS")
print("=" * 78)

display(
    all_seed_results_df.round(6)
)

print(
    "\nCombined table shape:",
    all_seed_results_df.shape
)

# ------------------------------------------------------------
# 9. Calculate mean, sample SD, minimum, and maximum
# ------------------------------------------------------------

summary_rows = []

for method in METHOD_ORDER:

    method_df = all_seed_results_df[
        all_seed_results_df[
            "Method"
        ] == method
    ].sort_values(
        "Holdout_Seed"
    )

    if len(method_df) != len(SEEDS):

        raise ValueError(
            f"{method} has {len(method_df)} seed rows; "
            f"expected {len(SEEDS)}."
        )

    summary_row = {
        "Dataset": "CIFAR-10",
        "Protocol": "Repeated fixed hold-out",
        "Method": method,
        "Number_of_Seeds": len(SEEDS),
        "Seeds": "42, 123, 2025"
    }

    for metric in METRIC_COLUMNS:

        metric_values = method_df[
            metric
        ].to_numpy(
            dtype=np.float64
        )

        summary_row[
            f"{metric}_Mean"
        ] = float(
            np.mean(
                metric_values
            )
        )

        # Sample standard deviation across three runs
        summary_row[
            f"{metric}_SD"
        ] = float(
            np.std(
                metric_values,
                ddof=1
            )
        )

        summary_row[
            f"{metric}_Min"
        ] = float(
            np.min(
                metric_values
            )
        )

        summary_row[
            f"{metric}_Max"
        ] = float(
            np.max(
                metric_values
            )
        )

    summary_rows.append(
        summary_row
    )

mean_sd_summary_df = pd.DataFrame(
    summary_rows
)

# ------------------------------------------------------------
# 10. Create compact publication-style table
# ------------------------------------------------------------

compact_rows = []

for _, summary_row in (
    mean_sd_summary_df.iterrows()
):

    compact_row = {
        "Method": summary_row[
            "Method"
        ]
    }

    for metric in METRIC_COLUMNS:

        mean_value = summary_row[
            f"{metric}_Mean"
        ]

        sd_value = summary_row[
            f"{metric}_SD"
        ]

        if metric in PERCENTAGE_METRICS:

            compact_row[
                metric
            ] = (
                f"{mean_value * 100:.3f} "
                f"± {sd_value * 100:.3f}"
            )

        else:

            compact_row[
                metric
            ] = (
                f"{mean_value:.6f} "
                f"± {sd_value:.6f}"
            )

    compact_rows.append(
        compact_row
    )

compact_table_df = pd.DataFrame(
    compact_rows
)

# ------------------------------------------------------------
# 11. Create concise accuracy, F1, AUC table
# ------------------------------------------------------------

accuracy_f1_auc_rows = []

for _, summary_row in (
    mean_sd_summary_df.iterrows()
):

    accuracy_f1_auc_rows.append(
        {
            "Method": summary_row[
                "Method"
            ],

            "Accuracy_Mean_Percent":
                summary_row[
                    "Accuracy_Mean"
                ] * 100,

            "Accuracy_SD_Percent":
                summary_row[
                    "Accuracy_SD"
                ] * 100,

            "Macro_F1_Mean_Percent":
                summary_row[
                    "Macro_F1_Mean"
                ] * 100,

            "Macro_F1_SD_Percent":
                summary_row[
                    "Macro_F1_SD"
                ] * 100,

            "Macro_ROC_AUC_Mean":
                summary_row[
                    "Macro_ROC_AUC_OVR_Mean"
                ],

            "Macro_ROC_AUC_SD":
                summary_row[
                    "Macro_ROC_AUC_OVR_SD"
                ],

            "Log_Loss_Mean":
                summary_row[
                    "Log_Loss_Mean"
                ],

            "Log_Loss_SD":
                summary_row[
                    "Log_Loss_SD"
                ],

            "ECE_Mean":
                summary_row[
                    "ECE_15_Bins_Mean"
                ],

            "ECE_SD":
                summary_row[
                    "ECE_15_Bins_SD"
                ]
        }
    )

accuracy_f1_auc_df = pd.DataFrame(
    accuracy_f1_auc_rows
)

# ------------------------------------------------------------
# 12. Save all output tables
# ------------------------------------------------------------

all_seed_results_path = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Repeated_Holdout_All_Seed_Results.csv"
)

mean_sd_summary_path = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Repeated_Holdout_Mean_SD_Summary.csv"
)

compact_table_path = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Repeated_Holdout_Compact_Table.csv"
)

accuracy_f1_auc_path = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Repeated_Holdout_Accuracy_F1_AUC.csv"
)

paper_compact_table_path = os.path.join(
    tables_dir,
    "CIFAR10_Repeated_Holdout_Mean_SD_Table.csv"
)

paper_accuracy_f1_auc_path = os.path.join(
    tables_dir,
    "CIFAR10_Repeated_Holdout_Accuracy_F1_AUC.csv"
)

all_seed_results_df.to_csv(
    all_seed_results_path,
    index=False
)

mean_sd_summary_df.to_csv(
    mean_sd_summary_path,
    index=False
)

compact_table_df.to_csv(
    compact_table_path,
    index=False
)

accuracy_f1_auc_df.to_csv(
    accuracy_f1_auc_path,
    index=False
)

compact_table_df.to_csv(
    paper_compact_table_path,
    index=False
)

accuracy_f1_auc_df.to_csv(
    paper_accuracy_f1_auc_path,
    index=False
)

# ------------------------------------------------------------
# 13. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("REPEATED HOLD-OUT NUMERICAL MEAN AND SD")
print("=" * 78)

display(
    mean_sd_summary_df.round(6)
)

print("\n" + "=" * 78)
print("PUBLICATION-STYLE MEAN ± SD TABLE")
print("=" * 78)

display(
    compact_table_df
)

print("\n" + "=" * 78)
print("CONCISE ACCURACY, F1, AUC AND CALIBRATION TABLE")
print("=" * 78)

display(
    accuracy_f1_auc_df.round(6)
)

# ------------------------------------------------------------
# 14. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("CIFAR-10 THREE-SEED SUMMARY COMPLETED SUCCESSFULLY")
print("=" * 78)

print(
    "\nTotal combined rows:",
    len(all_seed_results_df)
)

print(
    "Expected rows:",
    expected_row_count
)

print(
    "\nSample standard deviation was calculated "
    "across seeds 42, 123, and 2025 using ddof=1."
)

print("\nSaved files:")

print(all_seed_results_path)
print(mean_sd_summary_path)
print(compact_table_path)
print(accuracy_f1_auc_path)
print(paper_compact_table_path)
print(paper_accuracy_f1_auc_path)

In [ ]:
# ============================================================
# CIFAR-10
# REPEATED HOLD-OUT MEAN vs FIVE-FOLD OOF
# FINAL COMPARISON TABLES AND FIGURES
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

CIFAR_REPEATED_HOLDOUT_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Repeated_Holdout",
    "CIFAR10"
)

REPEATED_SUMMARY_PATH = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Repeated_Holdout_Mean_SD_Summary.csv"
)

ORIGINAL_COMPARISON_PATH = os.path.join(
    PROJECT_ROOT,
    "CIFAR10",
    "Results",
    "CIFAR10_Matched_Holdout_vs_OOF_Results.csv"
)

EXTENDED_EXPERIMENTS_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments"
)

TABLES_DIR = os.path.join(
    EXTENDED_EXPERIMENTS_DIR,
    "Tables"
)

FIGURES_DIR = os.path.join(
    EXTENDED_EXPERIMENTS_DIR,
    "Figures"
)

os.makedirs(
    TABLES_DIR,
    exist_ok=True
)

os.makedirs(
    FIGURES_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Check required files
# ------------------------------------------------------------

required_files = {
    "Repeated hold-out summary":
        REPEATED_SUMMARY_PATH,

    "Original hold-out/OOF results":
        ORIGINAL_COMPARISON_PATH
}

print("=" * 78)
print("INPUT FILE CHECK")
print("=" * 78)

for label, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{label:<32}: {exists}")
    print(path)

    if not exists:
        raise FileNotFoundError(
            f"\nMissing required file:\n{path}"
        )

# ------------------------------------------------------------
# 3. Method-name normalization
# ------------------------------------------------------------

METHOD_ORDER = [
    "Soft Voting",
    "Logistic Regression",
    "LightGBM"
]


def normalize_method_name(value):

    value = str(value).strip()

    replacements = {
        "SoftVoting": "Soft Voting",
        "Soft_Voting": "Soft Voting",
        "Soft-Voting": "Soft Voting",
        "Voting": "Soft Voting",

        "LR": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression Stacking":
            "Logistic Regression",

        "LGBM": "LightGBM",
        "Light GBM": "LightGBM",
        "LightGBM Stacking": "LightGBM"
    }

    return replacements.get(
        value,
        value
    )

# ------------------------------------------------------------
# 4. General column normalization
# ------------------------------------------------------------

def normalize_columns(dataframe):

    dataframe = dataframe.copy()

    rename_map = {
        "Model": "Method",
        "Meta_Learner": "Method",
        "Classifier": "Method",

        "Precision": "Macro_Precision",
        "Macro Precision": "Macro_Precision",
        "MacroPrecision": "Macro_Precision",

        "Recall": "Macro_Recall",
        "Macro Recall": "Macro_Recall",
        "MacroRecall": "Macro_Recall",

        "F1": "Macro_F1",
        "F1_Score": "Macro_F1",
        "Macro_F1_Score": "Macro_F1",
        "Macro F1": "Macro_F1",
        "MacroF1": "Macro_F1",

        "AUC": "Macro_ROC_AUC_OVR",
        "ROC_AUC": "Macro_ROC_AUC_OVR",
        "ROC_AUC_OVR": "Macro_ROC_AUC_OVR",
        "Macro_AUC": "Macro_ROC_AUC_OVR",
        "Macro ROC AUC": "Macro_ROC_AUC_OVR",

        "LogLoss": "Log_Loss",
        "Log Loss": "Log_Loss",

        "Brier": "Multiclass_Brier",
        "Brier_Score": "Multiclass_Brier",
        "Brier Score": "Multiclass_Brier",

        "ECE": "ECE_15_Bins",
        "ECE_15": "ECE_15_Bins"
    }

    for old_name, new_name in rename_map.items():

        if (
            old_name in dataframe.columns
            and new_name not in dataframe.columns
        ):

            dataframe = dataframe.rename(
                columns={
                    old_name: new_name
                }
            )

    if "Method" in dataframe.columns:

        dataframe["Method"] = (
            dataframe["Method"]
            .apply(
                normalize_method_name
            )
        )

    return dataframe

# ------------------------------------------------------------
# 5. Load repeated hold-out mean/SD summary
# ------------------------------------------------------------

repeated_summary_df = pd.read_csv(
    REPEATED_SUMMARY_PATH
)

repeated_summary_df = normalize_columns(
    repeated_summary_df
)

if "Method" not in repeated_summary_df.columns:
    raise ValueError(
        "Repeated summary has no Method column."
    )

repeated_summary_df = repeated_summary_df[
    repeated_summary_df["Method"].isin(
        METHOD_ORDER
    )
].copy()

repeated_required_columns = [
    "Method",
    "Accuracy_Mean",
    "Accuracy_SD",
    "Macro_F1_Mean",
    "Macro_F1_SD",
    "Macro_ROC_AUC_OVR_Mean",
    "Macro_ROC_AUC_OVR_SD"
]

missing_repeated_columns = [
    column
    for column in repeated_required_columns
    if column not in repeated_summary_df.columns
]

if missing_repeated_columns:
    raise ValueError(
        "Repeated summary is missing columns:\n"
        f"{missing_repeated_columns}\n\n"
        "Available columns:\n"
        f"{list(repeated_summary_df.columns)}"
    )

print("\nRepeated hold-out summary loaded:")
display(
    repeated_summary_df[
        repeated_required_columns
    ].round(6)
)

# ------------------------------------------------------------
# 6. Load original matched hold-out vs OOF results
# ------------------------------------------------------------

original_results_df = pd.read_csv(
    ORIGINAL_COMPARISON_PATH
)

print("\nOriginal comparison columns:")
print(
    list(original_results_df.columns)
)

original_results_df = normalize_columns(
    original_results_df
)

if "Method" not in original_results_df.columns:
    raise ValueError(
        "Original comparison table has no recognizable "
        "Method column."
    )

# ------------------------------------------------------------
# 7. Identify the protocol column
# ------------------------------------------------------------

protocol_candidates = [
    "Protocol",
    "Training_Protocol",
    "Evaluation_Protocol",
    "Approach",
    "Methodology",
    "Validation_Protocol",
    "Stacking_Protocol"
]

protocol_column = next(
    (
        column
        for column in protocol_candidates
        if column in original_results_df.columns
    ),
    None
)

if protocol_column is None:

    print(
        "\nNo explicit protocol column was found."
    )

    print(
        "Attempting to identify OOF rows from "
        "other text columns."
    )

    text_columns = [
        column
        for column in original_results_df.columns
        if original_results_df[
            column
        ].dtype == object
    ]

    combined_text = (
        original_results_df[
            text_columns
        ]
        .astype(str)
        .agg(
            " ".join,
            axis=1
        )
        .str.lower()
    )

else:

    print(
        "\nProtocol column:",
        protocol_column
    )

    combined_text = (
        original_results_df[
            protocol_column
        ]
        .astype(str)
        .str.lower()
    )

# ------------------------------------------------------------
# 8. Extract five-fold OOF rows
# ------------------------------------------------------------

oof_mask = (
    combined_text.str.contains(
        "oof",
        na=False
    )
    |
    combined_text.str.contains(
        "out-of-fold",
        na=False
    )
    |
    combined_text.str.contains(
        "out of fold",
        na=False
    )
    |
    combined_text.str.contains(
        "five-fold",
        na=False
    )
    |
    combined_text.str.contains(
        "5-fold",
        na=False
    )
)

oof_results_df = original_results_df[
    oof_mask
].copy()

oof_results_df = oof_results_df[
    oof_results_df["Method"].isin(
        METHOD_ORDER
    )
].copy()

oof_results_df = oof_results_df.drop_duplicates(
    subset=["Method"],
    keep="last"
)

print("\nExtracted OOF rows:")
display(oof_results_df)

missing_oof_methods = [
    method
    for method in METHOD_ORDER
    if method not in oof_results_df[
        "Method"
    ].values
]

if missing_oof_methods:

    print(
        "\nAvailable protocol/method combinations:"
    )

    available_columns = [
        column
        for column in [
            protocol_column,
            "Method",
            "Accuracy",
            "Macro_F1",
            "Macro_ROC_AUC_OVR"
        ]
        if column is not None
        and column in original_results_df.columns
    ]

    display(
        original_results_df[
            available_columns
        ]
    )

    raise ValueError(
        "OOF rows are missing these methods:\n"
        f"{missing_oof_methods}"
    )

# ------------------------------------------------------------
# 9. Validate OOF metric columns
# ------------------------------------------------------------

required_oof_metrics = [
    "Accuracy",
    "Macro_F1",
    "Macro_ROC_AUC_OVR"
]

missing_oof_metrics = [
    metric
    for metric in required_oof_metrics
    if metric not in oof_results_df.columns
]

if missing_oof_metrics:
    raise ValueError(
        "OOF results are missing metrics:\n"
        f"{missing_oof_metrics}\n\n"
        "Available columns:\n"
        f"{list(oof_results_df.columns)}"
    )

for metric in required_oof_metrics:

    oof_results_df[metric] = pd.to_numeric(
        oof_results_df[metric],
        errors="raise"
    )

# ------------------------------------------------------------
# 10. Build comparison table
# ------------------------------------------------------------

comparison_rows = []

for method in METHOD_ORDER:

    repeated_row = repeated_summary_df[
        repeated_summary_df[
            "Method"
        ] == method
    ]

    oof_row = oof_results_df[
        oof_results_df[
            "Method"
        ] == method
    ]

    if len(repeated_row) != 1:
        raise ValueError(
            f"Repeated hold-out summary has "
            f"{len(repeated_row)} rows for {method}."
        )

    if len(oof_row) != 1:
        raise ValueError(
            f"OOF results have "
            f"{len(oof_row)} rows for {method}."
        )

    repeated_row = repeated_row.iloc[0]
    oof_row = oof_row.iloc[0]

    comparison_rows.append(
        {
            "Dataset": "CIFAR-10",
            "Method": method,

            "Repeated_Holdout_Accuracy_Mean":
                repeated_row[
                    "Accuracy_Mean"
                ],

            "Repeated_Holdout_Accuracy_SD":
                repeated_row[
                    "Accuracy_SD"
                ],

            "OOF_Accuracy":
                oof_row[
                    "Accuracy"
                ],

            "OOF_minus_Holdout_Accuracy":
                oof_row[
                    "Accuracy"
                ]
                - repeated_row[
                    "Accuracy_Mean"
                ],

            "Repeated_Holdout_Macro_F1_Mean":
                repeated_row[
                    "Macro_F1_Mean"
                ],

            "Repeated_Holdout_Macro_F1_SD":
                repeated_row[
                    "Macro_F1_SD"
                ],

            "OOF_Macro_F1":
                oof_row[
                    "Macro_F1"
                ],

            "OOF_minus_Holdout_Macro_F1":
                oof_row[
                    "Macro_F1"
                ]
                - repeated_row[
                    "Macro_F1_Mean"
                ],

            "Repeated_Holdout_AUC_Mean":
                repeated_row[
                    "Macro_ROC_AUC_OVR_Mean"
                ],

            "Repeated_Holdout_AUC_SD":
                repeated_row[
                    "Macro_ROC_AUC_OVR_SD"
                ],

            "OOF_AUC":
                oof_row[
                    "Macro_ROC_AUC_OVR"
                ],

            "OOF_minus_Holdout_AUC":
                oof_row[
                    "Macro_ROC_AUC_OVR"
                ]
                - repeated_row[
                    "Macro_ROC_AUC_OVR_Mean"
                ]
        }
    )

comparison_df = pd.DataFrame(
    comparison_rows
)

# ------------------------------------------------------------
# 11. Publication-style comparison table
# ------------------------------------------------------------

publication_rows = []

for _, row in comparison_df.iterrows():

    publication_rows.append(
        {
            "Method": row["Method"],

            "Repeated Hold-out Accuracy (%)":
                (
                    f"{row['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Accuracy_SD'] * 100:.3f}"
                ),

            "Five-fold OOF Accuracy (%)":
                f"{row['OOF_Accuracy'] * 100:.3f}",

            "OOF − Hold-out Accuracy (pp)":
                (
                    row[
                        "OOF_minus_Holdout_Accuracy"
                    ] * 100
                ),

            "Repeated Hold-out Macro F1 (%)":
                (
                    f"{row['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}"
                ),

            "Five-fold OOF Macro F1 (%)":
                f"{row['OOF_Macro_F1'] * 100:.3f}",

            "OOF − Hold-out F1 (pp)":
                (
                    row[
                        "OOF_minus_Holdout_Macro_F1"
                    ] * 100
                ),

            "Repeated Hold-out AUC (%)":
                (
                    f"{row['Repeated_Holdout_AUC_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_AUC_SD'] * 100:.3f}"
                ),

            "Five-fold OOF AUC (%)":
                f"{row['OOF_AUC'] * 100:.3f}",

            "OOF − Hold-out AUC (pp)":
                (
                    row[
                        "OOF_minus_Holdout_AUC"
                    ] * 100
                )
        }
    )

publication_comparison_df = pd.DataFrame(
    publication_rows
)

# ------------------------------------------------------------
# 12. Save comparison tables
# ------------------------------------------------------------

numeric_comparison_path = os.path.join(
    TABLES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Comparison.csv"
)

publication_comparison_path = os.path.join(
    TABLES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Publication_Table.csv"
)

comparison_df.to_csv(
    numeric_comparison_path,
    index=False
)

publication_comparison_df.to_csv(
    publication_comparison_path,
    index=False
)

print("\n" + "=" * 78)
print("NUMERICAL COMPARISON")
print("=" * 78)

display(
    comparison_df.round(6)
)

print("\n" + "=" * 78)
print("PUBLICATION-STYLE COMPARISON")
print("=" * 78)

display(
    publication_comparison_df.round(3)
)

# ------------------------------------------------------------
# 13. Accuracy comparison figure
# ------------------------------------------------------------

methods = comparison_df[
    "Method"
].tolist()

x_positions = np.arange(
    len(methods)
)

bar_width = 0.36

holdout_accuracy = (
    comparison_df[
        "Repeated_Holdout_Accuracy_Mean"
    ].to_numpy() * 100
)

holdout_accuracy_sd = (
    comparison_df[
        "Repeated_Holdout_Accuracy_SD"
    ].to_numpy() * 100
)

oof_accuracy = (
    comparison_df[
        "OOF_Accuracy"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(10, 6)
)

holdout_bars = plt.bar(
    x_positions - bar_width / 2,
    holdout_accuracy,
    width=bar_width,
    yerr=holdout_accuracy_sd,
    capsize=5,
    label="Repeated hold-out mean ± SD"
)

oof_bars = plt.bar(
    x_positions + bar_width / 2,
    oof_accuracy,
    width=bar_width,
    label="Five-fold OOF"
)

plt.xticks(
    x_positions,
    methods
)

plt.ylabel(
    "Accuracy (%)"
)

plt.xlabel(
    "Ensemble method"
)

plt.title(
    "CIFAR-10: Repeated Hold-out vs Five-fold OOF Accuracy"
)

all_accuracy_values = np.concatenate(
    [
        holdout_accuracy
        - holdout_accuracy_sd,
        oof_accuracy
    ]
)

lower_limit = max(
    0,
    np.min(
        all_accuracy_values
    ) - 2
)

upper_limit = min(
    100,
    max(
        np.max(
            holdout_accuracy
            + holdout_accuracy_sd
        ),
        np.max(
            oof_accuracy
        )
    ) + 2
)

plt.ylim(
    lower_limit,
    upper_limit
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

for bars in [
    holdout_bars,
    oof_bars
]:

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.15,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

accuracy_figure_path = os.path.join(
    FIGURES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Accuracy.png"
)

plt.savefig(
    accuracy_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 14. Macro F1 comparison figure
# ------------------------------------------------------------

holdout_f1 = (
    comparison_df[
        "Repeated_Holdout_Macro_F1_Mean"
    ].to_numpy() * 100
)

holdout_f1_sd = (
    comparison_df[
        "Repeated_Holdout_Macro_F1_SD"
    ].to_numpy() * 100
)

oof_f1 = (
    comparison_df[
        "OOF_Macro_F1"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(10, 6)
)

holdout_f1_bars = plt.bar(
    x_positions - bar_width / 2,
    holdout_f1,
    width=bar_width,
    yerr=holdout_f1_sd,
    capsize=5,
    label="Repeated hold-out mean ± SD"
)

oof_f1_bars = plt.bar(
    x_positions + bar_width / 2,
    oof_f1,
    width=bar_width,
    label="Five-fold OOF"
)

plt.xticks(
    x_positions,
    methods
)

plt.ylabel(
    "Macro F1 (%)"
)

plt.xlabel(
    "Ensemble method"
)

plt.title(
    "CIFAR-10: Repeated Hold-out vs Five-fold OOF Macro F1"
)

all_f1_values = np.concatenate(
    [
        holdout_f1
        - holdout_f1_sd,
        oof_f1
    ]
)

plt.ylim(
    max(
        0,
        np.min(
            all_f1_values
        ) - 2
    ),
    min(
        100,
        max(
            np.max(
                holdout_f1
                + holdout_f1_sd
            ),
            np.max(
                oof_f1
            )
        ) + 2
    )
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

for bars in [
    holdout_f1_bars,
    oof_f1_bars
]:

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.15,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

f1_figure_path = os.path.join(
    FIGURES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Macro_F1.png"
)

plt.savefig(
    f1_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 15. Accuracy-difference figure
# ------------------------------------------------------------

accuracy_differences = (
    comparison_df[
        "OOF_minus_Holdout_Accuracy"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(9, 5.5)
)

difference_bars = plt.bar(
    methods,
    accuracy_differences
)

plt.axhline(
    y=0,
    linewidth=1
)

plt.ylabel(
    "OOF − repeated hold-out accuracy\n(percentage points)"
)

plt.xlabel(
    "Ensemble method"
)

plt.title(
    "CIFAR-10 Accuracy Difference Between Evaluation Protocols"
)

plt.grid(
    axis="y",
    alpha=0.3
)

for bar, value in zip(
    difference_bars,
    accuracy_differences
):

    vertical_alignment = (
        "bottom"
        if value >= 0
        else "top"
    )

    offset = (
        0.05
        if value >= 0
        else -0.05
    )

    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        value + offset,
        f"{value:+.3f}",
        ha="center",
        va=vertical_alignment,
        fontsize=10
    )

plt.tight_layout()

difference_figure_path = os.path.join(
    FIGURES_DIR,
    "CIFAR10_OOF_minus_Repeated_Holdout_Accuracy.png"
)

plt.savefig(
    difference_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 16. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("CIFAR-10 REPEATED HOLD-OUT vs OOF COMPARISON COMPLETED")
print("=" * 78)

print("\nSaved tables:")
print(numeric_comparison_path)
print(publication_comparison_path)

print("\nSaved figures:")
print(accuracy_figure_path)
print(f1_figure_path)
print(difference_figure_path)

In [ ]:
# ============================================================
# CIFAR-10
# REPEATED HOLD-OUT MEAN vs FIVE-FOLD OOF
# CORRECTED COMPARISON:
# Logistic Regression and LightGBM only
# Soft Voting retained separately as common baseline
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

CIFAR_REPEATED_HOLDOUT_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Repeated_Holdout",
    "CIFAR10"
)

REPEATED_SUMMARY_PATH = os.path.join(
    CIFAR_REPEATED_HOLDOUT_DIR,
    "CIFAR10_Repeated_Holdout_Mean_SD_Summary.csv"
)

ORIGINAL_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "CIFAR10",
    "Results",
    "CIFAR10_Matched_Holdout_vs_OOF_Results.csv"
)

EXTENDED_EXPERIMENTS_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments"
)

TABLES_DIR = os.path.join(
    EXTENDED_EXPERIMENTS_DIR,
    "Tables"
)

FIGURES_DIR = os.path.join(
    EXTENDED_EXPERIMENTS_DIR,
    "Figures"
)

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. Input verification
# ------------------------------------------------------------

required_files = {
    "Repeated hold-out summary":
        REPEATED_SUMMARY_PATH,

    "Original matched results":
        ORIGINAL_RESULTS_PATH
}

print("=" * 78)
print("INPUT FILE CHECK")
print("=" * 78)

for label, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{label:<30}: {exists}")
    print(path)

    if not exists:
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# 3. Configuration
# ------------------------------------------------------------

COMPARISON_METHODS = [
    "Logistic Regression",
    "LightGBM"
]

ALL_METHODS = [
    "Soft Voting",
    "Logistic Regression",
    "LightGBM"
]

# ------------------------------------------------------------
# 4. Method normalization
# ------------------------------------------------------------

def normalize_method_name(value):

    value = str(value).strip()

    replacements = {
        "SoftVoting": "Soft Voting",
        "Soft_Voting": "Soft Voting",
        "Soft-Voting": "Soft Voting",
        "Voting": "Soft Voting",

        "LR": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression Stacking":
            "Logistic Regression",

        "LGBM": "LightGBM",
        "Light GBM": "LightGBM",
        "LightGBM Stacking": "LightGBM"
    }

    return replacements.get(value, value)

# ------------------------------------------------------------
# 5. Column normalization
# ------------------------------------------------------------

def normalize_columns(dataframe):

    dataframe = dataframe.copy()

    rename_map = {
        "Model": "Method",
        "Meta_Learner": "Method",

        "Precision": "Macro_Precision",
        "Recall": "Macro_Recall",

        "F1": "Macro_F1",
        "F1_Score": "Macro_F1",
        "Macro_F1_Score": "Macro_F1",

        "AUC": "Macro_ROC_AUC_OVR",
        "ROC_AUC": "Macro_ROC_AUC_OVR",
        "ROC_AUC_OVR": "Macro_ROC_AUC_OVR",
        "Macro_AUC": "Macro_ROC_AUC_OVR",

        "LogLoss": "Log_Loss",
        "Brier": "Multiclass_Brier",
        "ECE": "ECE_15_Bins"
    }

    for old_name, new_name in rename_map.items():

        if (
            old_name in dataframe.columns
            and new_name not in dataframe.columns
        ):

            dataframe = dataframe.rename(
                columns={
                    old_name: new_name
                }
            )

    if "Method" in dataframe.columns:

        dataframe["Method"] = (
            dataframe["Method"]
            .apply(normalize_method_name)
        )

    return dataframe

# ------------------------------------------------------------
# 6. Load repeated hold-out summary
# ------------------------------------------------------------

repeated_df = pd.read_csv(
    REPEATED_SUMMARY_PATH
)

repeated_df = normalize_columns(
    repeated_df
)

repeated_df = repeated_df[
    repeated_df["Method"].isin(
        ALL_METHODS
    )
].copy()

required_repeated_columns = [
    "Method",
    "Accuracy_Mean",
    "Accuracy_SD",
    "Macro_F1_Mean",
    "Macro_F1_SD",
    "Macro_ROC_AUC_OVR_Mean",
    "Macro_ROC_AUC_OVR_SD"
]

missing_repeated_columns = [
    column
    for column in required_repeated_columns
    if column not in repeated_df.columns
]

if missing_repeated_columns:
    raise ValueError(
        "Repeated summary is missing columns:\n"
        f"{missing_repeated_columns}"
    )

print("\nRepeated hold-out summary:")

display(
    repeated_df[
        required_repeated_columns
    ].round(6)
)

# ------------------------------------------------------------
# 7. Load original matched results
# ------------------------------------------------------------

original_df = pd.read_csv(
    ORIGINAL_RESULTS_PATH
)

original_df = normalize_columns(
    original_df
)

required_original_columns = [
    "Protocol",
    "Method",
    "Accuracy",
    "Macro_F1",
    "Macro_ROC_AUC_OVR"
]

missing_original_columns = [
    column
    for column in required_original_columns
    if column not in original_df.columns
]

if missing_original_columns:
    raise ValueError(
        "Original results are missing columns:\n"
        f"{missing_original_columns}"
    )

original_df["Protocol"] = (
    original_df["Protocol"]
    .astype(str)
    .str.strip()
)

print("\nOriginal protocol rows:")

display(
    original_df[
        required_original_columns
    ].round(6)
)

# ------------------------------------------------------------
# 8. Extract five-fold OOF rows
# ------------------------------------------------------------

oof_df = original_df[
    original_df["Protocol"]
    .str.lower()
    .str.contains(
        "five-fold oof",
        na=False
    )
].copy()

oof_df = oof_df[
    oof_df["Method"].isin(
        COMPARISON_METHODS
    )
].copy()

oof_df = oof_df.drop_duplicates(
    subset=["Method"],
    keep="last"
)

missing_oof_methods = [
    method
    for method in COMPARISON_METHODS
    if method not in oof_df["Method"].values
]

if missing_oof_methods:
    raise ValueError(
        "OOF rows are missing methods:\n"
        f"{missing_oof_methods}"
    )

print("\nFive-fold OOF rows used for comparison:")

display(
    oof_df[
        required_original_columns
    ].round(6)
)

# ------------------------------------------------------------
# 9. Extract soft-voting common baseline separately
# ------------------------------------------------------------

soft_voting_original_df = original_df[
    original_df["Method"] == "Soft Voting"
].copy()

soft_voting_repeated_df = repeated_df[
    repeated_df["Method"] == "Soft Voting"
].copy()

if len(soft_voting_original_df) > 0:

    soft_voting_original_row = (
        soft_voting_original_df.iloc[0]
    )

    soft_voting_original_accuracy = (
        float(
            soft_voting_original_row[
                "Accuracy"
            ]
        )
    )

else:

    soft_voting_original_accuracy = np.nan

if len(soft_voting_repeated_df) == 1:

    soft_voting_repeated_row = (
        soft_voting_repeated_df.iloc[0]
    )

    soft_voting_repeated_accuracy_mean = (
        float(
            soft_voting_repeated_row[
                "Accuracy_Mean"
            ]
        )
    )

    soft_voting_repeated_accuracy_sd = (
        float(
            soft_voting_repeated_row[
                "Accuracy_SD"
            ]
        )
    )

else:

    soft_voting_repeated_accuracy_mean = np.nan
    soft_voting_repeated_accuracy_sd = np.nan

soft_voting_baseline_df = pd.DataFrame(
    [
        {
            "Dataset": "CIFAR-10",
            "Method": "Soft Voting",
            "Role": "Common non-trained baseline",
            "Original_Common_Baseline_Accuracy":
                soft_voting_original_accuracy,
            "Repeated_Holdout_Accuracy_Mean":
                soft_voting_repeated_accuracy_mean,
            "Repeated_Holdout_Accuracy_SD":
                soft_voting_repeated_accuracy_sd
        }
    ]
)

# ------------------------------------------------------------
# 10. Build matched protocol comparison
# ------------------------------------------------------------

comparison_rows = []

for method in COMPARISON_METHODS:

    repeated_row = repeated_df[
        repeated_df["Method"] == method
    ]

    oof_row = oof_df[
        oof_df["Method"] == method
    ]

    if len(repeated_row) != 1:
        raise ValueError(
            f"Repeated summary has "
            f"{len(repeated_row)} rows for {method}."
        )

    if len(oof_row) != 1:
        raise ValueError(
            f"OOF table has "
            f"{len(oof_row)} rows for {method}."
        )

    repeated_row = repeated_row.iloc[0]
    oof_row = oof_row.iloc[0]

    comparison_rows.append(
        {
            "Dataset": "CIFAR-10",
            "Method": method,

            "Repeated_Holdout_Accuracy_Mean":
                repeated_row[
                    "Accuracy_Mean"
                ],

            "Repeated_Holdout_Accuracy_SD":
                repeated_row[
                    "Accuracy_SD"
                ],

            "Five_Fold_OOF_Accuracy":
                oof_row[
                    "Accuracy"
                ],

            "OOF_minus_Holdout_Accuracy":
                oof_row[
                    "Accuracy"
                ]
                - repeated_row[
                    "Accuracy_Mean"
                ],

            "Repeated_Holdout_Macro_F1_Mean":
                repeated_row[
                    "Macro_F1_Mean"
                ],

            "Repeated_Holdout_Macro_F1_SD":
                repeated_row[
                    "Macro_F1_SD"
                ],

            "Five_Fold_OOF_Macro_F1":
                oof_row[
                    "Macro_F1"
                ],

            "OOF_minus_Holdout_Macro_F1":
                oof_row[
                    "Macro_F1"
                ]
                - repeated_row[
                    "Macro_F1_Mean"
                ],

            "Repeated_Holdout_AUC_Mean":
                repeated_row[
                    "Macro_ROC_AUC_OVR_Mean"
                ],

            "Repeated_Holdout_AUC_SD":
                repeated_row[
                    "Macro_ROC_AUC_OVR_SD"
                ],

            "Five_Fold_OOF_AUC":
                oof_row[
                    "Macro_ROC_AUC_OVR"
                ],

            "OOF_minus_Holdout_AUC":
                oof_row[
                    "Macro_ROC_AUC_OVR"
                ]
                - repeated_row[
                    "Macro_ROC_AUC_OVR_Mean"
                ]
        }
    )

comparison_df = pd.DataFrame(
    comparison_rows
)

# ------------------------------------------------------------
# 11. Publication-style table
# ------------------------------------------------------------

publication_rows = []

for _, row in comparison_df.iterrows():

    publication_rows.append(
        {
            "Method":
                row["Method"],

            "Repeated Hold-out Accuracy (%)":
                (
                    f"{row['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Accuracy_SD'] * 100:.3f}"
                ),

            "Five-fold OOF Accuracy (%)":
                f"{row['Five_Fold_OOF_Accuracy'] * 100:.3f}",

            "OOF − Hold-out Accuracy (pp)":
                (
                    row[
                        "OOF_minus_Holdout_Accuracy"
                    ] * 100
                ),

            "Repeated Hold-out Macro F1 (%)":
                (
                    f"{row['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}"
                ),

            "Five-fold OOF Macro F1 (%)":
                f"{row['Five_Fold_OOF_Macro_F1'] * 100:.3f}",

            "OOF − Hold-out F1 (pp)":
                (
                    row[
                        "OOF_minus_Holdout_Macro_F1"
                    ] * 100
                ),

            "Repeated Hold-out AUC (%)":
                (
                    f"{row['Repeated_Holdout_AUC_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_AUC_SD'] * 100:.3f}"
                ),

            "Five-fold OOF AUC (%)":
                f"{row['Five_Fold_OOF_AUC'] * 100:.3f}",

            "OOF − Hold-out AUC (pp)":
                (
                    row[
                        "OOF_minus_Holdout_AUC"
                    ] * 100
                )
        }
    )

publication_df = pd.DataFrame(
    publication_rows
)

# ------------------------------------------------------------
# 12. Save tables
# ------------------------------------------------------------

comparison_path = os.path.join(
    TABLES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Comparison.csv"
)

publication_path = os.path.join(
    TABLES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Publication_Table.csv"
)

soft_voting_path = os.path.join(
    TABLES_DIR,
    "CIFAR10_SoftVoting_Common_Baseline.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

publication_df.to_csv(
    publication_path,
    index=False
)

soft_voting_baseline_df.to_csv(
    soft_voting_path,
    index=False
)

print("\n" + "=" * 78)
print("MATCHED PROTOCOL COMPARISON")
print("=" * 78)

display(
    comparison_df.round(6)
)

print("\n" + "=" * 78)
print("PUBLICATION-STYLE TABLE")
print("=" * 78)

display(
    publication_df.round(3)
)

print("\n" + "=" * 78)
print("SOFT-VOTING COMMON BASELINE")
print("=" * 78)

display(
    soft_voting_baseline_df.round(6)
)

# ------------------------------------------------------------
# 13. Accuracy figure
# ------------------------------------------------------------

methods = comparison_df[
    "Method"
].tolist()

x_positions = np.arange(
    len(methods)
)

bar_width = 0.36

holdout_accuracy = (
    comparison_df[
        "Repeated_Holdout_Accuracy_Mean"
    ].to_numpy() * 100
)

holdout_accuracy_sd = (
    comparison_df[
        "Repeated_Holdout_Accuracy_SD"
    ].to_numpy() * 100
)

oof_accuracy = (
    comparison_df[
        "Five_Fold_OOF_Accuracy"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(8.5, 6)
)

holdout_bars = plt.bar(
    x_positions - bar_width / 2,
    holdout_accuracy,
    width=bar_width,
    yerr=holdout_accuracy_sd,
    capsize=5,
    label="Repeated hold-out mean ± SD"
)

oof_bars = plt.bar(
    x_positions + bar_width / 2,
    oof_accuracy,
    width=bar_width,
    label="Five-fold OOF"
)

plt.xticks(
    x_positions,
    methods
)

plt.ylabel(
    "Accuracy (%)"
)

plt.xlabel(
    "Meta-learner"
)

plt.title(
    "CIFAR-10: Repeated Hold-out vs Five-fold OOF Accuracy"
)

minimum_value = min(
    np.min(
        holdout_accuracy
        - holdout_accuracy_sd
    ),
    np.min(
        oof_accuracy
    )
)

maximum_value = max(
    np.max(
        holdout_accuracy
        + holdout_accuracy_sd
    ),
    np.max(
        oof_accuracy
    )
)

plt.ylim(
    max(0, minimum_value - 1.5),
    min(100, maximum_value + 1.5)
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

for bars in [
    holdout_bars,
    oof_bars
]:

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.10,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

accuracy_figure_path = os.path.join(
    FIGURES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Accuracy.png"
)

plt.savefig(
    accuracy_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 14. Macro F1 figure
# ------------------------------------------------------------

holdout_f1 = (
    comparison_df[
        "Repeated_Holdout_Macro_F1_Mean"
    ].to_numpy() * 100
)

holdout_f1_sd = (
    comparison_df[
        "Repeated_Holdout_Macro_F1_SD"
    ].to_numpy() * 100
)

oof_f1 = (
    comparison_df[
        "Five_Fold_OOF_Macro_F1"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(8.5, 6)
)

holdout_f1_bars = plt.bar(
    x_positions - bar_width / 2,
    holdout_f1,
    width=bar_width,
    yerr=holdout_f1_sd,
    capsize=5,
    label="Repeated hold-out mean ± SD"
)

oof_f1_bars = plt.bar(
    x_positions + bar_width / 2,
    oof_f1,
    width=bar_width,
    label="Five-fold OOF"
)

plt.xticks(
    x_positions,
    methods
)

plt.ylabel(
    "Macro F1 (%)"
)

plt.xlabel(
    "Meta-learner"
)

plt.title(
    "CIFAR-10: Repeated Hold-out vs Five-fold OOF Macro F1"
)

minimum_f1 = min(
    np.min(
        holdout_f1
        - holdout_f1_sd
    ),
    np.min(
        oof_f1
    )
)

maximum_f1 = max(
    np.max(
        holdout_f1
        + holdout_f1_sd
    ),
    np.max(
        oof_f1
    )
)

plt.ylim(
    max(0, minimum_f1 - 1.5),
    min(100, maximum_f1 + 1.5)
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

for bars in [
    holdout_f1_bars,
    oof_f1_bars
]:

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.10,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

f1_figure_path = os.path.join(
    FIGURES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Macro_F1.png"
)

plt.savefig(
    f1_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 15. Accuracy difference figure
# ------------------------------------------------------------

accuracy_difference = (
    comparison_df[
        "OOF_minus_Holdout_Accuracy"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(8, 5.5)
)

difference_bars = plt.bar(
    methods,
    accuracy_difference
)

plt.axhline(
    y=0,
    linewidth=1
)

plt.ylabel(
    "OOF − repeated hold-out accuracy\n(percentage points)"
)

plt.xlabel(
    "Meta-learner"
)

plt.title(
    "CIFAR-10 Accuracy Difference Between Evaluation Protocols"
)

plt.grid(
    axis="y",
    alpha=0.3
)

for bar, value in zip(
    difference_bars,
    accuracy_difference
):

    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        value + 0.03,
        f"{value:+.3f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()

difference_figure_path = os.path.join(
    FIGURES_DIR,
    "CIFAR10_OOF_minus_Repeated_Holdout_Accuracy.png"
)

plt.savefig(
    difference_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 16. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("CIFAR-10 PROTOCOL COMPARISON COMPLETED SUCCESSFULLY")
print("=" * 78)

print(
    "\nCompared methods:",
    COMPARISON_METHODS
)

print(
    "\nSoft Voting was retained separately as a "
    "common non-trained baseline because no distinct "
    "five-fold OOF row exists for it."
)

print("\nSaved tables:")
print(comparison_path)
print(publication_path)
print(soft_voting_path)

print("\nSaved figures:")
print(accuracy_figure_path)
print(f1_figure_path)
print(difference_figure_path)

In [ ]:
# ============================================================
# COMBINED CIFAR-10 + CT
# REPEATED HOLD-OUT MEAN vs FIVE-FOLD OOF
# COMMON METRICS ONLY:
# Accuracy, ROC-AUC, Log Loss, Brier Score, ECE
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

TABLES_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Tables"
)

os.makedirs(
    TABLES_DIR,
    exist_ok=True
)

CIFAR_COMPARISON_PATH = os.path.join(
    TABLES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Comparison.csv"
)

CT_COMPARISON_PATH = os.path.join(
    TABLES_DIR,
    "CT_Repeated_Holdout_vs_OOF_Comparison.csv"
)

CIFAR_REPEATED_SUMMARY_PATH = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Repeated_Holdout",
    "CIFAR10",
    "CIFAR10_Repeated_Holdout_Mean_SD_Summary.csv"
)

CT_REPEATED_SUMMARY_PATH = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Repeated_Holdout",
    "CT",
    "CT_Repeated_Holdout_Mean_SD_Summary.csv"
)

# ------------------------------------------------------------
# 2. Verify files
# ------------------------------------------------------------

required_files = {
    "CIFAR comparison":
        CIFAR_COMPARISON_PATH,

    "CT comparison":
        CT_COMPARISON_PATH,

    "CIFAR repeated summary":
        CIFAR_REPEATED_SUMMARY_PATH,

    "CT repeated summary":
        CT_REPEATED_SUMMARY_PATH
}

print("=" * 78)
print("INPUT FILE CHECK")
print("=" * 78)

for label, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{label:<28}: {exists}")
    print(path)

    if not exists:
        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )

# ------------------------------------------------------------
# 3. Load files
# ------------------------------------------------------------

cifar_comparison_df = pd.read_csv(
    CIFAR_COMPARISON_PATH
)

ct_comparison_df = pd.read_csv(
    CT_COMPARISON_PATH
)

cifar_summary_df = pd.read_csv(
    CIFAR_REPEATED_SUMMARY_PATH
)

ct_summary_df = pd.read_csv(
    CT_REPEATED_SUMMARY_PATH
)

print("\nCIFAR comparison columns:")
print(list(cifar_comparison_df.columns))

print("\nCT comparison columns:")
print(list(ct_comparison_df.columns))

print("\nCIFAR summary columns:")
print(list(cifar_summary_df.columns))

print("\nCT summary columns:")
print(list(ct_summary_df.columns))

# ------------------------------------------------------------
# 4. Normalize method names
# ------------------------------------------------------------

def normalize_method_name(value):

    value = str(value).strip()

    replacements = {
        "LR": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression Stacking":
            "Logistic Regression",

        "LGBM": "LightGBM",
        "Light GBM": "LightGBM",
        "LightGBM Stacking": "LightGBM"
    }

    return replacements.get(
        value,
        value
    )


for dataframe in [
    cifar_comparison_df,
    ct_comparison_df,
    cifar_summary_df,
    ct_summary_df
]:

    if "Method" in dataframe.columns:

        dataframe["Method"] = (
            dataframe["Method"]
            .apply(
                normalize_method_name
            )
        )

# ------------------------------------------------------------
# 5. Helper for safely retrieving one method row
# ------------------------------------------------------------

def get_method_row(
    dataframe,
    method,
    table_name
):

    method_rows = dataframe[
        dataframe["Method"] == method
    ]

    if len(method_rows) != 1:

        raise ValueError(
            f"{table_name} has {len(method_rows)} rows "
            f"for {method}; expected exactly 1."
        )

    return method_rows.iloc[0]

# ------------------------------------------------------------
# 6. Methods common to both datasets
# ------------------------------------------------------------

METHODS = [
    "Logistic Regression",
    "LightGBM"
]

# ------------------------------------------------------------
# 7. Build normalized CIFAR rows
# ------------------------------------------------------------

cifar_rows = []

for method in METHODS:

    comparison_row = get_method_row(
        cifar_comparison_df,
        method,
        "CIFAR comparison"
    )

    summary_row = get_method_row(
        cifar_summary_df,
        method,
        "CIFAR repeated summary"
    )

    cifar_rows.append(
        {
            "Dataset": "CIFAR-10",
            "Method": method,

            "Repeated_Holdout_Accuracy_Mean":
                float(
                    comparison_row[
                        "Repeated_Holdout_Accuracy_Mean"
                    ]
                ),

            "Repeated_Holdout_Accuracy_SD":
                float(
                    comparison_row[
                        "Repeated_Holdout_Accuracy_SD"
                    ]
                ),

            "Five_Fold_OOF_Accuracy":
                float(
                    comparison_row[
                        "Five_Fold_OOF_Accuracy"
                    ]
                ),

            "OOF_minus_Holdout_Accuracy":
                float(
                    comparison_row[
                        "OOF_minus_Holdout_Accuracy"
                    ]
                ),

            "Repeated_Holdout_ROC_AUC_Mean":
                float(
                    comparison_row[
                        "Repeated_Holdout_AUC_Mean"
                    ]
                ),

            "Repeated_Holdout_ROC_AUC_SD":
                float(
                    comparison_row[
                        "Repeated_Holdout_AUC_SD"
                    ]
                ),

            "Five_Fold_OOF_ROC_AUC":
                float(
                    comparison_row[
                        "Five_Fold_OOF_AUC"
                    ]
                ),

            "OOF_minus_Holdout_ROC_AUC":
                float(
                    comparison_row[
                        "OOF_minus_Holdout_AUC"
                    ]
                ),

            "Repeated_Holdout_Log_Loss_Mean":
                float(
                    summary_row[
                        "Log_Loss_Mean"
                    ]
                ),

            "Repeated_Holdout_Log_Loss_SD":
                float(
                    summary_row[
                        "Log_Loss_SD"
                    ]
                ),

            "Five_Fold_OOF_Log_Loss":
                np.nan,

            "OOF_minus_Holdout_Log_Loss":
                np.nan,

            "Repeated_Holdout_Brier_Mean":
                float(
                    summary_row[
                        "Multiclass_Brier_Mean"
                    ]
                ),

            "Repeated_Holdout_Brier_SD":
                float(
                    summary_row[
                        "Multiclass_Brier_SD"
                    ]
                ),

            "Five_Fold_OOF_Brier":
                np.nan,

            "OOF_minus_Holdout_Brier":
                np.nan,

            "Repeated_Holdout_ECE_Mean":
                float(
                    summary_row[
                        "ECE_15_Bins_Mean"
                    ]
                ),

            "Repeated_Holdout_ECE_SD":
                float(
                    summary_row[
                        "ECE_15_Bins_SD"
                    ]
                ),

            "Five_Fold_OOF_ECE":
                np.nan,

            "OOF_minus_Holdout_ECE":
                np.nan
        }
    )

cifar_normalized_df = pd.DataFrame(
    cifar_rows
)

# ------------------------------------------------------------
# 8. Build normalized CT rows
# ------------------------------------------------------------

ct_rows = []

for method in METHODS:

    comparison_row = get_method_row(
        ct_comparison_df,
        method,
        "CT comparison"
    )

    summary_row = get_method_row(
        ct_summary_df,
        method,
        "CT repeated summary"
    )

    # Support either uppercase/lowercase Minus spelling
    if "OOF_Minus_Holdout_Mean" in comparison_row.index:

        accuracy_difference = float(
            comparison_row[
                "OOF_Minus_Holdout_Mean"
            ]
        )

    else:

        accuracy_difference = (
            float(
                comparison_row[
                    "OOF_Accuracy"
                ]
            )
            - float(
                comparison_row[
                    "Repeated_Holdout_Accuracy_Mean"
                ]
            )
        )

    repeated_auc_sd = np.nan

    for candidate in [
        "ROC_AUC_SD",
        "AUC_SD",
        "Macro_ROC_AUC_OVR_SD"
    ]:

        if candidate in summary_row.index:

            repeated_auc_sd = float(
                summary_row[
                    candidate
                ]
            )

            break

    repeated_logloss_sd = np.nan

    for candidate in [
        "Log_Loss_SD",
        "LogLoss_SD"
    ]:

        if candidate in summary_row.index:

            repeated_logloss_sd = float(
                summary_row[
                    candidate
                ]
            )

            break

    repeated_brier_sd = np.nan

    for candidate in [
        "Brier_SD",
        "Brier_Score_SD",
        "Multiclass_Brier_SD"
    ]:

        if candidate in summary_row.index:

            repeated_brier_sd = float(
                summary_row[
                    candidate
                ]
            )

            break

    repeated_ece_sd = np.nan

    for candidate in [
        "ECE_SD",
        "ECE_15_Bins_SD"
    ]:

        if candidate in summary_row.index:

            repeated_ece_sd = float(
                summary_row[
                    candidate
                ]
            )

            break

    ct_rows.append(
        {
            "Dataset": "CT",
            "Method": method,

            "Repeated_Holdout_Accuracy_Mean":
                float(
                    comparison_row[
                        "Repeated_Holdout_Accuracy_Mean"
                    ]
                ),

            "Repeated_Holdout_Accuracy_SD":
                float(
                    comparison_row[
                        "Repeated_Holdout_Accuracy_SD"
                    ]
                ),

            "Five_Fold_OOF_Accuracy":
                float(
                    comparison_row[
                        "OOF_Accuracy"
                    ]
                ),

            "OOF_minus_Holdout_Accuracy":
                accuracy_difference,

            "Repeated_Holdout_ROC_AUC_Mean":
                float(
                    comparison_row[
                        "Repeated_Holdout_ROC_AUC_Mean"
                    ]
                ),

            "Repeated_Holdout_ROC_AUC_SD":
                repeated_auc_sd,

            "Five_Fold_OOF_ROC_AUC":
                float(
                    comparison_row[
                        "OOF_ROC_AUC"
                    ]
                ),

            "OOF_minus_Holdout_ROC_AUC":
                float(
                    comparison_row[
                        "OOF_ROC_AUC"
                    ]
                )
                - float(
                    comparison_row[
                        "Repeated_Holdout_ROC_AUC_Mean"
                    ]
                ),

            "Repeated_Holdout_Log_Loss_Mean":
                float(
                    comparison_row[
                        "Repeated_Holdout_Log_Loss_Mean"
                    ]
                ),

            "Repeated_Holdout_Log_Loss_SD":
                repeated_logloss_sd,

            "Five_Fold_OOF_Log_Loss":
                float(
                    comparison_row[
                        "OOF_Log_Loss"
                    ]
                ),

            "OOF_minus_Holdout_Log_Loss":
                float(
                    comparison_row[
                        "OOF_Log_Loss"
                    ]
                )
                - float(
                    comparison_row[
                        "Repeated_Holdout_Log_Loss_Mean"
                    ]
                ),

            "Repeated_Holdout_Brier_Mean":
                float(
                    comparison_row[
                        "Repeated_Holdout_Brier_Mean"
                    ]
                ),

            "Repeated_Holdout_Brier_SD":
                repeated_brier_sd,

            "Five_Fold_OOF_Brier":
                float(
                    comparison_row[
                        "OOF_Brier"
                    ]
                ),

            "OOF_minus_Holdout_Brier":
                float(
                    comparison_row[
                        "OOF_Brier"
                    ]
                )
                - float(
                    comparison_row[
                        "Repeated_Holdout_Brier_Mean"
                    ]
                ),

            "Repeated_Holdout_ECE_Mean":
                float(
                    comparison_row[
                        "Repeated_Holdout_ECE_Mean"
                    ]
                ),

            "Repeated_Holdout_ECE_SD":
                repeated_ece_sd,

            "Five_Fold_OOF_ECE":
                float(
                    comparison_row[
                        "OOF_ECE"
                    ]
                ),

            "OOF_minus_Holdout_ECE":
                float(
                    comparison_row[
                        "OOF_ECE"
                    ]
                )
                - float(
                    comparison_row[
                        "Repeated_Holdout_ECE_Mean"
                    ]
                )
        }
    )

ct_normalized_df = pd.DataFrame(
    ct_rows
)

# ------------------------------------------------------------
# 9. Combine both datasets
# ------------------------------------------------------------

combined_df = pd.concat(
    [
        cifar_normalized_df,
        ct_normalized_df
    ],
    ignore_index=True
)

dataset_rank = {
    "CIFAR-10": 0,
    "CT": 1
}

method_rank = {
    "Logistic Regression": 0,
    "LightGBM": 1
}

combined_df["_Dataset_Order"] = (
    combined_df["Dataset"]
    .map(dataset_rank)
)

combined_df["_Method_Order"] = (
    combined_df["Method"]
    .map(method_rank)
)

combined_df = (
    combined_df
    .sort_values(
        [
            "_Dataset_Order",
            "_Method_Order"
        ]
    )
    .drop(
        columns=[
            "_Dataset_Order",
            "_Method_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------------------
# 10. Publication-style accuracy and AUC table
# ------------------------------------------------------------

publication_rows = []

for _, row in combined_df.iterrows():

    auc_sd_text = (
        f"{row['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}"
        if np.isfinite(
            row[
                "Repeated_Holdout_ROC_AUC_SD"
            ]
        )
        else "NR"
    )

    publication_rows.append(
        {
            "Dataset":
                row["Dataset"],

            "Method":
                row["Method"],

            "Repeated hold-out accuracy (%)":
                (
                    f"{row['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Accuracy_SD'] * 100:.3f}"
                ),

            "Five-fold OOF accuracy (%)":
                f"{row['Five_Fold_OOF_Accuracy'] * 100:.3f}",

            "OOF − hold-out accuracy (pp)":
                row[
                    "OOF_minus_Holdout_Accuracy"
                ] * 100,

            "Repeated hold-out ROC-AUC (%)":
                (
                    f"{row['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
                    f"± {auc_sd_text}"
                ),

            "Five-fold OOF ROC-AUC (%)":
                f"{row['Five_Fold_OOF_ROC_AUC'] * 100:.3f}",

            "OOF − hold-out ROC-AUC (pp)":
                row[
                    "OOF_minus_Holdout_ROC_AUC"
                ] * 100
        }
    )

publication_df = pd.DataFrame(
    publication_rows
)

# ------------------------------------------------------------
# 11. Calibration and probability-quality table
# ------------------------------------------------------------

calibration_rows = []

for _, row in combined_df.iterrows():

    calibration_rows.append(
        {
            "Dataset":
                row["Dataset"],

            "Method":
                row["Method"],

            "Repeated hold-out log loss":
                row[
                    "Repeated_Holdout_Log_Loss_Mean"
                ],

            "Five-fold OOF log loss":
                row[
                    "Five_Fold_OOF_Log_Loss"
                ],

            "Repeated hold-out Brier":
                row[
                    "Repeated_Holdout_Brier_Mean"
                ],

            "Five-fold OOF Brier":
                row[
                    "Five_Fold_OOF_Brier"
                ],

            "Repeated hold-out ECE":
                row[
                    "Repeated_Holdout_ECE_Mean"
                ],

            "Five-fold OOF ECE":
                row[
                    "Five_Fold_OOF_ECE"
                ]
        }
    )

calibration_df = pd.DataFrame(
    calibration_rows
)

# ------------------------------------------------------------
# 12. Accuracy-gain summary
# ------------------------------------------------------------

accuracy_gain_summary_df = (
    combined_df
    .groupby(
        "Dataset",
        as_index=False
    )
    .agg(
        Mean_OOF_Accuracy_Gain=(
            "OOF_minus_Holdout_Accuracy",
            "mean"
        ),

        Minimum_OOF_Accuracy_Gain=(
            "OOF_minus_Holdout_Accuracy",
            "min"
        ),

        Maximum_OOF_Accuracy_Gain=(
            "OOF_minus_Holdout_Accuracy",
            "max"
        ),

        Mean_OOF_ROC_AUC_Gain=(
            "OOF_minus_Holdout_ROC_AUC",
            "mean"
        )
    )
)

for column in [
    "Mean_OOF_Accuracy_Gain",
    "Minimum_OOF_Accuracy_Gain",
    "Maximum_OOF_Accuracy_Gain",
    "Mean_OOF_ROC_AUC_Gain"
]:

    accuracy_gain_summary_df[
        column + "_pp"
    ] = (
        accuracy_gain_summary_df[
            column
        ] * 100
    )

# ------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------

numeric_output_path = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Common_Metric_Comparison.csv"
)

publication_output_path = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Accuracy_AUC_Publication_Table.csv"
)

calibration_output_path = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Calibration_Comparison.csv"
)

gain_summary_output_path = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_OOF_Gain_Summary.csv"
)

combined_df.to_csv(
    numeric_output_path,
    index=False
)

publication_df.to_csv(
    publication_output_path,
    index=False
)

calibration_df.to_csv(
    calibration_output_path,
    index=False
)

accuracy_gain_summary_df.to_csv(
    gain_summary_output_path,
    index=False
)

# ------------------------------------------------------------
# 14. Display
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("COMBINED NUMERICAL COMMON-METRIC TABLE")
print("=" * 78)

display(
    combined_df.round(6)
)

print("\n" + "=" * 78)
print("PUBLICATION-STYLE ACCURACY AND ROC-AUC TABLE")
print("=" * 78)

display(
    publication_df.round(3)
)

print("\n" + "=" * 78)
print("CALIBRATION AND PROBABILITY-QUALITY TABLE")
print("=" * 78)

display(
    calibration_df.round(6)
)

print("\n" + "=" * 78)
print("CROSS-DATASET OOF GAIN SUMMARY")
print("=" * 78)

display(
    accuracy_gain_summary_df.round(6)
)

# ------------------------------------------------------------
# 15. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("COMBINED CIFAR-10 AND CT COMPARISON COMPLETED")
print("=" * 78)

print("\nSaved files:")
print(numeric_output_path)
print(publication_output_path)
print(calibration_output_path)
print(gain_summary_output_path)

print(
    "\nNote: Macro F1 was excluded from the combined "
    "cross-dataset table because the CT comparison file "
    "does not contain macro F1 values."
)

In [ ]:
# ============================================================
# CELL 1
# REBUILD CT REPEATED HOLD-OUT vs FIVE-FOLD OOF COMPARISON
# INCLUDING MACRO F1
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

CT_REPEATED_SUMMARY_PATH = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Repeated_Holdout",
    "CT",
    "CT_Repeated_Holdout_Mean_SD_Summary.csv"
)

CT_ORIGINAL_RESULTS_PATH = os.path.join(
    PROJECT_ROOT,
    "CT",
    "Results",
    "CT_Matched_Holdout_vs_OOF_Results.csv"
)

TABLES_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Tables"
)

os.makedirs(
    TABLES_DIR,
    exist_ok=True
)

CT_UPDATED_COMPARISON_PATH = os.path.join(
    TABLES_DIR,
    "CT_Repeated_Holdout_vs_OOF_Comparison_With_Macro_F1.csv"
)

CT_UPDATED_PUBLICATION_PATH = os.path.join(
    TABLES_DIR,
    "CT_Repeated_Holdout_vs_OOF_Publication_Table_With_Macro_F1.csv"
)

# ------------------------------------------------------------
# 2. Verify input files
# ------------------------------------------------------------

required_files = {
    "CT repeated hold-out summary":
        CT_REPEATED_SUMMARY_PATH,

    "CT original matched results":
        CT_ORIGINAL_RESULTS_PATH
}

print("=" * 80)
print("CELL 1: INPUT FILE CHECK")
print("=" * 80)

for label, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{label:<34}: {exists}")
    print(path)

    if not exists:
        raise FileNotFoundError(
            f"\nRequired file not found:\n{path}"
        )

# ------------------------------------------------------------
# 3. Load input tables
# ------------------------------------------------------------

ct_repeated_df = pd.read_csv(
    CT_REPEATED_SUMMARY_PATH
)

ct_original_df = pd.read_csv(
    CT_ORIGINAL_RESULTS_PATH
)

print("\nCT repeated summary columns:")
print(list(ct_repeated_df.columns))

print("\nCT original comparison columns:")
print(list(ct_original_df.columns))

# ------------------------------------------------------------
# 4. Normalize method names
# ------------------------------------------------------------

def normalize_method_name(value):

    value = str(value).strip()

    replacements = {
        "LR": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression Stacking":
            "Logistic Regression",

        "LGBM": "LightGBM",
        "Light GBM": "LightGBM",
        "LightGBM Stacking": "LightGBM",

        "SoftVoting": "Soft Voting",
        "Soft_Voting": "Soft Voting"
    }

    return replacements.get(
        value,
        value
    )


for dataframe in [
    ct_repeated_df,
    ct_original_df
]:

    if "Method" not in dataframe.columns:
        raise ValueError(
            "A required input table does not contain "
            "a Method column."
        )

    dataframe["Method"] = (
        dataframe["Method"]
        .apply(
            normalize_method_name
        )
    )

# ------------------------------------------------------------
# 5. Normalize possible metric-column names
# ------------------------------------------------------------

original_rename_map = {
    "ROC_AUC_OVR": "Macro_ROC_AUC_OVR",
    "ROC_AUC": "Macro_ROC_AUC_OVR",
    "AUC": "Macro_ROC_AUC_OVR",

    "F1": "Macro_F1",
    "F1_Score": "Macro_F1",
    "Macro_F1_Score": "Macro_F1",

    "LogLoss": "Log_Loss",

    "Brier": "Brier_Score",
    "Multiclass_Brier": "Brier_Score",

    "ECE_15_Bins": "ECE"
}

for old_name, new_name in original_rename_map.items():

    if (
        old_name in ct_original_df.columns
        and new_name not in ct_original_df.columns
    ):

        ct_original_df = ct_original_df.rename(
            columns={
                old_name: new_name
            }
        )

# ------------------------------------------------------------
# 6. Validate repeated hold-out summary columns
# ------------------------------------------------------------

required_repeated_columns = [
    "Method",

    "Accuracy_Mean",
    "Accuracy_SD",

    "Macro_F1_Mean",
    "Macro_F1_SD",

    "ROC_AUC_Mean",
    "ROC_AUC_SD",

    "Log_Loss_Mean",
    "Log_Loss_SD",

    "Brier_Score_Mean",
    "Brier_Score_SD",

    "ECE_Mean",
    "ECE_SD"
]

missing_repeated_columns = [
    column
    for column in required_repeated_columns
    if column not in ct_repeated_df.columns
]

if missing_repeated_columns:
    raise ValueError(
        "CT repeated-holdout summary is missing:\n"
        f"{missing_repeated_columns}\n\n"
        "Available columns:\n"
        f"{list(ct_repeated_df.columns)}"
    )

# ------------------------------------------------------------
# 7. Identify the protocol column
# ------------------------------------------------------------

protocol_candidates = [
    "Protocol",
    "Training_Protocol",
    "Evaluation_Protocol",
    "Approach",
    "Validation_Protocol"
]

protocol_column = next(
    (
        column
        for column in protocol_candidates
        if column in ct_original_df.columns
    ),
    None
)

if protocol_column is None:
    raise ValueError(
        "No protocol column was found in the original "
        "CT comparison table.\n\n"
        f"Available columns:\n{list(ct_original_df.columns)}"
    )

ct_original_df[protocol_column] = (
    ct_original_df[protocol_column]
    .astype(str)
    .str.strip()
)

print("\nProtocol column identified:")
print(protocol_column)

print("\nAvailable CT protocol and method rows:")

display(
    ct_original_df[
        [
            protocol_column,
            "Method"
        ]
        +
        [
            column
            for column in [
                "Accuracy",
                "Macro_F1",
                "Macro_ROC_AUC_OVR",
                "Log_Loss",
                "Brier_Score",
                "ECE"
            ]
            if column in ct_original_df.columns
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 8. Extract five-fold OOF rows
# ------------------------------------------------------------

protocol_text = (
    ct_original_df[protocol_column]
    .astype(str)
    .str.lower()
)

oof_mask = (
    protocol_text.str.contains(
        "five-fold",
        na=False
    )
    |
    protocol_text.str.contains(
        "5-fold",
        na=False
    )
    |
    protocol_text.str.contains(
        "oof",
        na=False
    )
    |
    protocol_text.str.contains(
        "out-of-fold",
        na=False
    )
    |
    protocol_text.str.contains(
        "out of fold",
        na=False
    )
)

ct_oof_df = ct_original_df[
    oof_mask
].copy()

METHODS = [
    "Logistic Regression",
    "LightGBM"
]

ct_oof_df = ct_oof_df[
    ct_oof_df["Method"].isin(
        METHODS
    )
].copy()

ct_oof_df = ct_oof_df.drop_duplicates(
    subset=["Method"],
    keep="last"
)

missing_oof_methods = [
    method
    for method in METHODS
    if method not in ct_oof_df["Method"].values
]

if missing_oof_methods:
    raise ValueError(
        "The original CT table is missing OOF rows for:\n"
        f"{missing_oof_methods}"
    )

# ------------------------------------------------------------
# 9. Validate essential OOF metric columns
# ------------------------------------------------------------

essential_oof_columns = [
    "Accuracy",
    "Macro_F1",
    "Macro_ROC_AUC_OVR"
]

missing_oof_metrics = [
    column
    for column in essential_oof_columns
    if column not in ct_oof_df.columns
]

if missing_oof_metrics:
    raise ValueError(
        "The CT OOF table is missing essential metrics:\n"
        f"{missing_oof_metrics}\n\n"
        "Available columns:\n"
        f"{list(ct_oof_df.columns)}"
    )

# ------------------------------------------------------------
# 10. Helper for retrieving exactly one method row
# ------------------------------------------------------------

def get_method_row(
    dataframe,
    method,
    table_name
):

    rows = dataframe[
        dataframe["Method"] == method
    ]

    if len(rows) != 1:
        raise ValueError(
            f"{table_name} contains {len(rows)} rows "
            f"for {method}; expected exactly one."
        )

    return rows.iloc[0]

# ------------------------------------------------------------
# 11. Safely retrieve optional OOF metrics
# ------------------------------------------------------------

def get_optional_numeric(
    row,
    candidates
):

    for column in candidates:

        if column in row.index:

            value = row[column]

            if pd.notna(value):
                return float(value)

    return np.nan

# ------------------------------------------------------------
# 12. Build complete CT comparison table
# ------------------------------------------------------------

comparison_rows = []

for method in METHODS:

    repeated_row = get_method_row(
        ct_repeated_df,
        method,
        "CT repeated summary"
    )

    oof_row = get_method_row(
        ct_oof_df,
        method,
        "CT OOF results"
    )

    holdout_accuracy = float(
        repeated_row[
            "Accuracy_Mean"
        ]
    )

    oof_accuracy = float(
        oof_row[
            "Accuracy"
        ]
    )

    holdout_macro_f1 = float(
        repeated_row[
            "Macro_F1_Mean"
        ]
    )

    oof_macro_f1 = float(
        oof_row[
            "Macro_F1"
        ]
    )

    holdout_auc = float(
        repeated_row[
            "ROC_AUC_Mean"
        ]
    )

    oof_auc = float(
        oof_row[
            "Macro_ROC_AUC_OVR"
        ]
    )

    oof_log_loss = get_optional_numeric(
        oof_row,
        [
            "Log_Loss",
            "LogLoss"
        ]
    )

    oof_brier = get_optional_numeric(
        oof_row,
        [
            "Brier_Score",
            "Multiclass_Brier",
            "Brier"
        ]
    )

    oof_ece = get_optional_numeric(
        oof_row,
        [
            "ECE",
            "ECE_15_Bins"
        ]
    )

    comparison_rows.append(
        {
            "Dataset": "CT",
            "Method": method,

            "Repeated_Holdout_Accuracy_Mean":
                holdout_accuracy,

            "Repeated_Holdout_Accuracy_SD":
                float(
                    repeated_row[
                        "Accuracy_SD"
                    ]
                ),

            "Five_Fold_OOF_Accuracy":
                oof_accuracy,

            "OOF_minus_Holdout_Accuracy":
                oof_accuracy
                - holdout_accuracy,

            "Repeated_Holdout_Macro_F1_Mean":
                holdout_macro_f1,

            "Repeated_Holdout_Macro_F1_SD":
                float(
                    repeated_row[
                        "Macro_F1_SD"
                    ]
                ),

            "Five_Fold_OOF_Macro_F1":
                oof_macro_f1,

            "OOF_minus_Holdout_Macro_F1":
                oof_macro_f1
                - holdout_macro_f1,

            "Repeated_Holdout_ROC_AUC_Mean":
                holdout_auc,

            "Repeated_Holdout_ROC_AUC_SD":
                float(
                    repeated_row[
                        "ROC_AUC_SD"
                    ]
                ),

            "Five_Fold_OOF_ROC_AUC":
                oof_auc,

            "OOF_minus_Holdout_ROC_AUC":
                oof_auc
                - holdout_auc,

            "Repeated_Holdout_Log_Loss_Mean":
                float(
                    repeated_row[
                        "Log_Loss_Mean"
                    ]
                ),

            "Repeated_Holdout_Log_Loss_SD":
                float(
                    repeated_row[
                        "Log_Loss_SD"
                    ]
                ),

            "Five_Fold_OOF_Log_Loss":
                oof_log_loss,

            "OOF_minus_Holdout_Log_Loss":
                (
                    oof_log_loss
                    - float(
                        repeated_row[
                            "Log_Loss_Mean"
                        ]
                    )
                    if np.isfinite(oof_log_loss)
                    else np.nan
                ),

            "Repeated_Holdout_Brier_Mean":
                float(
                    repeated_row[
                        "Brier_Score_Mean"
                    ]
                ),

            "Repeated_Holdout_Brier_SD":
                float(
                    repeated_row[
                        "Brier_Score_SD"
                    ]
                ),

            "Five_Fold_OOF_Brier":
                oof_brier,

            "OOF_minus_Holdout_Brier":
                (
                    oof_brier
                    - float(
                        repeated_row[
                            "Brier_Score_Mean"
                        ]
                    )
                    if np.isfinite(oof_brier)
                    else np.nan
                ),

            "Repeated_Holdout_ECE_Mean":
                float(
                    repeated_row[
                        "ECE_Mean"
                    ]
                ),

            "Repeated_Holdout_ECE_SD":
                float(
                    repeated_row[
                        "ECE_SD"
                    ]
                ),

            "Five_Fold_OOF_ECE":
                oof_ece,

            "OOF_minus_Holdout_ECE":
                (
                    oof_ece
                    - float(
                        repeated_row[
                            "ECE_Mean"
                        ]
                    )
                    if np.isfinite(oof_ece)
                    else np.nan
                )
        }
    )

ct_updated_comparison_df = pd.DataFrame(
    comparison_rows
)

# ------------------------------------------------------------
# 13. Create publication-style table
# ------------------------------------------------------------

publication_rows = []

for _, row in ct_updated_comparison_df.iterrows():

    publication_rows.append(
        {
            "Method":
                row["Method"],

            "Repeated hold-out accuracy (%)":
                (
                    f"{row['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Accuracy_SD'] * 100:.3f}"
                ),

            "Five-fold OOF accuracy (%)":
                f"{row['Five_Fold_OOF_Accuracy'] * 100:.3f}",

            "OOF gain in accuracy (pp)":
                row[
                    "OOF_minus_Holdout_Accuracy"
                ] * 100,

            "Repeated hold-out macro F1 (%)":
                (
                    f"{row['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}"
                ),

            "Five-fold OOF macro F1 (%)":
                f"{row['Five_Fold_OOF_Macro_F1'] * 100:.3f}",

            "OOF gain in macro F1 (pp)":
                row[
                    "OOF_minus_Holdout_Macro_F1"
                ] * 100,

            "Repeated hold-out ROC-AUC (%)":
                (
                    f"{row['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}"
                ),

            "Five-fold OOF ROC-AUC (%)":
                f"{row['Five_Fold_OOF_ROC_AUC'] * 100:.3f}",

            "OOF gain in ROC-AUC (pp)":
                row[
                    "OOF_minus_Holdout_ROC_AUC"
                ] * 100
        }
    )

ct_publication_df = pd.DataFrame(
    publication_rows
)

# ------------------------------------------------------------
# 14. Save outputs
# ------------------------------------------------------------

ct_updated_comparison_df.to_csv(
    CT_UPDATED_COMPARISON_PATH,
    index=False
)

ct_publication_df.to_csv(
    CT_UPDATED_PUBLICATION_PATH,
    index=False
)

# ------------------------------------------------------------
# 15. Display results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("UPDATED CT NUMERICAL COMPARISON")
print("=" * 80)

display(
    ct_updated_comparison_df.round(6)
)

print("\n" + "=" * 80)
print("UPDATED CT PUBLICATION-STYLE TABLE")
print("=" * 80)

display(
    ct_publication_df.round(3)
)

# ------------------------------------------------------------
# 16. Final validation
# ------------------------------------------------------------

expected_methods = set(
    METHODS
)

actual_methods = set(
    ct_updated_comparison_df[
        "Method"
    ]
)

if actual_methods != expected_methods:
    raise ValueError(
        "Final method validation failed.\n"
        f"Expected: {expected_methods}\n"
        f"Found: {actual_methods}"
    )

required_final_metrics = [
    "Repeated_Holdout_Accuracy_Mean",
    "Five_Fold_OOF_Accuracy",
    "Repeated_Holdout_Macro_F1_Mean",
    "Five_Fold_OOF_Macro_F1",
    "Repeated_Holdout_ROC_AUC_Mean",
    "Five_Fold_OOF_ROC_AUC"
]

if ct_updated_comparison_df[
    required_final_metrics
].isna().any().any():

    raise ValueError(
        "One or more essential accuracy, macro F1, "
        "or ROC-AUC values are missing."
    )

print("\n" + "=" * 80)
print("CELL 1 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nSaved files:")
print(CT_UPDATED_COMPARISON_PATH)
print(CT_UPDATED_PUBLICATION_PATH)

print(
    "\nThe updated CT comparison now includes "
    "accuracy, macro F1, ROC-AUC, log loss, "
    "Brier score, and ECE where available."
)

In [ ]:
# ============================================================
# CELL 2
# COMBINE CIFAR-10 AND UPDATED CT RESULTS
# REPEATED HOLD-OUT vs FIVE-FOLD OOF
# INCLUDING ACCURACY, MACRO F1, AND ROC-AUC
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

TABLES_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Tables"
)

os.makedirs(
    TABLES_DIR,
    exist_ok=True
)

CIFAR_COMPARISON_PATH = os.path.join(
    TABLES_DIR,
    "CIFAR10_Repeated_Holdout_vs_OOF_Comparison.csv"
)

CT_UPDATED_COMPARISON_PATH = os.path.join(
    TABLES_DIR,
    "CT_Repeated_Holdout_vs_OOF_Comparison_With_Macro_F1.csv"
)

COMBINED_NUMERIC_PATH = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Repeated_Holdout_vs_OOF_With_Macro_F1.csv"
)

COMBINED_PUBLICATION_PATH = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Publication_Table_With_Macro_F1.csv"
)

COMBINED_GAIN_SUMMARY_PATH = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_OOF_Gain_Summary_With_Macro_F1.csv"
)

# ------------------------------------------------------------
# 2. Verify required files
# ------------------------------------------------------------

required_files = {
    "CIFAR-10 comparison":
        CIFAR_COMPARISON_PATH,

    "Updated CT comparison":
        CT_UPDATED_COMPARISON_PATH
}

print("=" * 80)
print("CELL 2: INPUT FILE CHECK")
print("=" * 80)

for label, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{label:<30}: {exists}")
    print(path)

    if not exists:
        raise FileNotFoundError(
            f"\nRequired file was not found:\n{path}"
        )

# ------------------------------------------------------------
# 3. Load input tables
# ------------------------------------------------------------

cifar_df = pd.read_csv(
    CIFAR_COMPARISON_PATH
)

ct_df = pd.read_csv(
    CT_UPDATED_COMPARISON_PATH
)

print("\nCIFAR-10 columns:")
print(list(cifar_df.columns))

print("\nUpdated CT columns:")
print(list(ct_df.columns))

# ------------------------------------------------------------
# 4. Normalize method names
# ------------------------------------------------------------

def normalize_method_name(value):

    value = str(value).strip()

    replacements = {
        "LR": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression Stacking":
            "Logistic Regression",

        "LGBM": "LightGBM",
        "Light GBM": "LightGBM",
        "LightGBM Stacking": "LightGBM"
    }

    return replacements.get(
        value,
        value
    )


for dataframe in [
    cifar_df,
    ct_df
]:

    if "Method" not in dataframe.columns:
        raise ValueError(
            "An input table does not contain a Method column."
        )

    dataframe["Method"] = (
        dataframe["Method"]
        .apply(
            normalize_method_name
        )
    )

# ------------------------------------------------------------
# 5. Normalize CIFAR-10 column names to the common schema
# ------------------------------------------------------------

cifar_rename_map = {
    "Repeated_Holdout_AUC_Mean":
        "Repeated_Holdout_ROC_AUC_Mean",

    "Repeated_Holdout_AUC_SD":
        "Repeated_Holdout_ROC_AUC_SD",

    "Five_Fold_OOF_AUC":
        "Five_Fold_OOF_ROC_AUC",

    "OOF_minus_Holdout_AUC":
        "OOF_minus_Holdout_ROC_AUC"
}

for old_name, new_name in cifar_rename_map.items():

    if (
        old_name in cifar_df.columns
        and new_name not in cifar_df.columns
    ):

        cifar_df = cifar_df.rename(
            columns={
                old_name: new_name
            }
        )

# ------------------------------------------------------------
# 6. Required common columns
# ------------------------------------------------------------

required_columns = [
    "Dataset",
    "Method",

    "Repeated_Holdout_Accuracy_Mean",
    "Repeated_Holdout_Accuracy_SD",
    "Five_Fold_OOF_Accuracy",
    "OOF_minus_Holdout_Accuracy",

    "Repeated_Holdout_Macro_F1_Mean",
    "Repeated_Holdout_Macro_F1_SD",
    "Five_Fold_OOF_Macro_F1",
    "OOF_minus_Holdout_Macro_F1",

    "Repeated_Holdout_ROC_AUC_Mean",
    "Repeated_Holdout_ROC_AUC_SD",
    "Five_Fold_OOF_ROC_AUC",
    "OOF_minus_Holdout_ROC_AUC"
]

for table_name, dataframe in {
    "CIFAR-10": cifar_df,
    "CT": ct_df
}.items():

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"\n{table_name} table is missing columns:\n"
            f"{missing_columns}\n\n"
            f"Available columns:\n"
            f"{list(dataframe.columns)}"
        )

# ------------------------------------------------------------
# 7. Retain only the two directly comparable meta-learners
# ------------------------------------------------------------

METHODS = [
    "Logistic Regression",
    "LightGBM"
]

cifar_df = cifar_df[
    cifar_df["Method"].isin(
        METHODS
    )
].copy()

ct_df = ct_df[
    ct_df["Method"].isin(
        METHODS
    )
].copy()

# Standardize dataset labels
cifar_df["Dataset"] = "CIFAR-10"
ct_df["Dataset"] = "CT"

# ------------------------------------------------------------
# 8. Validate one row per dataset-method combination
# ------------------------------------------------------------

def validate_method_rows(
    dataframe,
    dataset_name
):

    for method in METHODS:

        method_count = len(
            dataframe[
                dataframe["Method"] == method
            ]
        )

        if method_count != 1:

            raise ValueError(
                f"{dataset_name} contains {method_count} rows "
                f"for {method}; expected exactly one."
            )


validate_method_rows(
    cifar_df,
    "CIFAR-10"
)

validate_method_rows(
    ct_df,
    "CT"
)

# ------------------------------------------------------------
# 9. Combine both datasets
# ------------------------------------------------------------

combined_df = pd.concat(
    [
        cifar_df[
            required_columns
        ],
        ct_df[
            required_columns
        ]
    ],
    ignore_index=True
)

# Convert all metrics to numeric
metric_columns = [
    column
    for column in required_columns
    if column not in [
        "Dataset",
        "Method"
    ]
]

for column in metric_columns:

    combined_df[column] = pd.to_numeric(
        combined_df[column],
        errors="raise"
    )

    if not np.isfinite(
        combined_df[column]
    ).all():

        raise ValueError(
            f"Column {column} contains missing or "
            "non-finite values."
        )

# ------------------------------------------------------------
# 10. Sort rows
# ------------------------------------------------------------

dataset_order = {
    "CIFAR-10": 0,
    "CT": 1
}

method_order = {
    "Logistic Regression": 0,
    "LightGBM": 1
}

combined_df["_Dataset_Order"] = (
    combined_df["Dataset"]
    .map(dataset_order)
)

combined_df["_Method_Order"] = (
    combined_df["Method"]
    .map(method_order)
)

combined_df = (
    combined_df
    .sort_values(
        [
            "_Dataset_Order",
            "_Method_Order"
        ]
    )
    .drop(
        columns=[
            "_Dataset_Order",
            "_Method_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)

expected_rows = 4

if len(combined_df) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} combined rows, "
        f"but found {len(combined_df)}."
    )

# ------------------------------------------------------------
# 11. Create publication-style table
# ------------------------------------------------------------

publication_rows = []

for _, row in combined_df.iterrows():

    publication_rows.append(
        {
            "Dataset":
                row["Dataset"],

            "Method":
                row["Method"],

            "Repeated hold-out accuracy (%)":
                (
                    f"{row['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Accuracy_SD'] * 100:.3f}"
                ),

            "Five-fold OOF accuracy (%)":
                f"{row['Five_Fold_OOF_Accuracy'] * 100:.3f}",

            "Accuracy gain (pp)":
                row[
                    "OOF_minus_Holdout_Accuracy"
                ] * 100,

            "Repeated hold-out macro F1 (%)":
                (
                    f"{row['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}"
                ),

            "Five-fold OOF macro F1 (%)":
                f"{row['Five_Fold_OOF_Macro_F1'] * 100:.3f}",

            "Macro F1 gain (pp)":
                row[
                    "OOF_minus_Holdout_Macro_F1"
                ] * 100,

            "Repeated hold-out ROC-AUC (%)":
                (
                    f"{row['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}"
                ),

            "Five-fold OOF ROC-AUC (%)":
                f"{row['Five_Fold_OOF_ROC_AUC'] * 100:.3f}",

            "ROC-AUC gain (pp)":
                row[
                    "OOF_minus_Holdout_ROC_AUC"
                ] * 100
        }
    )

publication_df = pd.DataFrame(
    publication_rows
)

# ------------------------------------------------------------
# 12. Create cross-dataset gain summary
# ------------------------------------------------------------

gain_summary_df = (
    combined_df
    .groupby(
        "Dataset",
        as_index=False
    )
    .agg(
        Mean_Accuracy_Gain=(
            "OOF_minus_Holdout_Accuracy",
            "mean"
        ),

        Minimum_Accuracy_Gain=(
            "OOF_minus_Holdout_Accuracy",
            "min"
        ),

        Maximum_Accuracy_Gain=(
            "OOF_minus_Holdout_Accuracy",
            "max"
        ),

        Mean_Macro_F1_Gain=(
            "OOF_minus_Holdout_Macro_F1",
            "mean"
        ),

        Minimum_Macro_F1_Gain=(
            "OOF_minus_Holdout_Macro_F1",
            "min"
        ),

        Maximum_Macro_F1_Gain=(
            "OOF_minus_Holdout_Macro_F1",
            "max"
        ),

        Mean_ROC_AUC_Gain=(
            "OOF_minus_Holdout_ROC_AUC",
            "mean"
        )
    )
)

gain_columns = [
    "Mean_Accuracy_Gain",
    "Minimum_Accuracy_Gain",
    "Maximum_Accuracy_Gain",
    "Mean_Macro_F1_Gain",
    "Minimum_Macro_F1_Gain",
    "Maximum_Macro_F1_Gain",
    "Mean_ROC_AUC_Gain"
]

for column in gain_columns:

    gain_summary_df[
        column + "_pp"
    ] = (
        gain_summary_df[column]
        * 100
    )

# ------------------------------------------------------------
# 13. Add overall cross-dataset summary
# ------------------------------------------------------------

overall_summary = {
    "Dataset": "Overall",

    "Mean_Accuracy_Gain":
        combined_df[
            "OOF_minus_Holdout_Accuracy"
        ].mean(),

    "Minimum_Accuracy_Gain":
        combined_df[
            "OOF_minus_Holdout_Accuracy"
        ].min(),

    "Maximum_Accuracy_Gain":
        combined_df[
            "OOF_minus_Holdout_Accuracy"
        ].max(),

    "Mean_Macro_F1_Gain":
        combined_df[
            "OOF_minus_Holdout_Macro_F1"
        ].mean(),

    "Minimum_Macro_F1_Gain":
        combined_df[
            "OOF_minus_Holdout_Macro_F1"
        ].min(),

    "Maximum_Macro_F1_Gain":
        combined_df[
            "OOF_minus_Holdout_Macro_F1"
        ].max(),

    "Mean_ROC_AUC_Gain":
        combined_df[
            "OOF_minus_Holdout_ROC_AUC"
        ].mean()
}

for column in gain_columns:

    overall_summary[
        column + "_pp"
    ] = (
        overall_summary[column]
        * 100
    )

gain_summary_df = pd.concat(
    [
        gain_summary_df,
        pd.DataFrame(
            [overall_summary]
        )
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 14. Save output tables
# ------------------------------------------------------------

combined_df.to_csv(
    COMBINED_NUMERIC_PATH,
    index=False
)

publication_df.to_csv(
    COMBINED_PUBLICATION_PATH,
    index=False
)

gain_summary_df.to_csv(
    COMBINED_GAIN_SUMMARY_PATH,
    index=False
)

# ------------------------------------------------------------
# 15. Display results
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMBINED NUMERICAL TABLE")
print("=" * 80)

display(
    combined_df.round(6)
)

print("\n" + "=" * 80)
print("FINAL PUBLICATION-STYLE TABLE")
print("=" * 80)

display(
    publication_df.round(3)
)

print("\n" + "=" * 80)
print("CROSS-DATASET OOF GAIN SUMMARY")
print("=" * 80)

display(
    gain_summary_df.round(6)
)

# ------------------------------------------------------------
# 16. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 2 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(
    "\nCombined rows:",
    len(combined_df)
)

print(
    "Expected rows:",
    expected_rows
)

print("\nSaved files:")
print(COMBINED_NUMERIC_PATH)
print(COMBINED_PUBLICATION_PATH)
print(COMBINED_GAIN_SUMMARY_PATH)

print(
    "\nThe final combined table now includes "
    "accuracy, macro F1, and ROC-AUC for both "
    "CIFAR-10 and CT."
)

In [ ]:
# ============================================================
# CELL 3
# FINAL CROSS-DATASET FIGURES
# CIFAR-10 AND CT
# REPEATED HOLD-OUT vs FIVE-FOLD OOF
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

TABLES_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Tables"
)

FIGURES_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Figures"
)

os.makedirs(
    FIGURES_DIR,
    exist_ok=True
)

COMBINED_RESULTS_PATH = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Repeated_Holdout_vs_OOF_With_Macro_F1.csv"
)

print("=" * 80)
print("CELL 3: INPUT FILE CHECK")
print("=" * 80)

print(
    "Combined results file:",
    os.path.exists(
        COMBINED_RESULTS_PATH
    )
)

print(
    COMBINED_RESULTS_PATH
)

if not os.path.exists(
    COMBINED_RESULTS_PATH
):
    raise FileNotFoundError(
        COMBINED_RESULTS_PATH
    )

# ------------------------------------------------------------
# 2. Load final combined results
# ------------------------------------------------------------

results_df = pd.read_csv(
    COMBINED_RESULTS_PATH
)

required_columns = [
    "Dataset",
    "Method",

    "Repeated_Holdout_Accuracy_Mean",
    "Repeated_Holdout_Accuracy_SD",
    "Five_Fold_OOF_Accuracy",
    "OOF_minus_Holdout_Accuracy",

    "Repeated_Holdout_Macro_F1_Mean",
    "Repeated_Holdout_Macro_F1_SD",
    "Five_Fold_OOF_Macro_F1",
    "OOF_minus_Holdout_Macro_F1",

    "Repeated_Holdout_ROC_AUC_Mean",
    "Repeated_Holdout_ROC_AUC_SD",
    "Five_Fold_OOF_ROC_AUC",
    "OOF_minus_Holdout_ROC_AUC"
]

missing_columns = [
    column
    for column in required_columns
    if column not in results_df.columns
]

if missing_columns:
    raise ValueError(
        "The combined table is missing columns:\n"
        f"{missing_columns}"
    )

results_df = results_df[
    required_columns
].copy()

# ------------------------------------------------------------
# 3. Sort rows consistently
# ------------------------------------------------------------

dataset_order = {
    "CIFAR-10": 0,
    "CT": 1
}

method_order = {
    "Logistic Regression": 0,
    "LightGBM": 1
}

results_df["_Dataset_Order"] = (
    results_df["Dataset"]
    .map(dataset_order)
)

results_df["_Method_Order"] = (
    results_df["Method"]
    .map(method_order)
)

results_df = (
    results_df
    .sort_values(
        [
            "_Dataset_Order",
            "_Method_Order"
        ]
    )
    .drop(
        columns=[
            "_Dataset_Order",
            "_Method_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)

if len(results_df) != 4:
    raise ValueError(
        f"Expected 4 rows, found {len(results_df)}."
    )

display(
    results_df.round(6)
)

# ------------------------------------------------------------
# 4. Create combined category labels
# ------------------------------------------------------------

category_labels = [
    f"{row['Dataset']}\n{row['Method']}"
    for _, row in results_df.iterrows()
]

x_positions = np.arange(
    len(category_labels)
)

bar_width = 0.36

# ------------------------------------------------------------
# 5. Figure 1: Accuracy comparison
# ------------------------------------------------------------

holdout_accuracy = (
    results_df[
        "Repeated_Holdout_Accuracy_Mean"
    ].to_numpy() * 100
)

holdout_accuracy_sd = (
    results_df[
        "Repeated_Holdout_Accuracy_SD"
    ].to_numpy() * 100
)

oof_accuracy = (
    results_df[
        "Five_Fold_OOF_Accuracy"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(12, 6.5)
)

holdout_bars = plt.bar(
    x_positions - bar_width / 2,
    holdout_accuracy,
    width=bar_width,
    yerr=holdout_accuracy_sd,
    capsize=5,
    label="Repeated hold-out mean ± SD"
)

oof_bars = plt.bar(
    x_positions + bar_width / 2,
    oof_accuracy,
    width=bar_width,
    label="Five-fold OOF"
)

plt.xticks(
    x_positions,
    category_labels
)

plt.ylabel(
    "Accuracy (%)"
)

plt.xlabel(
    "Dataset and meta-learner"
)

plt.title(
    "Repeated Hold-out versus Five-fold OOF Accuracy"
)

minimum_accuracy = min(
    np.min(
        holdout_accuracy
        - holdout_accuracy_sd
    ),
    np.min(
        oof_accuracy
    )
)

maximum_accuracy = max(
    np.max(
        holdout_accuracy
        + holdout_accuracy_sd
    ),
    np.max(
        oof_accuracy
    )
)

plt.ylim(
    max(
        0,
        minimum_accuracy - 2
    ),
    min(
        100,
        maximum_accuracy + 2
    )
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.legend()

for bars in [
    holdout_bars,
    oof_bars
]:

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.12,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

accuracy_figure_path = os.path.join(
    FIGURES_DIR,
    "Combined_CIFAR10_CT_Repeated_Holdout_vs_OOF_Accuracy.png"
)

plt.savefig(
    accuracy_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 6. Figure 2: Macro F1 comparison
# ------------------------------------------------------------

holdout_f1 = (
    results_df[
        "Repeated_Holdout_Macro_F1_Mean"
    ].to_numpy() * 100
)

holdout_f1_sd = (
    results_df[
        "Repeated_Holdout_Macro_F1_SD"
    ].to_numpy() * 100
)

oof_f1 = (
    results_df[
        "Five_Fold_OOF_Macro_F1"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(12, 6.5)
)

holdout_f1_bars = plt.bar(
    x_positions - bar_width / 2,
    holdout_f1,
    width=bar_width,
    yerr=holdout_f1_sd,
    capsize=5,
    label="Repeated hold-out mean ± SD"
)

oof_f1_bars = plt.bar(
    x_positions + bar_width / 2,
    oof_f1,
    width=bar_width,
    label="Five-fold OOF"
)

plt.xticks(
    x_positions,
    category_labels
)

plt.ylabel(
    "Macro F1 (%)"
)

plt.xlabel(
    "Dataset and meta-learner"
)

plt.title(
    "Repeated Hold-out versus Five-fold OOF Macro F1"
)

minimum_f1 = min(
    np.min(
        holdout_f1
        - holdout_f1_sd
    ),
    np.min(
        oof_f1
    )
)

maximum_f1 = max(
    np.max(
        holdout_f1
        + holdout_f1_sd
    ),
    np.max(
        oof_f1
    )
)

plt.ylim(
    max(
        0,
        minimum_f1 - 2
    ),
    min(
        100,
        maximum_f1 + 2
    )
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.legend()

for bars in [
    holdout_f1_bars,
    oof_f1_bars
]:

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.12,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

f1_figure_path = os.path.join(
    FIGURES_DIR,
    "Combined_CIFAR10_CT_Repeated_Holdout_vs_OOF_Macro_F1.png"
)

plt.savefig(
    f1_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. Figure 3: ROC-AUC comparison
# ------------------------------------------------------------

holdout_auc = (
    results_df[
        "Repeated_Holdout_ROC_AUC_Mean"
    ].to_numpy() * 100
)

holdout_auc_sd = (
    results_df[
        "Repeated_Holdout_ROC_AUC_SD"
    ].to_numpy() * 100
)

oof_auc = (
    results_df[
        "Five_Fold_OOF_ROC_AUC"
    ].to_numpy() * 100
)

plt.figure(
    figsize=(12, 6.5)
)

holdout_auc_bars = plt.bar(
    x_positions - bar_width / 2,
    holdout_auc,
    width=bar_width,
    yerr=holdout_auc_sd,
    capsize=5,
    label="Repeated hold-out mean ± SD"
)

oof_auc_bars = plt.bar(
    x_positions + bar_width / 2,
    oof_auc,
    width=bar_width,
    label="Five-fold OOF"
)

plt.xticks(
    x_positions,
    category_labels
)

plt.ylabel(
    "ROC-AUC (%)"
)

plt.xlabel(
    "Dataset and meta-learner"
)

plt.title(
    "Repeated Hold-out versus Five-fold OOF ROC-AUC"
)

minimum_auc = min(
    np.min(
        holdout_auc
        - holdout_auc_sd
    ),
    np.min(
        oof_auc
    )
)

maximum_auc = max(
    np.max(
        holdout_auc
        + holdout_auc_sd
    ),
    np.max(
        oof_auc
    )
)

plt.ylim(
    max(
        0,
        minimum_auc - 1
    ),
    min(
        100,
        maximum_auc + 1
    )
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.legend()

for bars in [
    holdout_auc_bars,
    oof_auc_bars
]:

    for bar in bars:

        height = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            height + 0.05,
            f"{height:.2f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

auc_figure_path = os.path.join(
    FIGURES_DIR,
    "Combined_CIFAR10_CT_Repeated_Holdout_vs_OOF_ROC_AUC.png"
)

plt.savefig(
    auc_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 8. Figure 4: OOF performance gains
# ------------------------------------------------------------

accuracy_gain = (
    results_df[
        "OOF_minus_Holdout_Accuracy"
    ].to_numpy() * 100
)

f1_gain = (
    results_df[
        "OOF_minus_Holdout_Macro_F1"
    ].to_numpy() * 100
)

auc_gain = (
    results_df[
        "OOF_minus_Holdout_ROC_AUC"
    ].to_numpy() * 100
)

gain_bar_width = 0.25

plt.figure(
    figsize=(12, 6.5)
)

accuracy_gain_bars = plt.bar(
    x_positions - gain_bar_width,
    accuracy_gain,
    width=gain_bar_width,
    label="Accuracy gain"
)

f1_gain_bars = plt.bar(
    x_positions,
    f1_gain,
    width=gain_bar_width,
    label="Macro F1 gain"
)

auc_gain_bars = plt.bar(
    x_positions + gain_bar_width,
    auc_gain,
    width=gain_bar_width,
    label="ROC-AUC gain"
)

plt.xticks(
    x_positions,
    category_labels
)

plt.ylabel(
    "OOF improvement over repeated hold-out\n(percentage points)"
)

plt.xlabel(
    "Dataset and meta-learner"
)

plt.title(
    "Performance Gain from Five-fold OOF Stacking"
)

plt.axhline(
    y=0,
    linewidth=1
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.legend()

for bars in [
    accuracy_gain_bars,
    f1_gain_bars,
    auc_gain_bars
]:

    for bar in bars:

        value = bar.get_height()

        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            value + 0.03,
            f"{value:.2f}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.tight_layout()

gain_figure_path = os.path.join(
    FIGURES_DIR,
    "Combined_CIFAR10_CT_OOF_Performance_Gains.png"
)

plt.savefig(
    gain_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 9. Save figure-data table
# ------------------------------------------------------------

figure_data_df = pd.DataFrame(
    {
        "Dataset": results_df["Dataset"],
        "Method": results_df["Method"],

        "Repeated_Holdout_Accuracy_Percent":
            holdout_accuracy,

        "Repeated_Holdout_Accuracy_SD_Percent":
            holdout_accuracy_sd,

        "Five_Fold_OOF_Accuracy_Percent":
            oof_accuracy,

        "Accuracy_Gain_pp":
            accuracy_gain,

        "Repeated_Holdout_Macro_F1_Percent":
            holdout_f1,

        "Repeated_Holdout_Macro_F1_SD_Percent":
            holdout_f1_sd,

        "Five_Fold_OOF_Macro_F1_Percent":
            oof_f1,

        "Macro_F1_Gain_pp":
            f1_gain,

        "Repeated_Holdout_ROC_AUC_Percent":
            holdout_auc,

        "Repeated_Holdout_ROC_AUC_SD_Percent":
            holdout_auc_sd,

        "Five_Fold_OOF_ROC_AUC_Percent":
            oof_auc,

        "ROC_AUC_Gain_pp":
            auc_gain
    }
)

figure_data_path = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Final_Figure_Data.csv"
)

figure_data_df.to_csv(
    figure_data_path,
    index=False
)

# ------------------------------------------------------------
# 10. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 3 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nSaved figures:")
print(accuracy_figure_path)
print(f1_figure_path)
print(auc_figure_path)
print(gain_figure_path)

print("\nSaved figure-data table:")
print(figure_data_path)

In [ ]:
# ============================================================
# CELL 4
# GENERATE MANUSCRIPT-READY RESULTS TEXT
# FROM VERIFIED CIFAR-10 AND CT COMPARISON RESULTS
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

TABLES_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Tables"
)

TEXT_DIR = os.path.join(
    PROJECT_ROOT,
    "Extended_Experiments",
    "Manuscript_Text"
)

os.makedirs(
    TEXT_DIR,
    exist_ok=True
)

COMBINED_RESULTS_PATH = os.path.join(
    TABLES_DIR,
    "Combined_CIFAR10_CT_Repeated_Holdout_vs_OOF_With_Macro_F1.csv"
)

RESULTS_TEXT_PATH = os.path.join(
    TEXT_DIR,
    "Repeated_Holdout_vs_OOF_Results_Text.txt"
)

RESULTS_SUMMARY_PATH = os.path.join(
    TABLES_DIR,
    "Repeated_Holdout_vs_OOF_Manuscript_Summary.csv"
)

# ------------------------------------------------------------
# 2. Check the combined results file
# ------------------------------------------------------------

print("=" * 80)
print("CELL 4: INPUT FILE CHECK")
print("=" * 80)

print(
    "Combined results file:",
    os.path.exists(
        COMBINED_RESULTS_PATH
    )
)

print(
    COMBINED_RESULTS_PATH
)

if not os.path.exists(
    COMBINED_RESULTS_PATH
):
    raise FileNotFoundError(
        COMBINED_RESULTS_PATH
    )

# ------------------------------------------------------------
# 3. Load results
# ------------------------------------------------------------

results_df = pd.read_csv(
    COMBINED_RESULTS_PATH
)

required_columns = [
    "Dataset",
    "Method",

    "Repeated_Holdout_Accuracy_Mean",
    "Repeated_Holdout_Accuracy_SD",
    "Five_Fold_OOF_Accuracy",
    "OOF_minus_Holdout_Accuracy",

    "Repeated_Holdout_Macro_F1_Mean",
    "Repeated_Holdout_Macro_F1_SD",
    "Five_Fold_OOF_Macro_F1",
    "OOF_minus_Holdout_Macro_F1",

    "Repeated_Holdout_ROC_AUC_Mean",
    "Repeated_Holdout_ROC_AUC_SD",
    "Five_Fold_OOF_ROC_AUC",
    "OOF_minus_Holdout_ROC_AUC"
]

missing_columns = [
    column
    for column in required_columns
    if column not in results_df.columns
]

if missing_columns:
    raise ValueError(
        "The combined results table is missing:\n"
        f"{missing_columns}"
    )

# ------------------------------------------------------------
# 4. Helper for retrieving one result row
# ------------------------------------------------------------

def get_result_row(
    dataset,
    method
):

    selected_rows = results_df[
        (results_df["Dataset"] == dataset)
        &
        (results_df["Method"] == method)
    ]

    if len(selected_rows) != 1:
        raise ValueError(
            f"Expected one row for {dataset} / {method}, "
            f"but found {len(selected_rows)}."
        )

    return selected_rows.iloc[0]

# ------------------------------------------------------------
# 5. Retrieve verified rows
# ------------------------------------------------------------

cifar_lr = get_result_row(
    "CIFAR-10",
    "Logistic Regression"
)

cifar_lgbm = get_result_row(
    "CIFAR-10",
    "LightGBM"
)

ct_lr = get_result_row(
    "CT",
    "Logistic Regression"
)

ct_lgbm = get_result_row(
    "CT",
    "LightGBM"
)

# ------------------------------------------------------------
# 6. Calculate cross-dataset summaries
# ------------------------------------------------------------

cifar_df = results_df[
    results_df["Dataset"] == "CIFAR-10"
]

ct_df = results_df[
    results_df["Dataset"] == "CT"
]

overall_accuracy_gain = (
    results_df[
        "OOF_minus_Holdout_Accuracy"
    ].mean() * 100
)

overall_f1_gain = (
    results_df[
        "OOF_minus_Holdout_Macro_F1"
    ].mean() * 100
)

overall_auc_gain = (
    results_df[
        "OOF_minus_Holdout_ROC_AUC"
    ].mean() * 100
)

cifar_accuracy_gain = (
    cifar_df[
        "OOF_minus_Holdout_Accuracy"
    ].mean() * 100
)

cifar_f1_gain = (
    cifar_df[
        "OOF_minus_Holdout_Macro_F1"
    ].mean() * 100
)

cifar_auc_gain = (
    cifar_df[
        "OOF_minus_Holdout_ROC_AUC"
    ].mean() * 100
)

ct_accuracy_gain = (
    ct_df[
        "OOF_minus_Holdout_Accuracy"
    ].mean() * 100
)

ct_f1_gain = (
    ct_df[
        "OOF_minus_Holdout_Macro_F1"
    ].mean() * 100
)

ct_auc_gain = (
    ct_df[
        "OOF_minus_Holdout_ROC_AUC"
    ].mean() * 100
)

# ------------------------------------------------------------
# 7. Generate manuscript-ready Results text
# ------------------------------------------------------------

results_heading = (
    "Robustness analysis using repeated hold-out evaluation"
)

paragraph_1 = (
    "To examine whether the observed differences between fixed hold-out "
    "and five-fold out-of-fold (OOF) stacking were dependent on a single "
    "data partition, the fixed hold-out protocol was repeated using three "
    "stratified random seeds (42, 123, and 2025). The mean and sample "
    "standard deviation across these runs were then compared with the "
    "five-fold OOF results. This analysis was conducted for Logistic "
    "Regression and LightGBM meta-learners on both the CIFAR-10 and CT "
    "datasets."
)

paragraph_2 = (
    f"On CIFAR-10, repeated hold-out Logistic Regression achieved an "
    f"accuracy of {cifar_lr['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
    f"± {cifar_lr['Repeated_Holdout_Accuracy_SD'] * 100:.3f}%, a macro "
    f"F1 score of {cifar_lr['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
    f"± {cifar_lr['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}%, and a "
    f"ROC-AUC of {cifar_lr['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
    f"± {cifar_lr['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}%. Under the "
    f"five-fold OOF protocol, the corresponding values increased to "
    f"{cifar_lr['Five_Fold_OOF_Accuracy'] * 100:.3f}% accuracy, "
    f"{cifar_lr['Five_Fold_OOF_Macro_F1'] * 100:.3f}% macro F1, and "
    f"{cifar_lr['Five_Fold_OOF_ROC_AUC'] * 100:.3f}% ROC-AUC. These "
    f"changes represent improvements of "
    f"{cifar_lr['OOF_minus_Holdout_Accuracy'] * 100:.3f}, "
    f"{cifar_lr['OOF_minus_Holdout_Macro_F1'] * 100:.3f}, and "
    f"{cifar_lr['OOF_minus_Holdout_ROC_AUC'] * 100:.3f} percentage "
    f"points, respectively."
)

paragraph_3 = (
    f"For CIFAR-10 LightGBM, repeated hold-out evaluation produced "
    f"{cifar_lgbm['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
    f"± {cifar_lgbm['Repeated_Holdout_Accuracy_SD'] * 100:.3f}% accuracy, "
    f"{cifar_lgbm['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
    f"± {cifar_lgbm['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}% macro F1, "
    f"and {cifar_lgbm['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
    f"± {cifar_lgbm['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}% ROC-AUC. "
    f"The OOF protocol increased these values to "
    f"{cifar_lgbm['Five_Fold_OOF_Accuracy'] * 100:.3f}%, "
    f"{cifar_lgbm['Five_Fold_OOF_Macro_F1'] * 100:.3f}%, and "
    f"{cifar_lgbm['Five_Fold_OOF_ROC_AUC'] * 100:.3f}%, corresponding "
    f"to gains of "
    f"{cifar_lgbm['OOF_minus_Holdout_Accuracy'] * 100:.3f}, "
    f"{cifar_lgbm['OOF_minus_Holdout_Macro_F1'] * 100:.3f}, and "
    f"{cifar_lgbm['OOF_minus_Holdout_ROC_AUC'] * 100:.3f} percentage "
    f"points."
)

paragraph_4 = (
    f"On the CT dataset, repeated hold-out Logistic Regression achieved "
    f"{ct_lr['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
    f"± {ct_lr['Repeated_Holdout_Accuracy_SD'] * 100:.3f}% accuracy, "
    f"{ct_lr['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
    f"± {ct_lr['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}% macro F1, "
    f"and {ct_lr['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
    f"± {ct_lr['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}% ROC-AUC. "
    f"The five-fold OOF results were "
    f"{ct_lr['Five_Fold_OOF_Accuracy'] * 100:.3f}% accuracy, "
    f"{ct_lr['Five_Fold_OOF_Macro_F1'] * 100:.3f}% macro F1, and "
    f"{ct_lr['Five_Fold_OOF_ROC_AUC'] * 100:.3f}% ROC-AUC. The resulting "
    f"improvements were "
    f"{ct_lr['OOF_minus_Holdout_Accuracy'] * 100:.3f}, "
    f"{ct_lr['OOF_minus_Holdout_Macro_F1'] * 100:.3f}, and "
    f"{ct_lr['OOF_minus_Holdout_ROC_AUC'] * 100:.3f} percentage points."
)

paragraph_5 = (
    f"CT LightGBM showed a similar pattern. Its repeated hold-out accuracy "
    f"was {ct_lgbm['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
    f"± {ct_lgbm['Repeated_Holdout_Accuracy_SD'] * 100:.3f}%, with a "
    f"macro F1 score of "
    f"{ct_lgbm['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
    f"± {ct_lgbm['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}% and a "
    f"ROC-AUC of "
    f"{ct_lgbm['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
    f"± {ct_lgbm['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}%. Five-fold "
    f"OOF stacking increased these values to "
    f"{ct_lgbm['Five_Fold_OOF_Accuracy'] * 100:.3f}% accuracy, "
    f"{ct_lgbm['Five_Fold_OOF_Macro_F1'] * 100:.3f}% macro F1, and "
    f"{ct_lgbm['Five_Fold_OOF_ROC_AUC'] * 100:.3f}% ROC-AUC. The gains "
    f"were therefore "
    f"{ct_lgbm['OOF_minus_Holdout_Accuracy'] * 100:.3f}, "
    f"{ct_lgbm['OOF_minus_Holdout_Macro_F1'] * 100:.3f}, and "
    f"{ct_lgbm['OOF_minus_Holdout_ROC_AUC'] * 100:.3f} percentage points."
)

paragraph_6 = (
    f"Across the two CIFAR-10 meta-learners, the mean OOF improvements "
    f"were {cifar_accuracy_gain:.3f} percentage points for accuracy, "
    f"{cifar_f1_gain:.3f} percentage points for macro F1, and "
    f"{cifar_auc_gain:.3f} percentage points for ROC-AUC. The corresponding "
    f"mean improvements on CT were {ct_accuracy_gain:.3f}, "
    f"{ct_f1_gain:.3f}, and {ct_auc_gain:.3f} percentage points. When all "
    f"four dataset–meta-learner combinations were considered together, "
    f"five-fold OOF stacking improved accuracy by an average of "
    f"{overall_accuracy_gain:.3f} percentage points, macro F1 by "
    f"{overall_f1_gain:.3f} percentage points, and ROC-AUC by "
    f"{overall_auc_gain:.3f} percentage points."
)

paragraph_7 = (
    "Overall, the repeated hold-out results indicate that performance "
    "varied moderately across random partitions, but the five-fold OOF "
    "protocol consistently produced higher accuracy, macro F1, and "
    "ROC-AUC for both meta-learners and both datasets. The improvement was "
    "more pronounced on the CT dataset, suggesting that OOF meta-training "
    "may provide greater benefit when the available development dataset "
    "is smaller and the learned stacking model is more sensitive to the "
    "composition of a single hold-out partition."
)

manuscript_text = (
    results_heading
    + "\n\n"
    + paragraph_1
    + "\n\n"
    + paragraph_2
    + "\n\n"
    + paragraph_3
    + "\n\n"
    + paragraph_4
    + "\n\n"
    + paragraph_5
    + "\n\n"
    + paragraph_6
    + "\n\n"
    + paragraph_7
)

# ------------------------------------------------------------
# 8. Create concise summary table
# ------------------------------------------------------------

summary_rows = []

for _, row in results_df.iterrows():

    summary_rows.append(
        {
            "Dataset":
                row["Dataset"],

            "Method":
                row["Method"],

            "Repeated_Holdout_Accuracy":
                (
                    f"{row['Repeated_Holdout_Accuracy_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Accuracy_SD'] * 100:.3f}"
                ),

            "Five_Fold_OOF_Accuracy":
                f"{row['Five_Fold_OOF_Accuracy'] * 100:.3f}",

            "Accuracy_Gain_pp":
                row[
                    "OOF_minus_Holdout_Accuracy"
                ] * 100,

            "Repeated_Holdout_Macro_F1":
                (
                    f"{row['Repeated_Holdout_Macro_F1_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_Macro_F1_SD'] * 100:.3f}"
                ),

            "Five_Fold_OOF_Macro_F1":
                f"{row['Five_Fold_OOF_Macro_F1'] * 100:.3f}",

            "Macro_F1_Gain_pp":
                row[
                    "OOF_minus_Holdout_Macro_F1"
                ] * 100,

            "Repeated_Holdout_ROC_AUC":
                (
                    f"{row['Repeated_Holdout_ROC_AUC_Mean'] * 100:.3f} "
                    f"± "
                    f"{row['Repeated_Holdout_ROC_AUC_SD'] * 100:.3f}"
                ),

            "Five_Fold_OOF_ROC_AUC":
                f"{row['Five_Fold_OOF_ROC_AUC'] * 100:.3f}",

            "ROC_AUC_Gain_pp":
                row[
                    "OOF_minus_Holdout_ROC_AUC"
                ] * 100
        }
    )

summary_df = pd.DataFrame(
    summary_rows
)

# ------------------------------------------------------------
# 9. Save manuscript text and summary table
# ------------------------------------------------------------

with open(
    RESULTS_TEXT_PATH,
    "w",
    encoding="utf-8"
) as text_file:

    text_file.write(
        manuscript_text
    )

summary_df.to_csv(
    RESULTS_SUMMARY_PATH,
    index=False
)

# ------------------------------------------------------------
# 10. Display manuscript text
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MANUSCRIPT-READY RESULTS TEXT")
print("=" * 80)

print(
    manuscript_text
)

print("\n" + "=" * 80)
print("MANUSCRIPT SUMMARY TABLE")
print("=" * 80)

display(
    summary_df.round(3)
)

# ------------------------------------------------------------
# 11. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 4 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nSaved manuscript text:")
print(RESULTS_TEXT_PATH)

print("\nSaved manuscript summary table:")
print(RESULTS_SUMMARY_PATH)